In [1]:
# ============================================
# MAX-AUC PIPELINE (RUN-READY, SINGLE SCRIPT)  [WINDOWS SAFE + FAST + RESUMABLE]
# - Patient-level 5-fold CV on (train+val pool)
# - Multi-bag inference on VAL/TEST (avg logits)
# - Optional TTA at eval (noflip + hflip)
# - Gate temperature + gate entropy regularization
# - FULL RESUME per (seed, fold): model/opt/scaler/epoch/earlystop/RNG
# - Windows stability: NUM_WORKERS=0, main-guard, safe RNG restore
#
# IMPORTANT SPEED FIXES (avoid "no progress for hours"):
#   * K_VAL reduced (default 5)
#   * eval done every EVAL_EVERY epochs (default 2)
#   * TTA disabled during selection by default
#   * evaluation uses EVAL_BS small; train uses grad accumulation
#
# Outputs (in RUNS_DIR):
#   - cv_best_{BENCHMARK}_seed{seed}_fold{fold}.pt    (best ckpt by val AUC)
#   - progress_{BENCHMARK}_seed{seed}_fold{fold}.pt   (latest state for resume)
#   - summary_{BENCHMARK}_seed{seed}_fold{fold}.txt   (best epoch + best AUC)
#   - pred_fold{fold}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv (fold test logits)
#   - pred_seed{seed}_{BENCHMARK}_folds{a}_{b}_Ktest{K_TEST}_agg.csv (avg fold logits)
# ============================================

import os, json, math, random, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import torchvision
from torchvision import transforms as T


# ---------------------------
# Config (EDIT HERE)
# ---------------------------
ART_DIR = Path(r"D:\ENT\_challenge2_artifacts")  # <-- change if needed
RUNS_DIR = ART_DIR / "_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK = "res_shift"  # "standard" or "res_shift"

SPLITS = {
    "standard": {
        "train": ART_DIR / "split_standard_train_colab.csv",
        "val":   ART_DIR / "split_standard_val_colab.csv",
        "test":  ART_DIR / "split_standard_test_colab.csv",
    },
    "res_shift": {
        "train": ART_DIR / "split_res_shift_train_major_colab.csv",
        "val":   ART_DIR / "split_res_shift_val_major_colab.csv",
        "test":  ART_DIR / "split_res_shift_test_minor_colab.csv",
    }
}

SPLIT_TRAIN = SPLITS[BENCHMARK]["train"]
SPLIT_VAL   = SPLITS[BENCHMARK]["val"]
SPLIT_TEST  = SPLITS[BENCHMARK]["test"]

# ---------------------------
# Device
# ---------------------------
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
AMP = True

# ---------------------------
# Data
# ---------------------------
IMG_SIZE = 224

# Windows-safe: keep 0 unless you're 100% on Linux
NUM_WORKERS = 0
PIN_MEMORY = False

# ---------------------------
# Bag sizes
# ---------------------------
BAG_SIZE_NONLEUKO = 24
BAG_SIZE_LEUKO = 40

# ---------------------------
# Training
# ---------------------------
EPOCHS_MAX = 30
LR = 5e-5
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 6
MIN_DELTA = 1e-4

# Train batch size is "patients per step"
TRAIN_BS = 2
GRAD_ACCUM_STEPS = 4   # effective batch = TRAIN_BS * GRAD_ACCUM_STEPS

# Eval batch size
EVAL_BS = 2

# ---------------------------
# Selection / Eval
# ---------------------------
# Huge speed win: do NOT run K=20 + TTA every epoch.
K_VAL = 5           # selection K (fast). Increase later if needed.
K_TEST = 10         # keep final test as requested
USE_TTA_VAL = False # keep selection stable+fast
USE_TTA_TEST = True # if you want, keep test TTA
EVAL_EVERY = 2      # evaluate every N epochs (and always at ep 1)

# ---------------------------
# Aux leuko head
# ---------------------------
USE_AUX_LEUKO = True
LAMBDA_LEUKO = 0.30

# ---------------------------
# Gate warmup + stabilization
# ---------------------------
WARMUP_FORCE_HALF_GATE_EPOCHS = 2
GATE_TAU = 2.0
LAMBDA_GATE_ENT = 0.02

# Optional: stabilize early training
FREEZE_BACKBONE_EPOCHS = 2

# ---------------------------
# CV + seeds and fold range
# ---------------------------
N_FOLDS = 5
SEEDS = [43]      # <-- set to [43] then later [44], etc.
FOLD_START = 2    # <-- run folds 2..5 for seed=43
FOLD_END = 5


print(f"DEVICE: {DEVICE}")
print("=" * 70)
print(f"BENCHMARK={BENCHMARK}")
print(" train:", SPLIT_TRAIN)
print(" val  :", SPLIT_VAL)
print(" test :", SPLIT_TEST)
print(f"RUNNING seeds={SEEDS} folds={FOLD_START}..{FOLD_END} (of {N_FOLDS})")
print("=" * 70)


# ---------------------------
# Repro helpers
# ---------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))

def try_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def safe_int(x, default=0):
    try:
        return int(float(x))
    except Exception:
        return default

def parse_leuko(x):
    if x is None:
        return 0
    s = str(x).strip().lower()
    return 1 if s in ["yes", "1", "true", "y"] else 0

def parse_malign(x):
    return 1 if safe_int(x, 0) == 1 else 0


# ---------------------------
# RNG capture/restore (robust across torch versions)
# ---------------------------
def capture_rng_state():
    st = {"py": random.getstate(), "np": np.random.get_state()}
    try:
        st["torch"] = torch.get_rng_state()
    except Exception:
        st["torch"] = None
    try:
        st["cuda"] = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    except Exception:
        st["cuda"] = None
    return st

def _as_byte_tensor(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.to(dtype=torch.uint8, device="cpu")
    try:
        return torch.tensor(x, dtype=torch.uint8, device="cpu")
    except Exception:
        return None

def restore_rng_state(rng):
    if rng is None:
        return
    try:
        random.setstate(rng.get("py", random.getstate()))
    except Exception:
        pass
    try:
        np.random.set_state(rng.get("np", np.random.get_state()))
    except Exception:
        pass
    try:
        ts = _as_byte_tensor(rng.get("torch", None))
        if ts is not None:
            torch.set_rng_state(ts)
    except Exception as e:
        print(f"[WARN] could not restore torch RNG: {e}")
    try:
        cs = rng.get("cuda", None)
        if torch.cuda.is_available() and cs is not None:
            cs2 = []
            for s in cs:
                bt = _as_byte_tensor(s)
                if bt is None:
                    cs2 = None
                    break
                cs2.append(bt)
            if cs2 is not None:
                torch.cuda.set_rng_state_all(cs2)
    except Exception as e:
        print(f"[WARN] could not restore cuda RNG: {e}")


# ---------------------------
# Robust CSV loading
# ---------------------------
def load_csv_rows(csv_path):
    csv_path = str(csv_path)
    try:
        import pandas as pd
        df = pd.read_csv(csv_path)
        return df.to_dict(orient="records")
    except Exception as e:
        print(f"[WARN] pandas read_csv failed, fallback csv module: {e}")
        import csv
        rows = []
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for r in reader:
                rows.append(r)
        return rows

def ensure_exists(rows):
    kept, missing = [], 0
    for r in rows:
        fp = r.get("filepath", "")
        if fp and os.path.exists(fp):
            kept.append(r)
        else:
            missing += 1
    if missing > 0:
        print(f"[INFO] filtered missing files: {missing}")
    return kept


# ---------------------------
# Build patient index
# ---------------------------
def build_patient_index(rows):
    pid_to_frames = defaultdict(list)

    for r in rows:
        pid = r.get("Patient", None) or r.get("patient", None) or r.get("PID", None)
        if pid is None:
            fp = r.get("filepath", "")
            pid = "UNK"
            if "Patient" in fp:
                try:
                    pid = "Patient" + fp.split("Patient")[1].split(os.sep)[0]
                except Exception:
                    pid = "UNK"

        y_m = parse_malign(r.get("label_binary", 0))
        y_l = parse_leuko(r.get("Leukoplakia", 0))
        fp = r.get("filepath", "")

        pid_to_frames[pid].append({"filepath": fp, "y_m": y_m, "y_l": y_l})

    pid_to_labels = {}
    for pid, frs in pid_to_frames.items():
        y_m = 1 if any(f["y_m"] == 1 for f in frs) else 0
        y_l = 1 if any(f["y_l"] == 1 for f in frs) else 0
        pid_to_labels[pid] = (y_m, y_l)

    patients = sorted(pid_to_frames.keys())
    return patients, pid_to_frames, pid_to_labels


# ---------------------------
# Transforms
# ---------------------------
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
    T.ToTensor(),
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

eval_tf_hflip = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=1.0),
    T.ToTensor(),
])


# ---------------------------
# Dataset: patient bags
# ---------------------------
class PatientBagDataset(Dataset):
    def __init__(self, patients, pid_to_frames, pid_to_labels, train=True):
        self.patients = list(patients)
        self.pid_to_frames = pid_to_frames
        self.pid_to_labels = pid_to_labels
        self.train = train
        self.tf = train_tf if train else eval_tf
        self.eval_random = True

    def set_transform(self, tf):
        self.tf = tf

    def __len__(self):
        return len(self.patients)

    def _sample_frames(self, pid):
        frs = self.pid_to_frames[pid]
        y_m, y_l = self.pid_to_labels[pid]
        bag_size = BAG_SIZE_LEUKO if y_l == 1 else BAG_SIZE_NONLEUKO
        n = len(frs)

        if self.train or self.eval_random:
            idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size))
        else:
            if n >= bag_size:
                idxs = np.arange(bag_size)
            else:
                reps = int(math.ceil(bag_size / n))
                idxs = (np.tile(np.arange(n), reps))[:bag_size]

        return [frs[i] for i in idxs]

    def __getitem__(self, i):
        pid = self.patients[i]
        y_m, y_l = self.pid_to_labels[pid]
        sampled = self._sample_frames(pid)

        imgs, fps = [], []
        for f in sampled:
            fp = f["filepath"]
            fps.append(fp)
            im = Image.open(fp).convert("RGB")
            im = self.tf(im)
            imgs.append(im)

        X = torch.stack(imgs, dim=0)
        return X, y_m, y_l, pid, fps


def collate_patient_bags(batch):
    Xs, ym, yl, pids, fps = zip(*batch)
    lengths = [x.shape[0] for x in Xs]
    max_len = max(lengths)

    B = len(Xs)
    C, H, W = Xs[0].shape[1], Xs[0].shape[2], Xs[0].shape[3]

    Xpad = torch.zeros((B, max_len, C, H, W), dtype=Xs[0].dtype)
    mask = torch.zeros((B, max_len), dtype=torch.bool)

    for i, x in enumerate(Xs):
        n = x.shape[0]
        Xpad[i, :n] = x
        mask[i, :n] = True

    ym = torch.tensor(ym, dtype=torch.long)
    yl = torch.tensor(yl, dtype=torch.long)
    return Xpad, mask, ym, yl, list(pids), list(fps)


# ---------------------------
# Model
# ---------------------------
class ResNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(m.children())[:-1])
        self.out_dim = 2048

    def forward(self, x):
        h = self.backbone(x)
        return h.flatten(1)

class AttnPool(nn.Module):
    def __init__(self, d, h=256):
        super().__init__()
        self.V = nn.Linear(d, h)
        self.w = nn.Linear(h, 1)

    def forward(self, H, mask=None):
        A = self.w(torch.tanh(self.V(H)))  # (B,n,1)
        if mask is not None:
            A = A.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        A = torch.softmax(A, dim=1)
        Z = (A * H).sum(dim=1)
        return Z, A.squeeze(-1)

class GatedMILv2(nn.Module):
    def __init__(self, use_aux_leuko=True, gate_tau=2.0):
        super().__init__()
        self.use_aux_leuko = bool(use_aux_leuko)
        self.gate_tau = float(gate_tau)

        self.encoder = ResNetEncoder()
        d = self.encoder.out_dim

        self.pool_non = AttnPool(d, h=256)
        self.pool_leu = AttnPool(d, h=256)

        self.cls_non = nn.Linear(d, 1)
        self.cls_leu = nn.Linear(d, 1)

        self.gate = nn.Linear(2 * d, 1)

        if self.use_aux_leuko:
            self.leuko_head = nn.Sequential(
                nn.Linear(2 * d, 256),
                nn.ReLU(inplace=True),
                nn.Dropout(0.2),
                nn.Linear(256, 1),
            )

    def forward(self, X, mask=None, force_half_gate=False):
        B, N, C, H, W = X.shape
        x = X.view(B * N, C, H, W)
        h = self.encoder(x)
        d = h.shape[-1]
        Hbag = h.view(B, N, d)

        z_non, _ = self.pool_non(Hbag, mask=mask)
        z_leu, _ = self.pool_leu(Hbag, mask=mask)

        logit_non = self.cls_non(z_non)
        logit_leu = self.cls_leu(z_leu)

        gate_in = torch.cat([z_non, z_leu], dim=1)
        g_logit = self.gate(gate_in) / self.gate_tau
        g = torch.sigmoid(g_logit)

        if force_half_gate:
            g = 0.5 * torch.ones_like(g)

        logit_m = (1.0 - g) * logit_non + g * logit_leu

        logit_l = None
        if self.use_aux_leuko:
            logit_l = self.leuko_head(gate_in)

        return logit_m, logit_l, g


def set_backbone_trainable(model: nn.Module, trainable: bool):
    for p in model.encoder.backbone.parameters():
        p.requires_grad = bool(trainable)

def gate_entropy(g, eps=1e-6):
    g = torch.clamp(g, eps, 1.0 - eps)
    ent = -(g * torch.log(g) + (1.0 - g) * torch.log(1.0 - g))
    return ent.mean()


# ---------------------------
# Checkpoint helpers
# ---------------------------
def save_ckpt(path: Path, model, opt, scaler, epoch, best_auc, best_epoch, bad):
    ck = {
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": int(epoch),
        "best_auc": float(best_auc),
        "best_epoch": int(best_epoch),
        "bad": int(bad),
        "rng": capture_rng_state(),
    }
    torch.save(ck, str(path))

def load_ckpt(path: Path, model, opt, scaler):
    # PyTorch 2.6: default weights_only=True; we need full object.
    ck = torch.load(str(path), map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"], strict=True)
    if opt is not None and "opt" in ck:
        opt.load_state_dict(ck["opt"])
    if scaler is not None and ck.get("scaler") is not None:
        scaler.load_state_dict(ck["scaler"])
    restore_rng_state(ck.get("rng", None))
    return ck


# ---------------------------
# Train / Eval
# ---------------------------
def train_epoch(model, loader, opt, crit_m, crit_l, scaler, epoch):
    model.train()
    losses = []

    if FREEZE_BACKBONE_EPOCHS > 0:
        set_backbone_trainable(model, trainable=(epoch > FREEZE_BACKBONE_EPOCHS))

    opt.zero_grad(set_to_none=True)
    step = 0

    for it, (X, mask, ym, yl, _pids, _fps) in enumerate(loader):
        X = X.to(DEVICE, non_blocking=False)
        mask = mask.to(DEVICE, non_blocking=False)
        ym = ym.to(DEVICE, non_blocking=False).float().unsqueeze(1)
        yl = yl.to(DEVICE, non_blocking=False).float().unsqueeze(1)

        force_half = (epoch <= WARMUP_FORCE_HALF_GATE_EPOCHS)

        if scaler is not None:
            with torch.amp.autocast(device_type="cuda"):
                logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
                loss = crit_m(logit_m, ym)

                if USE_AUX_LEUKO and (logit_l is not None):
                    loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)

                if (not force_half) and (LAMBDA_GATE_ENT > 0):
                    loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

                loss = loss / float(GRAD_ACCUM_STEPS)

            scaler.scale(loss).backward()
        else:
            logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
            loss = crit_m(logit_m, ym)

            if USE_AUX_LEUKO and (logit_l is not None):
                loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)

            if (not force_half) and (LAMBDA_GATE_ENT > 0):
                loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

            loss = loss / float(GRAD_ACCUM_STEPS)
            loss.backward()

        step += 1
        if step % GRAD_ACCUM_STEPS == 0:
            if scaler is not None:
                scaler.step(opt)
                scaler.update()
            else:
                opt.step()
            opt.zero_grad(set_to_none=True)

        losses.append(float(loss.detach().cpu().item()) * float(GRAD_ACCUM_STEPS))

    # flush if remainder
    if step % GRAD_ACCUM_STEPS != 0:
        if scaler is not None:
            scaler.step(opt)
            scaler.update()
        else:
            opt.step()
        opt.zero_grad(set_to_none=True)

    return float(np.mean(losses)) if losses else float("nan")


@torch.no_grad()
def predict_logits_multibag(model, ds_eval, K=5, use_tta=False):
    model.eval()

    pid_list = ds_eval.patients
    pid_to_idx = {p: i for i, p in enumerate(pid_list)}
    n = len(pid_list)

    sum_logits = np.zeros((n,), dtype=np.float64)
    sum_gate = np.zeros((n,), dtype=np.float64)

    dl = DataLoader(
        ds_eval,
        batch_size=EVAL_BS,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_patient_bags
    )

    tta_tfs = [eval_tf]
    if use_tta:
        tta_tfs.append(eval_tf_hflip)

    denom = float(K * len(tta_tfs))

    for _k in range(K):
        ds_eval.train = False
        ds_eval.eval_random = True

        for tf in tta_tfs:
            ds_eval.set_transform(tf)

            for X, mask, _ym, _yl, pids, _fps in dl:
                X = X.to(DEVICE, non_blocking=False)
                mask = mask.to(DEVICE, non_blocking=False)

                logit_m, _logit_l, g = model(X, mask=mask, force_half_gate=False)
                logit_m = logit_m.detach().cpu().numpy().reshape(-1)
                g = g.detach().cpu().numpy().reshape(-1)

                for pid, lg, gg in zip(pids, logit_m, g):
                    j = pid_to_idx[pid]
                    sum_logits[j] += float(lg)
                    sum_gate[j] += float(gg)

    avg_logits = sum_logits / denom
    avg_gate = sum_gate / denom

    y = np.asarray([ds_eval.pid_to_labels[p][0] for p in pid_list], dtype=int)
    le = np.asarray([ds_eval.pid_to_labels[p][1] for p in pid_list], dtype=int)

    return pid_list, y, le, avg_logits, avg_gate


# ---------------------------
# CV folds
# ---------------------------
def make_patient_folds(patients, pid_to_labels, n_splits=5, seed=42):
    strat = np.array([2 * pid_to_labels[p][0] + pid_to_labels[p][1] for p in patients], dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idxs = np.arange(len(patients))
    folds = []
    for tr_idx, va_idx in skf.split(idxs, strat):
        tr_p = [patients[i] for i in tr_idx]
        va_p = [patients[i] for i in va_idx]
        folds.append((tr_p, va_p))
    return folds


# ---------------------------
# Utilities: write per-fold test prediction
# ---------------------------
def save_fold_test_csv(seed, fold_i, pids, y_true, leuko, logits, gates):
    out = RUNS_DIR / f"pred_fold{fold_i}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit,prob,gate\n")
        probs = sigmoid_np(logits)
        for pid, yt, lk, lg, pr, gg in zip(pids, y_true, leuko, logits, probs, gates):
            f.write(f"{pid},{int(yt)},{int(lk)},{float(lg):.8f},{float(pr):.8f},{float(gg):.6f}\n")
    print("Saved fold test preds:", out)
    return out


def aggregate_seed_folds(seed, folds_done):
    # average logits across folds (expects pred_fold{f}_... exists)
    import pandas as pd

    dfs = []
    for fidx in folds_done:
        p = RUNS_DIR / f"pred_fold{fidx}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")
        df = pd.read_csv(p).set_index("Patient")[["y_true", "leuko", "logit"]]
        dfs.append(df)

    # align index + mean
    base = dfs[0].copy()
    for d in dfs[1:]:
        base = base.join(d, how="inner", rsuffix="_r")

    # collect all logit columns
    logit_cols = [c for c in base.columns if c.startswith("logit")]
    # if only one fold, it is 'logit' only
    if "logit" not in logit_cols and "logit" in base.columns:
        logit_cols = ["logit"]

    logit_mean = base[logit_cols].mean(axis=1).astype(float).values
    prob_mean = sigmoid_np(logit_mean)

    out = RUNS_DIR / f"pred_seed{seed}_{BENCHMARK}_folds{min(folds_done)}_{max(folds_done)}_Ktest{K_TEST}_agg.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit_mean,prob_mean\n")
        for pid, (yt, lk), lg, pr in zip(base.index.tolist(),
                                        base[["y_true", "leuko"]].astype(int).values.tolist(),
                                        logit_mean.tolist(),
                                        prob_mean.tolist()):
            f.write(f"{pid},{int(yt)},{int(lk)},{float(lg):.8f},{float(pr):.8f}\n")

    auc = try_auc(base["y_true"].values.astype(int), prob_mean)
    print("Saved:", out)
    print(f"AGG TEST AUC (avg logits over folds {min(folds_done)}..{max(folds_done)}): {auc}")
    return out, float(auc)


# ---------------------------
# Main
# ---------------------------
def main():
    rows_tr = ensure_exists(load_csv_rows(SPLIT_TRAIN))
    rows_va = ensure_exists(load_csv_rows(SPLIT_VAL))
    rows_te = ensure_exists(load_csv_rows(SPLIT_TEST))

    rows_pool = rows_tr + rows_va
    patients_pool, pid_to_frames_pool, pid_to_labels_pool = build_patient_index(rows_pool)
    patients_pool = [p for p in patients_pool if len(pid_to_frames_pool[p]) >= 2]

    patients_test, pid_to_frames_test, pid_to_labels_test = build_patient_index(rows_te)
    patients_test = [p for p in patients_test if len(pid_to_frames_test[p]) >= 2]

    print(f"[POOL] patients={len(patients_pool)} (train+val)")
    print(f"[TEST] patients={len(patients_test)}")

    folds = make_patient_folds(patients_pool, pid_to_labels_pool, n_splits=N_FOLDS, seed=SEEDS[0])

    print("=" * 70)
    print(f"RUN seed(s)={SEEDS} | K_VAL={K_VAL} | K_TEST={K_TEST} | TTA_VAL={USE_TTA_VAL} | TTA_TEST={USE_TTA_TEST}")
    print(f"Gate: tau={GATE_TAU} | lambda_ent={LAMBDA_GATE_ENT} | freeze_backbone_epochs={FREEZE_BACKBONE_EPOCHS}")
    print(f"TRAIN_BS={TRAIN_BS} (accum {GRAD_ACCUM_STEPS}) | EVAL_BS={EVAL_BS} | EVAL_EVERY={EVAL_EVERY}")
    print("=" * 70)

    for seed in SEEDS:
        seed_all(seed)
        print("\n" + "=" * 70)
        print(f"RUN seed={seed} | benchmark={BENCHMARK} | folds={FOLD_START}..{FOLD_END}")
        print("=" * 70)

        folds_done = []

        for fold_i, (tr_pat, va_pat) in enumerate(folds, start=1):
            if fold_i < FOLD_START or fold_i > FOLD_END:
                continue

            print(f"\n--- Fold {fold_i}/{N_FOLDS} | train_pat={len(tr_pat)} val_pat={len(va_pat)} ---")

            ds_tr = PatientBagDataset(tr_pat, pid_to_frames_pool, pid_to_labels_pool, train=True)
            ds_va = PatientBagDataset(va_pat, pid_to_frames_pool, pid_to_labels_pool, train=False)
            ds_te = PatientBagDataset(patients_test, pid_to_frames_test, pid_to_labels_test, train=False)

            # pos_weight
            y_m_tr = np.array([pid_to_labels_pool[p][0] for p in tr_pat], dtype=int)
            y_l_tr = np.array([pid_to_labels_pool[p][1] for p in tr_pat], dtype=int)
            pos_m = int(y_m_tr.sum()); neg_m = int(len(y_m_tr) - pos_m)
            pos_l = int(y_l_tr.sum()); neg_l = int(len(y_l_tr) - pos_l)

            pos_weight_m = torch.tensor([neg_m / max(pos_m, 1)], dtype=torch.float32, device=DEVICE)
            pos_weight_l = torch.tensor([neg_l / max(pos_l, 1)], dtype=torch.float32, device=DEVICE)

            crit_m = nn.BCEWithLogitsLoss(pos_weight=pos_weight_m)
            crit_l = nn.BCEWithLogitsLoss(pos_weight=pos_weight_l)

            dl_tr = DataLoader(
                ds_tr,
                batch_size=TRAIN_BS,
                shuffle=True,
                num_workers=NUM_WORKERS,
                pin_memory=PIN_MEMORY,
                collate_fn=collate_patient_bags
            )

            model = GatedMILv2(use_aux_leuko=USE_AUX_LEUKO, gate_tau=GATE_TAU).to(DEVICE)
            print("Model on:", next(model.parameters()).device)

            opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scaler = torch.amp.GradScaler("cuda") if (AMP and DEVICE.type == "cuda") else None

            ckpt_best = RUNS_DIR / f"cv_best_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"
            ckpt_prog = RUNS_DIR / f"progress_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"
            summary_path = RUNS_DIR / f"summary_{BENCHMARK}_seed{seed}_fold{fold_i}.txt"

            best_auc = -1e9
            best_epoch = -1
            bad = 0
            start_ep = 1

            # Resume preference: progress (latest) else best
            resume_path = ckpt_prog if ckpt_prog.exists() else (ckpt_best if ckpt_best.exists() else None)
            if resume_path is not None:
                ck = load_ckpt(resume_path, model, opt, scaler)
                best_auc = float(ck.get("best_auc", -1e9))
                best_epoch = int(ck.get("best_epoch", ck.get("epoch", -1)))
                bad = int(ck.get("bad", 0))
                start_ep = int(ck.get("epoch", 0)) + 1
                print(f"[RESUME] seed={seed} fold={fold_i} from epoch {start_ep} | best_auc={best_auc:.4f} at ep {best_epoch} | bad={bad}")

            if start_ep > EPOCHS_MAX:
                print(f"[SKIP] fold already finished (last epoch {start_ep-1} >= {EPOCHS_MAX})")
            else:
                for ep in range(start_ep, EPOCHS_MAX + 1):
                    t0 = time.time()
                    tr_loss = train_epoch(model, dl_tr, opt, crit_m, crit_l, scaler, epoch=ep)
                    t_train = (time.time() - t0) / 60.0

                    do_eval = (ep == 1) or (ep % EVAL_EVERY == 0)
                    auc_va = float("nan")
                    gate_mean = float("nan")

                    if do_eval:
                        t1 = time.time()
                        _pids, yv, _lv, logits, gates = predict_logits_multibag(
                            model, ds_va, K=K_VAL, use_tta=USE_TTA_VAL
                        )
                        t_val = (time.time() - t1) / 60.0

                        auc_va = try_auc(yv, sigmoid_np(logits))
                        gate_mean = float(np.mean(gates)) if len(gates) else float("nan")

                        print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                              f"train_loss={tr_loss:.4f} | val_auc(K)={auc_va:.4f} | gate_mean={gate_mean:.3f} | "
                              f"time: train={t_train:.2f}m val={t_val:.2f}m")
                    else:
                        print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                              f"train_loss={tr_loss:.4f} | (no val this epoch) | time: train={t_train:.2f}m")

                    # Always save progress checkpoint for resume safety
                    save_ckpt(ckpt_prog, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)

                    if do_eval:
                        improved = (auc_va > best_auc + MIN_DELTA) or (best_epoch < 0)
                        if improved:
                            best_auc = float(auc_va)
                            best_epoch = int(ep)
                            bad = 0
                            save_ckpt(ckpt_best, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)
                            with open(summary_path, "w", encoding="utf-8") as f:
                                f.write(f"seed={seed}\nfold={fold_i}\nbest_epoch={best_epoch}\nbest_auc={best_auc:.6f}\n")
                            print(f"[SAVE] best updated -> {summary_path}")
                        else:
                            bad += 1
                            if bad >= EARLY_STOP_PATIENCE:
                                print(f"Early stop at ep {ep} (best_auc={best_auc:.4f} at ep {best_epoch}).")
                                break

            # Load best (strict) and predict TEST
            if ckpt_best.exists():
                ck = torch.load(str(ckpt_best), map_location=DEVICE, weights_only=False)
                model.load_state_dict(ck["model"], strict=True)
            model.eval()

            pids_te, y_te, le_te, lg_te, gg_te = predict_logits_multibag(
                model, ds_te, K=K_TEST, use_tta=USE_TTA_TEST
            )

            save_fold_test_csv(seed, fold_i, pids_te, y_te, le_te, lg_te, gg_te)
            folds_done.append(fold_i)

        if folds_done:
            aggregate_seed_folds(seed, folds_done)


if __name__ == "__main__":
    main()


DEVICE: cuda:0
BENCHMARK=res_shift
 train: D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv
 val  : D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv
 test : D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv
RUNNING seeds=[43] folds=2..5 (of 5)
[INFO] filtered missing files: 13
[POOL] patients=137 (train+val)
[TEST] patients=73
RUN seed(s)=[43] | K_VAL=5 | K_TEST=10 | TTA_VAL=False | TTA_TEST=True
Gate: tau=2.0 | lambda_ent=0.02 | freeze_backbone_epochs=2
TRAIN_BS=2 (accum 4) | EVAL_BS=2 | EVAL_EVERY=2

RUN seed=43 | benchmark=res_shift | folds=2..5

--- Fold 2/5 | train_pat=109 val_pat=28 ---
Model on: cuda:0
[RESUME] seed=43 fold=2 from epoch 12 | best_auc=0.9187 at ep 11 | bad=0
[res_shift] seed=43 fold=2 ep 12/30 | train_loss=0.8204 | val_auc(K)=0.8562 | gate_mean=0.805 | time: train=1.67m val=1.34m
[res_shift] seed=43 fold=2 ep 13/30 | train_loss=0.7263 | (no val this epoch) | time: train=1.44m
[res_shift] seed=43 fold=2 ep 14/30 | 

In [1]:
# ============================================
# RUN SEED=43 FOLD=1 ONLY + AGGREGATE 1..5
# Windows-safe + GPU + resumable
# ============================================

import os, json, math, random, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import torchvision
from torchvision import transforms as T


# ---------------------------
# Config
# ---------------------------
ART_DIR = Path(r"D:\ENT\_challenge2_artifacts")   # <-- change if needed
RUNS_DIR = ART_DIR / "_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK = "res_shift"

SPLITS = {
    "res_shift": {
        "train": ART_DIR / "split_res_shift_train_major_colab.csv",
        "val":   ART_DIR / "split_res_shift_val_major_colab.csv",
        "test":  ART_DIR / "split_res_shift_test_minor_colab.csv",
    }
}

SPLIT_TRAIN = SPLITS[BENCHMARK]["train"]
SPLIT_VAL   = SPLITS[BENCHMARK]["val"]
SPLIT_TEST  = SPLITS[BENCHMARK]["test"]

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
AMP = True

IMG_SIZE = 224

# Windows safe:
NUM_WORKERS = 0
PIN_MEMORY = False

BAG_SIZE_NONLEUKO = 24
BAG_SIZE_LEUKO = 40

EPOCHS_MAX = 30
LR = 5e-5
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 6
MIN_DELTA = 1e-4

TRAIN_BS = 2
GRAD_ACCUM_STEPS = 4
EVAL_BS = 2

# fast selection:
K_VAL = 5
K_TEST = 10
USE_TTA_VAL = False
USE_TTA_TEST = True
EVAL_EVERY = 2

USE_AUX_LEUKO = True
LAMBDA_LEUKO = 0.30

WARMUP_FORCE_HALF_GATE_EPOCHS = 2
GATE_TAU = 2.0
LAMBDA_GATE_ENT = 0.02
FREEZE_BACKBONE_EPOCHS = 2

# What we run now:
SEED = 43
RUN_FOLD = 1  # <-- run fold 1 only
AGG_FOLDS = [1, 2, 3, 4, 5]  # after fold1 is done, aggregate all

print(f"DEVICE: {DEVICE}")
print("=" * 70)
print(f"BENCHMARK={BENCHMARK}")
print(" train:", SPLIT_TRAIN)
print(" val  :", SPLIT_VAL)
print(" test :", SPLIT_TEST)
print(f"RUNNING seed={SEED} fold={RUN_FOLD} then aggregate folds {AGG_FOLDS}")
print("=" * 70)


# ---------------------------
# Helpers
# ---------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))

def try_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def safe_int(x, default=0):
    try:
        return int(float(x))
    except Exception:
        return default

def parse_leuko(x):
    if x is None:
        return 0
    s = str(x).strip().lower()
    return 1 if s in ["yes", "1", "true", "y"] else 0

def parse_malign(x):
    return 1 if safe_int(x, 0) == 1 else 0

def capture_rng_state():
    st = {"py": random.getstate(), "np": np.random.get_state()}
    try:
        st["torch"] = torch.get_rng_state()
    except Exception:
        st["torch"] = None
    try:
        st["cuda"] = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    except Exception:
        st["cuda"] = None
    return st

def _as_byte_tensor(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.to(dtype=torch.uint8, device="cpu")
    try:
        return torch.tensor(x, dtype=torch.uint8, device="cpu")
    except Exception:
        return None

def restore_rng_state(rng):
    if rng is None:
        return
    try:
        random.setstate(rng.get("py", random.getstate()))
    except Exception:
        pass
    try:
        np.random.set_state(rng.get("np", np.random.get_state()))
    except Exception:
        pass
    try:
        ts = _as_byte_tensor(rng.get("torch", None))
        if ts is not None:
            torch.set_rng_state(ts)
    except Exception as e:
        print(f"[WARN] could not restore torch RNG: {e}")
    try:
        cs = rng.get("cuda", None)
        if torch.cuda.is_available() and cs is not None:
            cs2 = []
            for s in cs:
                bt = _as_byte_tensor(s)
                if bt is None:
                    cs2 = None
                    break
                cs2.append(bt)
            if cs2 is not None:
                torch.cuda.set_rng_state_all(cs2)
    except Exception as e:
        print(f"[WARN] could not restore cuda RNG: {e}")

def load_csv_rows(csv_path):
    csv_path = str(csv_path)
    import pandas as pd
    df = pd.read_csv(csv_path)
    return df.to_dict(orient="records")

def ensure_exists(rows):
    kept, missing = [], 0
    for r in rows:
        fp = r.get("filepath", "")
        if fp and os.path.exists(fp):
            kept.append(r)
        else:
            missing += 1
    if missing > 0:
        print(f"[INFO] filtered missing files: {missing}")
    return kept

def build_patient_index(rows):
    pid_to_frames = defaultdict(list)
    for r in rows:
        pid = r.get("Patient", None) or r.get("patient", None) or r.get("PID", None)
        if pid is None:
            pid = "UNK"
        y_m = parse_malign(r.get("label_binary", 0))
        y_l = parse_leuko(r.get("Leukoplakia", 0))
        fp = r.get("filepath", "")
        pid_to_frames[pid].append({"filepath": fp, "y_m": y_m, "y_l": y_l})

    pid_to_labels = {}
    for pid, frs in pid_to_frames.items():
        y_m = 1 if any(f["y_m"] == 1 for f in frs) else 0
        y_l = 1 if any(f["y_l"] == 1 for f in frs) else 0
        pid_to_labels[pid] = (y_m, y_l)

    patients = sorted(pid_to_frames.keys())
    return patients, pid_to_frames, pid_to_labels


# ---------------------------
# Transforms
# ---------------------------
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
    T.ToTensor(),
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

eval_tf_hflip = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=1.0),
    T.ToTensor(),
])


# ---------------------------
# Dataset
# ---------------------------
class PatientBagDataset(Dataset):
    def __init__(self, patients, pid_to_frames, pid_to_labels, train=True):
        self.patients = list(patients)
        self.pid_to_frames = pid_to_frames
        self.pid_to_labels = pid_to_labels
        self.train = train
        self.tf = train_tf if train else eval_tf
        self.eval_random = True

    def set_transform(self, tf):
        self.tf = tf

    def __len__(self):
        return len(self.patients)

    def _sample_frames(self, pid):
        frs = self.pid_to_frames[pid]
        y_m, y_l = self.pid_to_labels[pid]
        bag_size = BAG_SIZE_LEUKO if y_l == 1 else BAG_SIZE_NONLEUKO
        n = len(frs)
        idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size))
        return [frs[i] for i in idxs]

    def __getitem__(self, i):
        pid = self.patients[i]
        y_m, y_l = self.pid_to_labels[pid]
        sampled = self._sample_frames(pid)

        imgs, fps = [], []
        for f in sampled:
            fp = f["filepath"]
            fps.append(fp)
            im = Image.open(fp).convert("RGB")
            im = self.tf(im)
            imgs.append(im)

        X = torch.stack(imgs, dim=0)
        return X, y_m, y_l, pid, fps

def collate_patient_bags(batch):
    Xs, ym, yl, pids, fps = zip(*batch)
    max_len = max([x.shape[0] for x in Xs])

    B = len(Xs)
    C, H, W = Xs[0].shape[1], Xs[0].shape[2], Xs[0].shape[3]

    Xpad = torch.zeros((B, max_len, C, H, W), dtype=Xs[0].dtype)
    mask = torch.zeros((B, max_len), dtype=torch.bool)

    for i, x in enumerate(Xs):
        n = x.shape[0]
        Xpad[i, :n] = x
        mask[i, :n] = True

    ym = torch.tensor(ym, dtype=torch.long)
    yl = torch.tensor(yl, dtype=torch.long)
    return Xpad, mask, ym, yl, list(pids), list(fps)


# ---------------------------
# Model
# ---------------------------
class ResNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(m.children())[:-1])
        self.out_dim = 2048

    def forward(self, x):
        h = self.backbone(x)
        return h.flatten(1)

class AttnPool(nn.Module):
    def __init__(self, d, h=256):
        super().__init__()
        self.V = nn.Linear(d, h)
        self.w = nn.Linear(h, 1)

    def forward(self, H, mask=None):
        A = self.w(torch.tanh(self.V(H)))
        if mask is not None:
            A = A.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        A = torch.softmax(A, dim=1)
        Z = (A * H).sum(dim=1)
        return Z, A.squeeze(-1)

class GatedMILv2(nn.Module):
    def __init__(self, use_aux_leuko=True, gate_tau=2.0):
        super().__init__()
        self.use_aux_leuko = bool(use_aux_leuko)
        self.gate_tau = float(gate_tau)

        self.encoder = ResNetEncoder()
        d = self.encoder.out_dim

        self.pool_non = AttnPool(d, h=256)
        self.pool_leu = AttnPool(d, h=256)

        self.cls_non = nn.Linear(d, 1)
        self.cls_leu = nn.Linear(d, 1)
        self.gate = nn.Linear(2 * d, 1)

        if self.use_aux_leuko:
            self.leuko_head = nn.Sequential(
                nn.Linear(2 * d, 256),
                nn.ReLU(inplace=True),
                nn.Dropout(0.2),
                nn.Linear(256, 1),
            )

    def forward(self, X, mask=None, force_half_gate=False):
        B, N, C, H, W = X.shape
        x = X.view(B * N, C, H, W)
        h = self.encoder(x)
        d = h.shape[-1]
        Hbag = h.view(B, N, d)

        z_non, _ = self.pool_non(Hbag, mask=mask)
        z_leu, _ = self.pool_leu(Hbag, mask=mask)

        logit_non = self.cls_non(z_non)
        logit_leu = self.cls_leu(z_leu)

        gate_in = torch.cat([z_non, z_leu], dim=1)
        g = torch.sigmoid(self.gate(gate_in) / self.gate_tau)

        if force_half_gate:
            g = 0.5 * torch.ones_like(g)

        logit_m = (1.0 - g) * logit_non + g * logit_leu

        logit_l = None
        if self.use_aux_leuko:
            logit_l = self.leuko_head(gate_in)

        return logit_m, logit_l, g

def set_backbone_trainable(model: nn.Module, trainable: bool):
    for p in model.encoder.backbone.parameters():
        p.requires_grad = bool(trainable)

def gate_entropy(g, eps=1e-6):
    g = torch.clamp(g, eps, 1.0 - eps)
    return (-(g * torch.log(g) + (1.0 - g) * torch.log(1.0 - g))).mean()


# ---------------------------
# CV folds
# ---------------------------
def make_patient_folds(patients, pid_to_labels, n_splits=5, seed=43):
    strat = np.array([2 * pid_to_labels[p][0] + pid_to_labels[p][1] for p in patients], dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idxs = np.arange(len(patients))
    folds = []
    for tr_idx, va_idx in skf.split(idxs, strat):
        tr_p = [patients[i] for i in tr_idx]
        va_p = [patients[i] for i in va_idx]
        folds.append((tr_p, va_p))
    return folds


# ---------------------------
# Checkpoints
# ---------------------------
def save_ckpt(path: Path, model, opt, scaler, epoch, best_auc, best_epoch, bad):
    ck = {
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": int(epoch),
        "best_auc": float(best_auc),
        "best_epoch": int(best_epoch),
        "bad": int(bad),
        "rng": capture_rng_state(),
    }
    torch.save(ck, str(path))

def load_ckpt(path: Path, model, opt, scaler):
    ck = torch.load(str(path), map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"], strict=True)
    opt.load_state_dict(ck["opt"])
    if scaler is not None and ck.get("scaler") is not None:
        scaler.load_state_dict(ck["scaler"])
    restore_rng_state(ck.get("rng", None))
    return ck


# ---------------------------
# Train/Eval
# ---------------------------
def train_epoch(model, loader, opt, crit_m, crit_l, scaler, epoch):
    model.train()

    if FREEZE_BACKBONE_EPOCHS > 0:
        set_backbone_trainable(model, trainable=(epoch > FREEZE_BACKBONE_EPOCHS))

    opt.zero_grad(set_to_none=True)
    losses = []
    step = 0

    for X, mask, ym, yl, _pids, _fps in loader:
        X = X.to(DEVICE)
        mask = mask.to(DEVICE)
        ym = ym.to(DEVICE).float().unsqueeze(1)
        yl = yl.to(DEVICE).float().unsqueeze(1)

        force_half = (epoch <= WARMUP_FORCE_HALF_GATE_EPOCHS)

        if scaler is not None:
            with torch.amp.autocast(device_type="cuda"):
                logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
                loss = crit_m(logit_m, ym)

                if USE_AUX_LEUKO and (logit_l is not None):
                    loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)

                if (not force_half) and (LAMBDA_GATE_ENT > 0):
                    loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

                loss = loss / float(GRAD_ACCUM_STEPS)

            scaler.scale(loss).backward()
        else:
            logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
            loss = crit_m(logit_m, ym)

            if USE_AUX_LEUKO and (logit_l is not None):
                loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)

            if (not force_half) and (LAMBDA_GATE_ENT > 0):
                loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

            loss = loss / float(GRAD_ACCUM_STEPS)
            loss.backward()

        step += 1
        if step % GRAD_ACCUM_STEPS == 0:
            if scaler is not None:
                scaler.step(opt)
                scaler.update()
            else:
                opt.step()
            opt.zero_grad(set_to_none=True)

        losses.append(float(loss.detach().cpu().item()) * float(GRAD_ACCUM_STEPS))

    if step % GRAD_ACCUM_STEPS != 0:
        if scaler is not None:
            scaler.step(opt)
            scaler.update()
        else:
            opt.step()
        opt.zero_grad(set_to_none=True)

    return float(np.mean(losses)) if losses else float("nan")


@torch.no_grad()
def predict_logits_multibag(model, ds_eval, K=5, use_tta=False):
    model.eval()

    pid_list = ds_eval.patients
    pid_to_idx = {p: i for i, p in enumerate(pid_list)}
    n = len(pid_list)

    sum_logits = np.zeros((n,), dtype=np.float64)
    sum_gate = np.zeros((n,), dtype=np.float64)

    dl = DataLoader(
        ds_eval,
        batch_size=EVAL_BS,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_patient_bags
    )

    tta_tfs = [eval_tf]
    if use_tta:
        tta_tfs.append(eval_tf_hflip)

    denom = float(K * len(tta_tfs))

    for _k in range(K):
        ds_eval.train = False
        ds_eval.eval_random = True

        for tf in tta_tfs:
            ds_eval.set_transform(tf)
            for X, mask, _ym, _yl, pids, _fps in dl:
                X = X.to(DEVICE)
                mask = mask.to(DEVICE)
                logit_m, _logit_l, g = model(X, mask=mask, force_half_gate=False)
                logit_m = logit_m.detach().cpu().numpy().reshape(-1)
                g = g.detach().cpu().numpy().reshape(-1)
                for pid, lg, gg in zip(pids, logit_m, g):
                    j = pid_to_idx[pid]
                    sum_logits[j] += float(lg)
                    sum_gate[j] += float(gg)

    avg_logits = sum_logits / denom
    avg_gate = sum_gate / denom

    y = np.asarray([ds_eval.pid_to_labels[p][0] for p in pid_list], dtype=int)
    le = np.asarray([ds_eval.pid_to_labels[p][1] for p in pid_list], dtype=int)
    return pid_list, y, le, avg_logits, avg_gate


def save_fold_test_csv(seed, fold_i, pids, y_true, leuko, logits, gates):
    out = RUNS_DIR / f"pred_fold{fold_i}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit,prob,gate\n")
        probs = sigmoid_np(logits)
        for pid, yt, lk, lg, pr, gg in zip(pids, y_true, leuko, logits, probs, gates):
            f.write(f"{pid},{int(yt)},{int(lk)},{float(lg):.8f},{float(pr):.8f},{float(gg):.6f}\n")
    print("Saved fold test preds:", out)
    return out


def aggregate_seed_folds(seed, folds_list):
    import pandas as pd
    dfs = []
    for fidx in folds_list:
        p = RUNS_DIR / f"pred_fold{fidx}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")
        df = pd.read_csv(p).set_index("Patient")[["y_true", "leuko", "logit"]]
        df = df.rename(columns={"logit": f"logit_fold{fidx}"})
        dfs.append(df)

    merged = dfs[0]
    for d in dfs[1:]:
        merged = merged.join(d, how="inner")

    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    logit_mean = merged[logit_cols].mean(axis=1).astype(float).values
    prob_mean = sigmoid_np(logit_mean)

    out = RUNS_DIR / f"pred_seed{seed}_{BENCHMARK}_folds{min(folds_list)}_{max(folds_list)}_Ktest{K_TEST}_agg.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit_mean,prob_mean\n")
        for pid, yt, lk, lg, pr in zip(
            merged.index.tolist(),
            merged["y_true"].astype(int).values.tolist(),
            merged["leuko"].astype(int).values.tolist(),
            logit_mean.tolist(),
            prob_mean.tolist()
        ):
            f.write(f"{pid},{yt},{lk},{lg:.8f},{pr:.8f}\n")

    auc = try_auc(merged["y_true"].values.astype(int), prob_mean)
    print("Saved:", out)
    print(f"AGG TEST AUC (avg logits over folds {min(folds_list)}..{max(folds_list)}): {auc}")
    return out, float(auc)


# ---------------------------
# Main
# ---------------------------
def main():
    seed_all(SEED)

    rows_tr = ensure_exists(load_csv_rows(SPLIT_TRAIN))
    rows_va = ensure_exists(load_csv_rows(SPLIT_VAL))
    rows_te = ensure_exists(load_csv_rows(SPLIT_TEST))

    rows_pool = rows_tr + rows_va
    patients_pool, pid_to_frames_pool, pid_to_labels_pool = build_patient_index(rows_pool)
    patients_pool = [p for p in patients_pool if len(pid_to_frames_pool[p]) >= 2]

    patients_test, pid_to_frames_test, pid_to_labels_test = build_patient_index(rows_te)
    patients_test = [p for p in patients_test if len(pid_to_frames_test[p]) >= 2]

    print(f"[POOL] patients={len(patients_pool)} (train+val)")
    print(f"[TEST] patients={len(patients_test)}")

    folds = make_patient_folds(patients_pool, pid_to_labels_pool, n_splits=5, seed=SEED)
    tr_pat, va_pat = folds[RUN_FOLD - 1]

    print("=" * 70)
    print(f"RUN seed={SEED} fold={RUN_FOLD}/5 | K_VAL={K_VAL} | K_TEST={K_TEST} | TTA_VAL={USE_TTA_VAL} | TTA_TEST={USE_TTA_TEST}")
    print(f"TRAIN_BS={TRAIN_BS} (accum {GRAD_ACCUM_STEPS}) | EVAL_BS={EVAL_BS} | EVAL_EVERY={EVAL_EVERY}")
    print("=" * 70)

    ds_tr = PatientBagDataset(tr_pat, pid_to_frames_pool, pid_to_labels_pool, train=True)
    ds_va = PatientBagDataset(va_pat, pid_to_frames_pool, pid_to_labels_pool, train=False)
    ds_te = PatientBagDataset(patients_test, pid_to_frames_test, pid_to_labels_test, train=False)

    y_m_tr = np.array([pid_to_labels_pool[p][0] for p in tr_pat], dtype=int)
    y_l_tr = np.array([pid_to_labels_pool[p][1] for p in tr_pat], dtype=int)
    pos_m = int(y_m_tr.sum()); neg_m = int(len(y_m_tr) - pos_m)
    pos_l = int(y_l_tr.sum()); neg_l = int(len(y_l_tr) - pos_l)

    pos_weight_m = torch.tensor([neg_m / max(pos_m, 1)], dtype=torch.float32, device=DEVICE)
    pos_weight_l = torch.tensor([neg_l / max(pos_l, 1)], dtype=torch.float32, device=DEVICE)

    crit_m = nn.BCEWithLogitsLoss(pos_weight=pos_weight_m)
    crit_l = nn.BCEWithLogitsLoss(pos_weight=pos_weight_l)

    dl_tr = DataLoader(
        ds_tr,
        batch_size=TRAIN_BS,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_patient_bags
    )

    model = GatedMILv2(use_aux_leuko=USE_AUX_LEUKO, gate_tau=GATE_TAU).to(DEVICE)
    print("Model on:", next(model.parameters()).device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda") if (AMP and DEVICE.type == "cuda") else None

    ckpt_best = RUNS_DIR / f"cv_best_{BENCHMARK}_seed{SEED}_fold{RUN_FOLD}.pt"
    ckpt_prog = RUNS_DIR / f"progress_{BENCHMARK}_seed{SEED}_fold{RUN_FOLD}.pt"
    summary_path = RUNS_DIR / f"summary_{BENCHMARK}_seed{SEED}_fold{RUN_FOLD}.txt"

    best_auc = -1e9
    best_epoch = -1
    bad = 0
    start_ep = 1

    resume_path = ckpt_prog if ckpt_prog.exists() else (ckpt_best if ckpt_best.exists() else None)
    if resume_path is not None:
        ck = load_ckpt(resume_path, model, opt, scaler)
        best_auc = float(ck.get("best_auc", -1e9))
        best_epoch = int(ck.get("best_epoch", ck.get("epoch", -1)))
        bad = int(ck.get("bad", 0))
        start_ep = int(ck.get("epoch", 0)) + 1
        print(f"[RESUME] seed={SEED} fold={RUN_FOLD} from epoch {start_ep} | best_auc={best_auc:.4f} at ep {best_epoch} | bad={bad}")

    for ep in range(start_ep, EPOCHS_MAX + 1):
        t0 = time.time()
        tr_loss = train_epoch(model, dl_tr, opt, crit_m, crit_l, scaler, epoch=ep)
        t_train = (time.time() - t0) / 60.0

        do_eval = (ep == 1) or (ep % EVAL_EVERY == 0)
        if do_eval:
            t1 = time.time()
            _pids, yv, _lv, logits, gates = predict_logits_multibag(model, ds_va, K=K_VAL, use_tta=USE_TTA_VAL)
            t_val = (time.time() - t1) / 60.0

            auc_va = try_auc(yv, sigmoid_np(logits))
            gate_mean = float(np.mean(gates)) if len(gates) else float("nan")

            print(f"[{BENCHMARK}] seed={SEED} fold={RUN_FOLD} ep {ep:02d}/{EPOCHS_MAX} | "
                  f"train_loss={tr_loss:.4f} | val_auc(K)={auc_va:.4f} | gate_mean={gate_mean:.3f} | "
                  f"time: train={t_train:.2f}m val={t_val:.2f}m")

            improved = (auc_va > best_auc + MIN_DELTA) or (best_epoch < 0)
            if improved:
                best_auc = float(auc_va)
                best_epoch = int(ep)
                bad = 0
                save_ckpt(ckpt_best, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)
                with open(summary_path, "w", encoding="utf-8") as f:
                    f.write(f"seed={SEED}\nfold={RUN_FOLD}\nbest_epoch={best_epoch}\nbest_auc={best_auc:.6f}\n")
                print(f"[SAVE] best updated -> {summary_path}")
            else:
                bad += 1
                if bad >= EARLY_STOP_PATIENCE:
                    print(f"Early stop at ep {ep} (best_auc={best_auc:.4f} at ep {best_epoch}).")
                    break
        else:
            print(f"[{BENCHMARK}] seed={SEED} fold={RUN_FOLD} ep {ep:02d}/{EPOCHS_MAX} | "
                  f"train_loss={tr_loss:.4f} | (no val this epoch) | time: train={t_train:.2f}m")

        # always save progress
        save_ckpt(ckpt_prog, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)

    # load best and do TEST preds for this fold
    if ckpt_best.exists():
        ck = torch.load(str(ckpt_best), map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"], strict=True)

    pids_te, y_te, le_te, lg_te, gg_te = predict_logits_multibag(model, ds_te, K=K_TEST, use_tta=USE_TTA_TEST)
    save_fold_test_csv(SEED, RUN_FOLD, pids_te, y_te, le_te, lg_te, gg_te)

    # now aggregate 1..5 (requires pred_fold1..5 all exist)
    aggregate_seed_folds(SEED, AGG_FOLDS)


if __name__ == "__main__":
    main()

DEVICE: cuda:0
BENCHMARK=res_shift
 train: D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv
 val  : D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv
 test : D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv
RUNNING seed=43 fold=1 then aggregate folds [1, 2, 3, 4, 5]
[INFO] filtered missing files: 13
[POOL] patients=137 (train+val)
[TEST] patients=73
RUN seed=43 fold=1/5 | K_VAL=5 | K_TEST=10 | TTA_VAL=False | TTA_TEST=True
TRAIN_BS=2 (accum 4) | EVAL_BS=2 | EVAL_EVERY=2
Model on: cuda:0
[RESUME] seed=43 fold=1 from epoch 4 | best_auc=0.7875 at ep 3 | bad=0
[res_shift] seed=43 fold=1 ep 04/30 | train_loss=1.2826 | val_auc(K)=0.9250 | gate_mean=0.510 | time: train=1.81m val=1.35m
[SAVE] best updated -> D:\ENT\_challenge2_artifacts\_runs\summary_res_shift_seed43_fold1.txt
[res_shift] seed=43 fold=1 ep 05/30 | train_loss=1.1669 | (no val this epoch) | time: train=1.47m
[res_shift] seed=43 fold=1 ep 06/30 | train_loss=1.0865 | val_auc(K)=0.9

ValueError: columns overlap but no suffix specified: Index(['y_true', 'leuko'], dtype='object')

In [1]:
# ============================================
# MAX-AUC PIPELINE (SINGLE SCRIPT, WINDOWS/GPU, RESUMABLE)
# - Patient-level 5-fold CV on (train+val pool)
# - Multi-bag inference on VAL/TEST (avg logits)
# - Optional TTA on VAL/TEST
# - Gate temperature + gate entropy regularization
# - FULL RESUME per (seed, fold):
#     * model + optimizer + scaler + epoch + earlystop counters
#     * (best-effort) RNG restore (handles torch>=2.6 safely)
# - OOM-safe defaults:
#     * TRAIN_BS=2 with grad-accum
#     * EVAL_BS=2
#     * eval every EVAL_EVERY epochs
# - Produces:
#     * cv_best_{BENCHMARK}_seed{seed}_fold{fold}.pt (resumable)
#     * summary_{BENCHMARK}_seed{seed}_fold{fold}.txt
#     * pred_fold{fold}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv
#     * pred_seed{seed}_{BENCHMARK}_folds{a}_{b}_Ktest{K_TEST}_agg.csv
#
# NOTE:
# - Aggregation FIXED (no y_true/leuko column overlap).
# - Loads checkpoints with torch.load(..., weights_only=False) for torch>=2.6 behavior.
#   Only do this for your own trusted checkpoints (which these are).
# ============================================

import os, json, math, random, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import torchvision
from torchvision import transforms as T

# ---------------------------
# Paths / Config (EDIT THESE)
# ---------------------------
# Windows example:
ART_DIR = Path(r"D:\ENT\_challenge2_artifacts")
# Colab example:
# ART_DIR = Path("/content/drive/MyDrive/ENT/Data/_challenge2_artifacts")

RUNS_DIR = ART_DIR / "_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK = "res_shift"  # "standard" or "res_shift"

SPLITS = {
    "standard": {
        "train": ART_DIR / "split_standard_train_colab.csv",
        "val":   ART_DIR / "split_standard_val_colab.csv",
        "test":  ART_DIR / "split_standard_test_colab.csv",
    },
    "res_shift": {
        "train": ART_DIR / "split_res_shift_train_major_colab.csv",
        "val":   ART_DIR / "split_res_shift_val_major_colab.csv",
        "test":  ART_DIR / "split_res_shift_test_minor_colab.csv",
    }
}
SPLIT_TRAIN = SPLITS[BENCHMARK]["train"]
SPLIT_VAL   = SPLITS[BENCHMARK]["val"]
SPLIT_TEST  = SPLITS[BENCHMARK]["test"]

# ---------------------------
# Device
# ---------------------------
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
AMP = True

# ---------------------------
# Data
# ---------------------------
IMG_SIZE = 224
NUM_WORKERS = 0

# OOM-safe batch sizes
TRAIN_BS = 2
GRAD_ACCUM = 4           # effective patients/batch = TRAIN_BS * GRAD_ACCUM
EVAL_BS = 2              # OOM-safe

# Bag sizes
BAG_SIZE_NONLEUKO = 24
BAG_SIZE_LEUKO = 40

# ---------------------------
# Training
# ---------------------------
EPOCHS_MAX = 30
LR = 5e-5
WEIGHT_DECAY = 1e-4

EARLY_STOP_PATIENCE = 6
MIN_DELTA = 1e-4

# validate less often to speed up (set 1 to validate every epoch)
EVAL_EVERY = 2

# Multi-bag K
K_VAL = 5                # was 20; keep smaller on Windows for speed
K_TEST = 10

# TTA options
TTA_VAL = False          # turn off for speed
TTA_TEST = True          # can keep on for final test predictions

# Aux leuko head (kept)
USE_AUX_LEUKO = True
LAMBDA_LEUKO = 0.30

# Gate warmup: force g=0.5 for first epochs
WARMUP_FORCE_HALF_GATE_EPOCHS = 2

# Gate stabilization
GATE_TAU = 2.0
LAMBDA_GATE_ENT = 0.02

# Optional: stabilize early training
FREEZE_BACKBONE_EPOCHS = 2

# CV + seeds
N_FOLDS = 5
SEEDS = [43]            # set to [42,43,44,45,46] when ready

# Which folds to run this time (inclusive)
FOLD_START = 1
FOLD_END = 5

# Which folds to aggregate at the end (must all exist)
AGG_FOLDS = [1, 2, 3, 4, 5]

print("DEVICE:", DEVICE)
print("=" * 70)
print(f"BENCHMARK={BENCHMARK}")
print(" train:", SPLIT_TRAIN)
print(" val  :", SPLIT_VAL)
print(" test :", SPLIT_TEST)
print(f"RUNNING seeds={SEEDS} folds={FOLD_START}..{FOLD_END} (of {N_FOLDS})")
print("=" * 70)

# ---------------------------
# Repro helpers
# ---------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _to_torch_byte_tensor(x):
    # robust conversion for torch RNG states saved as list/np arrays
    if x is None:
        return None
    if isinstance(x, torch.ByteTensor) or (isinstance(x, torch.Tensor) and x.dtype == torch.uint8):
        return x
    if isinstance(x, (bytes, bytearray)):
        return torch.tensor(list(x), dtype=torch.uint8)
    if isinstance(x, (list, tuple, np.ndarray)):
        arr = np.asarray(x, dtype=np.uint8).reshape(-1)
        return torch.from_numpy(arr)
    return None

def capture_rng_state():
    rng = {
        "py": random.getstate(),
        "np": np.random.get_state(),
        "torch": torch.get_rng_state().clone().cpu(),
        "cuda": None
    }
    if torch.cuda.is_available():
        rng["cuda"] = [s.clone().cpu() for s in torch.cuda.get_rng_state_all()]
    return rng

def restore_rng_state(rng):
    if rng is None:
        return
    try:
        random.setstate(rng.get("py"))
    except Exception as e:
        print("[WARN] could not restore python RNG:", e)
    try:
        np.random.set_state(rng.get("np"))
    except Exception as e:
        print("[WARN] could not restore numpy RNG:", e)

    # torch RNG
    try:
        t = _to_torch_byte_tensor(rng.get("torch"))
        if t is not None:
            torch.set_rng_state(t)
        else:
            raise TypeError("RNG state must be a torch.ByteTensor")
    except Exception as e:
        print("[WARN] could not restore torch RNG:", e)

    # cuda RNG
    if torch.cuda.is_available():
        try:
            cuda_states = rng.get("cuda", None)
            if cuda_states is not None:
                fixed = []
                for s in cuda_states:
                    t = _to_torch_byte_tensor(s)
                    if t is None:
                        raise TypeError("RNG state must be a torch.ByteTensor")
                    fixed.append(t)
                torch.cuda.set_rng_state_all(fixed)
        except Exception as e:
            print("[WARN] could not restore cuda RNG:", e)

def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))

def try_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def safe_int(x, default=0):
    try:
        return int(float(x))
    except Exception:
        return default

def parse_leuko(x):
    if x is None:
        return 0
    s = str(x).strip().lower()
    return 1 if s in ["yes", "1", "true", "y"] else 0

def parse_malign(x):
    return 1 if safe_int(x, 0) == 1 else 0

# ---------------------------
# Robust CSV loading
# ---------------------------
def load_csv_rows(csv_path):
    csv_path = str(csv_path)
    try:
        import pandas as pd
        df = pd.read_csv(csv_path)
        return df.to_dict(orient="records")
    except Exception as e:
        print(f"[WARN] pandas read_csv failed, fallback csv module: {e}")
        import csv
        rows = []
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for r in reader:
                rows.append(r)
        return rows

def ensure_exists(rows):
    kept, missing = [], 0
    for r in rows:
        fp = r.get("filepath", "")
        if fp and os.path.exists(fp):
            kept.append(r)
        else:
            missing += 1
    if missing > 0:
        print(f"[INFO] filtered missing files: {missing}")
    return kept

# ---------------------------
# Build patient index
# ---------------------------
def build_patient_index(rows):
    pid_to_frames = defaultdict(list)

    for r in rows:
        pid = r.get("Patient", None) or r.get("patient", None) or r.get("PID", None)
        if pid is None:
            fp = r.get("filepath", "")
            pid = "UNK"
            if "Patient" in fp:
                try:
                    pid = "Patient" + fp.split("Patient")[1].split("/")[0]
                except Exception:
                    pid = "UNK"

        y_m = parse_malign(r.get("label_binary", 0))
        y_l = parse_leuko(r.get("Leukoplakia", 0))
        fp = r.get("filepath", "")

        pid_to_frames[pid].append({"filepath": fp, "y_m": y_m, "y_l": y_l})

    pid_to_labels = {}
    for pid, frs in pid_to_frames.items():
        y_m = 1 if any(f["y_m"] == 1 for f in frs) else 0
        y_l = 1 if any(f["y_l"] == 1 for f in frs) else 0
        pid_to_labels[pid] = (y_m, y_l)

    patients = sorted(pid_to_frames.keys())
    return patients, pid_to_frames, pid_to_labels

# ---------------------------
# Transforms
# ---------------------------
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
    T.ToTensor(),
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

eval_tf_hflip = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=1.0),
    T.ToTensor(),
])

# ---------------------------
# Dataset: patient bags
# ---------------------------
class PatientBagDataset(Dataset):
    def __init__(self, patients, pid_to_frames, pid_to_labels, train=True):
        self.patients = list(patients)
        self.pid_to_frames = pid_to_frames
        self.pid_to_labels = pid_to_labels
        self.train = train
        self.tf = train_tf if train else eval_tf
        self.eval_random = True

    def set_transform(self, tf):
        self.tf = tf

    def __len__(self):
        return len(self.patients)

    def _sample_frames(self, pid):
        frs = self.pid_to_frames[pid]
        _ym, yl = self.pid_to_labels[pid]
        bag_size = BAG_SIZE_LEUKO if yl == 1 else BAG_SIZE_NONLEUKO
        n = len(frs)

        if self.train:
            idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size))
        else:
            idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size)) if self.eval_random else np.arange(min(n, bag_size))
            if len(idxs) < bag_size:
                reps = int(math.ceil(bag_size / max(len(idxs), 1)))
                idxs = (np.tile(idxs, reps))[:bag_size]

        return [frs[i] for i in idxs]

    def __getitem__(self, i):
        pid = self.patients[i]
        ym, yl = self.pid_to_labels[pid]
        sampled = self._sample_frames(pid)

        imgs = []
        for f in sampled:
            fp = f["filepath"]
            im = Image.open(fp).convert("RGB")
            im = self.tf(im)
            imgs.append(im)

        X = torch.stack(imgs, dim=0)
        return X, ym, yl, pid

def collate_patient_bags(batch):
    Xs, ym, yl, pids = zip(*batch)
    lengths = [x.shape[0] for x in Xs]
    max_len = max(lengths)

    B = len(Xs)
    C, H, W = Xs[0].shape[1], Xs[0].shape[2], Xs[0].shape[3]

    Xpad = torch.zeros((B, max_len, C, H, W), dtype=Xs[0].dtype)
    mask = torch.zeros((B, max_len), dtype=torch.bool)

    for i, x in enumerate(Xs):
        n = x.shape[0]
        Xpad[i, :n] = x
        mask[i, :n] = True

    ym = torch.tensor(ym, dtype=torch.float32).unsqueeze(1)
    yl = torch.tensor(yl, dtype=torch.float32).unsqueeze(1)
    return Xpad, mask, ym, yl, list(pids)

# ---------------------------
# Model
# ---------------------------
class ResNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(m.children())[:-1])
        self.out_dim = 2048

    def forward(self, x):
        h = self.backbone(x)
        return h.flatten(1)

class AttnPool(nn.Module):
    def __init__(self, d, h=256):
        super().__init__()
        self.V = nn.Linear(d, h)
        self.w = nn.Linear(h, 1)

    def forward(self, H, mask=None):
        A = self.w(torch.tanh(self.V(H)))
        if mask is not None:
            A = A.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        A = torch.softmax(A, dim=1)
        Z = (A * H).sum(dim=1)
        return Z, A.squeeze(-1)

class GatedMILv2(nn.Module):
    def __init__(self, use_aux_leuko=True, gate_tau=2.0):
        super().__init__()
        self.use_aux_leuko = bool(use_aux_leuko)
        self.gate_tau = float(gate_tau)

        self.encoder = ResNetEncoder()
        d = self.encoder.out_dim

        self.pool_non = AttnPool(d, h=256)
        self.pool_leu = AttnPool(d, h=256)

        self.cls_non = nn.Linear(d, 1)
        self.cls_leu = nn.Linear(d, 1)

        self.gate = nn.Linear(2 * d, 1)

        if self.use_aux_leuko:
            self.leuko_head = nn.Sequential(
                nn.Linear(2 * d, 256),
                nn.ReLU(inplace=True),
                nn.Dropout(0.2),
                nn.Linear(256, 1),
            )

    def forward(self, X, mask=None, force_half_gate=False):
        B, N, C, H, W = X.shape
        x = X.view(B * N, C, H, W)
        h = self.encoder(x)
        d = h.shape[-1]
        Hbag = h.view(B, N, d)

        z_non, _ = self.pool_non(Hbag, mask=mask)
        z_leu, _ = self.pool_leu(Hbag, mask=mask)

        logit_non = self.cls_non(z_non)
        logit_leu = self.cls_leu(z_leu)

        gate_in = torch.cat([z_non, z_leu], dim=1)
        g_logit = self.gate(gate_in) / self.gate_tau
        g = torch.sigmoid(g_logit)

        if force_half_gate:
            g = 0.5 * torch.ones_like(g)

        logit_m = (1.0 - g) * logit_non + g * logit_leu

        logit_l = None
        if self.use_aux_leuko:
            logit_l = self.leuko_head(gate_in)

        return logit_m, logit_l, g

def set_backbone_trainable(model: nn.Module, trainable: bool):
    for p in model.encoder.backbone.parameters():
        p.requires_grad = bool(trainable)

# ---------------------------
# Gate entropy regularizer
# ---------------------------
def gate_entropy(g, eps=1e-6):
    g = torch.clamp(g, eps, 1.0 - eps)
    ent = -(g * torch.log(g) + (1.0 - g) * torch.log(1.0 - g))
    return ent.mean()

# ---------------------------
# Checkpoint helpers (FULL RESUME)
# ---------------------------
def save_ckpt(path: Path, model, opt, scaler, epoch, best_auc, best_epoch, bad):
    ck = {
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": int(epoch),
        "best_auc": float(best_auc),
        "best_epoch": int(best_epoch),
        "bad": int(bad),
        "rng": capture_rng_state(),
    }
    torch.save(ck, str(path))

def load_ckpt(path: Path, model, opt, scaler):
    ck = torch.load(str(path), map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"], strict=True)
    if opt is not None and "opt" in ck:
        opt.load_state_dict(ck["opt"])
    if scaler is not None and ck.get("scaler") is not None:
        scaler.load_state_dict(ck["scaler"])
    restore_rng_state(ck.get("rng", None))
    return ck

# ---------------------------
# Train + multi-bag inference
# ---------------------------
def train_epoch(model, loader, opt, crit_m, crit_l, scaler, epoch):
    model.train()
    losses = []

    if FREEZE_BACKBONE_EPOCHS > 0:
        set_backbone_trainable(model, trainable=(epoch > FREEZE_BACKBONE_EPOCHS))

    opt.zero_grad(set_to_none=True)

    for step, (X, mask, ym, yl, _pids) in enumerate(loader, start=1):
        X = X.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        ym = ym.to(DEVICE, non_blocking=True)
        yl = yl.to(DEVICE, non_blocking=True)

        force_half = (epoch <= WARMUP_FORCE_HALF_GATE_EPOCHS)

        if scaler is not None:
            with torch.amp.autocast(device_type="cuda"):
                logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
                loss = crit_m(logit_m, ym)

                if USE_AUX_LEUKO and (logit_l is not None):
                    loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)

                if (not force_half) and (LAMBDA_GATE_ENT > 0):
                    loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

                loss = loss / float(GRAD_ACCUM)

            scaler.scale(loss).backward()

            if step % GRAD_ACCUM == 0:
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)

        else:
            logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
            loss = crit_m(logit_m, ym)
            if USE_AUX_LEUKO and (logit_l is not None):
                loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)
            if (not force_half) and (LAMBDA_GATE_ENT > 0):
                loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

            loss = loss / float(GRAD_ACCUM)
            loss.backward()

            if step % GRAD_ACCUM == 0:
                opt.step()
                opt.zero_grad(set_to_none=True)

        losses.append(float(loss.detach().cpu().item()) * float(GRAD_ACCUM))

    # flush remainder grads
    if (len(loader) % GRAD_ACCUM) != 0:
        if scaler is not None:
            scaler.step(opt); scaler.update()
        else:
            opt.step()
        opt.zero_grad(set_to_none=True)

    return float(np.nanmean(losses)) if losses else float("nan")

@torch.no_grad()
def predict_logits_multibag(model, ds_eval, K=10, use_tta=True):
    model.eval()

    pid_list = ds_eval.patients
    pid_to_idx = {p: i for i, p in enumerate(pid_list)}
    n = len(pid_list)

    sum_logits = np.zeros((n,), dtype=np.float64)
    sum_gate = np.zeros((n,), dtype=np.float64)

    dl = DataLoader(
        ds_eval,
        batch_size=EVAL_BS,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=collate_patient_bags
    )

    tta_tfs = [eval_tf]
    if use_tta:
        tta_tfs.append(eval_tf_hflip)

    denom = float(K * len(tta_tfs))

    for _k in range(K):
        ds_eval.train = False
        ds_eval.eval_random = True

        for tf in tta_tfs:
            ds_eval.set_transform(tf)

            for X, mask, _ym, _yl, pids in dl:
                X = X.to(DEVICE, non_blocking=True)
                mask = mask.to(DEVICE, non_blocking=True)

                logit_m, _logit_l, g = model(X, mask=mask, force_half_gate=False)

                logit_m = logit_m.detach().float().cpu().numpy().reshape(-1)
                g = g.detach().float().cpu().numpy().reshape(-1)

                for pid, lg, gg in zip(pids, logit_m, g):
                    j = pid_to_idx[pid]
                    sum_logits[j] += float(lg)
                    sum_gate[j] += float(gg)

    avg_logits = sum_logits / denom
    avg_gate = sum_gate / denom

    y = np.asarray([ds_eval.pid_to_labels[p][0] for p in pid_list], dtype=int)
    le = np.asarray([ds_eval.pid_to_labels[p][1] for p in pid_list], dtype=int)

    return avg_logits, avg_gate, pid_list, y, le

# ---------------------------
# CV folds (patient-level stratified)
# ---------------------------
def make_patient_folds(patients, pid_to_labels, n_splits=5, seed=42):
    strat = np.array([2 * pid_to_labels[p][0] + pid_to_labels[p][1] for p in patients], dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idxs = np.arange(len(patients))
    folds = []
    for tr_idx, va_idx in skf.split(idxs, strat):
        tr_p = [patients[i] for i in tr_idx]
        va_p = [patients[i] for i in va_idx]
        folds.append((tr_p, va_p))
    return folds

# ---------------------------
# Saving predictions
# ---------------------------
def save_fold_test_csv(seed, fold_i, pids, y_true, leuko, logits, gates):
    out = RUNS_DIR / f"pred_fold{fold_i}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit,prob,gate\n")
        for pid, yt, lk, lg, gg in zip(pids, y_true, leuko, logits, gates):
            pr = float(sigmoid_np(lg))
            f.write(f"{pid},{int(yt)},{int(lk)},{float(lg):.8f},{pr:.8f},{float(gg):.6f}\n")
    print("Saved fold test preds:", out)

def aggregate_seed_folds(seed, folds_list):
    """
    FIXED: avoids y_true/leuko overlap by keeping them only from the first fold.
    Joins only the logit columns for the remaining folds.
    """
    import pandas as pd

    base = None
    logit_cols = []

    for i, fidx in enumerate(folds_list):
        p = RUNS_DIR / f"pred_fold{fidx}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")

        df = pd.read_csv(p).set_index("Patient")
        col_logit = f"logit_fold{fidx}"
        df = df.rename(columns={"logit": col_logit})

        if i == 0:
            base = df[["y_true", "leuko", col_logit]].copy()
        else:
            base = base.join(df[[col_logit]], how="inner")

        logit_cols.append(col_logit)

    logit_mean = base[logit_cols].mean(axis=1).astype(float).values
    prob_mean = sigmoid_np(logit_mean)

    out = RUNS_DIR / f"pred_seed{seed}_{BENCHMARK}_folds{min(folds_list)}_{max(folds_list)}_Ktest{K_TEST}_agg.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit_mean,prob_mean\n")
        for pid, yt, lk, lg, pr in zip(
            base.index.tolist(),
            base["y_true"].astype(int).values.tolist(),
            base["leuko"].astype(int).values.tolist(),
            logit_mean.tolist(),
            prob_mean.tolist()
        ):
            f.write(f"{pid},{yt},{lk},{lg:.8f},{pr:.8f}\n")

    auc = try_auc(base["y_true"].values.astype(int), prob_mean)
    print("Saved:", out)
    print(f"AGG TEST AUC (avg logits over folds {min(folds_list)}..{max(folds_list)}): {auc}")
    return out, float(auc)

# ---------------------------
# Main
# ---------------------------
def main():
    rows_tr = ensure_exists(load_csv_rows(SPLIT_TRAIN))
    rows_va = ensure_exists(load_csv_rows(SPLIT_VAL))
    rows_te = ensure_exists(load_csv_rows(SPLIT_TEST))

    rows_pool = rows_tr + rows_va
    patients_pool, pid_to_frames_pool, pid_to_labels_pool = build_patient_index(rows_pool)
    patients_pool = [p for p in patients_pool if len(pid_to_frames_pool[p]) >= 2]

    patients_test, pid_to_frames_test, pid_to_labels_test = build_patient_index(rows_te)
    patients_test = [p for p in patients_test if len(pid_to_frames_test[p]) >= 2]

    print(f"[POOL] patients={len(patients_pool)} (train+val)")
    print(f"[TEST] patients={len(patients_test)}")
    print("=" * 70)
    print(f"RUN seed(s)={SEEDS} | K_VAL={K_VAL} | K_TEST={K_TEST} | TTA_VAL={TTA_VAL} | TTA_TEST={TTA_TEST}")
    print(f"Gate: tau={GATE_TAU} | lambda_ent={LAMBDA_GATE_ENT} | freeze_backbone_epochs={FREEZE_BACKBONE_EPOCHS}")
    print(f"TRAIN_BS={TRAIN_BS} (accum {GRAD_ACCUM}) | EVAL_BS={EVAL_BS} | EVAL_EVERY={EVAL_EVERY}")
    print("=" * 70)

    folds = make_patient_folds(patients_pool, pid_to_labels_pool, n_splits=N_FOLDS, seed=SEEDS[0])

    for seed in SEEDS:
        seed_all(seed)
        print("\n" + "=" * 70)
        print(f"RUN seed={seed} | benchmark={BENCHMARK} | folds={FOLD_START}..{FOLD_END}")
        print("=" * 70)

        for fold_i, (tr_pat, va_pat) in enumerate(folds, start=1):
            if fold_i < FOLD_START or fold_i > FOLD_END:
                continue

            print(f"\n--- Fold {fold_i}/{N_FOLDS} | train_pat={len(tr_pat)} val_pat={len(va_pat)} ---")

            ds_tr = PatientBagDataset(tr_pat, pid_to_frames_pool, pid_to_labels_pool, train=True)
            ds_va = PatientBagDataset(va_pat, pid_to_frames_pool, pid_to_labels_pool, train=False)
            ds_te = PatientBagDataset(patients_test, pid_to_frames_test, pid_to_labels_test, train=False)

            # train loader
            dl_tr = DataLoader(
                ds_tr,
                batch_size=TRAIN_BS,
                shuffle=True,
                num_workers=NUM_WORKERS,
                pin_memory=True,
                collate_fn=collate_patient_bags
            )

            # pos_weight from train fold
            y_m_tr = np.array([pid_to_labels_pool[p][0] for p in tr_pat], dtype=int)
            y_l_tr = np.array([pid_to_labels_pool[p][1] for p in tr_pat], dtype=int)
            pos_m = int(y_m_tr.sum()); neg_m = int(len(y_m_tr) - pos_m)
            pos_l = int(y_l_tr.sum()); neg_l = int(len(y_l_tr) - pos_l)

            pos_weight_m = torch.tensor([neg_m / max(pos_m, 1)], dtype=torch.float32, device=DEVICE)
            pos_weight_l = torch.tensor([neg_l / max(pos_l, 1)], dtype=torch.float32, device=DEVICE)

            crit_m = nn.BCEWithLogitsLoss(pos_weight=pos_weight_m)
            crit_l = nn.BCEWithLogitsLoss(pos_weight=pos_weight_l)

            model = GatedMILv2(use_aux_leuko=USE_AUX_LEUKO, gate_tau=GATE_TAU).to(DEVICE)
            print("Model on:", next(model.parameters()).device)

            opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scaler = torch.amp.GradScaler("cuda") if (AMP and DEVICE.startswith("cuda")) else None

            ckpt_path = RUNS_DIR / f"cv_best_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"

            best_auc = -1e9
            best_epoch = -1
            bad = 0
            start_ep = 1

            if ckpt_path.exists():
                ck = load_ckpt(ckpt_path, model, opt, scaler)
                best_auc = float(ck.get("best_auc", -1e9))
                best_epoch = int(ck.get("best_epoch", ck.get("epoch", -1)))
                bad = int(ck.get("bad", 0))
                start_ep = int(ck.get("epoch", 0)) + 1
                print(f"[RESUME] seed={seed} fold={fold_i} from epoch {start_ep} | best_auc={best_auc:.4f} at ep {best_epoch} | bad={bad}")

            for ep in range(start_ep, EPOCHS_MAX + 1):
                t0 = time.time()
                tr_loss = train_epoch(model, dl_tr, opt, crit_m, crit_l, scaler, epoch=ep)
                t_train = time.time() - t0

                do_val = (ep == start_ep) or (ep % EVAL_EVERY == 0) or (ep == EPOCHS_MAX)
                if do_val:
                    t1 = time.time()
                    lg_va, gg_va, _pids, y_va, _le = predict_logits_multibag(model, ds_va, K=K_VAL, use_tta=TTA_VAL)
                    auc_va = try_auc(y_va, sigmoid_np(lg_va))
                    gate_mean = float(np.mean(gg_va)) if len(gg_va) else float("nan")
                    t_val = time.time() - t1

                    print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                          f"train_loss={tr_loss:.4f} | val_auc(K)={auc_va:.4f} | gate_mean={gate_mean:.3f} | "
                          f"time: train={t_train/60:.2f}m val={t_val/60:.2f}m")

                    improved = (auc_va > best_auc + MIN_DELTA) or (best_epoch < 0)
                    if improved:
                        best_auc = float(auc_va)
                        best_epoch = int(ep)
                        bad = 0
                        save_ckpt(ckpt_path, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)

                        summary_path = RUNS_DIR / f"summary_{BENCHMARK}_seed{seed}_fold{fold_i}.txt"
                        with open(summary_path, "w", encoding="utf-8") as f:
                            f.write(f"seed={seed}\nfold={fold_i}\nbest_epoch={best_epoch}\nbest_auc={best_auc:.6f}\n")
                        print(f"[SAVE] best updated -> {summary_path}")
                    else:
                        bad += 1
                        if bad >= EARLY_STOP_PATIENCE:
                            print(f"Early stop at ep {ep} (best_auc={best_auc:.4f} at ep {best_epoch}).")
                            break
                else:
                    print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                          f"train_loss={tr_loss:.4f} | (no val this epoch) | time: train={t_train/60:.2f}m")

            # load best for TEST prediction for this fold
            if ckpt_path.exists():
                ck = torch.load(str(ckpt_path), map_location=DEVICE, weights_only=False)
                model.load_state_dict(ck["model"], strict=True)
            model.eval()

            lg_te, gg_te, pids_te, y_te, le_te = predict_logits_multibag(model, ds_te, K=K_TEST, use_tta=TTA_TEST)
            save_fold_test_csv(seed, fold_i, pids_te, y_te, le_te, lg_te, gg_te)

        # aggregate after running selected folds (only if all folds exist)
        try:
            aggregate_seed_folds(seed, AGG_FOLDS)
        except FileNotFoundError as e:
            print("[AGG] skipped:", e)

if __name__ == "__main__":
    main()

DEVICE: cuda:0
BENCHMARK=res_shift
 train: D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv
 val  : D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv
 test : D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv
RUNNING seeds=[43] folds=1..5 (of 5)
[INFO] filtered missing files: 13
[POOL] patients=137 (train+val)
[TEST] patients=73
RUN seed(s)=[43] | K_VAL=5 | K_TEST=10 | TTA_VAL=False | TTA_TEST=True
Gate: tau=2.0 | lambda_ent=0.02 | freeze_backbone_epochs=2
TRAIN_BS=2 (accum 4) | EVAL_BS=2 | EVAL_EVERY=2

RUN seed=43 | benchmark=res_shift | folds=1..5

--- Fold 1/5 | train_pat=109 val_pat=28 ---
Model on: cuda:0
[WARN] could not restore torch RNG: RNG state must be a torch.ByteTensor
[WARN] could not restore cuda RNG: RNG state must be a torch.ByteTensor
[RESUME] seed=43 fold=1 from epoch 21 | best_auc=1.0000 at ep 20 | bad=0
[res_shift] seed=43 fold=1 ep 21/30 | train_loss=0.5013 | val_auc(K)=0.9062 | gate_mean=0.914 | time: train=1.78m v

In [1]:
# ============================================================
# MAX-AUC PIPELINE (SINGLE SCRIPT) — RESUMABLE + GPU + WINDOWS/DRIVE
# - Patient-level 5-fold CV on (train+val pool)
# - Multi-bag inference on VAL/TEST (avg logits)
# - Optional TTA at eval (noflip + hflip)
# - Gate temperature + gate entropy regularization
# - FULL RESUME per (seed, fold) from progress OR best checkpoints
# - Writes:
#     * cv_best_{benchmark}_seed{seed}_fold{fold}.pt
#     * progress_{benchmark}_seed{seed}_fold{fold}.pt
#     * summary_{benchmark}_seed{seed}_fold{fold}.txt
#     * pred_fold{fold}_{benchmark}_seed{seed}_Ktest{K_TEST}.csv
#     * pred_seed{seed}_{benchmark}_folds{a}_{b}_Ktest{K_TEST}_agg.csv
# ============================================================

import os, json, math, random, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import torchvision
from torchvision import transforms as T

# ------------------------------------------------------------
# USER CONFIG
# ------------------------------------------------------------

# Root folder that contains split CSVs and _runs/
# Example:
#   Windows: r"D:\ENT\_challenge2_artifacts"
#   Colab:   "/content/drive/MyDrive/ENT/Data/_challenge2_artifacts"
ART_DIR = Path(r"D:\ENT\_challenge2_artifacts")

BENCHMARK = "res_shift"  # "standard" or "res_shift"

SPLITS = {
    "standard": {
        "train": ART_DIR / "split_standard_train_colab.csv",
        "val":   ART_DIR / "split_standard_val_colab.csv",
        "test":  ART_DIR / "split_standard_test_colab.csv",
    },
    "res_shift": {
        "train": ART_DIR / "split_res_shift_train_major_colab.csv",
        "val":   ART_DIR / "split_res_shift_val_major_colab.csv",
        "test":  ART_DIR / "split_res_shift_test_minor_colab.csv",
    }
}

# Seeds to run (you can put [42], [43], or multiple)
SEEDS = [43]

# Which folds to run (1..5). Set start/end to run a subset.
FOLD_START = 1
FOLD_END   = 5

# After running, aggregate these folds for each seed (must exist)
AGG_FOLDS = [1, 2, 3, 4, 5]

# ------------------------------------------------------------
# TRAIN/EVAL CONFIG
# ------------------------------------------------------------

# GPU
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
AMP = True
torch.backends.cudnn.benchmark = True

# Data
IMG_SIZE = 224
NUM_WORKERS = 0

# Batch sizes (patient-bag batches). Keep low if VRAM tight.
TRAIN_BS = 2
EVAL_BS  = 2

# Gradient accumulation (effective batch = TRAIN_BS * ACCUM_STEPS)
ACCUM_STEPS = 4

# Bag sizes
BAG_SIZE_NONLEUKO = 24
BAG_SIZE_LEUKO = 40

# Training
EPOCHS_MAX = 30
LR = 5e-5
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 6
MIN_DELTA = 1e-4

# Optional leuko auxiliary head (keep True if your checkpoints were trained with it)
USE_AUX_LEUKO = True
LAMBDA_LEUKO = 0.30

# Gate warmup: force g=0.5 for first epochs
WARMUP_FORCE_HALF_GATE_EPOCHS = 2

# Gate stabilization
GATE_TAU = 2.0
LAMBDA_GATE_ENT = 0.02

# Backbone freezing
FREEZE_BACKBONE_EPOCHS = 2

# Multi-bag + TTA
K_VAL = 5           # smaller makes it faster; you used 5 successfully
K_TEST = 10
TTA_VAL = False     # faster
TTA_TEST = True     # more stable test estimate

# Evaluate on val every N epochs (saves time). Use 1 for classic behavior.
EVAL_EVERY = 2

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
RUNS_DIR = ART_DIR / "_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_TRAIN = SPLITS[BENCHMARK]["train"]
SPLIT_VAL   = SPLITS[BENCHMARK]["val"]
SPLIT_TEST  = SPLITS[BENCHMARK]["test"]

print(f"DEVICE: {DEVICE}")
print("="*70)
print(f"BENCHMARK={BENCHMARK}")
print(" train:", SPLIT_TRAIN)
print(" val  :", SPLIT_VAL)
print(" test :", SPLIT_TEST)
print(f"RUNNING seeds={SEEDS} folds={FOLD_START}..{FOLD_END} (of 5)")
print("="*70)
print(f"RUN seed(s)={SEEDS} | K_VAL={K_VAL} | K_TEST={K_TEST} | "
      f"TTA_VAL={TTA_VAL} | TTA_TEST={TTA_TEST}")
print(f"Gate: tau={GATE_TAU} | lambda_ent={LAMBDA_GATE_ENT} | freeze_backbone_epochs={FREEZE_BACKBONE_EPOCHS}")
print(f"TRAIN_BS={TRAIN_BS} (accum {ACCUM_STEPS}) | EVAL_BS={EVAL_BS} | EVAL_EVERY={EVAL_EVERY}")
print("="*70)

# ------------------------------------------------------------
# REPRO HELPERS
# ------------------------------------------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def capture_rng_state():
    rng = {
        "py": random.getstate(),
        "np": np.random.get_state(),
        "torch": torch.get_rng_state().clone().cpu().to(torch.uint8),
        "cuda": None
    }
    if torch.cuda.is_available():
        rng["cuda"] = [s.clone().cpu().to(torch.uint8) for s in torch.cuda.get_rng_state_all()]
    return rng

def restore_rng_state(rng):
    if rng is None:
        return
    try:
        random.setstate(rng.get("py"))
    except Exception as e:
        print("[WARN] could not restore python RNG:", e)
    try:
        np.random.set_state(rng.get("np"))
    except Exception as e:
        print("[WARN] could not restore numpy RNG:", e)

    try:
        t = rng.get("torch", None)
        if isinstance(t, torch.Tensor) and t.dtype == torch.uint8:
            torch.set_rng_state(t)
    except Exception as e:
        print("[WARN] could not restore torch RNG:", e)

    if torch.cuda.is_available():
        try:
            cuda_states = rng.get("cuda", None)
            if cuda_states is not None:
                fixed = []
                for s in cuda_states:
                    if isinstance(s, torch.Tensor) and s.dtype == torch.uint8:
                        fixed.append(s)
                if len(fixed) == torch.cuda.device_count():
                    torch.cuda.set_rng_state_all(fixed)
        except Exception as e:
            print("[WARN] could not restore cuda RNG:", e)

def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))

def try_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def safe_int(x, default=0):
    try:
        return int(float(x))
    except Exception:
        return default

def parse_leuko(x):
    if x is None:
        return 0
    s = str(x).strip().lower()
    return 1 if s in ["yes", "1", "true", "y"] else 0

def parse_malign(x):
    return 1 if safe_int(x, 0) == 1 else 0

# ------------------------------------------------------------
# ROBUST CSV LOADING
# ------------------------------------------------------------
def load_csv_rows(csv_path):
    csv_path = str(csv_path)
    try:
        import pandas as pd
        df = pd.read_csv(csv_path)
        return df.to_dict(orient="records")
    except Exception as e:
        print(f"[WARN] pandas read_csv failed, fallback csv module: {e}")
        import csv
        rows = []
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for r in reader:
                rows.append(r)
        return rows

def ensure_exists(rows):
    kept, missing = [], 0
    for r in rows:
        fp = r.get("filepath", "")
        if fp and os.path.exists(fp):
            kept.append(r)
        else:
            missing += 1
    if missing > 0:
        print(f"[INFO] filtered missing files: {missing}")
    return kept

# ------------------------------------------------------------
# PATIENT INDEX
# ------------------------------------------------------------
def build_patient_index(rows):
    pid_to_frames = defaultdict(list)

    for r in rows:
        pid = r.get("Patient", None) or r.get("patient", None) or r.get("PID", None)
        if pid is None:
            fp = r.get("filepath", "")
            pid = "UNK"
            if "Patient" in fp:
                try:
                    pid = "Patient" + fp.split("Patient")[1].split("/")[0]
                except Exception:
                    pid = "UNK"

        y_m = parse_malign(r.get("label_binary", 0))
        y_l = parse_leuko(r.get("Leukoplakia", 0))
        fp = r.get("filepath", "")

        pid_to_frames[pid].append({"filepath": fp, "y_m": y_m, "y_l": y_l})

    pid_to_labels = {}
    for pid, frs in pid_to_frames.items():
        y_m = 1 if any(f["y_m"] == 1 for f in frs) else 0
        y_l = 1 if any(f["y_l"] == 1 for f in frs) else 0
        pid_to_labels[pid] = (y_m, y_l)

    patients = sorted(pid_to_frames.keys())
    return patients, pid_to_frames, pid_to_labels

# ------------------------------------------------------------
# TRANSFORMS
# ------------------------------------------------------------
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
    T.ToTensor(),
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

eval_tf_hflip = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=1.0),
    T.ToTensor(),
])

# ------------------------------------------------------------
# DATASET (PATIENT BAGS)
# ------------------------------------------------------------
class PatientBagDataset(Dataset):
    def __init__(self, patients, pid_to_frames, pid_to_labels, train=True):
        self.patients = list(patients)
        self.pid_to_frames = pid_to_frames
        self.pid_to_labels = pid_to_labels
        self.train = train
        self.tf = train_tf if train else eval_tf
        self.eval_random = True

    def set_transform(self, tf):
        self.tf = tf

    def __len__(self):
        return len(self.patients)

    def _sample_frames(self, pid):
        frs = self.pid_to_frames[pid]
        y_m, y_l = self.pid_to_labels[pid]
        bag_size = BAG_SIZE_LEUKO if y_l == 1 else BAG_SIZE_NONLEUKO
        n = len(frs)

        if self.train:
            idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size))
        else:
            if self.eval_random:
                idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size))
            else:
                if n >= bag_size:
                    idxs = np.arange(bag_size)
                else:
                    reps = int(math.ceil(bag_size / n))
                    idxs = (np.tile(np.arange(n), reps))[:bag_size]

        return [frs[i] for i in idxs]

    def __getitem__(self, i):
        pid = self.patients[i]
        y_m, y_l = self.pid_to_labels[pid]
        sampled = self._sample_frames(pid)

        imgs, fps = [], []
        for f in sampled:
            fp = f["filepath"]
            fps.append(fp)
            im = Image.open(fp).convert("RGB")
            im = self.tf(im)
            imgs.append(im)

        X = torch.stack(imgs, dim=0)  # (bag, C, H, W)
        return X, y_m, y_l, pid, fps

def collate_patient_bags(batch):
    Xs, ym, yl, pids, fps = zip(*batch)
    lengths = [x.shape[0] for x in Xs]
    max_len = max(lengths)

    B = len(Xs)
    C, H, W = Xs[0].shape[1], Xs[0].shape[2], Xs[0].shape[3]

    Xpad = torch.zeros((B, max_len, C, H, W), dtype=Xs[0].dtype)
    mask = torch.zeros((B, max_len), dtype=torch.bool)

    for i, x in enumerate(Xs):
        n = x.shape[0]
        Xpad[i, :n] = x
        mask[i, :n] = True

    ym = torch.tensor(ym, dtype=torch.long)
    yl = torch.tensor(yl, dtype=torch.long)
    return Xpad, mask, ym, yl, list(pids), list(fps)

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------
class ResNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(m.children())[:-1])
        self.out_dim = 2048

    def forward(self, x):
        h = self.backbone(x)
        return h.flatten(1)

class AttnPool(nn.Module):
    def __init__(self, d, h=256):
        super().__init__()
        self.V = nn.Linear(d, h)
        self.w = nn.Linear(h, 1)

    def forward(self, H, mask=None):
        A = self.w(torch.tanh(self.V(H)))  # (B,n,1)
        if mask is not None:
            A = A.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        A = torch.softmax(A, dim=1)
        Z = (A * H).sum(dim=1)
        return Z, A.squeeze(-1)

class GatedMILv2(nn.Module):
    def __init__(self, use_aux_leuko=True, gate_tau=2.0):
        super().__init__()
        self.use_aux_leuko = bool(use_aux_leuko)
        self.gate_tau = float(gate_tau)

        self.encoder = ResNetEncoder()
        d = self.encoder.out_dim

        self.pool_non = AttnPool(d, h=256)
        self.pool_leu = AttnPool(d, h=256)

        self.cls_non = nn.Linear(d, 1)
        self.cls_leu = nn.Linear(d, 1)

        self.gate = nn.Linear(2 * d, 1)

        if self.use_aux_leuko:
            self.leuko_head = nn.Sequential(
                nn.Linear(2 * d, 256),
                nn.ReLU(inplace=True),
                nn.Dropout(0.2),
                nn.Linear(256, 1),
            )

    def forward(self, X, mask=None, force_half_gate=False):
        B, N, C, H, W = X.shape
        x = X.view(B * N, C, H, W)
        h = self.encoder(x)
        d = h.shape[-1]
        Hbag = h.view(B, N, d)

        z_non, _ = self.pool_non(Hbag, mask=mask)
        z_leu, _ = self.pool_leu(Hbag, mask=mask)

        logit_non = self.cls_non(z_non)
        logit_leu = self.cls_leu(z_leu)

        gate_in = torch.cat([z_non, z_leu], dim=1)
        g_logit = self.gate(gate_in) / self.gate_tau
        g = torch.sigmoid(g_logit)
        if force_half_gate:
            g = 0.5 * torch.ones_like(g)

        logit_m = (1.0 - g) * logit_non + g * logit_leu

        logit_l = None
        if self.use_aux_leuko:
            logit_l = self.leuko_head(gate_in)

        return logit_m, logit_l, g

def set_backbone_trainable(model: nn.Module, trainable: bool):
    for p in model.encoder.backbone.parameters():
        p.requires_grad = bool(trainable)

# ------------------------------------------------------------
# REGULARIZER
# ------------------------------------------------------------
def gate_entropy(g, eps=1e-6):
    g = torch.clamp(g, eps, 1.0 - eps)
    ent = -(g * torch.log(g) + (1.0 - g) * torch.log(1.0 - g))
    return ent.mean()

# ------------------------------------------------------------
# CHECKPOINT HELPERS
# ------------------------------------------------------------
def save_ckpt(path: Path, model, opt, scaler, epoch, best_auc, best_epoch, bad):
    ck = {
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": int(epoch),
        "best_auc": float(best_auc),
        "best_epoch": int(best_epoch),
        "bad": int(bad),
        "rng": capture_rng_state(),
        "meta": {
            "use_aux_leuko": bool(USE_AUX_LEUKO),
            "gate_tau": float(GATE_TAU),
        }
    }
    torch.save(ck, str(path))

def load_ckpt(path: Path, model, opt, scaler):
    # PyTorch 2.6+ defaults weights_only=True; force full load (trusted local file)
    ck = torch.load(str(path), map_location=DEVICE, weights_only=False)

    # Allow loading even if head config changes (but keep USE_AUX_LEUKO consistent ideally)
    missing, unexpected = model.load_state_dict(ck["model"], strict=False)
    if len(unexpected) > 0:
        print("[WARN] unexpected keys in ckpt (ignored):", unexpected[:6], "..." if len(unexpected) > 6 else "")
    if len(missing) > 0:
        print("[WARN] missing keys in ckpt (will keep init):", missing[:6], "..." if len(missing) > 6 else "")

    if "opt" in ck and opt is not None:
        try:
            opt.load_state_dict(ck["opt"])
        except Exception as e:
            print("[WARN] optimizer state not loaded:", e)

    if scaler is not None and ck.get("scaler") is not None:
        try:
            scaler.load_state_dict(ck["scaler"])
        except Exception as e:
            print("[WARN] scaler state not loaded:", e)

    restore_rng_state(ck.get("rng", None))
    return ck

# ------------------------------------------------------------
# CV FOLDS
# ------------------------------------------------------------
def make_patient_folds(patients, pid_to_labels, n_splits=5, seed=42):
    strat = np.array([2 * pid_to_labels[p][0] + pid_to_labels[p][1] for p in patients], dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idxs = np.arange(len(patients))
    folds = []
    for tr_idx, va_idx in skf.split(idxs, strat):
        tr_p = [patients[i] for i in tr_idx]
        va_p = [patients[i] for i in va_idx]
        folds.append((tr_p, va_p))
    return folds

# ------------------------------------------------------------
# TRAIN + PREDICT
# ------------------------------------------------------------
def train_epoch(model, loader, opt, crit_m, crit_l, scaler, epoch):
    model.train()
    losses = []

    if FREEZE_BACKBONE_EPOCHS > 0:
        set_backbone_trainable(model, trainable=(epoch > FREEZE_BACKBONE_EPOCHS))

    opt.zero_grad(set_to_none=True)

    for step_i, (X, mask, ym, yl, _pids, _fps) in enumerate(loader, start=1):
        X = X.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        ym = ym.to(DEVICE, non_blocking=True).float().unsqueeze(1)
        yl = yl.to(DEVICE, non_blocking=True).float().unsqueeze(1)

        force_half = (epoch <= WARMUP_FORCE_HALF_GATE_EPOCHS)

        if scaler is not None:
            with torch.amp.autocast(device_type="cuda"):
                logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
                loss = crit_m(logit_m, ym)
                if USE_AUX_LEUKO and (logit_l is not None):
                    loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)
                if (not force_half) and (LAMBDA_GATE_ENT > 0):
                    loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

            loss = loss / float(ACCUM_STEPS)
            scaler.scale(loss).backward()

            if (step_i % ACCUM_STEPS) == 0:
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)

        else:
            logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
            loss = crit_m(logit_m, ym)
            if USE_AUX_LEUKO and (logit_l is not None):
                loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)
            if (not force_half) and (LAMBDA_GATE_ENT > 0):
                loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

            loss = loss / float(ACCUM_STEPS)
            loss.backward()

            if (step_i % ACCUM_STEPS) == 0:
                opt.step()
                opt.zero_grad(set_to_none=True)

        losses.append(float(loss.detach().cpu().item()) * float(ACCUM_STEPS))

    # flush if steps not divisible
    if (len(loader) % ACCUM_STEPS) != 0:
        if scaler is not None:
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
        else:
            opt.step()
            opt.zero_grad(set_to_none=True)

    return float(np.mean(losses)) if losses else float("nan")

@torch.no_grad()
def predict_logits_multibag(model, ds_eval, K=10, use_tta=True):
    model.eval()

    pid_list = ds_eval.patients
    pid_to_idx = {p: i for i, p in enumerate(pid_list)}
    n = len(pid_list)

    sum_logits = np.zeros((n,), dtype=np.float64)
    sum_gate = np.zeros((n,), dtype=np.float64)

    dl = DataLoader(
        ds_eval,
        batch_size=EVAL_BS,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.startswith("cuda")),
        collate_fn=collate_patient_bags
    )

    tta_tfs = [eval_tf]
    if use_tta:
        tta_tfs.append(eval_tf_hflip)

    denom = float(K * len(tta_tfs))

    for _k in range(K):
        ds_eval.train = False
        ds_eval.eval_random = True

        for tf in tta_tfs:
            ds_eval.set_transform(tf)

            for X, mask, _ym, _yl, pids, _fps in dl:
                X = X.to(DEVICE, non_blocking=True)
                mask = mask.to(DEVICE, non_blocking=True)

                logit_m, _logit_l, g = model(X, mask=mask, force_half_gate=False)

                logit_m = logit_m.detach().cpu().numpy().reshape(-1)
                g = g.detach().cpu().numpy().reshape(-1)

                for pid, lg, gg in zip(pids, logit_m, g):
                    j = pid_to_idx[pid]
                    sum_logits[j] += float(lg)
                    sum_gate[j] += float(gg)

    avg_logits = sum_logits / denom
    avg_gate = sum_gate / denom

    y = np.asarray([ds_eval.pid_to_labels[p][0] for p in pid_list], dtype=int)
    le = np.asarray([ds_eval.pid_to_labels[p][1] for p in pid_list], dtype=int)

    return avg_logits, avg_gate, pid_list, y, le

# ------------------------------------------------------------
# CSV WRITERS + AGGREGATION
# ------------------------------------------------------------
def save_fold_test_csv(seed, fold, pids, y, le, logits, gates):
    out = RUNS_DIR / f"pred_fold{fold}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit,prob,gate\n")
        for pid, yt, lk, lg, gg in zip(pids, y, le, logits, gates):
            pr = float(sigmoid_np(lg))
            f.write(f"{pid},{int(yt)},{int(lk)},{float(lg):.8f},{pr:.8f},{float(gg):.6f}\n")
    print("Saved fold test preds:", out)

def aggregate_seed_folds(seed, folds_list):
    import pandas as pd

    paths = [RUNS_DIR / f"pred_fold{f}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv" for f in folds_list]
    for p in paths:
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")

    # Use Patient as index; do NOT join y_true/leuko multiple times
    base = None
    for i, p in enumerate(paths):
        df = pd.read_csv(p)
        df = df.set_index("Patient")
        if base is None:
            base = df[["y_true", "leuko"]].copy()
        else:
            # align and verify
            base = base.join(df[["y_true", "leuko"]], how="inner", rsuffix=f"_dup{i}")
            # If duplicates exist, drop them after a sanity check
            if f"y_true_dup{i}" in base.columns:
                mismatch = (base["y_true"] != base[f"y_true_dup{i}"]).sum()
                if mismatch:
                    print(f"[WARN] y_true mismatch count after join: {mismatch}")
                base = base.drop(columns=[f"y_true_dup{i}", f"leuko_dup{i}"], errors="ignore")

    # Now add logits per fold as separate columns
    merged = base.copy()
    for f in folds_list:
        df = pd.read_csv(RUNS_DIR / f"pred_fold{f}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv").set_index("Patient")
        merged = merged.join(df[["logit"]].rename(columns={"logit": f"logit_fold{f}"}), how="inner")

    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    logit_mean = merged[logit_cols].mean(axis=1).astype(float).values
    prob_mean = sigmoid_np(logit_mean)

    y_true = merged["y_true"].astype(int).values
    auc = try_auc(y_true, prob_mean)

    a, b = min(folds_list), max(folds_list)
    out = RUNS_DIR / f"pred_seed{seed}_{BENCHMARK}_folds{a}_{b}_Ktest{K_TEST}_agg.csv"

    out_df = merged[["y_true", "leuko"]].copy()
    out_df["logit_mean"] = logit_mean
    out_df["prob_mean"] = prob_mean
    out_df.to_csv(out)

    print("Saved:", out)
    print(f"AGG TEST AUC (avg logits over folds {a}..{b}): {auc:.10f}")
    return out, auc

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main():
    # load CSVs
    rows_tr = ensure_exists(load_csv_rows(SPLIT_TRAIN))
    rows_va = ensure_exists(load_csv_rows(SPLIT_VAL))
    rows_te = ensure_exists(load_csv_rows(SPLIT_TEST))

    rows_pool = rows_tr + rows_va
    patients_pool, pid_to_frames_pool, pid_to_labels_pool = build_patient_index(rows_pool)
    patients_pool = [p for p in patients_pool if len(pid_to_frames_pool[p]) >= 2]

    patients_test, pid_to_frames_test, pid_to_labels_test = build_patient_index(rows_te)
    patients_test = [p for p in patients_test if len(pid_to_frames_test[p]) >= 2]

    print(f"[POOL] patients={len(patients_pool)} (train+val)")
    print(f"[TEST] patients={len(patients_test)}")

    # Folds fixed across seeds using first seed in list (stable)
    folds = make_patient_folds(patients_pool, pid_to_labels_pool, n_splits=5, seed=SEEDS[0])

    for seed in SEEDS:
        seed_all(seed)
        print("\n" + "="*70)
        print(f"RUN seed={seed} | benchmark={BENCHMARK} | folds={FOLD_START}..{FOLD_END}")
        print("="*70)

        for fold_i in range(FOLD_START, FOLD_END + 1):
            tr_pat, va_pat = folds[fold_i - 1]
            print(f"\n--- Fold {fold_i}/5 | train_pat={len(tr_pat)} val_pat={len(va_pat)} ---")

            ds_tr = PatientBagDataset(tr_pat, pid_to_frames_pool, pid_to_labels_pool, train=True)
            ds_va = PatientBagDataset(va_pat, pid_to_frames_pool, pid_to_labels_pool, train=False)
            ds_te = PatientBagDataset(patients_test, pid_to_frames_test, pid_to_labels_test, train=False)

            # pos_weight from train fold
            y_m_tr = np.array([pid_to_labels_pool[p][0] for p in tr_pat], dtype=int)
            y_l_tr = np.array([pid_to_labels_pool[p][1] for p in tr_pat], dtype=int)
            pos_m = int(y_m_tr.sum()); neg_m = int(len(y_m_tr) - pos_m)
            pos_l = int(y_l_tr.sum()); neg_l = int(len(y_l_tr) - pos_l)

            pos_weight_m = torch.tensor([neg_m / max(pos_m, 1)], dtype=torch.float32, device=DEVICE)
            pos_weight_l = torch.tensor([neg_l / max(pos_l, 1)], dtype=torch.float32, device=DEVICE)
            crit_m = nn.BCEWithLogitsLoss(pos_weight=pos_weight_m)
            crit_l = nn.BCEWithLogitsLoss(pos_weight=pos_weight_l)

            dl_tr = DataLoader(
                ds_tr,
                batch_size=TRAIN_BS,
                shuffle=True,
                num_workers=NUM_WORKERS,
                pin_memory=(DEVICE.startswith("cuda")),
                collate_fn=collate_patient_bags
            )

            model = GatedMILv2(use_aux_leuko=USE_AUX_LEUKO, gate_tau=GATE_TAU).to(DEVICE)
            print("Model on:", next(model.parameters()).device)

            opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scaler = torch.amp.GradScaler("cuda") if (AMP and DEVICE.startswith("cuda")) else None

            best_path = RUNS_DIR / f"cv_best_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"
            prog_path = RUNS_DIR / f"progress_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"
            summary_path = RUNS_DIR / f"summary_{BENCHMARK}_seed{seed}_fold{fold_i}.txt"

            best_auc = -1e9
            best_epoch = -1
            bad = 0
            start_ep = 1

            # Resume priority: progress > best
            resume_from = None
            if prog_path.exists():
                resume_from = prog_path
            elif best_path.exists():
                resume_from = best_path

            if resume_from is not None:
                ck = load_ckpt(resume_from, model, opt, scaler)
                best_auc = float(ck.get("best_auc", -1e9))
                best_epoch = int(ck.get("best_epoch", ck.get("epoch", -1)))
                bad = int(ck.get("bad", 0))
                start_ep = int(ck.get("epoch", 0)) + 1
                print(f"[RESUME] seed={seed} fold={fold_i} from epoch {start_ep} | best_auc={best_auc:.4f} at ep {best_epoch} | bad={bad}")

            if start_ep > EPOCHS_MAX:
                print(f"[SKIP] fold={fold_i}: already finished (epoch {start_ep-1} >= {EPOCHS_MAX}).")
            else:
                for ep in range(start_ep, EPOCHS_MAX + 1):
                    t0 = time.time()
                    tr_loss = train_epoch(model, dl_tr, opt, crit_m, crit_l, scaler, epoch=ep)
                    t_train = (time.time() - t0) / 60.0

                    do_val = (ep % EVAL_EVERY == 0) or (ep == 1)

                    if do_val:
                        t1 = time.time()
                        lg_va, gg_va, _pids_va, y_va, _le_va = predict_logits_multibag(
                            model, ds_va, K=K_VAL, use_tta=TTA_VAL
                        )
                        auc_va = try_auc(y_va, sigmoid_np(lg_va))
                        gate_mean = float(np.mean(gg_va)) if len(gg_va) else float("nan")
                        t_val = (time.time() - t1) / 60.0

                        print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                              f"train_loss={tr_loss:.4f} | val_auc(K)={auc_va:.4f} | gate_mean={gate_mean:.3f} | "
                              f"time: train={t_train:.2f}m val={t_val:.2f}m")

                        improved = (auc_va > best_auc + MIN_DELTA) or (best_epoch < 0)
                        if improved:
                            best_auc = float(auc_va)
                            best_epoch = int(ep)
                            bad = 0
                            save_ckpt(best_path, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)
                            with open(summary_path, "w", encoding="utf-8") as f:
                                f.write(f"seed={seed}\nfold={fold_i}\nbest_epoch={best_epoch}\nbest_auc={best_auc:.6f}\n")
                            print(f"[SAVE] best updated -> {summary_path}")
                        else:
                            bad += 1

                        # always save progress for resume safety
                        save_ckpt(prog_path, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)

                        if bad >= EARLY_STOP_PATIENCE:
                            print(f"Early stop at ep {ep} (best_auc={best_auc:.4f} at ep {best_epoch}).")
                            break

                    else:
                        print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                              f"train_loss={tr_loss:.4f} | (no val this epoch) | time: train={t_train:.2f}m")
                        save_ckpt(prog_path, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)

                # cleanup progress if best exists and training done/early-stopped
                # (optional) keep progress files if you want

            # Load best checkpoint for test prediction
            if best_path.exists():
                ck_best = torch.load(str(best_path), map_location=DEVICE, weights_only=False)
                _missing, _unexpected = model.load_state_dict(ck_best["model"], strict=False)
            else:
                print(f"[WARN] no best checkpoint found for fold {fold_i}; using current model.")

            # Predict TEST
            lg_te, gg_te, pids_te, y_te, le_te = predict_logits_multibag(
                model, ds_te, K=K_TEST, use_tta=TTA_TEST
            )
            save_fold_test_csv(seed, fold_i, pids_te, y_te, le_te, lg_te, gg_te)

        # After all folds in range, aggregate if requested and all files exist
        try:
            aggregate_seed_folds(seed, AGG_FOLDS)
        except Exception as e:
            print("[WARN] aggregation skipped/failed:", e)

if __name__ == "__main__":
    main()

DEVICE: cuda:0
BENCHMARK=res_shift
 train: D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv
 val  : D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv
 test : D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv
RUNNING seeds=[43] folds=1..5 (of 5)
RUN seed(s)=[43] | K_VAL=5 | K_TEST=10 | TTA_VAL=False | TTA_TEST=True
Gate: tau=2.0 | lambda_ent=0.02 | freeze_backbone_epochs=2
TRAIN_BS=2 (accum 4) | EVAL_BS=2 | EVAL_EVERY=2
[INFO] filtered missing files: 13
[POOL] patients=137 (train+val)
[TEST] patients=73

RUN seed=43 | benchmark=res_shift | folds=1..5

--- Fold 1/5 | train_pat=109 val_pat=28 ---
Model on: cuda:0
[WARN] could not restore torch RNG: RNG state must be a torch.ByteTensor
[WARN] could not restore cuda RNG: RNG state must be a torch.ByteTensor
[RESUME] seed=43 fold=1 from epoch 31 | best_auc=1.0000 at ep 20 | bad=5
[SKIP] fold=1: already finished (epoch 30 >= 30).
Saved fold test preds: D:\ENT\_challenge2_artifacts\_runs\pred_fo

In [2]:
# ============================================================
# RESUMABLE 5-FOLD CV + MULTIBAG TEST PREDICTIONS (GPU)
# Windows/Colab friendly, no RNG restore (avoids ByteTensor warnings)
# ============================================================

import os, math, random, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import torchvision
from torchvision import transforms as T

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

# ------------------------------------------------------------
# USER CONFIG
# ------------------------------------------------------------

ART_DIR = Path(r"D:\ENT\_challenge2_artifacts")  # <-- change if needed
BENCHMARK = "res_shift"

SPLITS = {
    "res_shift": {
        "train": ART_DIR / "split_res_shift_train_major_colab.csv",
        "val":   ART_DIR / "split_res_shift_val_major_colab.csv",
        "test":  ART_DIR / "split_res_shift_test_minor_colab.csv",
    }
}

SEEDS = [43]
FOLD_START = 1
FOLD_END   = 5
AGG_FOLDS  = [1,2,3,4,5]

# ------------------------------------------------------------
# TRAIN/EVAL CONFIG
# ------------------------------------------------------------

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
AMP = True
torch.backends.cudnn.benchmark = True

IMG_SIZE = 224
NUM_WORKERS = 0

TRAIN_BS = 2
EVAL_BS  = 2
ACCUM_STEPS = 4

BAG_SIZE_NONLEUKO = 24
BAG_SIZE_LEUKO    = 40

EPOCHS_MAX = 30
LR = 5e-5
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 6
MIN_DELTA = 1e-4

USE_AUX_LEUKO = True
LAMBDA_LEUKO = 0.30

WARMUP_FORCE_HALF_GATE_EPOCHS = 2

GATE_TAU = 2.0
LAMBDA_GATE_ENT = 0.02

FREEZE_BACKBONE_EPOCHS = 2

K_VAL  = 5
K_TEST = 10
TTA_VAL  = False
TTA_TEST = True
EVAL_EVERY = 2

# If you want training to continue after resume without instantly early-stopping:
RESET_BAD_ON_RESUME = False  # set True if you prefer

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
RUNS_DIR = ART_DIR / "_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_TRAIN = SPLITS[BENCHMARK]["train"]
SPLIT_VAL   = SPLITS[BENCHMARK]["val"]
SPLIT_TEST  = SPLITS[BENCHMARK]["test"]

print(f"DEVICE: {DEVICE}")
print("="*70)
print(f"BENCHMARK={BENCHMARK}")
print(" train:", SPLIT_TRAIN)
print(" val  :", SPLIT_VAL)
print(" test :", SPLIT_TEST)
print(f"RUNNING seeds={SEEDS} folds={FOLD_START}..{FOLD_END} (of 5)")
print("="*70)
print(f"RUN seed(s)={SEEDS} | K_VAL={K_VAL} | K_TEST={K_TEST} | TTA_VAL={TTA_VAL} | TTA_TEST={TTA_TEST}")
print(f"Gate: tau={GATE_TAU} | lambda_ent={LAMBDA_GATE_ENT} | freeze_backbone_epochs={FREEZE_BACKBONE_EPOCHS}")
print(f"TRAIN_BS={TRAIN_BS} (accum {ACCUM_STEPS}) | EVAL_BS={EVAL_BS} | EVAL_EVERY={EVAL_EVERY}")
print("="*70)

# ------------------------------------------------------------
# UTILS
# ------------------------------------------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))

def try_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def safe_int(x, default=0):
    try:
        return int(float(x))
    except Exception:
        return default

def parse_leuko(x):
    if x is None:
        return 0
    s = str(x).strip().lower()
    return 1 if s in ["yes", "1", "true", "y"] else 0

def parse_malign(x):
    return 1 if safe_int(x, 0) == 1 else 0

# ------------------------------------------------------------
# CSV LOADING
# ------------------------------------------------------------
def load_csv_rows(csv_path):
    csv_path = str(csv_path)
    import pandas as pd
    df = pd.read_csv(csv_path)
    return df.to_dict(orient="records")

def ensure_exists(rows):
    kept, missing = [], 0
    for r in rows:
        fp = r.get("filepath", "")
        if fp and os.path.exists(fp):
            kept.append(r)
        else:
            missing += 1
    if missing > 0:
        print(f"[INFO] filtered missing files: {missing}")
    return kept

# ------------------------------------------------------------
# PATIENT INDEX
# ------------------------------------------------------------
def build_patient_index(rows):
    pid_to_frames = defaultdict(list)

    for r in rows:
        pid = r.get("Patient", None) or r.get("patient", None) or r.get("PID", None)
        if pid is None:
            pid = "UNK"

        y_m = parse_malign(r.get("label_binary", 0))
        y_l = parse_leuko(r.get("Leukoplakia", 0))
        fp  = r.get("filepath", "")

        pid_to_frames[pid].append({"filepath": fp, "y_m": y_m, "y_l": y_l})

    pid_to_labels = {}
    for pid, frs in pid_to_frames.items():
        y_m = 1 if any(f["y_m"] == 1 for f in frs) else 0
        y_l = 1 if any(f["y_l"] == 1 for f in frs) else 0
        pid_to_labels[pid] = (y_m, y_l)

    patients = sorted(pid_to_frames.keys())
    return patients, pid_to_frames, pid_to_labels

# ------------------------------------------------------------
# TRANSFORMS
# ------------------------------------------------------------
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
    T.ToTensor(),
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

eval_tf_hflip = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=1.0),
    T.ToTensor(),
])

# ------------------------------------------------------------
# DATASET
# ------------------------------------------------------------
class PatientBagDataset(Dataset):
    def __init__(self, patients, pid_to_frames, pid_to_labels, train=True):
        self.patients = list(patients)
        self.pid_to_frames = pid_to_frames
        self.pid_to_labels = pid_to_labels
        self.train = train
        self.tf = train_tf if train else eval_tf
        self.eval_random = True

    def set_transform(self, tf):
        self.tf = tf

    def __len__(self):
        return len(self.patients)

    def _sample_frames(self, pid):
        frs = self.pid_to_frames[pid]
        y_m, y_l = self.pid_to_labels[pid]
        bag_size = BAG_SIZE_LEUKO if y_l == 1 else BAG_SIZE_NONLEUKO
        n = len(frs)

        if self.train:
            idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size))
        else:
            if self.eval_random:
                idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size))
            else:
                if n >= bag_size:
                    idxs = np.arange(bag_size)
                else:
                    reps = int(math.ceil(bag_size / n))
                    idxs = (np.tile(np.arange(n), reps))[:bag_size]
        return [frs[i] for i in idxs]

    def __getitem__(self, i):
        pid = self.patients[i]
        y_m, y_l = self.pid_to_labels[pid]
        sampled = self._sample_frames(pid)

        imgs = []
        for f in sampled:
            fp = f["filepath"]
            im = Image.open(fp).convert("RGB")
            im = self.tf(im)
            imgs.append(im)

        X = torch.stack(imgs, dim=0)
        return X, y_m, y_l, pid

def collate_patient_bags(batch):
    Xs, ym, yl, pids = zip(*batch)
    lengths = [x.shape[0] for x in Xs]
    max_len = max(lengths)

    B = len(Xs)
    C, H, W = Xs[0].shape[1], Xs[0].shape[2], Xs[0].shape[3]

    Xpad = torch.zeros((B, max_len, C, H, W), dtype=Xs[0].dtype)
    mask = torch.zeros((B, max_len), dtype=torch.bool)

    for i, x in enumerate(Xs):
        n = x.shape[0]
        Xpad[i, :n] = x
        mask[i, :n] = True

    ym = torch.tensor(ym, dtype=torch.float32).unsqueeze(1)
    yl = torch.tensor(yl, dtype=torch.float32).unsqueeze(1)
    return Xpad, mask, ym, yl, list(pids)

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------
class ResNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(m.children())[:-1])
        self.out_dim = 2048

    def forward(self, x):
        h = self.backbone(x)
        return h.flatten(1)

class AttnPool(nn.Module):
    def __init__(self, d, h=256):
        super().__init__()
        self.V = nn.Linear(d, h)
        self.w = nn.Linear(h, 1)

    def forward(self, H, mask=None):
        A = self.w(torch.tanh(self.V(H)))
        if mask is not None:
            A = A.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        A = torch.softmax(A, dim=1)
        Z = (A * H).sum(dim=1)
        return Z, A.squeeze(-1)

class GatedMILv2(nn.Module):
    def __init__(self, use_aux_leuko=True, gate_tau=2.0):
        super().__init__()
        self.use_aux_leuko = bool(use_aux_leuko)
        self.gate_tau = float(gate_tau)

        self.encoder = ResNetEncoder()
        d = self.encoder.out_dim

        self.pool_non = AttnPool(d, h=256)
        self.pool_leu = AttnPool(d, h=256)

        self.cls_non = nn.Linear(d, 1)
        self.cls_leu = nn.Linear(d, 1)

        self.gate = nn.Linear(2 * d, 1)

        if self.use_aux_leuko:
            self.leuko_head = nn.Sequential(
                nn.Linear(2 * d, 256),
                nn.ReLU(inplace=True),
                nn.Dropout(0.2),
                nn.Linear(256, 1),
            )

    def forward(self, X, mask=None, force_half_gate=False):
        B, N, C, H, W = X.shape
        x = X.view(B * N, C, H, W)
        h = self.encoder(x)
        d = h.shape[-1]
        Hbag = h.view(B, N, d)

        z_non, _ = self.pool_non(Hbag, mask=mask)
        z_leu, _ = self.pool_leu(Hbag, mask=mask)

        logit_non = self.cls_non(z_non)
        logit_leu = self.cls_leu(z_leu)

        gate_in = torch.cat([z_non, z_leu], dim=1)
        g_logit = self.gate(gate_in) / self.gate_tau
        g = torch.sigmoid(g_logit)
        if force_half_gate:
            g = 0.5 * torch.ones_like(g)

        logit_m = (1.0 - g) * logit_non + g * logit_leu

        logit_l = None
        if self.use_aux_leuko:
            logit_l = self.leuko_head(gate_in)

        return logit_m, logit_l, g

def set_backbone_trainable(model: nn.Module, trainable: bool):
    for p in model.encoder.backbone.parameters():
        p.requires_grad = bool(trainable)

def gate_entropy(g, eps=1e-6):
    g = torch.clamp(g, eps, 1.0 - eps)
    ent = -(g * torch.log(g) + (1.0 - g) * torch.log(1.0 - g))
    return ent.mean()

# ------------------------------------------------------------
# CHECKPOINTS (NO RNG SAVE/RESTORE)
# ------------------------------------------------------------
def save_ckpt(path: Path, model, opt, scaler, epoch, best_auc, best_epoch, bad):
    ck = {
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": int(epoch),
        "best_auc": float(best_auc),
        "best_epoch": int(best_epoch),
        "bad": int(bad),
        "meta": {"use_aux_leuko": bool(USE_AUX_LEUKO), "gate_tau": float(GATE_TAU)},
    }
    torch.save(ck, str(path))

def load_ckpt(path: Path, model, opt, scaler):
    ck = torch.load(str(path), map_location=DEVICE, weights_only=False)
    _missing, _unexpected = model.load_state_dict(ck["model"], strict=False)

    if "opt" in ck and opt is not None:
        try:
            opt.load_state_dict(ck["opt"])
        except Exception as e:
            print("[WARN] optimizer state not loaded:", e)

    if scaler is not None and ck.get("scaler") is not None:
        try:
            scaler.load_state_dict(ck["scaler"])
        except Exception as e:
            print("[WARN] scaler state not loaded:", e)

    return ck

# ------------------------------------------------------------
# CV FOLDS
# ------------------------------------------------------------
def make_patient_folds(patients, pid_to_labels, n_splits=5, seed=42):
    strat = np.array([2 * pid_to_labels[p][0] + pid_to_labels[p][1] for p in patients], dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idxs = np.arange(len(patients))
    folds = []
    for tr_idx, va_idx in skf.split(idxs, strat):
        tr_p = [patients[i] for i in tr_idx]
        va_p = [patients[i] for i in va_idx]
        folds.append((tr_p, va_p))
    return folds

# ------------------------------------------------------------
# TRAIN / PREDICT
# ------------------------------------------------------------
def train_epoch(model, loader, opt, crit_m, crit_l, scaler, epoch):
    model.train()
    losses = []

    set_backbone_trainable(model, trainable=(epoch > FREEZE_BACKBONE_EPOCHS))

    opt.zero_grad(set_to_none=True)

    for step_i, (X, mask, ym, yl, _pids) in enumerate(loader, start=1):
        X = X.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        ym = ym.to(DEVICE, non_blocking=True)
        yl = yl.to(DEVICE, non_blocking=True)

        force_half = (epoch <= WARMUP_FORCE_HALF_GATE_EPOCHS)

        if scaler is not None:
            with torch.amp.autocast(device_type="cuda"):
                logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
                loss = crit_m(logit_m, ym)
                if USE_AUX_LEUKO and (logit_l is not None):
                    loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)
                if (not force_half) and (LAMBDA_GATE_ENT > 0):
                    loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

            loss = loss / float(ACCUM_STEPS)
            scaler.scale(loss).backward()

            if (step_i % ACCUM_STEPS) == 0:
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
        else:
            logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
            loss = crit_m(logit_m, ym)
            if USE_AUX_LEUKO and (logit_l is not None):
                loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)
            if (not force_half) and (LAMBDA_GATE_ENT > 0):
                loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

            loss = loss / float(ACCUM_STEPS)
            loss.backward()

            if (step_i % ACCUM_STEPS) == 0:
                opt.step()
                opt.zero_grad(set_to_none=True)

        losses.append(float(loss.detach().cpu().item()) * float(ACCUM_STEPS))

    if (len(loader) % ACCUM_STEPS) != 0:
        if scaler is not None:
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
        else:
            opt.step()
            opt.zero_grad(set_to_none=True)

    return float(np.mean(losses)) if losses else float("nan")

@torch.no_grad()
def predict_logits_multibag(model, ds_eval, K=10, use_tta=True):
    model.eval()

    pid_list = ds_eval.patients
    pid_to_idx = {p: i for i, p in enumerate(pid_list)}
    n = len(pid_list)

    sum_logits = np.zeros((n,), dtype=np.float64)
    sum_gate   = np.zeros((n,), dtype=np.float64)

    dl = DataLoader(
        ds_eval,
        batch_size=EVAL_BS,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.startswith("cuda")),
        collate_fn=collate_patient_bags
    )

    tta_tfs = [eval_tf]
    if use_tta:
        tta_tfs.append(eval_tf_hflip)

    denom = float(K * len(tta_tfs))

    for _k in range(K):
        ds_eval.train = False
        ds_eval.eval_random = True

        for tf in tta_tfs:
            ds_eval.set_transform(tf)

            for X, mask, _ym, _yl, pids in dl:
                X = X.to(DEVICE, non_blocking=True)
                mask = mask.to(DEVICE, non_blocking=True)

                logit_m, _logit_l, g = model(X, mask=mask, force_half_gate=False)

                logit_m = logit_m.detach().cpu().numpy().reshape(-1)
                g = g.detach().cpu().numpy().reshape(-1)

                for pid, lg, gg in zip(pids, logit_m, g):
                    j = pid_to_idx[pid]
                    sum_logits[j] += float(lg)
                    sum_gate[j]   += float(gg)

    avg_logits = sum_logits / denom
    avg_gate   = sum_gate / denom

    y  = np.asarray([ds_eval.pid_to_labels[p][0] for p in pid_list], dtype=int)
    le = np.asarray([ds_eval.pid_to_labels[p][1] for p in pid_list], dtype=int)

    return avg_logits, avg_gate, pid_list, y, le

# ------------------------------------------------------------
# OUTPUTS
# ------------------------------------------------------------
def save_fold_test_csv(seed, fold, pids, y, le, logits, gates):
    out = RUNS_DIR / f"pred_fold{fold}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit,prob,gate\n")
        for pid, yt, lk, lg, gg in zip(pids, y, le, logits, gates):
            pr = float(sigmoid_np(lg))
            f.write(f"{pid},{int(yt)},{int(lk)},{float(lg):.8f},{pr:.8f},{float(gg):.6f}\n")
    print("Saved fold test preds:", out)

def aggregate_seed_folds(seed, folds_list):
    import pandas as pd

    paths = [RUNS_DIR / f"pred_fold{f}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv" for f in folds_list]
    for p in paths:
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")

    base = None
    for i, p in enumerate(paths):
        df = pd.read_csv(p).set_index("Patient")
        if base is None:
            base = df[["y_true", "leuko"]].copy()
        else:
            tmp = df[["y_true", "leuko"]].rename(columns={"y_true": f"y_true_dup{i}", "leuko": f"leuko_dup{i}"})
            base = base.join(tmp, how="inner")
            mismatch = (base["y_true"] != base[f"y_true_dup{i}"]).sum()
            if mismatch:
                print(f"[WARN] y_true mismatch after join: {mismatch}")
            base = base.drop(columns=[f"y_true_dup{i}", f"leuko_dup{i}"], errors="ignore")

    merged = base.copy()
    for f in folds_list:
        df = pd.read_csv(RUNS_DIR / f"pred_fold{f}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv").set_index("Patient")
        merged = merged.join(df[["logit"]].rename(columns={"logit": f"logit_fold{f}"}), how="inner")

    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    logit_mean = merged[logit_cols].mean(axis=1).astype(float).values
    prob_mean  = sigmoid_np(logit_mean)

    y_true = merged["y_true"].astype(int).values
    auc = try_auc(y_true, prob_mean)

    a, b = min(folds_list), max(folds_list)
    out = RUNS_DIR / f"pred_seed{seed}_{BENCHMARK}_folds{a}_{b}_Ktest{K_TEST}_agg.csv"

    out_df = merged[["y_true", "leuko"]].copy()
    out_df["logit_mean"] = logit_mean
    out_df["prob_mean"]  = prob_mean
    out_df.to_csv(out)

    print("Saved:", out)
    print(f"AGG TEST AUC (avg logits over folds {a}..{b}): {auc:.10f}")
    return out, auc

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main():
    rows_tr = ensure_exists(load_csv_rows(SPLIT_TRAIN))
    rows_va = ensure_exists(load_csv_rows(SPLIT_VAL))
    rows_te = ensure_exists(load_csv_rows(SPLIT_TEST))

    rows_pool = rows_tr + rows_va

    patients_pool, pid_to_frames_pool, pid_to_labels_pool = build_patient_index(rows_pool)
    patients_pool = [p for p in patients_pool if len(pid_to_frames_pool[p]) >= 2]

    patients_test, pid_to_frames_test, pid_to_labels_test = build_patient_index(rows_te)
    patients_test = [p for p in patients_test if len(pid_to_frames_test[p]) >= 2]

    print(f"[POOL] patients={len(patients_pool)} (train+val)")
    print(f"[TEST] patients={len(patients_test)}")

    folds = make_patient_folds(patients_pool, pid_to_labels_pool, n_splits=5, seed=SEEDS[0])

    for seed in SEEDS:
        seed_all(seed)

        print("\n" + "="*70)
        print(f"RUN seed={seed} | benchmark={BENCHMARK} | folds={FOLD_START}..{FOLD_END}")
        print("="*70)

        for fold_i in range(FOLD_START, FOLD_END + 1):
            tr_pat, va_pat = folds[fold_i - 1]
            print(f"\n--- Fold {fold_i}/5 | train_pat={len(tr_pat)} val_pat={len(va_pat)} ---")

            ds_tr = PatientBagDataset(tr_pat, pid_to_frames_pool, pid_to_labels_pool, train=True)
            ds_va = PatientBagDataset(va_pat, pid_to_frames_pool, pid_to_labels_pool, train=False)
            ds_te = PatientBagDataset(patients_test, pid_to_frames_test, pid_to_labels_test, train=False)

            y_m_tr = np.array([pid_to_labels_pool[p][0] for p in tr_pat], dtype=int)
            y_l_tr = np.array([pid_to_labels_pool[p][1] for p in tr_pat], dtype=int)
            pos_m = int(y_m_tr.sum()); neg_m = int(len(y_m_tr) - pos_m)
            pos_l = int(y_l_tr.sum()); neg_l = int(len(y_l_tr) - pos_l)

            pos_weight_m = torch.tensor([neg_m / max(pos_m, 1)], dtype=torch.float32, device=DEVICE)
            pos_weight_l = torch.tensor([neg_l / max(pos_l, 1)], dtype=torch.float32, device=DEVICE)
            crit_m = nn.BCEWithLogitsLoss(pos_weight=pos_weight_m)
            crit_l = nn.BCEWithLogitsLoss(pos_weight=pos_weight_l)

            dl_tr = DataLoader(
                ds_tr,
                batch_size=TRAIN_BS,
                shuffle=True,
                num_workers=NUM_WORKERS,
                pin_memory=(DEVICE.startswith("cuda")),
                collate_fn=collate_patient_bags
            )

            model = GatedMILv2(use_aux_leuko=USE_AUX_LEUKO, gate_tau=GATE_TAU).to(DEVICE)
            print("Model on:", next(model.parameters()).device)

            opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scaler = torch.amp.GradScaler("cuda") if (AMP and DEVICE.startswith("cuda")) else None

            best_path   = RUNS_DIR / f"cv_best_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"
            prog_path   = RUNS_DIR / f"progress_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"
            summary_txt = RUNS_DIR / f"summary_{BENCHMARK}_seed{seed}_fold{fold_i}.txt"

            best_auc   = -1e9
            best_epoch = -1
            bad        = 0
            start_ep   = 1

            resume_from = prog_path if prog_path.exists() else (best_path if best_path.exists() else None)

            if resume_from is not None:
                ck = load_ckpt(resume_from, model, opt, scaler)
                best_auc   = float(ck.get("best_auc", -1e9))
                best_epoch = int(ck.get("best_epoch", ck.get("epoch", -1)))
                bad        = int(ck.get("bad", 0))
                start_ep   = int(ck.get("epoch", 0)) + 1
                if RESET_BAD_ON_RESUME:
                    bad = 0
                print(f"[RESUME] seed={seed} fold={fold_i} from epoch {start_ep} | best_auc={best_auc:.4f} at ep {best_epoch} | bad={bad}")

            if start_ep > EPOCHS_MAX:
                print(f"[SKIP] fold={fold_i}: already finished (epoch {start_ep-1} >= {EPOCHS_MAX}).")
            else:
                for ep in range(start_ep, EPOCHS_MAX + 1):
                    t0 = time.time()
                    tr_loss = train_epoch(model, dl_tr, opt, crit_m, crit_l, scaler, epoch=ep)
                    t_train = (time.time() - t0) / 60.0

                    do_val = (ep % EVAL_EVERY == 0) or (ep == 1)

                    if do_val:
                        t1 = time.time()
                        lg_va, gg_va, _pids_va, y_va, _le_va = predict_logits_multibag(
                            model, ds_va, K=K_VAL, use_tta=TTA_VAL
                        )
                        auc_va = try_auc(y_va, sigmoid_np(lg_va))
                        gate_mean = float(np.mean(gg_va)) if len(gg_va) else float("nan")
                        t_val = (time.time() - t1) / 60.0

                        print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                              f"train_loss={tr_loss:.4f} | val_auc(K)={auc_va:.4f} | gate_mean={gate_mean:.3f} | "
                              f"time: train={t_train:.2f}m val={t_val:.2f}m")

                        improved = (auc_va > best_auc + MIN_DELTA) or (best_epoch < 0)
                        if improved:
                            best_auc = float(auc_va)
                            best_epoch = int(ep)
                            bad = 0
                            save_ckpt(best_path, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)
                            with open(summary_txt, "w", encoding="utf-8") as f:
                                f.write(f"seed={seed}\nfold={fold_i}\nbest_epoch={best_epoch}\nbest_auc={best_auc:.6f}\n")
                            print(f"[SAVE] best updated -> {summary_txt}")
                        else:
                            bad += 1

                        save_ckpt(prog_path, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)

                        if bad >= EARLY_STOP_PATIENCE:
                            print(f"Early stop at ep {ep} (best_auc={best_auc:.4f} at ep {best_epoch}).")
                            break
                    else:
                        print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                              f"train_loss={tr_loss:.4f} | (no val this epoch) | time: train={t_train:.2f}m")
                        save_ckpt(prog_path, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)

            # Load best for test
            if best_path.exists():
                ck_best = torch.load(str(best_path), map_location=DEVICE, weights_only=False)
                model.load_state_dict(ck_best["model"], strict=False)

            lg_te, gg_te, pids_te, y_te, le_te = predict_logits_multibag(
                model, ds_te, K=K_TEST, use_tta=TTA_TEST
            )
            save_fold_test_csv(seed, fold_i, pids_te, y_te, le_te, lg_te, gg_te)

        # aggregate
        try:
            aggregate_seed_folds(seed, AGG_FOLDS)
        except Exception as e:
            print("[WARN] aggregation skipped/failed:", e)

if __name__ == "__main__":
    main()

DEVICE: cuda:0
BENCHMARK=res_shift
 train: D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv
 val  : D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv
 test : D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv
RUNNING seeds=[43] folds=1..5 (of 5)
RUN seed(s)=[43] | K_VAL=5 | K_TEST=10 | TTA_VAL=False | TTA_TEST=True
Gate: tau=2.0 | lambda_ent=0.02 | freeze_backbone_epochs=2
TRAIN_BS=2 (accum 4) | EVAL_BS=2 | EVAL_EVERY=2
[INFO] filtered missing files: 13
[POOL] patients=137 (train+val)
[TEST] patients=73

RUN seed=43 | benchmark=res_shift | folds=1..5

--- Fold 1/5 | train_pat=109 val_pat=28 ---
Model on: cuda:0
[RESUME] seed=43 fold=1 from epoch 31 | best_auc=1.0000 at ep 20 | bad=5
[SKIP] fold=1: already finished (epoch 30 >= 30).
Saved fold test preds: D:\ENT\_challenge2_artifacts\_runs\pred_fold1_res_shift_seed43_Ktest10.csv

--- Fold 2/5 | train_pat=109 val_pat=28 ---
Model on: cuda:0
[RESUME] seed=43 fold=2 from epoch 25 | best_auc=0

In [1]:
# ============================================
# MAX-AUC PIPELINE (RUN-READY, SINGLE SCRIPT)  [WINDOWS/GDRIVE OK]
# UPDATED FIXES:
#   ✅ Resume works on PyTorch 2.6+ (weights_only=False)
#   ✅ RNG restore is safe (handles ByteTensor vs list/np issues)
#   ✅ Avoid "instant early stop" after resume (RESET_BAD_ON_RESUME=True)
#   ✅ Stable TEST preds: ALWAYS load BEST checkpoint for test inference
#   ✅ OOM-safe defaults: TRAIN_BS=2, ACCUM_STEPS=4, EVAL_BS=2
#   ✅ Faster: validate every EVAL_EVERY epochs; K_VAL smaller on local
#   ✅ Aggregation fixed: no pandas join column overlap
# ============================================

import os, json, math, random, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import torchvision
from torchvision import transforms as T


# ---------------------------
# Config (edit these)
# ---------------------------

# Root artifact directory (Windows or Colab)
ART_DIR = Path(r"D:\ENT\_challenge2_artifacts")  # <-- change if needed
RUNS_DIR = ART_DIR / "_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK = "res_shift"  # "standard" or "res_shift"

SPLITS = {
    "standard": {
        "train": ART_DIR / "split_standard_train_colab.csv",
        "val":   ART_DIR / "split_standard_val_colab.csv",
        "test":  ART_DIR / "split_standard_test_colab.csv",
    },
    "res_shift": {
        "train": ART_DIR / "split_res_shift_train_major_colab.csv",
        "val":   ART_DIR / "split_res_shift_val_major_colab.csv",
        "test":  ART_DIR / "split_res_shift_test_minor_colab.csv",
    }
}

SPLIT_TRAIN = SPLITS[BENCHMARK]["train"]
SPLIT_VAL   = SPLITS[BENCHMARK]["val"]
SPLIT_TEST  = SPLITS[BENCHMARK]["test"]

# GPU
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
AMP = True

# Data / loaders
IMG_SIZE = 224
NUM_WORKERS = 0  # if Windows issues, set 0
PIN_MEMORY = True if "cuda" in DEVICE else False

# Bag sizes
BAG_SIZE_NONLEUKO = 24
BAG_SIZE_LEUKO = 40

# Training
EPOCHS_MAX = 30
LR = 5e-5
WEIGHT_DECAY = 1e-4

EARLY_STOP_PATIENCE = 6
MIN_DELTA = 1e-4

# Aux leuko head
USE_AUX_LEUKO = True
LAMBDA_LEUKO = 0.30

# Gate
WARMUP_FORCE_HALF_GATE_EPOCHS = 2
GATE_TAU = 2.0
LAMBDA_GATE_ENT = 0.02

# Stabilize early training
FREEZE_BACKBONE_EPOCHS = 2

# Eval controls
K_VAL = 5          # local faster; you can raise back to 20
K_TEST = 10
TTA_VAL = False
TTA_TEST = True
EVAL_EVERY = 2      # validate every N epochs (speed)

# Batch controls (OOM-safe)
TRAIN_BS = 2
ACCUM_STEPS = 4     # effective batch = TRAIN_BS * ACCUM_STEPS
EVAL_BS = 2

# CV / seeds
N_FOLDS = 5
SEEDS_TO_RUN = [43]     # set e.g. [42,43,44,45,46]
RUN_FOLDS = (1, 5)      # inclusive: (start_fold, end_fold)

# Resume behavior
RESET_BAD_ON_RESUME = True     # ✅ prevents immediate early stop after resume
SKIP_IF_FINISHED = True        # if epoch>=EPOCHS_MAX, skip training
ONLY_PREDICT_IF_BEST_EXISTS = False  # set True to only generate test preds

# Torch load safety (PyTorch 2.6+)
TORCH_LOAD_WEIGHTS_ONLY_FALSE = True  # we load our own checkpoints only


print(f"DEVICE: {DEVICE}")
print("=" * 70)
print(f"BENCHMARK={BENCHMARK}")
print(f" train: {SPLIT_TRAIN}")
print(f" val  : {SPLIT_VAL}")
print(f" test : {SPLIT_TEST}")
print(f"RUNNING seeds={SEEDS_TO_RUN} folds={RUN_FOLDS[0]}..{RUN_FOLDS[1]} (of {N_FOLDS})")
print("=" * 70)
print(f"RUN seed(s)={SEEDS_TO_RUN} | K_VAL={K_VAL} | K_TEST={K_TEST} | TTA_VAL={TTA_VAL} | TTA_TEST={TTA_TEST}")
print(f"Gate: tau={GATE_TAU} | lambda_ent={LAMBDA_GATE_ENT} | freeze_backbone_epochs={FREEZE_BACKBONE_EPOCHS}")
print(f"TRAIN_BS={TRAIN_BS} (accum {ACCUM_STEPS}) | EVAL_BS={EVAL_BS} | EVAL_EVERY={EVAL_EVERY}")
print("=" * 70)


# ---------------------------
# Repro helpers
# ---------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _to_torch_byte_tensor(x):
    # handle torch ByteTensor, list, numpy arrays, etc.
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.to(dtype=torch.uint8, device="cpu")
    if isinstance(x, (list, tuple)):
        return torch.tensor(list(x), dtype=torch.uint8, device="cpu")
    if isinstance(x, np.ndarray):
        return torch.from_numpy(x.astype(np.uint8)).cpu()
    return None

def capture_rng_state():
    st = {"py": random.getstate(), "np": np.random.get_state()}
    try:
        st["torch"] = torch.get_rng_state()
    except Exception:
        st["torch"] = None
    if torch.cuda.is_available():
        try:
            st["cuda"] = [s.clone().cpu() for s in torch.cuda.get_rng_state_all()]
        except Exception:
            st["cuda"] = None
    else:
        st["cuda"] = None
    return st

def restore_rng_state(rng):
    if rng is None:
        return
    try:
        random.setstate(rng.get("py"))
    except Exception:
        pass
    try:
        np.random.set_state(rng.get("np"))
    except Exception:
        pass

    # torch cpu rng
    try:
        t = _to_torch_byte_tensor(rng.get("torch"))
        if t is not None:
            torch.set_rng_state(t)
    except Exception as e:
        print(f"[WARN] could not restore torch RNG: {e}")

    # cuda rng
    if torch.cuda.is_available():
        try:
            cuda_states = rng.get("cuda", None)
            if isinstance(cuda_states, (list, tuple)) and len(cuda_states) > 0:
                fixed = []
                for s in cuda_states:
                    ts = _to_torch_byte_tensor(s)
                    if ts is None:
                        raise TypeError("RNG state must be a torch.ByteTensor")
                    fixed.append(ts)
                torch.cuda.set_rng_state_all(fixed)
        except Exception as e:
            print(f"[WARN] could not restore cuda RNG: {e}")

def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))

def try_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def safe_int(x, default=0):
    try:
        return int(float(x))
    except Exception:
        return default

def parse_leuko(x):
    if x is None:
        return 0
    s = str(x).strip().lower()
    return 1 if s in ["yes", "1", "true", "y"] else 0

def parse_malign(x):
    return 1 if safe_int(x, 0) == 1 else 0


# ---------------------------
# Robust CSV loading
# ---------------------------
def load_csv_rows(csv_path):
    csv_path = str(csv_path)
    try:
        import pandas as pd
        df = pd.read_csv(csv_path)
        return df.to_dict(orient="records")
    except Exception as e:
        print(f"[WARN] pandas read_csv failed, fallback csv module: {e}")
        import csv
        rows = []
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for r in reader:
                rows.append(r)
        return rows

def ensure_exists(rows):
    kept, missing = [], 0
    for r in rows:
        fp = r.get("filepath", "")
        if fp and os.path.exists(fp):
            kept.append(r)
        else:
            missing += 1
    if missing > 0:
        print(f"[INFO] filtered missing files: {missing}")
    return kept


# ---------------------------
# Build patient index
# ---------------------------
def build_patient_index(rows):
    pid_to_frames = defaultdict(list)

    for r in rows:
        pid = r.get("Patient", None) or r.get("patient", None) or r.get("PID", None)
        if pid is None:
            fp = r.get("filepath", "")
            pid = "UNK"
            if "Patient" in fp:
                try:
                    pid = "Patient" + fp.split("Patient")[1].split("/")[0]
                except Exception:
                    pid = "UNK"

        y_m = parse_malign(r.get("label_binary", 0))
        y_l = parse_leuko(r.get("Leukoplakia", 0))
        fp = r.get("filepath", "")

        pid_to_frames[pid].append({"filepath": fp, "y_m": y_m, "y_l": y_l})

    pid_to_labels = {}
    for pid, frs in pid_to_frames.items():
        y_m = 1 if any(f["y_m"] == 1 for f in frs) else 0
        y_l = 1 if any(f["y_l"] == 1 for f in frs) else 0
        pid_to_labels[pid] = (y_m, y_l)

    patients = sorted(pid_to_frames.keys())
    return patients, pid_to_frames, pid_to_labels


# ---------------------------
# Transforms
# ---------------------------
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
    T.ToTensor(),
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

eval_tf_hflip = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=1.0),
    T.ToTensor(),
])


# ---------------------------
# Dataset: patient bags
# ---------------------------
class PatientBagDataset(Dataset):
    def __init__(self, patients, pid_to_frames, pid_to_labels, train=True):
        self.patients = list(patients)
        self.pid_to_frames = pid_to_frames
        self.pid_to_labels = pid_to_labels
        self.train = train
        self.tf = train_tf if train else eval_tf
        self.eval_random = True

    def set_transform(self, tf):
        self.tf = tf

    def __len__(self):
        return len(self.patients)

    def _sample_frames(self, pid):
        frs = self.pid_to_frames[pid]
        y_m, y_l = self.pid_to_labels[pid]
        bag_size = BAG_SIZE_LEUKO if y_l == 1 else BAG_SIZE_NONLEUKO
        n = len(frs)

        if self.train:
            idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size))
        else:
            idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size)) if self.eval_random else np.arange(min(bag_size, n))
            if (not self.eval_random) and (n < bag_size):
                reps = int(math.ceil(bag_size / n))
                idxs = (np.tile(np.arange(n), reps))[:bag_size]

        return [frs[i] for i in idxs]

    def __getitem__(self, i):
        pid = self.patients[i]
        y_m, y_l = self.pid_to_labels[pid]
        sampled = self._sample_frames(pid)

        imgs, fps = [], []
        for f in sampled:
            fp = f["filepath"]
            fps.append(fp)
            im = Image.open(fp).convert("RGB")
            im = self.tf(im)
            imgs.append(im)

        X = torch.stack(imgs, dim=0)
        return X, y_m, y_l, pid, fps


def collate_patient_bags(batch):
    Xs, ym, yl, pids, fps = zip(*batch)
    lengths = [x.shape[0] for x in Xs]
    max_len = max(lengths)

    B = len(Xs)
    C, H, W = Xs[0].shape[1], Xs[0].shape[2], Xs[0].shape[3]

    Xpad = torch.zeros((B, max_len, C, H, W), dtype=Xs[0].dtype)
    mask = torch.zeros((B, max_len), dtype=torch.bool)

    for i, x in enumerate(Xs):
        n = x.shape[0]
        Xpad[i, :n] = x
        mask[i, :n] = True

    ym = torch.tensor(ym, dtype=torch.long)
    yl = torch.tensor(yl, dtype=torch.long)
    return Xpad, mask, ym, yl, list(pids), list(fps)


# ---------------------------
# Model
# ---------------------------
class ResNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(m.children())[:-1])
        self.out_dim = 2048

    def forward(self, x):
        h = self.backbone(x)
        return h.flatten(1)

class AttnPool(nn.Module):
    def __init__(self, d, h=256):
        super().__init__()
        self.V = nn.Linear(d, h)
        self.w = nn.Linear(h, 1)

    def forward(self, H, mask=None):
        A = self.w(torch.tanh(self.V(H)))
        if mask is not None:
            A = A.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        A = torch.softmax(A, dim=1)
        Z = (A * H).sum(dim=1)
        return Z, A.squeeze(-1)

class GatedMILv2(nn.Module):
    def __init__(self, use_aux_leuko=True, gate_tau=2.0):
        super().__init__()
        self.use_aux_leuko = bool(use_aux_leuko)
        self.gate_tau = float(gate_tau)

        self.encoder = ResNetEncoder()
        d = self.encoder.out_dim

        self.pool_non = AttnPool(d, h=256)
        self.pool_leu = AttnPool(d, h=256)

        self.cls_non = nn.Linear(d, 1)
        self.cls_leu = nn.Linear(d, 1)

        self.gate = nn.Linear(2 * d, 1)

        if self.use_aux_leuko:
            self.leuko_head = nn.Sequential(
                nn.Linear(2 * d, 256),
                nn.ReLU(inplace=True),
                nn.Dropout(0.2),
                nn.Linear(256, 1),
            )

    def forward(self, X, mask=None, force_half_gate=False):
        B, N, C, H, W = X.shape
        x = X.view(B * N, C, H, W)
        h = self.encoder(x)
        d = h.shape[-1]
        Hbag = h.view(B, N, d)

        z_non, _ = self.pool_non(Hbag, mask=mask)
        z_leu, _ = self.pool_leu(Hbag, mask=mask)

        logit_non = self.cls_non(z_non)
        logit_leu = self.cls_leu(z_leu)

        gate_in = torch.cat([z_non, z_leu], dim=1)
        g_logit = self.gate(gate_in) / self.gate_tau
        g = torch.sigmoid(g_logit)
        if force_half_gate:
            g = 0.5 * torch.ones_like(g)

        logit_m = (1.0 - g) * logit_non + g * logit_leu

        logit_l = None
        if self.use_aux_leuko:
            logit_l = self.leuko_head(gate_in)

        return logit_m, logit_l, g

def set_backbone_trainable(model: nn.Module, trainable: bool):
    for p in model.encoder.backbone.parameters():
        p.requires_grad = bool(trainable)


# ---------------------------
# Gate entropy regularizer
# ---------------------------
def gate_entropy(g, eps=1e-6):
    g = torch.clamp(g, eps, 1.0 - eps)
    ent = -(g * torch.log(g) + (1.0 - g) * torch.log(1.0 - g))
    return ent.mean()


# ---------------------------
# Train + multi-bag inference
# ---------------------------
def train_epoch(model, loader, opt, crit_m, crit_l, scaler, epoch):
    model.train()
    losses = []
    opt.zero_grad(set_to_none=True)

    if FREEZE_BACKBONE_EPOCHS > 0:
        set_backbone_trainable(model, trainable=(epoch > FREEZE_BACKBONE_EPOCHS))

    for step, (X, mask, ym, yl, _pids, _fps) in enumerate(loader, start=1):
        X = X.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        ym = ym.to(DEVICE, non_blocking=True).float().unsqueeze(1)
        yl = yl.to(DEVICE, non_blocking=True).float().unsqueeze(1)

        force_half = (epoch <= WARMUP_FORCE_HALF_GATE_EPOCHS)

        if scaler is not None:
            with torch.amp.autocast(device_type="cuda"):
                logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
                loss = crit_m(logit_m, ym)
                if USE_AUX_LEUKO and (logit_l is not None):
                    loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)
                if (not force_half) and (LAMBDA_GATE_ENT > 0):
                    loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

                loss = loss / float(ACCUM_STEPS)

            scaler.scale(loss).backward()

            if step % ACCUM_STEPS == 0:
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)

        else:
            logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
            loss = crit_m(logit_m, ym)
            if USE_AUX_LEUKO and (logit_l is not None):
                loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)
            if (not force_half) and (LAMBDA_GATE_ENT > 0):
                loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

            loss = loss / float(ACCUM_STEPS)
            loss.backward()

            if step % ACCUM_STEPS == 0:
                opt.step()
                opt.zero_grad(set_to_none=True)

        losses.append(float(loss.detach().cpu().item()) * float(ACCUM_STEPS))

    if len(loader) % ACCUM_STEPS != 0:
        if scaler is not None:
            scaler.step(opt)
            scaler.update()
        else:
            opt.step()
        opt.zero_grad(set_to_none=True)

    return float(np.mean(losses)) if losses else float("nan")


@torch.no_grad()
def predict_logits_multibag(model, ds_eval, K=10, use_tta=True):
    model.eval()

    pid_list = ds_eval.patients
    pid_to_idx = {p: i for i, p in enumerate(pid_list)}
    n = len(pid_list)

    sum_logits = np.zeros((n,), dtype=np.float64)
    sum_gate = np.zeros((n,), dtype=np.float64)

    dl = DataLoader(
        ds_eval,
        batch_size=EVAL_BS,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_patient_bags
    )

    tta_tfs = [eval_tf]
    if use_tta:
        tta_tfs.append(eval_tf_hflip)

    denom = float(K * len(tta_tfs))

    for _k in range(K):
        ds_eval.train = False
        ds_eval.eval_random = True

        for tf in tta_tfs:
            ds_eval.set_transform(tf)

            for X, mask, _ym, _yl, pids, _fps in dl:
                X = X.to(DEVICE, non_blocking=True)
                mask = mask.to(DEVICE, non_blocking=True)

                logit_m, _logit_l, g = model(X, mask=mask, force_half_gate=False)
                logit_m = logit_m.detach().cpu().numpy().reshape(-1)
                g = g.detach().cpu().numpy().reshape(-1)

                for pid, lg, gg in zip(pids, logit_m, g):
                    j = pid_to_idx[pid]
                    sum_logits[j] += float(lg)
                    sum_gate[j] += float(gg)

    avg_logits = sum_logits / denom
    avg_gate = sum_gate / denom

    y = np.asarray([ds_eval.pid_to_labels[p][0] for p in pid_list], dtype=int)
    le = np.asarray([ds_eval.pid_to_labels[p][1] for p in pid_list], dtype=int)

    return avg_logits, avg_gate, pid_list, y, le


# ---------------------------
# CV folds (patient-level stratified)
# ---------------------------
def make_patient_folds(patients, pid_to_labels, n_splits=5, seed=42):
    strat = np.array([2 * pid_to_labels[p][0] + pid_to_labels[p][1] for p in patients], dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idxs = np.arange(len(patients))
    folds = []
    for tr_idx, va_idx in skf.split(idxs, strat):
        tr_p = [patients[i] for i in tr_idx]
        va_p = [patients[i] for i in va_idx]
        folds.append((tr_p, va_p))
    return folds


# ---------------------------
# Checkpoint helpers
# ---------------------------
def torch_load(path: Path, map_location):
    # PyTorch 2.6 default is weights_only=True; we need full dict for resume
    if TORCH_LOAD_WEIGHTS_ONLY_FALSE:
        return torch.load(str(path), map_location=map_location, weights_only=False)
    return torch.load(str(path), map_location=map_location)

def save_ckpt(path: Path, model, opt, scaler, epoch, best_auc, best_epoch, bad):
    ck = {
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": int(epoch),
        "best_auc": float(best_auc),
        "best_epoch": int(best_epoch),
        "bad": int(bad),
        "rng": capture_rng_state(),
    }
    torch.save(ck, str(path))

def load_ckpt(path: Path, model, opt, scaler):
    ck = torch_load(path, map_location=DEVICE)
    # strict=False prevents issues if you change aux head flags
    model.load_state_dict(ck["model"], strict=False)
    if "opt" in ck and opt is not None:
        opt.load_state_dict(ck["opt"])
    if scaler is not None and ck.get("scaler") is not None:
        scaler.load_state_dict(ck["scaler"])
    restore_rng_state(ck.get("rng", None))
    return ck


# ---------------------------
# Output helpers
# ---------------------------
def save_fold_test_csv(seed, fold, pids, y_true, leuko, logits, gates):
    out = RUNS_DIR / f"pred_fold{fold}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit,prob,gate\n")
        for pid, yt, lk, lg, gg in zip(pids, y_true, leuko, logits.tolist(), gates.tolist()):
            pr = float(sigmoid_np(lg))
            f.write(f"{pid},{int(yt)},{int(lk)},{lg:.8f},{pr:.8f},{gg:.6f}\n")
    print("Saved fold test preds:", out)
    return out

def aggregate_seed_folds(seed, folds_list):
    # robust aggregation without column overlap
    import pandas as pd

    dfs = []
    for f in folds_list:
        p = RUNS_DIR / f"pred_fold{f}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")
        df = pd.read_csv(p)
        df = df.set_index("Patient")

        # keep y_true/leuko only from first fold; keep logit per fold
        df = df[["y_true", "leuko", "logit"]].rename(columns={"logit": f"logit_fold{f}"})
        dfs.append(df)

    merged = dfs[0]
    for d in dfs[1:]:
        merged = merged.join(d[[c for c in d.columns if c.startswith("logit_fold")]], how="inner")

    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    logit_mean = merged[logit_cols].mean(axis=1).astype(float).values
    prob_mean = sigmoid_np(logit_mean)

    y_true = merged["y_true"].astype(int).values
    leuko = merged["leuko"].astype(int).values

    out = RUNS_DIR / f"pred_seed{seed}_{BENCHMARK}_folds{min(folds_list)}_{max(folds_list)}_Ktest{K_TEST}_agg.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit_mean,prob_mean\n")
        for pid, yt, lk, lg, pr in zip(merged.index.tolist(), y_true, leuko, logit_mean.tolist(), prob_mean.tolist()):
            f.write(f"{pid},{yt},{lk},{lg:.8f},{pr:.8f}\n")

    auc = try_auc(y_true, prob_mean)
    print("Saved:", out)
    print(f"AGG TEST AUC (avg logits over folds {min(folds_list)}..{max(folds_list)}): {auc}")
    return out, auc


# ---------------------------
# Main
# ---------------------------
def main():
    rows_tr = ensure_exists(load_csv_rows(SPLIT_TRAIN))
    rows_va = ensure_exists(load_csv_rows(SPLIT_VAL))
    rows_te = ensure_exists(load_csv_rows(SPLIT_TEST))

    rows_pool = rows_tr + rows_va
    patients_pool, pid_to_frames_pool, pid_to_labels_pool = build_patient_index(rows_pool)
    patients_pool = [p for p in patients_pool if len(pid_to_frames_pool[p]) >= 2]

    patients_test, pid_to_frames_test, pid_to_labels_test = build_patient_index(rows_te)
    patients_test = [p for p in patients_test if len(pid_to_frames_test[p]) >= 2]

    print(f"[POOL] patients={len(patients_pool)} (train+val)")
    print(f"[TEST] patients={len(patients_test)}")

    folds = make_patient_folds(patients_pool, pid_to_labels_pool, n_splits=N_FOLDS, seed=SEEDS_TO_RUN[0])

    fold_start, fold_end = RUN_FOLDS
    folds_to_run = list(range(fold_start, fold_end + 1))

    for seed in SEEDS_TO_RUN:
        seed_all(seed)
        print("\n" + "=" * 70)
        print(f"RUN seed={seed} | benchmark={BENCHMARK} | folds={fold_start}..{fold_end}")
        print("=" * 70)

        for fold_i in folds_to_run:
            tr_pat, va_pat = folds[fold_i - 1]
            print(f"\n--- Fold {fold_i}/{N_FOLDS} | train_pat={len(tr_pat)} val_pat={len(va_pat)} ---")

            ds_tr = PatientBagDataset(tr_pat, pid_to_frames_pool, pid_to_labels_pool, train=True)
            ds_va = PatientBagDataset(va_pat, pid_to_frames_pool, pid_to_labels_pool, train=False)
            ds_te = PatientBagDataset(patients_test, pid_to_frames_test, pid_to_labels_test, train=False)

            y_m_tr = np.array([pid_to_labels_pool[p][0] for p in tr_pat], dtype=int)
            y_l_tr = np.array([pid_to_labels_pool[p][1] for p in tr_pat], dtype=int)
            pos_m = int(y_m_tr.sum()); neg_m = int(len(y_m_tr) - pos_m)
            pos_l = int(y_l_tr.sum()); neg_l = int(len(y_l_tr) - pos_l)

            pos_weight_m = torch.tensor([neg_m / max(pos_m, 1)], dtype=torch.float32, device=DEVICE)
            pos_weight_l = torch.tensor([neg_l / max(pos_l, 1)], dtype=torch.float32, device=DEVICE)
            crit_m = nn.BCEWithLogitsLoss(pos_weight=pos_weight_m)
            crit_l = nn.BCEWithLogitsLoss(pos_weight=pos_weight_l)

            dl_tr = DataLoader(
                ds_tr,
                batch_size=TRAIN_BS,
                shuffle=True,
                num_workers=NUM_WORKERS,
                pin_memory=PIN_MEMORY,
                collate_fn=collate_patient_bags
            )

            model = GatedMILv2(use_aux_leuko=USE_AUX_LEUKO, gate_tau=GATE_TAU).to(DEVICE)
            print("Model on:", next(model.parameters()).device)

            opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scaler = torch.amp.GradScaler("cuda") if (AMP and "cuda" in DEVICE) else None

            best_path = RUNS_DIR / f"cv_best_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"
            prog_path = RUNS_DIR / f"progress_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"
            summary_path = RUNS_DIR / f"summary_{BENCHMARK}_seed{seed}_fold{fold_i}.txt"

            best_auc = -1e9
            best_epoch = -1
            bad = 0
            start_ep = 1

            # Resume from BEST if exists (preferred), else from progress
            resume_from = None
            if best_path.exists():
                resume_from = best_path
            elif prog_path.exists():
                resume_from = prog_path

            if resume_from is not None:
                ck = load_ckpt(resume_from, model, opt, scaler)
                best_auc = float(ck.get("best_auc", -1e9))
                best_epoch = int(ck.get("best_epoch", ck.get("epoch", -1)))
                bad = int(ck.get("bad", 0))
                start_ep = int(ck.get("epoch", 0)) + 1

                if RESET_BAD_ON_RESUME:
                    bad = 0

                print(f"[RESUME] seed={seed} fold={fold_i} from epoch {start_ep} | best_auc={best_auc:.4f} at ep {best_epoch} | bad={bad}")

            if SKIP_IF_FINISHED and start_ep > EPOCHS_MAX:
                print(f"[SKIP] fold={fold_i}: already finished (epoch {start_ep-1} >= {EPOCHS_MAX}).")
                # still save test preds from BEST
                if best_path.exists():
                    ck_best = torch_load(best_path, map_location=DEVICE)
                    model.load_state_dict(ck_best["model"], strict=False)
                lg_te, gg_te, pids_te, y_te, le_te = predict_logits_multibag(model, ds_te, K=K_TEST, use_tta=TTA_TEST)
                save_fold_test_csv(seed, fold_i, pids_te, y_te, le_te, lg_te, gg_te)
                continue

            if ONLY_PREDICT_IF_BEST_EXISTS and best_path.exists():
                ck_best = torch_load(best_path, map_location=DEVICE)
                model.load_state_dict(ck_best["model"], strict=False)
                lg_te, gg_te, pids_te, y_te, le_te = predict_logits_multibag(model, ds_te, K=K_TEST, use_tta=TTA_TEST)
                save_fold_test_csv(seed, fold_i, pids_te, y_te, le_te, lg_te, gg_te)
                continue

            # Training loop
            for ep in range(start_ep, EPOCHS_MAX + 1):
                t0 = time.time()
                tr_loss = train_epoch(model, dl_tr, opt, crit_m, crit_l, scaler, epoch=ep)
                t_train = (time.time() - t0)

                do_val = (ep == start_ep) or (ep % EVAL_EVERY == 0) or (ep == EPOCHS_MAX)
                if do_val:
                    t1 = time.time()
                    lg_va, gg_va, pids_va, y_va, le_va = predict_logits_multibag(model, ds_va, K=K_VAL, use_tta=TTA_VAL)
                    auc_va = try_auc(y_va, sigmoid_np(lg_va))
                    gate_mean = float(np.mean(gg_va)) if len(gg_va) else float("nan")
                    t_val = (time.time() - t1)

                    print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                          f"train_loss={tr_loss:.4f} | val_auc(K)={auc_va:.4f} | gate_mean={gate_mean:.3f} | "
                          f"time: train={t_train/60:.2f}m val={t_val/60:.2f}m")

                    improved = (auc_va > best_auc + MIN_DELTA) or (best_epoch < 0)
                    if improved:
                        best_auc = float(auc_va)
                        best_epoch = int(ep)
                        bad = 0
                        save_ckpt(best_path, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)
                        with open(summary_path, "w", encoding="utf-8") as f:
                            f.write(f"seed={seed}\nfold={fold_i}\nbest_epoch={best_epoch}\nbest_auc={best_auc:.6f}\n")
                        print(f"[SAVE] best updated -> {summary_path}")
                    else:
                        bad += 1
                        save_ckpt(prog_path, model, opt, scaler, epoch=ep, best_auc=best_auc, best_epoch=best_epoch, bad=bad)

                        if bad >= EARLY_STOP_PATIENCE:
                            print(f"Early stop at ep {ep} (best_auc={best_auc:.4f} at ep {best_epoch}).")
                            break
                else:
                    print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                          f"train_loss={tr_loss:.4f} | (no val this epoch) | time: train={t_train/60:.2f}m")

            # TEST prediction ALWAYS from BEST
            if best_path.exists():
                ck_best = torch_load(best_path, map_location=DEVICE)
                model.load_state_dict(ck_best["model"], strict=False)

            lg_te, gg_te, pids_te, y_te, le_te = predict_logits_multibag(model, ds_te, K=K_TEST, use_tta=TTA_TEST)
            save_fold_test_csv(seed, fold_i, pids_te, y_te, le_te, lg_te, gg_te)

        # aggregate (only if we ran full 1..5)
        if fold_start == 1 and fold_end == N_FOLDS:
            aggregate_seed_folds(seed, list(range(1, N_FOLDS + 1)))


if __name__ == "__main__":
    main()

DEVICE: cuda:0
BENCHMARK=res_shift
 train: D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv
 val  : D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv
 test : D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv
RUNNING seeds=[43] folds=1..5 (of 5)
RUN seed(s)=[43] | K_VAL=5 | K_TEST=10 | TTA_VAL=False | TTA_TEST=True
Gate: tau=2.0 | lambda_ent=0.02 | freeze_backbone_epochs=2
TRAIN_BS=2 (accum 4) | EVAL_BS=2 | EVAL_EVERY=2
[INFO] filtered missing files: 13
[POOL] patients=137 (train+val)
[TEST] patients=73

RUN seed=43 | benchmark=res_shift | folds=1..5

--- Fold 1/5 | train_pat=109 val_pat=28 ---
Model on: cuda:0
[RESUME] seed=43 fold=1 from epoch 21 | best_auc=1.0000 at ep 20 | bad=0
[res_shift] seed=43 fold=1 ep 21/30 | train_loss=0.4386 | val_auc(K)=0.9625 | gate_mean=0.902 | time: train=1.90m val=1.38m
[res_shift] seed=43 fold=1 ep 22/30 | train_loss=0.5726 | val_auc(K)=0.9437 | gate_mean=0.903 | time: train=1.45m val=1.00m
[res_shift] 

In [3]:
# ============================================
# RUN-READY SINGLE SCRIPT (Windows / Colab)
# FIXES ADDED (based on your latest logs):
#   ✅ If BEST already exists, you can SKIP training and ONLY (re)make preds
#   ✅ Avoid wasting time resuming after best_epoch (common when you already peaked)
#   ✅ Robust NaN/Inf handling (skip bad batch, clip grads)
#   ✅ TEST preds ALWAYS from BEST checkpoint (not last/progress)
#   ✅ Aggregation fixed (no y_true/leuko column overlap)
#   ✅ Resume works on PyTorch 2.6+ (weights_only=False)
# ============================================

import os, math, time, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import torchvision
from torchvision import transforms as T


# ---------------------------
# Config (EDIT HERE)
# ---------------------------

ART_DIR = Path(r"D:\ENT\_challenge2_artifacts")   # <-- change if needed
RUNS_DIR = ART_DIR / "_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK = "res_shift"

SPLITS = {
    "res_shift": {
        "train": ART_DIR / "split_res_shift_train_major_colab.csv",
        "val":   ART_DIR / "split_res_shift_val_major_colab.csv",
        "test":  ART_DIR / "split_res_shift_test_minor_colab.csv",
    }
}
SPLIT_TRAIN = SPLITS[BENCHMARK]["train"]
SPLIT_VAL   = SPLITS[BENCHMARK]["val"]
SPLIT_TEST  = SPLITS[BENCHMARK]["test"]

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
AMP = True

IMG_SIZE = 224
NUM_WORKERS = 0        # if Windows deadlock -> set 0
PIN_MEMORY = True if "cuda" in DEVICE else False

BAG_SIZE_NONLEUKO = 24
BAG_SIZE_LEUKO = 40

EPOCHS_MAX = 30
LR = 5e-5
WEIGHT_DECAY = 1e-4

EARLY_STOP_PATIENCE = 6
MIN_DELTA = 1e-4

USE_AUX_LEUKO = True
LAMBDA_LEUKO = 0.30

WARMUP_FORCE_HALF_GATE_EPOCHS = 2
GATE_TAU = 2.0
LAMBDA_GATE_ENT = 0.02
FREEZE_BACKBONE_EPOCHS = 2

K_VAL = 5
K_TEST = 10
TTA_VAL = False
TTA_TEST = True
EVAL_EVERY = 2

TRAIN_BS = 2
ACCUM_STEPS = 4
EVAL_BS = 2

N_FOLDS = 5
SEEDS_TO_RUN = [43]
RUN_FOLDS = (1, 5)

# Resume / skip policy
RESET_BAD_ON_RESUME = True

# ✅ NEW: if BEST exists, do not continue training after best epoch; just produce preds
PRED_ONLY_IF_BEST_EXISTS = True

# ✅ NEW: hard stop training if current epoch > best_epoch + (patience + 1) on resume
STOP_IF_PAST_BEST = True

# ✅ NEW: if pred_fold*.csv exists, skip recomputing it unless you want refresh
SKIP_IF_PRED_EXISTS = True

# Training stability
GRAD_CLIP_NORM = 1.0
SKIP_NAN_BATCH = True

# PyTorch 2.6+ load
TORCH_LOAD_WEIGHTS_ONLY_FALSE = True


print(f"DEVICE: {DEVICE}")
print("=" * 70)
print(f"BENCHMARK={BENCHMARK}")
print(f" train: {SPLIT_TRAIN}")
print(f" val  : {SPLIT_VAL}")
print(f" test : {SPLIT_TEST}")
print(f"RUNNING seeds={SEEDS_TO_RUN} folds={RUN_FOLDS[0]}..{RUN_FOLDS[1]} (of {N_FOLDS})")
print("=" * 70)
print(f"RUN seed(s)={SEEDS_TO_RUN} | K_VAL={K_VAL} | K_TEST={K_TEST} | TTA_VAL={TTA_VAL} | TTA_TEST={TTA_TEST}")
print(f"Gate: tau={GATE_TAU} | lambda_ent={LAMBDA_GATE_ENT} | freeze_backbone_epochs={FREEZE_BACKBONE_EPOCHS}")
print(f"TRAIN_BS={TRAIN_BS} (accum {ACCUM_STEPS}) | EVAL_BS={EVAL_BS} | EVAL_EVERY={EVAL_EVERY}")
print("=" * 70)


# ---------------------------
# Utils
# ---------------------------

def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))

def try_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def safe_int(x, default=0):
    try:
        return int(float(x))
    except Exception:
        return default

def parse_leuko(x):
    if x is None:
        return 0
    s = str(x).strip().lower()
    return 1 if s in ["yes", "1", "true", "y"] else 0

def parse_malign(x):
    return 1 if safe_int(x, 0) == 1 else 0

def load_csv_rows(csv_path):
    csv_path = str(csv_path)
    try:
        import pandas as pd
        df = pd.read_csv(csv_path)
        return df.to_dict(orient="records")
    except Exception:
        import csv
        rows = []
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for r in reader:
                rows.append(r)
        return rows

def ensure_exists(rows):
    kept, missing = [], 0
    for r in rows:
        fp = r.get("filepath", "")
        if fp and os.path.exists(fp):
            kept.append(r)
        else:
            missing += 1
    if missing > 0:
        print(f"[INFO] filtered missing files: {missing}")
    return kept

def build_patient_index(rows):
    pid_to_frames = defaultdict(list)
    for r in rows:
        pid = r.get("Patient", None) or r.get("patient", None) or r.get("PID", None)
        if pid is None:
            pid = "UNK"
        y_m = parse_malign(r.get("label_binary", 0))
        y_l = parse_leuko(r.get("Leukoplakia", 0))
        fp = r.get("filepath", "")
        pid_to_frames[pid].append({"filepath": fp, "y_m": y_m, "y_l": y_l})

    pid_to_labels = {}
    for pid, frs in pid_to_frames.items():
        y_m = 1 if any(f["y_m"] == 1 for f in frs) else 0
        y_l = 1 if any(f["y_l"] == 1 for f in frs) else 0
        pid_to_labels[pid] = (y_m, y_l)

    patients = sorted(pid_to_frames.keys())
    return patients, pid_to_frames, pid_to_labels


# ---------------------------
# Transforms
# ---------------------------
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
    T.ToTensor(),
])

eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])
eval_tf_hflip = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(p=1.0), T.ToTensor()])


# ---------------------------
# Dataset
# ---------------------------
class PatientBagDataset(Dataset):
    def __init__(self, patients, pid_to_frames, pid_to_labels, train=True):
        self.patients = list(patients)
        self.pid_to_frames = pid_to_frames
        self.pid_to_labels = pid_to_labels
        self.train = train
        self.tf = train_tf if train else eval_tf
        self.eval_random = True

    def set_transform(self, tf):
        self.tf = tf

    def __len__(self):
        return len(self.patients)

    def _sample_frames(self, pid):
        frs = self.pid_to_frames[pid]
        y_m, y_l = self.pid_to_labels[pid]
        bag_size = BAG_SIZE_LEUKO if y_l == 1 else BAG_SIZE_NONLEUKO
        n = len(frs)
        idxs = np.random.choice(n, size=bag_size, replace=(n < bag_size))
        return [frs[i] for i in idxs]

    def __getitem__(self, i):
        pid = self.patients[i]
        y_m, y_l = self.pid_to_labels[pid]
        sampled = self._sample_frames(pid)

        imgs = []
        for f in sampled:
            fp = f["filepath"]
            im = Image.open(fp).convert("RGB")
            imgs.append(self.tf(im))
        X = torch.stack(imgs, dim=0)
        return X, y_m, y_l, pid

def collate_patient_bags(batch):
    Xs, ym, yl, pids = zip(*batch)
    lens = [x.shape[0] for x in Xs]
    max_len = max(lens)
    B = len(Xs)
    C, H, W = Xs[0].shape[1:]

    Xpad = torch.zeros((B, max_len, C, H, W), dtype=Xs[0].dtype)
    mask = torch.zeros((B, max_len), dtype=torch.bool)
    for i, x in enumerate(Xs):
        n = x.shape[0]
        Xpad[i, :n] = x
        mask[i, :n] = True

    ym = torch.tensor(ym, dtype=torch.long)
    yl = torch.tensor(yl, dtype=torch.long)
    return Xpad, mask, ym, yl, list(pids)


# ---------------------------
# Model
# ---------------------------
class ResNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(m.children())[:-1])
        self.out_dim = 2048

    def forward(self, x):
        h = self.backbone(x)
        return h.flatten(1)

class AttnPool(nn.Module):
    def __init__(self, d, h=256):
        super().__init__()
        self.V = nn.Linear(d, h)
        self.w = nn.Linear(h, 1)

    def forward(self, H, mask=None):
        A = self.w(torch.tanh(self.V(H)))
        if mask is not None:
            A = A.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        A = torch.softmax(A, dim=1)
        Z = (A * H).sum(dim=1)
        return Z, A.squeeze(-1)

class GatedMILv2(nn.Module):
    def __init__(self, use_aux_leuko=True, gate_tau=2.0):
        super().__init__()
        self.use_aux_leuko = bool(use_aux_leuko)
        self.gate_tau = float(gate_tau)

        self.encoder = ResNetEncoder()
        d = self.encoder.out_dim

        self.pool_non = AttnPool(d, h=256)
        self.pool_leu = AttnPool(d, h=256)

        self.cls_non = nn.Linear(d, 1)
        self.cls_leu = nn.Linear(d, 1)
        self.gate = nn.Linear(2 * d, 1)

        if self.use_aux_leuko:
            self.leuko_head = nn.Sequential(
                nn.Linear(2 * d, 256),
                nn.ReLU(inplace=True),
                nn.Dropout(0.2),
                nn.Linear(256, 1),
            )

    def forward(self, X, mask=None, force_half_gate=False):
        B, N, C, H, W = X.shape
        x = X.view(B * N, C, H, W)
        h = self.encoder(x)
        d = h.shape[-1]
        Hbag = h.view(B, N, d)

        z_non, _ = self.pool_non(Hbag, mask=mask)
        z_leu, _ = self.pool_leu(Hbag, mask=mask)

        logit_non = self.cls_non(z_non)
        logit_leu = self.cls_leu(z_leu)

        gate_in = torch.cat([z_non, z_leu], dim=1)
        g = torch.sigmoid((self.gate(gate_in) / self.gate_tau))

        if force_half_gate:
            g = 0.5 * torch.ones_like(g)

        logit_m = (1.0 - g) * logit_non + g * logit_leu

        logit_l = None
        if self.use_aux_leuko:
            logit_l = self.leuko_head(gate_in)

        return logit_m, logit_l, g

def set_backbone_trainable(model: nn.Module, trainable: bool):
    for p in model.encoder.backbone.parameters():
        p.requires_grad = bool(trainable)

def gate_entropy(g, eps=1e-6):
    g = torch.clamp(g, eps, 1.0 - eps)
    ent = -(g * torch.log(g) + (1.0 - g) * torch.log(1.0 - g))
    return ent.mean()


# ---------------------------
# Torch load/save (PyTorch 2.6+)
# ---------------------------
def torch_load(path: Path, map_location):
    if TORCH_LOAD_WEIGHTS_ONLY_FALSE:
        return torch.load(str(path), map_location=map_location, weights_only=False)
    return torch.load(str(path), map_location=map_location)

def save_ckpt(path: Path, model, opt, scaler, epoch, best_auc, best_epoch, bad):
    ck = {
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": int(epoch),
        "best_auc": float(best_auc),
        "best_epoch": int(best_epoch),
        "bad": int(bad),
    }
    torch.save(ck, str(path))

def load_ckpt(path: Path, model, opt, scaler):
    ck = torch_load(path, map_location=DEVICE)
    model.load_state_dict(ck["model"], strict=False)
    if opt is not None and "opt" in ck:
        opt.load_state_dict(ck["opt"])
    if scaler is not None and ck.get("scaler") is not None:
        scaler.load_state_dict(ck["scaler"])
    return ck


# ---------------------------
# Train / Eval
# ---------------------------
def train_epoch(model, loader, opt, crit_m, crit_l, scaler, epoch):
    model.train()
    opt.zero_grad(set_to_none=True)

    if FREEZE_BACKBONE_EPOCHS > 0:
        set_backbone_trainable(model, trainable=(epoch > FREEZE_BACKBONE_EPOCHS))

    losses = []

    for step, (X, mask, ym, yl, _pids) in enumerate(loader, start=1):
        X = X.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        ym = ym.to(DEVICE, non_blocking=True).float().unsqueeze(1)
        yl = yl.to(DEVICE, non_blocking=True).float().unsqueeze(1)

        force_half = (epoch <= WARMUP_FORCE_HALF_GATE_EPOCHS)

        if scaler is not None:
            with torch.amp.autocast(device_type="cuda"):
                logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
                loss = crit_m(logit_m, ym)
                if USE_AUX_LEUKO and (logit_l is not None):
                    loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)
                if (not force_half) and (LAMBDA_GATE_ENT > 0):
                    loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

                if torch.isnan(loss) or torch.isinf(loss):
                    if SKIP_NAN_BATCH:
                        opt.zero_grad(set_to_none=True)
                        continue
                    else:
                        loss = torch.nan_to_num(loss, nan=0.0, posinf=0.0, neginf=0.0)

                loss = loss / float(ACCUM_STEPS)

            scaler.scale(loss).backward()

            if step % ACCUM_STEPS == 0:
                scaler.unscale_(opt)
                if GRAD_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)

        else:
            logit_m, logit_l, g = model(X, mask=mask, force_half_gate=force_half)
            loss = crit_m(logit_m, ym)
            if USE_AUX_LEUKO and (logit_l is not None):
                loss = loss + LAMBDA_LEUKO * crit_l(logit_l, yl)
            if (not force_half) and (LAMBDA_GATE_ENT > 0):
                loss = loss + float(LAMBDA_GATE_ENT) * gate_entropy(g)

            if torch.isnan(loss) or torch.isinf(loss):
                if SKIP_NAN_BATCH:
                    opt.zero_grad(set_to_none=True)
                    continue
                else:
                    loss = torch.nan_to_num(loss, nan=0.0, posinf=0.0, neginf=0.0)

            loss = loss / float(ACCUM_STEPS)
            loss.backward()

            if step % ACCUM_STEPS == 0:
                if GRAD_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                opt.step()
                opt.zero_grad(set_to_none=True)

        losses.append(float(loss.detach().cpu().item()) * float(ACCUM_STEPS))

    if len(loader) % ACCUM_STEPS != 0:
        if scaler is not None:
            scaler.unscale_(opt)
            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(opt)
            scaler.update()
        else:
            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            opt.step()
        opt.zero_grad(set_to_none=True)

    return float(np.mean(losses)) if losses else float("nan")

@torch.no_grad()
def predict_logits_multibag(model, ds_eval, K=10, use_tta=True):
    model.eval()
    pid_list = ds_eval.patients
    pid_to_idx = {p: i for i, p in enumerate(pid_list)}
    n = len(pid_list)

    sum_logits = np.zeros((n,), dtype=np.float64)
    sum_gate = np.zeros((n,), dtype=np.float64)

    dl = DataLoader(
        ds_eval, batch_size=EVAL_BS, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        collate_fn=collate_patient_bags
    )

    tta_tfs = [eval_tf] + ([eval_tf_hflip] if use_tta else [])
    denom = float(K * len(tta_tfs))

    for _ in range(K):
        ds_eval.train = False
        ds_eval.eval_random = True
        for tf in tta_tfs:
            ds_eval.set_transform(tf)
            for X, mask, _ym, _yl, pids in dl:
                X = X.to(DEVICE, non_blocking=True)
                mask = mask.to(DEVICE, non_blocking=True)
                logit_m, _logit_l, g = model(X, mask=mask, force_half_gate=False)
                logit_m = logit_m.detach().cpu().numpy().reshape(-1)
                g = g.detach().cpu().numpy().reshape(-1)
                for pid, lg, gg in zip(pids, logit_m, g):
                    j = pid_to_idx[pid]
                    sum_logits[j] += float(lg)
                    sum_gate[j] += float(gg)

    avg_logits = sum_logits / denom
    avg_gate = sum_gate / denom

    y = np.asarray([ds_eval.pid_to_labels[p][0] for p in pid_list], dtype=int)
    le = np.asarray([ds_eval.pid_to_labels[p][1] for p in pid_list], dtype=int)
    return avg_logits, avg_gate, pid_list, y, le


# ---------------------------
# CV folds
# ---------------------------
def make_patient_folds(patients, pid_to_labels, n_splits=5, seed=42):
    strat = np.array([2 * pid_to_labels[p][0] + pid_to_labels[p][1] for p in patients], dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idxs = np.arange(len(patients))
    folds = []
    for tr_idx, va_idx in skf.split(idxs, strat):
        tr_p = [patients[i] for i in tr_idx]
        va_p = [patients[i] for i in va_idx]
        folds.append((tr_p, va_p))
    return folds


# ---------------------------
# Output + aggregation (no overlap)
# ---------------------------
def save_fold_test_csv(seed, fold, pids, y_true, leuko, logits, gates):
    out = RUNS_DIR / f"pred_fold{fold}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit,prob,gate\n")
        for pid, yt, lk, lg, gg in zip(pids, y_true, leuko, logits.tolist(), gates.tolist()):
            pr = float(sigmoid_np(lg))
            f.write(f"{pid},{int(yt)},{int(lk)},{lg:.8f},{pr:.8f},{gg:.6f}\n")
    print("Saved fold test preds:", out)
    return out

def aggregate_seed_folds(seed, folds_list):
    import pandas as pd

    base = None
    for f in folds_list:
        p = RUNS_DIR / f"pred_fold{f}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")
        df = pd.read_csv(p).set_index("Patient")
        if base is None:
            base = df[["y_true", "leuko"]].copy()
        base[f"logit_fold{f}"] = df["logit"].astype(float)

    logit_cols = [c for c in base.columns if c.startswith("logit_fold")]
    logit_mean = base[logit_cols].mean(axis=1).astype(float).values
    prob_mean = sigmoid_np(logit_mean)

    y_true = base["y_true"].astype(int).values
    leuko = base["leuko"].astype(int).values

    out = RUNS_DIR / f"pred_seed{seed}_{BENCHMARK}_folds{min(folds_list)}_{max(folds_list)}_Ktest{K_TEST}_agg.csv"
    with open(out, "w", encoding="utf-8") as f:
        f.write("Patient,y_true,leuko,logit_mean,prob_mean\n")
        for pid, yt, lk, lg, pr in zip(base.index.tolist(), y_true, leuko, logit_mean.tolist(), prob_mean.tolist()):
            f.write(f"{pid},{yt},{lk},{lg:.8f},{pr:.8f}\n")

    auc = try_auc(y_true, prob_mean)
    print("Saved:", out)
    print(f"AGG TEST AUC (avg logits over folds {min(folds_list)}..{max(folds_list)}): {auc}")
    return out, auc


# ---------------------------
# Main
# ---------------------------
def main():
    rows_tr = ensure_exists(load_csv_rows(SPLIT_TRAIN))
    rows_va = ensure_exists(load_csv_rows(SPLIT_VAL))
    rows_te = ensure_exists(load_csv_rows(SPLIT_TEST))

    rows_pool = rows_tr + rows_va
    patients_pool, pid_to_frames_pool, pid_to_labels_pool = build_patient_index(rows_pool)
    patients_pool = [p for p in patients_pool if len(pid_to_frames_pool[p]) >= 2]

    patients_test, pid_to_frames_test, pid_to_labels_test = build_patient_index(rows_te)
    patients_test = [p for p in patients_test if len(pid_to_frames_test[p]) >= 2]

    print(f"[POOL] patients={len(patients_pool)} (train+val)")
    print(f"[TEST] patients={len(patients_test)}")

    folds = make_patient_folds(patients_pool, pid_to_labels_pool, n_splits=N_FOLDS, seed=SEEDS_TO_RUN[0])

    fold_start, fold_end = RUN_FOLDS
    folds_to_run = list(range(fold_start, fold_end + 1))

    for seed in SEEDS_TO_RUN:
        seed_all(seed)
        print("\n" + "=" * 70)
        print(f"RUN seed={seed} | benchmark={BENCHMARK} | folds={fold_start}..{fold_end}")
        print("=" * 70)

        for fold_i in folds_to_run:
            tr_pat, va_pat = folds[fold_i - 1]
            print(f"\n--- Fold {fold_i}/{N_FOLDS} | train_pat={len(tr_pat)} val_pat={len(va_pat)} ---")

            pred_path = RUNS_DIR / f"pred_fold{fold_i}_{BENCHMARK}_seed{seed}_Ktest{K_TEST}.csv"
            best_path = RUNS_DIR / f"cv_best_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"
            prog_path = RUNS_DIR / f"progress_{BENCHMARK}_seed{seed}_fold{fold_i}.pt"
            summary_path = RUNS_DIR / f"summary_{BENCHMARK}_seed{seed}_fold{fold_i}.txt"

            if SKIP_IF_PRED_EXISTS and pred_path.exists() and best_path.exists():
                print(f"[SKIP] preds exist -> {pred_path}")
                continue

            ds_tr = PatientBagDataset(tr_pat, pid_to_frames_pool, pid_to_labels_pool, train=True)
            ds_va = PatientBagDataset(va_pat, pid_to_frames_pool, pid_to_labels_pool, train=False)
            ds_te = PatientBagDataset(patients_test, pid_to_frames_test, pid_to_labels_test, train=False)

            y_m_tr = np.array([pid_to_labels_pool[p][0] for p in tr_pat], dtype=int)
            y_l_tr = np.array([pid_to_labels_pool[p][1] for p in tr_pat], dtype=int)
            pos_m = int(y_m_tr.sum()); neg_m = int(len(y_m_tr) - pos_m)
            pos_l = int(y_l_tr.sum()); neg_l = int(len(y_l_tr) - pos_l)

            pos_weight_m = torch.tensor([neg_m / max(pos_m, 1)], dtype=torch.float32, device=DEVICE)
            pos_weight_l = torch.tensor([neg_l / max(pos_l, 1)], dtype=torch.float32, device=DEVICE)
            crit_m = nn.BCEWithLogitsLoss(pos_weight=pos_weight_m)
            crit_l = nn.BCEWithLogitsLoss(pos_weight=pos_weight_l)

            dl_tr = DataLoader(
                ds_tr, batch_size=TRAIN_BS, shuffle=True,
                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                collate_fn=collate_patient_bags
            )

            model = GatedMILv2(use_aux_leuko=USE_AUX_LEUKO, gate_tau=GATE_TAU).to(DEVICE)
            print("Model on:", next(model.parameters()).device)
            opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scaler = torch.amp.GradScaler("cuda") if (AMP and "cuda" in DEVICE) else None

            # If best exists and we only want preds, skip training.
            if PRED_ONLY_IF_BEST_EXISTS and best_path.exists():
                ck_best = torch_load(best_path, map_location=DEVICE)
                model.load_state_dict(ck_best["model"], strict=False)
                lg_te, gg_te, pids_te, y_te, le_te = predict_logits_multibag(model, ds_te, K=K_TEST, use_tta=TTA_TEST)
                save_fold_test_csv(seed, fold_i, pids_te, y_te, le_te, lg_te, gg_te)
                continue

            # Resume (prefer progress if you want to continue training)
            best_auc = -1e9
            best_epoch = -1
            bad = 0
            start_ep = 1

            resume_from = prog_path if prog_path.exists() else (best_path if best_path.exists() else None)
            if resume_from is not None:
                ck = load_ckpt(resume_from, model, opt, scaler)
                best_auc = float(ck.get("best_auc", -1e9))
                best_epoch = int(ck.get("best_epoch", ck.get("epoch", -1)))
                bad = int(ck.get("bad", 0))
                start_ep = int(ck.get("epoch", 0)) + 1
                if RESET_BAD_ON_RESUME:
                    bad = 0
                print(f"[RESUME] seed={seed} fold={fold_i} from epoch {start_ep} | best_auc={best_auc:.4f} at ep {best_epoch} | bad={bad}")

                if STOP_IF_PAST_BEST and best_epoch > 0 and start_ep > (best_epoch + EARLY_STOP_PATIENCE + 1) and best_path.exists():
                    print(f"[STOP] past best_epoch+patience -> load BEST and predict.")
                    ck_best = torch_load(best_path, map_location=DEVICE)
                    model.load_state_dict(ck_best["model"], strict=False)
                    lg_te, gg_te, pids_te, y_te, le_te = predict_logits_multibag(model, ds_te, K=K_TEST, use_tta=TTA_TEST)
                    save_fold_test_csv(seed, fold_i, pids_te, y_te, le_te, lg_te, gg_te)
                    continue

            # Train
            for ep in range(start_ep, EPOCHS_MAX + 1):
                t0 = time.time()
                tr_loss = train_epoch(model, dl_tr, opt, crit_m, crit_l, scaler, epoch=ep)
                t_train = time.time() - t0

                do_val = (ep == start_ep) or (ep % EVAL_EVERY == 0) or (ep == EPOCHS_MAX)
                if not do_val:
                    print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | train_loss={tr_loss:.4f} | (no val) | time={t_train/60:.2f}m")
                    save_ckpt(prog_path, model, opt, scaler, ep, best_auc, best_epoch, bad)
                    continue

                t1 = time.time()
                lg_va, gg_va, _pids_va, y_va, _le_va = predict_logits_multibag(model, ds_va, K=K_VAL, use_tta=TTA_VAL)
                auc_va = try_auc(y_va, sigmoid_np(lg_va))
                gate_mean = float(np.mean(gg_va)) if len(gg_va) else float("nan")
                t_val = time.time() - t1

                print(f"[{BENCHMARK}] seed={seed} fold={fold_i} ep {ep:02d}/{EPOCHS_MAX} | "
                      f"train_loss={tr_loss:.4f} | val_auc(K)={auc_va:.4f} | gate_mean={gate_mean:.3f} | "
                      f"time: train={t_train/60:.2f}m val={t_val/60:.2f}m")

                improved = (auc_va > best_auc + MIN_DELTA) or (best_epoch < 0)
                if improved:
                    best_auc = float(auc_va)
                    best_epoch = int(ep)
                    bad = 0
                    save_ckpt(best_path, model, opt, scaler, ep, best_auc, best_epoch, bad)
                    with open(summary_path, "w", encoding="utf-8") as f:
                        f.write(f"seed={seed}\nfold={fold_i}\nbest_epoch={best_epoch}\nbest_auc={best_auc:.6f}\n")
                    print(f"[SAVE] best updated -> {summary_path}")
                else:
                    bad += 1
                    save_ckpt(prog_path, model, opt, scaler, ep, best_auc, best_epoch, bad)
                    if bad >= EARLY_STOP_PATIENCE:
                        print(f"Early stop at ep {ep} (best_auc={best_auc:.4f} at ep {best_epoch}).")
                        break

            # Predict TEST from BEST
            if best_path.exists():
                ck_best = torch_load(best_path, map_location=DEVICE)
                model.load_state_dict(ck_best["model"], strict=False)
            lg_te, gg_te, pids_te, y_te, le_te = predict_logits_multibag(model, ds_te, K=K_TEST, use_tta=TTA_TEST)
            save_fold_test_csv(seed, fold_i, pids_te, y_te, le_te, lg_te, gg_te)

        # Aggregate for 1..5
        if fold_start == 1 and fold_end == N_FOLDS:
            aggregate_seed_folds(seed, list(range(1, N_FOLDS + 1)))


if __name__ == "__main__":
    main()

DEVICE: cuda:0
BENCHMARK=res_shift
 train: D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv
 val  : D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv
 test : D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv
RUNNING seeds=[43] folds=1..5 (of 5)
RUN seed(s)=[43] | K_VAL=5 | K_TEST=10 | TTA_VAL=False | TTA_TEST=True
Gate: tau=2.0 | lambda_ent=0.02 | freeze_backbone_epochs=2
TRAIN_BS=2 (accum 4) | EVAL_BS=2 | EVAL_EVERY=2
[INFO] filtered missing files: 13
[POOL] patients=137 (train+val)
[TEST] patients=73

RUN seed=43 | benchmark=res_shift | folds=1..5

--- Fold 1/5 | train_pat=109 val_pat=28 ---
[SKIP] preds exist -> D:\ENT\_challenge2_artifacts\_runs\pred_fold1_res_shift_seed43_Ktest10.csv

--- Fold 2/5 | train_pat=109 val_pat=28 ---
[SKIP] preds exist -> D:\ENT\_challenge2_artifacts\_runs\pred_fold2_res_shift_seed43_Ktest10.csv

--- Fold 3/5 | train_pat=110 val_pat=27 ---
[SKIP] preds exist -> D:\ENT\_challenge2_artifacts\_runs\pred_fold

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

RUNS_DIR = Path(r"D:\ENT\_challenge2_artifacts\_runs")
BENCHMARK = "res_shift"
K_TEST = 10
SEEDS = [42, 43]

def sigmoid(x): return 1/(1+np.exp(-x))

dfs = []
for s in SEEDS:
    p = RUNS_DIR / f"pred_seed{s}_{BENCHMARK}_folds1_5_Ktest{K_TEST}_agg.csv"
    df = pd.read_csv(p).set_index("Patient")
    # expects: y_true, leuko, logit_mean, prob_mean
    dfs.append(df[["y_true","leuko","logit_mean"]].rename(columns={"logit_mean": f"logit_seed{s}"}))

merged = dfs[0]
for d in dfs[1:]:
    merged = merged.join(d[[c for c in d.columns if c.startswith("logit_seed")]], how="inner")

logit_cols = [c for c in merged.columns if c.startswith("logit_seed")]
merged["logit_ens"] = merged[logit_cols].mean(axis=1).astype(float)
merged["prob_ens"]  = sigmoid(merged["logit_ens"].values)

y = merged["y_true"].astype(int).values
auc = roc_auc_score(y, merged["prob_ens"].values) if len(set(y)) > 1 else float("nan")

out = RUNS_DIR / f"pred_ens_seeds{SEEDS[0]}_{SEEDS[1]}_{BENCHMARK}_Ktest{K_TEST}.csv"
merged.reset_index()[["Patient","y_true","leuko","logit_ens","prob_ens"]].to_csv(out, index=False)

print("Saved:", out)
print("ENSEMBLE AUC:", auc)
print("Patients used:", merged.shape[0])

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\ENT\\_challenge2_artifacts\\_runs\\pred_seed42_res_shift_folds1_5_Ktest10_agg.csv'

In [5]:
from pathlib import Path

RUNS_DIR = Path(r"D:\ENT\_challenge2_artifacts\_runs")

print("\nPrediction CSV files:\n")

for f in sorted(RUNS_DIR.glob("pred*.csv")):
    print(f.name)


Prediction CSV files:

pred_fold1_res_shift_seed43_Ktest10.csv
pred_fold2_res_shift_seed43_Ktest10.csv
pred_fold3_res_shift_seed43_Ktest10.csv
pred_fold4_res_shift_seed42_Ktest10.csv
pred_fold4_res_shift_seed43_Ktest10.csv
pred_fold5_res_shift_seed42_Ktest10.csv
pred_fold5_res_shift_seed43_Ktest10.csv
pred_resnet50_res_shift_test.csv
pred_resnet50_res_shift_test_patient.csv
pred_resnet50_standard_test.csv
pred_resnet50_standard_test_patient.csv
pred_resnet50GATED_standard_test_patient.csv
pred_resnet50GATED_SUP_standard_test_patient.csv
pred_resnet50GATEDv2_res_shift_seed42_test_patient.csv
pred_resnet50GATEDv2_res_shift_seed43_test_patient.csv
pred_resnet50GATEDv2_res_shift_test_patient.csv
pred_resnet50GATEDv2_standard_test_patient.csv
pred_resnet50GRL_standard_test.csv
pred_resnet50GRL_standard_test_patient.csv
pred_resnet50Hybrid_standard_test_patient.csv
pred_resnet50MIL_standard_test_patient.csv
pred_resnet50MT_res_shift_test.csv
pred_resnet50MT_res_shift_test_patient.csv
pred_r

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

RUNS_DIR = Path(r"D:\ENT\_challenge2_artifacts\_runs")
BENCHMARK = "res_shift"
K_TEST = 10
SEED = 43

def sigmoid(x):
    return 1/(1+np.exp(-x))

# Load aggregated seed43 file
p = RUNS_DIR / f"pred_seed{SEED}_{BENCHMARK}_folds1_5_Ktest{K_TEST}_agg.csv"
df = pd.read_csv(p)

# Ensure probability exists
if "prob_mean" not in df.columns:
    df["prob_mean"] = sigmoid(df["logit_mean"].values)

# Save clean submission
out = RUNS_DIR / f"FINAL_seed{SEED}_{BENCHMARK}.csv"
df[["Patient","prob_mean"]].to_csv(out, index=False)

print("Saved final submission:", out)
print("Patients:", df.shape[0])

Saved final submission: D:\ENT\_challenge2_artifacts\_runs\FINAL_seed43_res_shift.csv
Patients: 73


In [11]:
import os
import re
import math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# =========================
# CONFIG
# =========================

# A) Your uploaded Excel (works in this ChatGPT sandbox / local Jupyter):
EXCEL_PATH = Path("D:/ENT/patients_list_clean.csv")

# B) Set ONE dataset root that exists on your machine.
# Examples:
#   Windows: r"D:\ENT\_challenge2_artifacts"
#   Colab  : "/content/drive/MyDrive/ENT/Data/_challenge2_artifacts"
DATA_ROOT = Path(r"D:/ENT")   # <-- CHANGE ME if needed

# If you know where actual images live (optional):
# If None, we only summarize files under DATA_ROOT
IMAGES_ROOT = None  # e.g. Path(r"D:\ENT\images") or Path("/content/drive/MyDrive/...")

# For fold generation:
MAKE_FOLDS = False
N_FOLDS = 5
RANDOM_STATE = 42
OUT_SPLITS_DIR = DATA_ROOT / "_splits_patient_level"   # will be created


# =========================
# HELPERS
# =========================

def human_bytes(n: int) -> str:
    if n is None:
        return "NA"
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for u in units:
        if x < 1024 or u == units[-1]:
            return f"{x:.2f} {u}"
        x /= 1024.0

def safe_read_excel(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Excel not found: {path}")
    df = pd.read_csv(path)
    # Normalize column names
    df.columns = [str(c).strip() for c in df.columns]
    return df

def guess_columns(df: pd.DataFrame):
    cols = {c.lower(): c for c in df.columns}

    # patient id candidates
    pid_candidates = ["patient", "patient_id", "patientid", "pid", "case", "case_id"]
    pid_col = next((cols[c] for c in cols if any(k == c or k in c for k in pid_candidates)), None)

    # label candidates
    y_candidates = ["label", "y", "target", "class", "diagnosis", "benign_malignant", "malignant"]
    y_col = next((cols[c] for c in cols if any(k == c or k in c for k in y_candidates)), None)

    # images per patient candidates
    nimg_candidates = ["n_images", "num_images", "images", "image_count", "count"]
    nimg_col = next((cols[c] for c in cols if any(k == c or k in c for k in nimg_candidates)), None)

    # path candidates (if present)
    path_candidates = ["path", "filepath", "file", "image_path", "img_path", "folder"]
    path_col = next((cols[c] for c in cols if any(k == c or k in c for k in path_candidates)), None)

    return pid_col, y_col, nimg_col, path_col

def normalize_label(x):
    """
    Normalize label into {0,1} if possible:
      benign -> 0
      malignant -> 1
    If already numeric, keep.
    """
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        # assume 0/1 already
        return int(x)
    s = str(x).strip().lower()
    if s in ["benign", "b", "0", "false", "neg", "negative"]:
        return 0
    if s in ["malignant", "m", "1", "true", "pos", "positive", "cancer"]:
        return 1
    # try to extract 0/1
    m = re.fullmatch(r"[01]", s)
    if m:
        return int(s)
    return x  # leave as-is

def print_patient_stats(df: pd.DataFrame, pid_col: str, y_col: str, nimg_col: str):
    print("\n======================")
    print("PATIENT-LEVEL STATS")
    print("======================")

    if pid_col is None:
        raise ValueError("Could not detect a patient id column. Rename a column to 'Patient' or 'patient_id'.")

    df = df.copy()
    df[pid_col] = df[pid_col].astype(str).str.strip()

    n_pat = df[pid_col].nunique()
    print(f"Unique patients: {n_pat}")

    if y_col is not None:
        df["_y_norm"] = df[y_col].map(normalize_label)
        # If still non-numeric, just count unique values
        if pd.api.types.is_numeric_dtype(df["_y_norm"]):
            counts = df.groupby("_y_norm")[pid_col].nunique().to_dict()
            benign = counts.get(0, 0)
            malignant = counts.get(1, 0)
            print(f"Benign patients   (0): {benign}")
            print(f"Malignant patients(1): {malignant}")
            if (benign + malignant) > 0:
                print(f"Class ratio benign/malignant: {benign}/{malignant} = {benign/(malignant+1e-9):.2f}")
        else:
            counts = df.groupby(y_col)[pid_col].nunique().sort_values(ascending=False)
            print("\nLabel values (patient counts):")
            print(counts.to_string())
    else:
        print("No label column detected; skipping class balance.")

    if nimg_col is not None and pd.api.types.is_numeric_dtype(df[nimg_col]):
        per_pat = df.groupby(pid_col)[nimg_col].max()  # if repeated rows per patient, keep max
        desc = per_pat.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
        print("\nImages-per-patient summary:")
        print(desc.to_string())
        # Outliers
        top = per_pat.sort_values(ascending=False).head(10)
        print("\nTop 10 patients by image count:")
        print(top.to_string())
    else:
        print("No numeric image-count column detected; skipping images-per-patient stats.")

def scan_tree(root: Path, max_examples_per_ext=5):
    print("\n======================")
    print("FILES/FOLDERS INVENTORY")
    print("======================")

    if not root.exists():
        print(f"[WARN] DATA_ROOT does not exist: {root}")
        return

    n_files = 0
    n_dirs = 0
    total_bytes = 0

    ext_counter = Counter()
    ext_sizes = defaultdict(int)
    ext_examples = defaultdict(list)

    for p in root.rglob("*"):
        if p.is_dir():
            n_dirs += 1
            continue
        n_files += 1
        try:
            sz = p.stat().st_size
        except Exception:
            sz = 0
        total_bytes += sz

        ext = p.suffix.lower() if p.suffix else "<noext>"
        ext_counter[ext] += 1
        ext_sizes[ext] += sz
        if len(ext_examples[ext]) < max_examples_per_ext:
            ext_examples[ext].append(str(p.relative_to(root)))

    print(f"Root: {root}")
    print(f"Dirs : {n_dirs}")
    print(f"Files: {n_files}")
    print(f"Total size: {human_bytes(total_bytes)}")

    # top extensions
    print("\nTop file types by count:")
    for ext, c in ext_counter.most_common(15):
        print(f"  {ext:>8}  count={c:>7}  size={human_bytes(ext_sizes[ext])}")

    print("\nExample files (per extension):")
    for ext, _ in ext_counter.most_common(8):
        print(f"\n[{ext}]")
        for ex in ext_examples[ext]:
            print("  ", ex)

def map_patient_to_files(images_root: Path, patient_ids: list[str]):
    """
    Optional: if you have a folder of images and filenames contain patient ids,
    this gives you counts per patient. This is heuristic and depends on naming.
    """
    if images_root is None:
        return None
    if not images_root.exists():
        print(f"[WARN] IMAGES_ROOT does not exist: {images_root}")
        return None

    pid_set = set(patient_ids)
    counts = Counter()

    # heuristic: count any file whose name contains a patient id token
    for p in images_root.rglob("*"):
        if not p.is_file():
            continue
        name = p.stem
        for pid in pid_set:
            if pid in name:
                counts[pid] += 1
                break
    return counts

def make_stratified_group_folds(df: pd.DataFrame, pid_col: str, y_col: str, out_dir: Path, n_folds=5, seed=42):
    """
    Patient-level Stratified Group K-Fold:
      - groups = patient id
      - stratify by label at patient level
    Outputs:
      fold_i_train.csv / fold_i_val.csv with patient IDs
    """
    if y_col is None:
        raise ValueError("Need a label column to do stratified folds.")
    out_dir.mkdir(parents=True, exist_ok=True)

    df = df.copy()
    df[pid_col] = df[pid_col].astype(str).str.strip()
    df["_y"] = df[y_col].map(normalize_label)

    # reduce to one row per patient (patient-level label)
    pat = df.groupby(pid_col)["_y"].agg(lambda s: s.dropna().iloc[0] if len(s.dropna()) else np.nan).reset_index()
    pat = pat.dropna(subset=["_y"])
    pat["_y"] = pat["_y"].astype(int)

    # sklearn StratifiedGroupKFold if available
    try:
        from sklearn.model_selection import StratifiedGroupKFold
        sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        X = pat[pid_col].values
        y = pat["_y"].values
        groups = pat[pid_col].values

        folds = []
        for fold_idx, (tr, va) in enumerate(sgkf.split(X, y, groups=groups), start=1):
            tr_ids = pat.iloc[tr][pid_col].tolist()
            va_ids = pat.iloc[va][pid_col].tolist()

            pd.DataFrame({pid_col: tr_ids}).to_csv(out_dir / f"fold{fold_idx}_train_patients.csv", index=False)
            pd.DataFrame({pid_col: va_ids}).to_csv(out_dir / f"fold{fold_idx}_val_patients.csv", index=False)

            folds.append((len(tr_ids), len(va_ids), int((pat.iloc[va]["_y"]==1).sum())))
        print("\nSaved folds to:", out_dir)
        print("Fold summary: (n_train, n_val, n_val_malignant)")
        for i, t in enumerate(folds, start=1):
            print(f"  fold{i}: {t}")
    except ImportError:
        raise ImportError("scikit-learn not installed in this environment.")


# =========================
# MAIN
# =========================

def main():
    print("Excel:", EXCEL_PATH)
    df = safe_read_excel(EXCEL_PATH)

    print("\nColumns:")
    print(df.columns.tolist())

    pid_col, y_col, nimg_col, path_col = guess_columns(df)
    print("\nDetected columns:")
    print("  patient id:", pid_col)
    print("  label     :", y_col)
    print("  n_images  :", nimg_col)
    print("  path      :", path_col)

    print_patient_stats(df, pid_col, y_col, nimg_col)

    # Inventory your run/artifact folder (or any root you set)
    scan_tree(DATA_ROOT)

    # Optional: estimate image counts by filename pattern if you set IMAGES_ROOT
    if IMAGES_ROOT is not None:
        print("\n======================")
        print("HEURISTIC IMAGE COUNTS")
        print("======================")
        patient_ids = df[pid_col].astype(str).str.strip().unique().tolist()
        counts = map_patient_to_files(IMAGES_ROOT, patient_ids)
        if counts is not None:
            arr = np.array(list(counts.values()), dtype=int)
            print(f"Matched patients: {len(counts)}/{len(patient_ids)}")
            if len(arr):
                print("Image count stats (matched patients):")
                print(pd.Series(arr).describe().to_string())

    if MAKE_FOLDS:
        make_stratified_group_folds(df, pid_col, y_col, OUT_SPLITS_DIR, n_folds=N_FOLDS, seed=RANDOM_STATE)

if __name__ == "__main__":
    main()

Excel: D:\ENT\patients_list_clean.csv

Columns:
['Patient', 'Histopathology', 'Type of Lesion', 'Leukoplakia', 'Number of Images', 'Image Size']

Detected columns:
  patient id: Patient
  label     : Histopathology
  n_images  : Number of Images
  path      : Histopathology

PATIENT-LEVEL STATS
Unique patients: 210

Label values (patient counts):
Histopathology
Reinke’s edema          45
SCC                     30
Low grade dysplasia     24
Papilloma               22
Polyp                   20
High grade dysplasia    18
Hyperkeratosis          14
Carcinoma in situ       12
Cyst                     9
Squamous hyperplasia     5
Amyloidosis              3
Granuloma                3
Hemangioma               2
Bamboo nodes             1
Inflammation             1
Nodule                   1

Images-per-patient summary:
count    210.000000
mean      53.066667
std       31.603939
min        2.000000
50%       50.000000
75%       74.000000
90%       90.100000
95%       99.100000
99%      137.73

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

EXCEL_PATH = Path(r"D:\ENT\Patients_List_Updated_Final.xlsx")

df = pd.read_excel(EXCEL_PATH)
df.columns = [str(c).strip() for c in df.columns]

# --- CLEAN PATIENT COLUMN ---
df["Patient_raw"] = df["Patient"]
df["Patient"] = df["Patient"].astype(str).str.strip()

# Remove rows where Patient is NaN or becomes "nan"/"None"/empty
bad = df["Patient"].isin(["", "nan", "None", "NaT"]) | df["Patient"].isna()
print(f"Rows total: {len(df)}")
print(f"Rows with missing/invalid Patient: {bad.sum()}")
df = df.loc[~bad].copy()

# Ensure numeric image counts
df["Number of Images"] = pd.to_numeric(df["Number of Images"], errors="coerce")

print("\n======================")
print("PATIENT-LEVEL STATS (CLEANED)")
print("======================")
print("Unique patients:", df["Patient"].nunique())

# label distribution by patient
label_by_patient = (
    df.groupby("Patient")["Histopathology"]
      .agg(lambda s: s.dropna().iloc[0] if len(s.dropna()) else np.nan)
      .dropna()
)
print("\nLabel distribution (patients):")
print(label_by_patient.value_counts().to_string())

# images per patient (if one row per patient, this is direct; if repeated rows, use max/sum depending on meaning)
img_by_patient = (
    df.groupby("Patient")["Number of Images"]
      .max()
      .dropna()
)

print("\nImages-per-patient summary:")
print(img_by_patient.describe(percentiles=[0.5,0.75,0.9,0.95,0.99]).to_string())

print("\nTop 10 patients by image count:")
print(img_by_patient.sort_values(ascending=False).head(10).to_string())

# duplicates in Excel (more than one row per patient)
dup_pat = df["Patient"].value_counts()
dup_pat = dup_pat[dup_pat > 1]
print("\nPatients appearing >1 row in Excel:", len(dup_pat))
if len(dup_pat):
    print(dup_pat.head(20).to_string())

Rows total: 1048539
Rows with missing/invalid Patient: 1048329

PATIENT-LEVEL STATS (CLEANED)
Unique patients: 210

Label distribution (patients):
Histopathology
Reinke’s edema          45
SCC                     30
Low grade dysplasia     24
Papilloma               22
Polyp                   20
High grade dysplasia    18
Hyperkeratosis          14
Carcinoma in situ       12
Cyst                     9
Squamous hyperplasia     5
Granuloma                3
Amyloidosis              3
Hemangioma               2
Nodule                   1
Inflammation             1
Bamboo nodes             1

Images-per-patient summary:
count    210.000000
mean      53.066667
std       31.603939
min        2.000000
50%       50.000000
75%       74.000000
90%       90.100000
95%       99.100000
99%      137.730000
max      186.000000

Top 10 patients by image count:
Patient
Patient099    186.0
Patient105    146.0
Patient148    138.0
Patient100    135.0
Patient176    127.0
Patient170    125.0
Patient188    12

In [14]:
from pathlib import Path
from collections import Counter, defaultdict
import re
import numpy as np
import pandas as pd

IMAGES_ROOT = Path(r"D:\ENT")  # <-- CHANGE THIS to your actual images folder
EXCEL_PATH  = Path(r"D:\ENT\Patients_List_Updated_Final.xlsx")

df = pd.read_excel(EXCEL_PATH)
df["Patient"] = df["Patient"].astype(str).str.strip()
df = df[~df["Patient"].isin(["", "nan", "None", "NaT"])].copy()
patients = sorted(df["Patient"].unique().tolist())
patient_set = set(patients)

if not IMAGES_ROOT.exists():
    raise FileNotFoundError(f"IMAGES_ROOT not found: {IMAGES_ROOT}")

# Collect all files
all_files = [p for p in IMAGES_ROOT.rglob("*") if p.is_file()]
exts = Counter([p.suffix.lower() for p in all_files])

print("IMAGES_ROOT:", IMAGES_ROOT)
print("Total files:", len(all_files))
print("Top extensions:", exts.most_common(10))

# Pattern A: patient as folder name segment
counts_folder = Counter()
for p in all_files:
    parts = set(p.parts)
    hit = next((pid for pid in patient_set if pid in parts), None)
    if hit:
        counts_folder[hit] += 1

# Pattern B: patient in filename
counts_name = Counter()
for p in all_files:
    name = p.stem
    hit = next((pid for pid in patient_set if pid in name), None)
    if hit:
        counts_name[hit] += 1

# Choose best mapping automatically (whichever matches more patients)
map_used = counts_folder if len(counts_folder) >= len(counts_name) else counts_name
method = "folder-segment" if map_used is counts_folder else "filename-substring"

print("\nMapping method used:", method)
print("Matched patients:", len(map_used), "/", len(patients))

missing_patients = [pid for pid in patients if pid not in map_used]
print("Patients with ZERO matched files:", len(missing_patients))
print("Example missing (first 30):", missing_patients[:30])

arr = np.array(list(map_used.values()), dtype=int)
if len(arr):
    print("\nImages-per-patient stats (from filesystem):")
    print(pd.Series(arr).describe(percentiles=[0.5,0.75,0.9,0.95,0.99]).to_string())

    top = map_used.most_common(15)
    print("\nTop 15 patients by file count:")
    for pid, c in top:
        print(f"  {pid}: {c}")

IMAGES_ROOT: D:\ENT
Total files: 11298
Top extensions: [('.jpg', 11157), ('.csv', 62), ('.pt', 36), ('.json', 32), ('.txt', 9), ('.ipynb', 1), ('.xlsx', 1)]

Mapping method used: folder-segment
Matched patients: 210 / 210
Patients with ZERO matched files: 0
Example missing (first 30): []

Images-per-patient stats (from filesystem):
count    210.000000
mean      53.128571
std       31.611996
min        2.000000
50%       50.000000
75%       74.000000
90%       90.100000
95%       99.100000
99%      137.730000
max      186.000000

Top 15 patients by file count:
  Patient099: 186
  Patient105: 146
  Patient148: 138
  Patient100: 135
  Patient176: 127
  Patient170: 125
  Patient188: 125
  Patient153: 119
  Patient098: 115
  Patient108: 110
  Patient141: 100
  Patient194: 98
  Patient178: 97
  Patient136: 96
  Patient122: 95


In [15]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd
import numpy as np
from pathlib import Path

CSV_PATH = r"D:\ENT\patients_list_clean.csv"
OUT_DIR = Path(r"D:\ENT\patient_folds")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH)
df["Patient"] = df["Patient"].astype(str).str.strip()

# Binary label for screening task
df["label"] = df["Histopathology"].apply(
    lambda x: 1 if str(x).strip().lower() == "scc" else 0
)

# Patient-level reduction
pat = df.groupby("Patient")["label"].max().reset_index()

X = pat["Patient"].values
y = pat["label"].values
groups = pat["Patient"].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr, val) in enumerate(sgkf.split(X, y, groups), start=1):
    tr_ids = pat.iloc[tr]["Patient"]
    val_ids = pat.iloc[val]["Patient"]

    pd.DataFrame({"Patient": tr_ids}).to_csv(OUT_DIR / f"fold{fold}_train.csv", index=False)
    pd.DataFrame({"Patient": val_ids}).to_csv(OUT_DIR / f"fold{fold}_val.csv", index=False)

print("Folds saved to:", OUT_DIR)

Folds saved to: D:\ENT\patient_folds


In [17]:
pat_counts = df.groupby("Patient")["Number of Images"].max()

high = pat_counts[pat_counts > 75].index
low  = pat_counts[pat_counts <= 50].index

print("High-volume patients:", len(high))
print("Low-volume patients :", len(low))

High-volume patients: 51
Low-volume patients : 107


In [19]:
import numpy as np

def cam_entropy(cam):
    p = cam.flatten()
    p = p / (p.sum() + 1e-8)
    return -np.sum(p * np.log(p + 1e-8))

In [20]:
# ============================================================
# FINAL PIPELINE FOR TASK B (Explainability) + TASK C (Domain Shift)
# Works with your patient-level ENT dataset + your trained checkpoints
# ============================================================

import os, re, json, math, time, glob, random
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, balanced_accuracy_score, roc_curve
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
from PIL import Image

# ----------------------------
# CONFIG
# ----------------------------
@dataclass
class CFG:
    # Paths
    IMAGES_ROOT: Path = Path(r"D:\ENT")   # folder that contains PatientXXX subfolders/images
    RUNS_DIR: Path   = Path(r"D:\ENT\_challenge2_artifacts\_runs")
    META_XLSX: Path  = Path(r"D:\ENT\Patients_List_Updated_Final.xlsx")  # or CSV path
    META_CSV_OUT: Path = Path(r"D:\ENT\_challenge2_artifacts\patients_meta.xlsx")

    # Labels
    POS_LABEL: str = "Malignant"  # from Type of Lesion
    TYPE_COL: str = "Type of Lesion"
    HISTO_COL: str = "Histopathology"
    LEUKO_COL: str = "Leukoplakia"
    PID_COL: str = "Patient"

    # Inference / evaluation
    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    IMG_EXTS: tuple = (".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp")
    TTA_TEST: bool = True

    # Explainability (deletion)
    DELETE_FRAC: float = 0.20   # remove top 20% highest-attention images
    MIN_KEEP: int = 5           # never reduce below this many images

    # Calibration
    ECE_BINS: int = 15

    # Bootstrapping
    BOOT: int = 2000
    BOOT_SEED: int = 123

cfg = CFG()

cfg.RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("DEVICE:", cfg.DEVICE)
print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
print("RUNS_DIR:", cfg.RUNS_DIR)

# ----------------------------
# UTIL: load metadata (Excel OR CSV)
# ----------------------------
def load_metadata_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        df = pd.read_excel(path)
    # normalize columns (strip)
    df.columns = [str(c).strip() for c in df.columns]
    return df

def clean_patient_id(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    # normalize like Patient001
    s = re.sub(r"\s+", "", s)
    return s

def normalize_leuko(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s in ["1","yes","y","true","t","pos","positive"]:
        return 1
    if s in ["0","no","n","false","f","neg","negative"]:
        return 0
    return None

def type_to_binary(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if "malig" in s:
        return 1
    if "benign" in s:
        return 0
    return None

# ----------------------------
# UTIL: filesystem inventory + patient->images mapping
# folder-segment method: ...\Patient123\*.jpg
# ----------------------------
def index_images(images_root: Path, exts=cfg.IMG_EXTS):
    images_root = Path(images_root)
    all_files = []
    for ext in exts:
        all_files.extend(images_root.rglob(f"*{ext}"))
    all_files = [p for p in all_files if p.is_file()]
    return all_files

def patient_from_path(p: Path):
    # look for segment like Patient### in path parts
    parts = [str(x) for x in p.parts]
    for seg in parts[::-1]:
        if re.match(r"^Patient\d+$", seg):
            return seg
    return None

def build_patient_image_map(images_root: Path):
    files = index_images(images_root)
    mp = {}
    for p in files:
        pid = patient_from_path(p)
        if pid is None:
            continue
        mp.setdefault(pid, []).append(p)
    # sort for deterministic order
    for k in mp:
        mp[k] = sorted(mp[k])
    return mp, files

# ----------------------------
# METRICS
# ----------------------------
def youden_threshold(y_true, prob):
    fpr, tpr, thr = roc_curve(y_true, prob)
    j = tpr - fpr
    idx = int(np.argmax(j))
    return thr[idx]

def ece_score(y_true, prob, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (prob >= lo) & (prob < hi) if i < n_bins-1 else (prob >= lo) & (prob <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = prob[m].mean()
        ece += (m.mean()) * abs(acc - conf)
    return float(ece)

def bootstrap_ci(y_true, prob, fn, n=2000, seed=123):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    prob = np.asarray(prob)
    N = len(y_true)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, N, size=N)
        try:
            vals.append(fn(y_true[idx], prob[idx]))
        except Exception:
            continue
    vals = np.asarray(vals, dtype=float)
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def compute_report(y_true, prob):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    auc = roc_auc_score(y_true, prob)
    thr = youden_threshold(y_true, prob)
    pred = (prob >= thr).astype(int)
    bacc = balanced_accuracy_score(y_true, pred)
    ece = ece_score(y_true, prob, cfg.ECE_BINS)

    # sens/spec
    tp = ((pred==1) & (y_true==1)).sum()
    tn = ((pred==0) & (y_true==0)).sum()
    fp = ((pred==1) & (y_true==0)).sum()
    fn = ((pred==0) & (y_true==1)).sum()
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)

    return {
        "AUC": float(auc),
        "BAcc": float(bacc),
        "ECE": float(ece),
        "YoudenThr": float(thr),
        "Sens": float(sens),
        "Spec": float(spec),
    }

# ----------------------------
# MODEL INFERENCE (drop-in)
# IMPORTANT:
# If you already have your own image transforms/model forward,
# replace predict_patient_with_gates() with your existing function.
# ----------------------------
def default_preprocess(img: Image.Image, size=224):
    # simple ImageNet-ish preprocessing; replace with your training preprocessing if needed
    img = img.convert("RGB").resize((size, size))
    x = np.asarray(img).astype(np.float32) / 255.0
    x = (x - np.array([0.485,0.456,0.406])) / np.array([0.229,0.224,0.225])
    x = np.transpose(x, (2,0,1))
    return torch.tensor(x, dtype=torch.float32)

@torch.no_grad()
def predict_patient_with_gates(model, image_paths, device, tta=True):
    """
    Returns:
      logit (float) patient-level logit
      prob  (float) patient-level prob
      gates (np.ndarray) per-image attention weights (sum to 1)
    Requirements:
      model should return (logit, gates) OR dict containing them.
    If your model API differs, adapt here.
    """
    model.eval()
    xs = []
    for p in image_paths:
        img = Image.open(p)
        x = default_preprocess(img)  # [3,H,W]
        xs.append(x)
        if tta:
            # simple horizontal flip TTA
            xf = torch.flip(x, dims=[2])
            xs.append(xf)
    X = torch.stack(xs, dim=0).to(device)  # [N,3,H,W]

    out = model(X)  # adapt if needed

    # Handle different possible outputs
    if isinstance(out, dict):
        logit = out.get("logit", None)
        gates = out.get("gates", None)
    elif isinstance(out, (tuple, list)) and len(out) >= 2:
        logit, gates = out[0], out[1]
    else:
        raise RuntimeError("Model output format not recognized. Adjust predict_patient_with_gates().")

    # logit could be [1] or scalar tensor
    logit = float(torch.squeeze(logit).detach().cpu().item())

    # gates expected per-image; if TTA doubled N, you can collapse by averaging every 2
    gates = torch.squeeze(gates).detach().cpu().float().numpy()
    gates = np.asarray(gates, dtype=float)

    # If TTA doubled, collapse pairs (original, flip)
    if tta and len(gates) == 2*len(image_paths):
        gates = 0.5*(gates[0::2] + gates[1::2])

    # normalize to sum to 1
    gates = np.clip(gates, 1e-12, None)
    gates = gates / gates.sum()

    prob = float(1.0 / (1.0 + math.exp(-logit)))
    return logit, prob, gates

# ----------------------------
# LOAD YOUR MODEL CHECKPOINT (adapt)
# ----------------------------
def load_fold_model(ckpt_path: Path, device: str):
    """
    Expected ckpt format: torch.save({'model': state_dict, ...})
    If your ckpt saves full model, adapt here.
    """
    ck = torch.load(str(ckpt_path), map_location=device, weights_only=False)

    # ---- IMPORTANT: you must import/define your model class before calling this ----
    # from your_code import GatedMILv2
    model = GatedMILv2(use_aux_leuko=False, gate_tau=2.0).to(device)

    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    # tolerate key mismatch (leuko head etc.)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if len(unexpected) > 0:
        print("[WARN] unexpected keys:", unexpected[:8], ("..." if len(unexpected)>8 else ""))
    if len(missing) > 0:
        print("[WARN] missing keys:", missing[:8], ("..." if len(missing)>8 else ""))
    model.eval()
    return model

# ----------------------------
# BUILD CANONICAL PATIENT META (from Excel/CSV + filesystem)
# ----------------------------
def build_canonical_meta(meta_path: Path, images_root: Path) -> pd.DataFrame:
    df = load_metadata_table(meta_path)

    # standardize columns (use your known names)
    need = [cfg.PID_COL, cfg.TYPE_COL, cfg.HISTO_COL, cfg.LEUKO_COL]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing column in metadata: {c}. Found: {df.columns.tolist()}")

    df[cfg.PID_COL] = df[cfg.PID_COL].apply(clean_patient_id)
    df = df.dropna(subset=[cfg.PID_COL]).copy()

    df["y_bin"] = df[cfg.TYPE_COL].apply(type_to_binary)
    df["leuko_bin"] = df[cfg.LEUKO_COL].apply(normalize_leuko)

    # filesystem map
    p2imgs, all_files = build_patient_image_map(images_root)
    df["fs_n_images"] = df[cfg.PID_COL].map(lambda pid: len(p2imgs.get(pid, [])))
    df["has_files"] = df["fs_n_images"].fillna(0).astype(int) > 0

    # keep only patients that have images
    df = df[df["has_files"]].copy()

    # canonical columns
    keep = [
        cfg.PID_COL,
        cfg.TYPE_COL,
        cfg.HISTO_COL,
        cfg.LEUKO_COL,
        "y_bin",
        "leuko_bin",
        "fs_n_images"
    ]
    df = df[keep].reset_index(drop=True)

    # save canonical csv for reproducibility
    cfg.META_CSV_OUT.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(cfg.META_CSV_OUT, index=False)
    print("Saved canonical meta:", cfg.META_CSV_OUT)
    print("Patients:", len(df), "| malignant:", int(df["y_bin"].sum()), "| benign:", int((df["y_bin"]==0).sum()))
    return df, p2imgs

# ----------------------------
# TASK C: DOMAIN SPLITS (patient-level)
# ----------------------------
def domain_split_leukoplakia(df):
    a = df[df["leuko_bin"] == 0].copy()
    b = df[df["leuko_bin"] == 1].copy()
    return a, b

def domain_split_early_vs_scc(df):
    # define early = HGD + CIS ; advanced = SCC
    hist = df[cfg.HISTO_COL].astype(str).str.lower()

    is_scc = hist.str.contains("scc")
    is_cis = hist.str.contains("carcinoma in situ") | hist.str.contains("cis")
    is_hgd = hist.str.contains("high grade dysplasia") | hist.str.contains("high-grade dysplasia")

    early = df[is_cis | is_hgd].copy()
    scc = df[is_scc].copy()
    return scc, early

def domain_split_volume(df, q=0.5):
    # split by median by default
    thr = float(df["fs_n_images"].quantile(q))
    hi = df[df["fs_n_images"] > thr].copy()
    lo = df[df["fs_n_images"] <= thr].copy()
    return hi, lo, thr

# ----------------------------
# INFERENCE OVER A SET OF PATIENTS (using fold ensemble)
# ----------------------------
def infer_patients_with_folds(patients_df, p2imgs, ckpt_paths, device):
    """
    ckpt_paths: list of fold checkpoint paths
    returns dataframe with Patient, y_true, prob_mean, logit_mean, (optional) gates stats
    """
    preds = []
    models = [load_fold_model(p, device) for p in ckpt_paths]

    for i, row in patients_df.iterrows():
        pid = row[cfg.PID_COL]
        y = int(row["y_bin"])
        imgs = p2imgs.get(pid, [])
        if len(imgs) == 0:
            continue

        fold_probs = []
        fold_logits = []
        fold_ent = []
        fold_topk = []

        for m in models:
            logit, prob, gates = predict_patient_with_gates(m, imgs, device, tta=cfg.TTA_TEST)
            fold_probs.append(prob)
            fold_logits.append(logit)

            # explainability stats at patient-level
            ent = float(-(gates * np.log(gates + 1e-12)).sum())              # entropy
            topk = float(np.sort(gates)[-max(1, int(0.1*len(gates))) :].sum())  # top10% mass
            fold_ent.append(ent)
            fold_topk.append(topk)

        prob_mean = float(np.mean(fold_probs))
        logit_mean = float(np.mean(fold_logits))

        preds.append({
            "Patient": pid,
            "y_true": y,
            "prob_mean": prob_mean,
            "logit_mean": logit_mean,
            "gate_entropy_mean": float(np.mean(fold_ent)),
            "gate_top10_mass_mean": float(np.mean(fold_topk)),
            "n_images": int(len(imgs)),
        })

    return pd.DataFrame(preds)

# ----------------------------
# TASK B: FAITHFULNESS (Deletion)
# ----------------------------
def deletion_test(patients_df, p2imgs, ckpt_paths, device, delete_frac=0.2, min_keep=5):
    """
    1) run model once -> get gates
    2) remove top-attended images
    3) run again -> get prob drop
    """
    models = [load_fold_model(p, device) for p in ckpt_paths]
    rows = []

    for _, row in patients_df.iterrows():
        pid = row[cfg.PID_COL]
        imgs = p2imgs.get(pid, [])
        if len(imgs) < (min_keep + 2):
            continue

        # baseline ensemble prob + average gates ranking from folds
        all_gate_scores = np.zeros(len(imgs), dtype=float)
        base_probs = []

        for m in models:
            _, prob, gates = predict_patient_with_gates(m, imgs, device, tta=cfg.TTA_TEST)
            base_probs.append(prob)
            all_gate_scores += gates

        all_gate_scores /= max(1, len(models))
        base_prob = float(np.mean(base_probs))

        # delete top fraction
        n_del = int(round(delete_frac * len(imgs)))
        n_del = max(1, n_del)
        keep_n = max(min_keep, len(imgs) - n_del)
        idx_sorted = np.argsort(-all_gate_scores)  # descending
        keep_idx = np.sort(idx_sorted[:keep_n])
        imgs_kept = [imgs[i] for i in keep_idx.tolist()]

        # re-run ensemble
        del_probs = []
        for m in models:
            _, prob2, _ = predict_patient_with_gates(m, imgs_kept, device, tta=cfg.TTA_TEST)
            del_probs.append(prob2)
        del_prob = float(np.mean(del_probs))

        rows.append({
            "Patient": pid,
            "y_true": int(row["y_bin"]),
            "base_prob": base_prob,
            "deleted_prob": del_prob,
            "prob_drop": float(base_prob - del_prob),
            "n_images": int(len(imgs)),
            "n_kept": int(len(imgs_kept))
        })

    out = pd.DataFrame(rows)
    return out

# ----------------------------
# MAIN: run Task C and Task B
# ----------------------------
def run_all(seed: int, benchmark="res_shift"):
    # 1) canonical meta + image map
    df, p2imgs = build_canonical_meta(cfg.META_XLSX, cfg.IMAGES_ROOT)

    # 2) collect fold checkpoints (seed-specific)
    # You saved folds as: cv_best_res_shift_seed43_fold{f}.pt (adjust if yours differs)
    ckpts = []
    for f in [1,2,3,4,5]:
        p = cfg.RUNS_DIR / f"cv_best_{benchmark}_seed{seed}_fold{f}.pt"
        if p.exists():
            ckpts.append(p)
    if len(ckpts) == 0:
        # fallback: if you only kept best per-seed model and still saved fold preds, you must keep fold ckpts to run deletion.
        raise FileNotFoundError(f"No fold checkpoints found for seed={seed} in {cfg.RUNS_DIR}")

    print("Using fold ckpts:")
    for p in ckpts:
        print(" ", p.name)

    # ---------------- Task C: Domain shift ----------------
    results = []

    # C1 leukoplakia shift
    tr, te = domain_split_leukoplakia(df)
    if len(tr) > 0 and len(te) > 0:
        pred_te = infer_patients_with_folds(te, p2imgs, ckpts, cfg.DEVICE)
        rep = compute_report(pred_te["y_true"], pred_te["prob_mean"])
        rep["Experiment"] = "C1_leuko_train0_test1"
        rep["N_test"] = len(pred_te)
        rep["AUC_CI"] = bootstrap_ci(pred_te["y_true"], pred_te["prob_mean"], roc_auc_score, cfg.BOOT, cfg.BOOT_SEED)
        results.append(rep)

    # C2 early vs SCC shift (train SCC, test early)
    scc, early = domain_split_early_vs_scc(df)
    if len(scc) > 0 and len(early) > 0:
        pred_te = infer_patients_with_folds(early, p2imgs, ckpts, cfg.DEVICE)
        rep = compute_report(pred_te["y_true"], pred_te["prob_mean"])
        rep["Experiment"] = "C2_trainSCC_testEarly"
        rep["N_test"] = len(pred_te)
        rep["AUC_CI"] = bootstrap_ci(pred_te["y_true"], pred_te["prob_mean"], roc_auc_score, cfg.BOOT, cfg.BOOT_SEED)
        results.append(rep)

    # C3 volume shift
    hi, lo, thr = domain_split_volume(df, q=0.5)
    if len(hi) > 0 and len(lo) > 0:
        pred_te = infer_patients_with_folds(lo, p2imgs, ckpts, cfg.DEVICE)
        rep = compute_report(pred_te["y_true"], pred_te["prob_mean"])
        rep["Experiment"] = f"C3_trainHighVol_testLowVol(thr={thr:.1f})"
        rep["N_test"] = len(pred_te)
        rep["AUC_CI"] = bootstrap_ci(pred_te["y_true"], pred_te["prob_mean"], roc_auc_score, cfg.BOOT, cfg.BOOT_SEED)
        results.append(rep)

    res_df = pd.DataFrame(results)
    out_res = cfg.RUNS_DIR / f"TASKC_domain_shift_seed{seed}_{benchmark}.csv"
    res_df.to_csv(out_res, index=False)
    print("Saved Task C results:", out_res)

    # ---------------- Task B: Explainability ----------------
    # B1: gate statistics on full test pool (all patients)
    pred_all = infer_patients_with_folds(df, p2imgs, ckpts, cfg.DEVICE)
    out_gate = cfg.RUNS_DIR / f"TASKB_gate_stats_seed{seed}_{benchmark}.csv"
    pred_all.to_csv(out_gate, index=False)
    print("Saved Task B gate stats:", out_gate)

    # B2: deletion faithfulness (subset; can be slow)
    del_df = deletion_test(df, p2imgs, ckpts, cfg.DEVICE, delete_frac=cfg.DELETE_FRAC, min_keep=cfg.MIN_KEEP)
    out_del = cfg.RUNS_DIR / f"TASKB_deletion_seed{seed}_{benchmark}_del{int(cfg.DELETE_FRAC*100)}.csv"
    del_df.to_csv(out_del, index=False)
    print("Saved Task B deletion:", out_del)

    # Summaries for paper
    if len(del_df) > 0:
        summary = {
            "mean_prob_drop": float(del_df["prob_drop"].mean()),
            "median_prob_drop": float(del_df["prob_drop"].median()),
            "mean_prob_drop_malignant": float(del_df[del_df["y_true"]==1]["prob_drop"].mean()) if (del_df["y_true"]==1).any() else None,
            "mean_prob_drop_benign": float(del_df[del_df["y_true"]==0]["prob_drop"].mean()) if (del_df["y_true"]==0).any() else None,
            "N": int(len(del_df)),
        }
        out_sum = cfg.RUNS_DIR / f"TASKB_deletion_summary_seed{seed}_{benchmark}.json"
        with open(out_sum, "w") as f:
            json.dump(summary, f, indent=2)
        print("Saved deletion summary:", out_sum)

    return res_df, pred_all, del_df

# ============================================================
# RUN (choose the seed + benchmark you finalized)
# ============================================================
# Example:
# res_df, gate_df, del_df = run_all(seed=43, benchmark="res_shift")

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs


In [3]:
seed = 43
benchmark = "res_shift"

for f in [1,2,3,4,5]:
    p = cfg.RUNS_DIR / f"cv_best_{benchmark}_seed{seed}_fold{f}.pt"
    print(f"fold{f}:", "OK" if p.exists() else "MISSING", "->", p)

fold1: OK -> D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold1.pt
fold2: OK -> D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold2.pt
fold3: OK -> D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold3.pt
fold4: OK -> D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold4.pt
fold5: OK -> D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold5.pt


In [8]:
import torch
import torch.nn as nn
from torchvision import models

class DualBranchGatedMIL(nn.Module):
    def __init__(self, gate_tau=2.0):
        super().__init__()
        self.gate_tau = gate_tau

        # Encoder
        base = models.resnet50(weights=None)
        self.encoder = nn.Module()
        self.encoder.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 2048

        # Gate
        self.gate = nn.Sequential(
            nn.Linear(self.feat_dim, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )

        # Pool branches
        self.pool_non = nn.Identity()
        self.pool_leu = nn.Identity()

        # Classifiers
        self.cls_non = nn.Linear(self.feat_dim, 1)
        self.cls_leu = nn.Linear(self.feat_dim, 1)

        # Auxiliary leuko head
        self.leuko_head = nn.Sequential(
            nn.Linear(self.feat_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1)
        )

    def forward(self, x, leuko_flag=None):
        # x: [N,3,H,W]
        f = self.encoder.backbone(x)
        f = f.flatten(1)  # [N,2048]

        g = self.gate(f).squeeze(1)
        g = g / max(self.gate_tau, 1e-6)
        w = torch.softmax(g, dim=0)

        bag = (w.unsqueeze(1) * f).sum(0, keepdim=True)

        if leuko_flag is None:
            logit = self.cls_non(bag)
        else:
            if leuko_flag == 0:
                logit = self.cls_non(bag)
            else:
                logit = self.cls_leu(bag)

        logit_leu = self.leuko_head(bag)

        return logit.squeeze(1), w, logit_leu.squeeze(1)

In [11]:
import torch
import torch.nn as nn
from torchvision import models

class DualBranchGatedMIL(nn.Module):
    def __init__(self, gate_tau=2.0):
        super().__init__()
        self.gate_tau = gate_tau

        # Backbone
        base = models.resnet50(weights=None)
        self.encoder = nn.Module()
        self.encoder.backbone = nn.Sequential(*list(base.children())[:-1])

        # Projection to 4096 (CRITICAL FIX)
        self.encoder.proj = nn.Linear(2048, 4096)

        self.feat_dim = 4096

        # Gate
        self.gate = nn.Sequential(
            nn.Linear(self.feat_dim, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )

        # Classifiers
        self.cls_non = nn.Linear(self.feat_dim, 1)
        self.cls_leu = nn.Linear(self.feat_dim, 1)

        # Auxiliary leuko head (MATCH CHECKPOINT)
        self.leuko_head = nn.Sequential(
            nn.Linear(self.feat_dim, 256),  # matches [256, 4096]
            nn.ReLU(inplace=True),
            nn.Linear(256, 1)
        )

    def forward(self, x, leuko_flag=None):
        f = self.encoder.backbone(x)
        f = f.flatten(1)
        f = self.encoder.proj(f)  # project to 4096

        g = self.gate(f).squeeze(1)
        g = g / max(self.gate_tau, 1e-6)
        w = torch.softmax(g, dim=0)

        bag = (w.unsqueeze(1) * f).sum(0, keepdim=True)

        if leuko_flag is None:
            logit = self.cls_non(bag)
        else:
            logit = self.cls_leu(bag) if leuko_flag else self.cls_non(bag)

        logit_leu = self.leuko_head(bag)

        return logit.squeeze(1), w, logit_leu.squeeze(1)

In [16]:
import re
import torch
import torch.nn as nn
from torchvision import models


# -------------------------
# Attention pooling: Linear -> Tanh -> Linear
# -------------------------
class AttnPool(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, feats: torch.Tensor):
        # feats: [N, D]
        a = self.net(feats).squeeze(1)     # [N]
        w = torch.softmax(a, dim=0)        # [N]
        bag = (w.unsqueeze(1) * feats).sum(0, keepdim=True)  # [1, D]
        return bag, w


# -------------------------
# Model that matches your checkpoint behavior
# - instance encoder: ResNet50 -> 2048
# - pool_non / pool_leu: attention pooling on 2048
# - cls_non/cls_leu: linear on 2048
# - gate: linear on concat(bag_non, bag_leu) => 4096
# - leuko_head: MLP on concat(bag_non, bag_leu) => 4096
# -------------------------
class DualBranchGatedMIL(nn.Module):
    def __init__(
        self,
        instance_dim=2048,
        pool_hidden=256,
        gate_tau=2.0,
        gate_in_dim=4096,
        leuko_hidden=256,
        leuko_in_dim=4096,
    ):
        super().__init__()
        self.gate_tau = float(gate_tau)

        base = models.resnet50(weights=None)
        self.encoder = nn.Module()
        self.encoder.backbone = nn.Sequential(*list(base.children())[:-1])  # [B,2048,1,1]

        self.pool_non = AttnPool(in_dim=instance_dim, hidden_dim=pool_hidden)
        self.pool_leu = AttnPool(in_dim=instance_dim, hidden_dim=pool_hidden)

        self.cls_non = nn.Linear(instance_dim, 1)
        self.cls_leu = nn.Linear(instance_dim, 1)

        # IMPORTANT: gate is on concat of pooled features (checkpoint shows 4096)
        self.gate = nn.Linear(gate_in_dim, 1)

        # leuko head on concat (checkpoint shows 4096 -> 256 -> 1)
        self.leuko_head = nn.Sequential(
            nn.Linear(leuko_in_dim, leuko_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(leuko_hidden, 1),
        )

    def encode_instances(self, x: torch.Tensor) -> torch.Tensor:
        f = self.encoder.backbone(x).flatten(1)  # [N, instance_dim]
        return f

    def forward(self, x: torch.Tensor):
        feats = self.encode_instances(x)                    # [N, D]
        bag_non, attn_non = self.pool_non(feats)            # [1, D]
        bag_leu, attn_leu = self.pool_leu(feats)            # [1, D]

        logit_non = self.cls_non(bag_non)                   # [1, 1]
        logit_leu = self.cls_leu(bag_leu)                   # [1, 1]

        concat = torch.cat([bag_non, bag_leu], dim=1)        # [1, 2D] usually 4096
        alpha = torch.sigmoid(self.gate(concat) / max(self.gate_tau, 1e-6))  # [1,1]

        logit = (1 - alpha) * logit_non + alpha * logit_leu  # [1,1]

        leuko_logit = self.leuko_head(concat)               # [1,1]

        return logit.squeeze(), alpha.squeeze(), attn_non, attn_leu, leuko_logit.squeeze()


# -------------------------
# Robust inference helpers
# -------------------------
def _infer_pool_hidden_from_sd(sd: dict) -> int:
    # Find a 2D weight under pool_non that isn't the final [1, hidden] layer.
    candidates = []
    for k, v in sd.items():
        if not k.startswith("pool_non"):
            continue
        if torch.is_tensor(v) and v.ndim == 2:
            candidates.append((k, v.shape))
    if not candidates:
        raise KeyError("Could not infer pool_hidden: no 2D weight found under 'pool_non*'.")

    non_final = [c for c in candidates if c[1][0] != 1]
    key, shape = (non_final[0] if non_final else candidates[0])
    return int(shape[0])


def _remap_pool_keys(sd: dict, model_keys: set) -> dict:
    """
    Remap checkpoint pool keys to model's pool_* .net.* keys.
    """
    out = {}
    for k, v in sd.items():
        # Already matches
        if k in model_keys:
            out[k] = v
            continue

        # pool_non.X -> pool_non.net.X
        if k.startswith("pool_non.") and ("pool_non.net." + k[len("pool_non."):]) in model_keys:
            out["pool_non.net." + k[len("pool_non."):]] = v
            continue

        # pool_leu.X -> pool_leu.net.X
        if k.startswith("pool_leu.") and ("pool_leu.net." + k[len("pool_leu."):]) in model_keys:
            out["pool_leu.net." + k[len("pool_leu."):]] = v
            continue

        # pool_non.attn.0.weight -> pool_non.net.0.weight, etc.
        m = re.match(r"^(pool_(non|leu))\.(attn|net)\.(.+)$", k)
        if m:
            pref = m.group(1)
            tail = m.group(4)
            cand = f"{pref}.net.{tail}"
            if cand in model_keys:
                out[cand] = v
                continue

        # fc1/fc2 -> net.0/net.2
        m = re.match(r"^(pool_(non|leu))\.(fc1|fc2)\.(weight|bias)$", k)
        if m:
            pref = m.group(1)
            which = m.group(3)
            wb = m.group(4)
            idx = "0" if which == "fc1" else "2"
            cand = f"{pref}.net.{idx}.{wb}"
            if cand in model_keys:
                out[cand] = v
                continue

        # keep original if no mapping rule hit
        out[k] = v

    return out


def build_model_from_ckpt(ckpt_path: str, device="cuda:0", gate_tau=2.0):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    sd = ckpt["model"]

    # infer dims directly from checkpoint tensors
    instance_dim = int(sd["cls_non.weight"].shape[1])           # expected 2048
    gate_in_dim  = int(sd["gate.weight"].shape[1])              # your ckpt: 4096
    leuko_in_dim = int(sd["leuko_head.0.weight"].shape[1])      # expected 4096
    leuko_hidden = int(sd["leuko_head.0.weight"].shape[0])      # expected 256
    pool_hidden  = _infer_pool_hidden_from_sd(sd)

    model = DualBranchGatedMIL(
        instance_dim=instance_dim,
        pool_hidden=pool_hidden,
        gate_tau=gate_tau,
        gate_in_dim=gate_in_dim,
        leuko_hidden=leuko_hidden,
        leuko_in_dim=leuko_in_dim,
    ).to(device)

    # remap pool keys if needed
    model_keys = set(model.state_dict().keys())
    sd2 = _remap_pool_keys(sd, model_keys)

    missing, unexpected = model.load_state_dict(sd2, strict=False)
    return model, ckpt, missing, unexpected


# -------------------------
# Usage
# -------------------------
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
CKPT_PATH = r"D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold1.pt"

model, ckpt, missing, unexpected = build_model_from_ckpt(CKPT_PATH, device=DEVICE, gate_tau=2.0)

print("Loaded on:", DEVICE)
print("Missing:", len(missing))
print("Unexpected:", len(unexpected))
print("Missing sample:", missing[:20])
print("Unexpected sample:", unexpected[:20])

# sanity dims
print("instance_dim:", model.cls_non.in_features)
print("pool_hidden:", model.pool_non.net[0].out_features)
print("gate_in_dim:", model.gate.in_features)
print("leuko_in_dim:", model.leuko_head[0].in_features)

model.eval()

Loaded on: cuda:0
Missing: 10
Unexpected: 10
Missing sample: ['pool_non.net.0.weight', 'pool_non.net.0.bias', 'pool_non.net.2.weight', 'pool_non.net.2.bias', 'pool_leu.net.0.weight', 'pool_leu.net.0.bias', 'pool_leu.net.2.weight', 'pool_leu.net.2.bias', 'leuko_head.2.weight', 'leuko_head.2.bias']
Unexpected sample: ['pool_non.V.weight', 'pool_non.V.bias', 'pool_non.w.weight', 'pool_non.w.bias', 'pool_leu.V.weight', 'pool_leu.V.bias', 'pool_leu.w.weight', 'pool_leu.w.bias', 'leuko_head.3.weight', 'leuko_head.3.bias']
instance_dim: 2048
pool_hidden: 256
gate_in_dim: 4096
leuko_in_dim: 4096


DualBranchGatedMIL(
  (encoder): Module(
    (backbone): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)

In [28]:
import os
import re
import json
import time
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import roc_auc_score
from tqdm import tqdm


# -----------------------------
# Utils
# -----------------------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def safe_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))

def sigmoid(x: float) -> float:
    return float(1.0 / (1.0 + np.exp(-x)))

def to_binary_label(series: pd.Series) -> pd.Series:
    """
    Convert common label strings to binary if needed.
    You can customize this mapping for your dataset.
    """
    if series.dtype != object:
        return series

    s = series.astype(str).str.strip().str.lower()

    # Example mapping (adjust to your challenge):
    # malignant: scc/cis/high-grade dysplasia
    malignant = {"scc", "carcinoma in situ", "high grade dysplasia", "high-grade dysplasia"}
    y = s.apply(lambda x: 1 if x in malignant else 0)
    return y


# -----------------------------
# Config
# -----------------------------
@dataclass
class CFG:
    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"

    BENCHMARK: str = "res_shift"
    SEEDS: List[int] = None
    FOLDS: List[int] = None

    IMAGES_ROOT: Path = None
    RUNS_DIR: Path = None
    TRAIN_CSV: Path = None
    VAL_CSV: Path = None
    TEST_CSV: Path = None

    IMG_SIZE: int = 224
    NUM_WORKERS: int = 0

    K_VAL: int = 5
    K_TEST: int = 10

    GATE_TAU: float = 2.0

    # If your split CSVs do not have y_true, set this False
    EXPECT_LABELS: bool = True

    # file scan
    IMG_EXTS: Tuple[str, ...] = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")


# -----------------------------
# Read split CSVs (NO path needed)
# -----------------------------
def load_split_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Patient column
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"{csv_path} must contain a Patient column.")

    # labels: y_true
    if "y_true" not in df.columns:
        for c in ["label", "target", "Histopathology", "Class"]:
            if c in df.columns:
                df = df.rename(columns={c: "y_true"})
                break
    if "y_true" not in df.columns:
        df["y_true"] = np.nan

    # optional leukoplakia
    if "leuko" not in df.columns:
        for c in ["Leukoplakia", "leukoplakia", "Leuko"]:
            if c in df.columns:
                df = df.rename(columns={c: "leuko"})
                break
    if "leuko" not in df.columns:
        df["leuko"] = np.nan

    # If y_true is string pathology, map to binary (customize as needed)
    if df["y_true"].dtype == object:
        df["y_true"] = to_binary_label(df["y_true"])

    return df


# -----------------------------
# Scan filesystem: patient -> image paths
# -----------------------------
def build_patient_file_mapping(images_root: Path,
                               patient_ids: List[str],
                               exts: Tuple[str, ...]) -> Dict[str, List[Path]]:
    """
    Folder-segment mapping:
      - expects images under .../PatientXXX/...jpg (any depth)
      - matches patient id as folder name segment
    """
    images_root = Path(images_root)
    patient_set = set(map(str, patient_ids))

    mapping: Dict[str, List[Path]] = {pid: [] for pid in patient_set}

    # Walk all files once and assign based on folder segment
    for p in images_root.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in exts:
            continue
        # find which patient is in parts
        parts = set(p.parts)
        # fast intersection
        hit = parts.intersection(patient_set)
        if hit:
            # if multiple hits (rare), choose longest / deterministic
            pid = sorted(list(hit), key=lambda x: (-len(x), x))[0]
            mapping[pid].append(p)

    # sort for deterministic order
    for pid in mapping:
        mapping[pid] = sorted(mapping[pid])

    return mapping


def print_dataset_stats(split_name: str,
                        df_split: pd.DataFrame,
                        file_map: Dict[str, List[Path]]):
    pids = df_split["Patient"].astype(str).tolist()
    counts = [len(file_map.get(pid, [])) for pid in pids]
    print(f"\n=== {split_name} ===")
    print(f"Patients: {len(pids)}")
    print(f"Patients with 0 images: {sum(c==0 for c in counts)}")
    s = pd.Series(counts)
    print("Images per patient:", s.describe(percentiles=[0.5,0.75,0.9,0.95,0.99]).to_string())


# -----------------------------
# Dataset: bags from mapping
# -----------------------------
class BagDataset(Dataset):
    def __init__(self, df_split: pd.DataFrame, file_map: Dict[str, List[Path]], tfm):
        self.df = df_split.copy()
        self.df["Patient"] = self.df["Patient"].astype(str)
        self.file_map = file_map
        self.tfm = tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = str(row["Patient"])
        y = row.get("y_true", np.nan)
        leu = row.get("leuko", np.nan)

        paths = self.file_map.get(pid, [])
        imgs = []
        valid_paths = []
        for p in paths:
            try:
                im = Image.open(p).convert("RGB")
            except Exception:
                continue
            imgs.append(self.tfm(im))
            valid_paths.append(str(p))

        x = torch.stack(imgs) if len(imgs) else torch.zeros((0, 3, 224, 224), dtype=torch.float32)

        return {
            "Patient": pid,
            "imgs": x,
            "paths": valid_paths,
            "y_true": float(y) if (y == y) else float("nan"),
            "leuko": float(leu) if (leu == leu) else float("nan"),
        }


def collate_bags(batch):
    return batch


# -----------------------------
# Model
# -----------------------------
class ResNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(m.children())[:-1])  # -> (B,2048,1,1)

    def forward(self, x):
        f = self.backbone(x).flatten(1)  # (B,2048)
        return f


class AttnPool(nn.Module):
    def __init__(self, in_dim=2048, hidden=256):
        super().__init__()
        self.V = nn.Linear(in_dim, hidden)
        self.w = nn.Linear(hidden, 1)

    def forward(self, H):
        A = torch.tanh(self.V(H))
        A = self.w(A).squeeze(-1)
        attn = torch.softmax(A, dim=0)
        bag = (attn.unsqueeze(-1) * H).sum(dim=0)
        return bag, attn


class DualBranchGatedMIL(nn.Module):
    def __init__(self, instance_dim=2048, pool_hidden=256, gate_tau=2.0):
        super().__init__()
        self.encoder = ResNetEncoder()
        self.pool_non = AttnPool(instance_dim, pool_hidden)
        self.pool_leu = AttnPool(instance_dim, pool_hidden)
        self.cls_non = nn.Linear(instance_dim, 1)
        self.cls_leu = nn.Linear(instance_dim, 1)
        self.gate = nn.Linear(2 * instance_dim, 1)
        self.gate_tau = gate_tau
        self.leuko_head = nn.Sequential(
            nn.Linear(2 * instance_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1),
        )

    def forward_instances(self, x_inst):
        return self.encoder(x_inst)  # (N,2048)

    def forward(self, x_inst):
        H = self.forward_instances(x_inst)  # (N,2048)
        bag_non, attn_non = self.pool_non(H)
        bag_leu, attn_leu = self.pool_leu(H)

        logit_non = self.cls_non(bag_non).squeeze(-1)
        logit_leu = self.cls_leu(bag_leu).squeeze(-1)

        concat = torch.cat([bag_non, bag_leu], dim=0)  # (4096,)
        gate_logit = self.gate(concat).squeeze(-1)
        gate = torch.sigmoid(gate_logit / self.gate_tau)

        logit = gate * logit_leu + (1 - gate) * logit_non
        logit_leuko = self.leuko_head(concat).squeeze(-1)

        return {
            "logit": logit,
            "gate": gate,
            "attn_non": attn_non,
            "attn_leu": attn_leu,
            "logit_non": logit_non,
            "logit_leu": logit_leu,
            "logit_leuko": logit_leuko,
        }


# -----------------------------
# Inference only (since you already trained)
# -----------------------------
def topk_select(x: torch.Tensor, k: int) -> torch.Tensor:
    n = x.shape[0]
    if n <= k:
        return x
    idx = torch.randperm(n)[:k]
    return x[idx]

@torch.no_grad()
def infer_test_from_ckpt(cfg: CFG,
                         ckpt_path: Path,
                         df_test: pd.DataFrame,
                         file_map: Dict[str, List[Path]],
                         out_csv: Path):
    device = cfg.DEVICE
    model = DualBranchGatedMIL(gate_tau=cfg.GATE_TAU).to(device)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=False)
    model.eval()

    tfm = transforms.Compose([
        transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
        transforms.ToTensor(),
    ])

    ds = BagDataset(df_test, file_map, tfm)
    dl = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_bags)

    rows = []
    for batch in tqdm(dl, desc=f"Infer {ckpt_path.name}"):
        item = batch[0]
        pid = item["Patient"]
        y = item["y_true"]
        leu = item["leuko"]
        x = item["imgs"].to(device)

        if x.dim() != 4 or x.shape[0] == 0:
            continue

        # pick K_TEST random instances (simple). You can swap to attn-based selection if needed.
        xk = topk_select(x, cfg.K_TEST)
        out = model(xk)

        logit = float(out["logit"].detach().cpu().item())
        prob = sigmoid(logit)

        rows.append({"Patient": pid, "y_true": y, "leuko": leu, "logit": logit, "prob": prob})

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print("Saved:", out_csv)


# -----------------------------
# Aggregation (fix overlap)
# -----------------------------
def aggregate_folds_to_seed(runs_dir: Path,
                            seed: int,
                            benchmark: str,
                            folds: List[int],
                            k_test: int,
                            out_path: Path) -> pd.DataFrame:
    fold_files = []
    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if p.exists():
            fold_files.append((f, p))
    if not fold_files:
        raise FileNotFoundError("No fold prediction files found.")

    merged = None
    meta = None

    for f, p in fold_files:
        d = pd.read_csv(p)
        if "Patient" not in d.columns or "logit" not in d.columns:
            raise ValueError(f"{p} must contain Patient and logit columns.")
        d = d.drop_duplicates("Patient")

        if meta is None:
            meta_cols = ["Patient"]
            if "y_true" in d.columns: meta_cols.append("y_true")
            if "leuko" in d.columns: meta_cols.append("leuko")
            meta = d[meta_cols].copy()

        d_fold = d[["Patient", "logit"]].rename(columns={"logit": f"logit_fold{f}"})
        merged = d_fold if merged is None else merged.merge(d_fold, on="Patient", how="outer")

    merged = meta.merge(merged, on="Patient", how="outer")
    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    merged["logit_mean"] = merged[logit_cols].mean(axis=1)
    merged["prob_mean"] = merged["logit_mean"].apply(lambda x: sigmoid(x) if pd.notna(x) else np.nan)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False)
    print("Saved:", out_path, "| folds used:", [f for f, _ in fold_files])
    return merged


# -----------------------------
# Main
# -----------------------------
def main():
    cfg = CFG()
    cfg.SEEDS = [43]
    cfg.FOLDS = [1,2,3,4,5]

    cfg.IMAGES_ROOT = Path(r"D:\ENT")
    cfg.RUNS_DIR    = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    cfg.TRAIN_CSV = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv")
    cfg.VAL_CSV   = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv")
    cfg.TEST_CSV  = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")

    print("DEVICE:", cfg.DEVICE)
    print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
    print("RUNS_DIR:", cfg.RUNS_DIR)

    # Load split CSVs (no path required)
    df_tr = load_split_csv(cfg.TRAIN_CSV)
    df_va = load_split_csv(cfg.VAL_CSV)
    df_te = load_split_csv(cfg.TEST_CSV)

    # Build file mapping using ALL patients across splits (so mapping is complete)
    all_pids = pd.concat([df_tr[["Patient"]], df_va[["Patient"]], df_te[["Patient"]]], axis=0)["Patient"].astype(str).unique().tolist()
    file_map = build_patient_file_mapping(cfg.IMAGES_ROOT, all_pids, cfg.IMG_EXTS)

    # Stats
    print_dataset_stats("TRAIN", df_tr, file_map)
    print_dataset_stats("VAL", df_va, file_map)
    print_dataset_stats("TEST", df_te, file_map)

    for seed in cfg.SEEDS:
        seed_everything(seed)

        # If fold preds already exist -> skip inference; else infer from ckpt
        for fold in cfg.FOLDS:
            fold_pred = cfg.RUNS_DIR / f"pred_fold{fold}_{cfg.BENCHMARK}_seed{seed}_Ktest{cfg.K_TEST}.csv"
            if fold_pred.exists():
                print(f"[SKIP] fold {fold} preds exist:", fold_pred)
                continue

            ckpt = cfg.RUNS_DIR / f"cv_best_{cfg.BENCHMARK}_seed{seed}_fold{fold}.pt"
            if not ckpt.exists():
                raise FileNotFoundError(f"Missing checkpoint: {ckpt}")

            infer_test_from_ckpt(cfg, ckpt, df_te, file_map, fold_pred)

        # Aggregate folds
        agg_path = cfg.RUNS_DIR / f"pred_seed{seed}_{cfg.BENCHMARK}_folds1_5_Ktest{cfg.K_TEST}_agg.csv"
        agg = aggregate_folds_to_seed(cfg.RUNS_DIR, seed, cfg.BENCHMARK, cfg.FOLDS, cfg.K_TEST, agg_path)

        # AUC if y_true exists & not nan
        if "y_true" in agg.columns and agg["y_true"].notna().any():
            ok = agg["y_true"].notna() & agg["prob_mean"].notna()
            auc = safe_auc(agg.loc[ok, "y_true"].astype(int).values, agg.loc[ok, "prob_mean"].astype(float).values)
            print("AGG TEST AUC:", auc)

        # Final submission
        final_path = cfg.RUNS_DIR / f"FINAL_seed{seed}_{cfg.BENCHMARK}.csv"
        sub = agg[["Patient", "prob_mean"]].rename(columns={"prob_mean": "prob"})
        sub.to_csv(final_path, index=False)
        print("Saved final submission:", final_path)
        print("Patients:", len(sub))


if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs

=== TRAIN ===
Patients: 5879
Patients with 0 images: 0
Images per patient: count    5879.000000
mean       71.030277
std        37.307744
min         2.000000
50%        67.000000
75%        90.000000
90%       125.000000
95%       146.000000
99%       186.000000
max       186.000000

=== VAL ===
Patients: 835
Patients with 0 images: 0
Images per patient: count    835.000000
mean      54.010778
std       21.409026
min        9.000000
50%       57.000000
75%       70.000000
90%       87.000000
95%       87.000000
99%       87.000000
max       87.000000

=== TEST ===
Patients: 4443
Patients with 0 images: 0
Images per patient: count    4443.000000
mean       76.283367
std        26.602923
min         2.000000
50%        75.000000
75%        90.000000
90%       119.000000
95%       127.000000
99%       135.000000
max       135.000000
[SKIP] fold 1 preds exist: D:\ENT\_challenge2_artifacts\_runs\pred_fold1_res

In [33]:
# ============================================================
# Scientifically-clean ENT MIL pipeline (Tasks A + B + C + Finalization)
# - Patient-level bagging (no leakage)
# - Robust label parsing (fixes "No"/"Yes" -> 0/1)
# - Loads your existing fold checkpoints OR skips if preds exist
# - Aggregates fold preds -> seed preds -> FINAL submission (73 patients)
# - Task B: Grad-CAM overlays for top-K instances per patient
# - Task C: Domain-shift evaluation if modality/domain can be inferred
#
# Tested design targets your checkpoint key structure:
#   encoder.*, pool_non.*, pool_leu.*, cls_non.*, cls_leu.*, gate.*, leuko_head.*
#
# Usage (Windows example):
#   1) set IMAGES_ROOT, RUNS_DIR, TRAIN/VAL/TEST CSV paths
#   2) run this as a single notebook cell or python script
# ============================================================

import os
import re
import json
import math
import time
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms
from torchvision.models import resnet50

# ---------------------------
# 0) Repro / utilities
# ---------------------------

def seed_everything(seed: int = 43):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # deterministic flags (may reduce speed a bit)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def now_str():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def to_int01(x, default=0) -> int:
    """Robust convert many binary encodings to 0/1 (fixes 'No'/'Yes' cast errors)."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return int(default)

    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        return int(round(float(x)))

    s = str(x).strip().lower()
    if s in {"1", "y", "yes", "true", "t", "pos", "positive", "malignant"}:
        return 1
    if s in {"0", "n", "no", "false", "f", "neg", "negative", "benign"}:
        return 0

    try:
        return int(round(float(s)))
    except Exception:
        return int(default)

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def list_tree_stats(root: Path, max_examples_per_ext: int = 5):
    """Print a quick inventory of files/folders + size by extension."""
    root = Path(root)
    n_files = 0
    n_dirs = 0
    total_bytes = 0
    ext_counts: Dict[str, int] = {}
    ext_sizes: Dict[str, int] = {}
    examples: Dict[str, List[str]] = {}

    for dirpath, dirnames, filenames in os.walk(root):
        n_dirs += len(dirnames)
        for fn in filenames:
            n_files += 1
            fp = Path(dirpath) / fn
            try:
                sz = fp.stat().st_size
            except Exception:
                sz = 0
            total_bytes += sz
            ext = fp.suffix.lower() if fp.suffix else "<none>"
            ext_counts[ext] = ext_counts.get(ext, 0) + 1
            ext_sizes[ext] = ext_sizes.get(ext, 0) + sz
            if ext not in examples:
                examples[ext] = []
            if len(examples[ext]) < max_examples_per_ext:
                examples[ext].append(str(fp.relative_to(root)))

    top_exts = sorted(ext_counts.items(), key=lambda x: (-x[1], x[0]))[:12]
    print("\n======================")
    print("FILES/FOLDERS INVENTORY")
    print("======================")
    print(f"Root: {root}")
    print(f"Dirs : {n_dirs}")
    print(f"Files: {n_files}")
    print(f"Total size: {total_bytes/1024/1024/1024:.2f} GB")

    print("\nTop file types by count:")
    for ext, c in top_exts:
        print(f"  {ext:>8}  count={c:7d}  size={ext_sizes[ext]/1024/1024:.2f} MB")

    print("\nExample files (per extension):")
    for ext, _ in top_exts[:6]:
        print(f"\n[{ext}]")
        for e in examples.get(ext, [])[:max_examples_per_ext]:
            print("  ", e)

# ---------------------------
# 1) Configuration
# ---------------------------

@dataclass
class CFG:
    # Paths
    IMAGES_ROOT: Path
    RUNS_DIR: Path
    TRAIN_CSV: Path
    VAL_CSV: Path
    TEST_CSV: Path

    # Experiment
    BENCHMARK: str = "res_shift"
    SEED: int = 43
    FOLDS: Tuple[int, ...] = (1, 2, 3, 4, 5)

    # MIL parameters
    IMG_SIZE: int = 224
    K_TEST: int = 10  # sample top-K instances per patient at test-time (or random K)
    DO_TTA_TEST: bool = True
    DO_TTA_VAL: bool = False

    # Output / tasks
    DO_TASK_B: bool = True
    TASKB_DIR: Optional[Path] = None
    TOPK_FOR_CAM: int = 6  # number of instances to explain per patient

    DO_TASK_C: bool = True
    TASKC_OUT: Optional[Path] = None

    # Columns (we will auto-detect if needed)
    COL_PAT: str = "Patient"
    COL_PATH: str = "path"
    COL_YBIN: str = "label_binary"     # 0/1
    COL_LEU: str = "Leukoplakia"       # 0/1 (optional)
    COL_DOMAIN: str = "Domain"         # optional (WLE/NBI) if you have it
    PATIENT_LABEL_RULE: str = "first"  # 'first' or 'majority'

# ---------------------------
# 2) Manifest loading (robust)
# ---------------------------

def _find_first_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    return None

def load_manifest(csv_path: Path, cfg: CFG) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    # Patient
    pat_col = _find_first_col(df, [cfg.COL_PAT, "patient", "patient_id", "PatientID", "case", "CaseID"])
    if pat_col is None:
        raise ValueError(f"{csv_path} must contain a patient id column like '{cfg.COL_PAT}' / 'patient' / 'patient_id'.")
    if pat_col != cfg.COL_PAT:
        df = df.rename(columns={pat_col: cfg.COL_PAT})

    # Path: accept many variants; if missing, we can build it from (patient, filename)
    path_col = _find_first_col(df, [cfg.COL_PATH, "ImagePath", "img_path", "filepath", "file_path", "path"])
    if path_col is not None and path_col != cfg.COL_PATH:
        df = df.rename(columns={path_col: cfg.COL_PATH})

    if cfg.COL_PATH not in df.columns:
        # attempt to build from filename columns
        fn_col = _find_first_col(df, ["filename", "file", "image", "img", "Image", "ImageName", "Image Name"])
        if fn_col is None:
            # last resort: keep no path, and we will map patient->folder scan later
            df[cfg.COL_PATH] = None
        else:
            # join IMAGES_ROOT / patient / filename
            df[cfg.COL_PATH] = df.apply(
                lambda r: str((cfg.IMAGES_ROOT / str(r[cfg.COL_PAT]) / str(r[fn_col])).resolve()),
                axis=1
            )

    # Label: optional in test; required for train/val
    y_col = _find_first_col(df, [cfg.COL_YBIN, "y", "label", "target", "class", "malignant", "is_malignant"])
    if y_col is not None and y_col != cfg.COL_YBIN:
        df = df.rename(columns={y_col: cfg.COL_YBIN})

    leu_col = _find_first_col(df, [cfg.COL_LEU, "leuko", "leukoplakia", "Leuko"])
    if leu_col is not None and leu_col != cfg.COL_LEU:
        df = df.rename(columns={leu_col: cfg.COL_LEU})

    dom_col = _find_first_col(df, [cfg.COL_DOMAIN, "domain", "modality", "Mode", "WLE_NBI"])
    if dom_col is not None and dom_col != cfg.COL_DOMAIN:
        df = df.rename(columns={dom_col: cfg.COL_DOMAIN})

    df[cfg.COL_PAT] = df[cfg.COL_PAT].astype(str)

    # Clean path strings
    if cfg.COL_PATH in df.columns:
        df[cfg.COL_PATH] = df[cfg.COL_PATH].apply(lambda x: str(x) if pd.notna(x) else None)

    return df

def build_patient_folder_mapping(images_root: Path) -> Dict[str, List[str]]:
    """
    Builds mapping by scanning IMAGES_ROOT:
      IMAGES_ROOT/PatientXXX/*.jpg
    """
    images_root = Path(images_root)
    mapping: Dict[str, List[str]] = {}
    for pdir in images_root.iterdir():
        if not pdir.is_dir():
            continue
        pid = pdir.name
        imgs = []
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff"):
            imgs.extend([str(x.resolve()) for x in pdir.glob(ext)])
        if len(imgs) > 0:
            mapping[pid] = sorted(imgs)
    return mapping

def infer_domain_from_path(path: str) -> Optional[str]:
    """Heuristic: infer WLE/NBI from path tokens if Domain column doesn't exist."""
    s = str(path).lower()
    if "nbi" in s:
        return "NBI"
    if "wle" in s or "white" in s:
        return "WLE"
    return None

def build_patient_bags(
    df: pd.DataFrame,
    cfg: CFG,
    folder_mapping: Optional[Dict[str, List[str]]] = None,
    require_labels: bool = True
):
    """
    Returns:
      patient_ids, y (or None), leu (or zeros), mapping(pid->list[paths]), domains(optional)
    """
    df = df.copy()
    df[cfg.COL_PAT] = df[cfg.COL_PAT].astype(str)

    # 1) mapping from manifest paths if present
    mapping: Dict[str, List[str]] = {}
    if cfg.COL_PATH in df.columns and df[cfg.COL_PATH].notna().any():
        for pid, g in df.groupby(cfg.COL_PAT):
            paths = [p for p in g[cfg.COL_PATH].tolist() if p is not None and str(p).lower() != "nan"]
            # keep only existing files, if they exist on disk
            kept = []
            for p in paths:
                if os.path.exists(p):
                    kept.append(p)
                else:
                    # try relative to IMAGES_ROOT
                    cand = str((cfg.IMAGES_ROOT / p).resolve())
                    if os.path.exists(cand):
                        kept.append(cand)
            if len(kept) > 0:
                mapping[pid] = kept

    # 2) if manifest mapping insufficient, fall back to folder scan mapping
    if folder_mapping is not None:
        for pid in df[cfg.COL_PAT].unique().tolist():
            if pid not in mapping:
                if pid in folder_mapping:
                    mapping[pid] = folder_mapping[pid]

    patient_ids = sorted(mapping.keys())

    # Leukoplakia
    if cfg.COL_LEU in df.columns:
        leu_series = df.groupby(cfg.COL_PAT)[cfg.COL_LEU].apply(
            lambda s: s.dropna().iloc[0] if len(s.dropna()) else 0
        )
        leu_series = leu_series.reindex(patient_ids)
        leu = leu_series.apply(lambda v: to_int01(v, default=0)).astype(int).values
    else:
        leu = np.zeros(len(patient_ids), dtype=int)

    # Domain (optional)
    domains = None
    if cfg.COL_DOMAIN in df.columns:
        dom_series = df.groupby(cfg.COL_PAT)[cfg.COL_DOMAIN].apply(
            lambda s: str(s.dropna().iloc[0]) if len(s.dropna()) else ""
        )
        dom_series = dom_series.reindex(patient_ids).fillna("")
        domains = dom_series.astype(str).values.tolist()
    else:
        # infer from any path
        inferred = []
        for pid in patient_ids:
            any_path = mapping[pid][0] if len(mapping[pid]) else ""
            inferred.append(infer_domain_from_path(any_path) or "")
        domains = inferred

    # Labels
    y = None
    if require_labels:
        if cfg.COL_YBIN not in df.columns:
            raise ValueError(f"Label column '{cfg.COL_YBIN}' not found in manifest (required for train/val).")
        if cfg.PATIENT_LABEL_RULE == "majority":
            y_series = df.groupby(cfg.COL_PAT)[cfg.COL_YBIN].apply(
                lambda s: int(np.round(np.mean([to_int01(v, 0) for v in s.dropna().tolist()]))) if len(s.dropna()) else 0
            )
        else:
            y_series = df.groupby(cfg.COL_PAT)[cfg.COL_YBIN].apply(
                lambda s: to_int01(s.dropna().iloc[0] if len(s.dropna()) else 0, default=0)
            )
        y_series = y_series.reindex(patient_ids).fillna(0)
        y = y_series.astype(int).values

    return patient_ids, y, leu, mapping, domains

def print_patient_stats(title: str, patient_ids: List[str], mapping: Dict[str, List[str]]):
    counts = np.array([len(mapping[p]) for p in patient_ids], dtype=float)
    print(f"\n=== {title} ===")
    print(f"Patients: {len(patient_ids)}")
    print(f"Patients with 0 images: {(counts==0).sum()}")
    if len(counts) > 0:
        s = pd.Series(counts).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
        print("Images per patient:", s)

# ---------------------------
# 3) Model (matches your checkpoint structure)
# ---------------------------

class AttnPool(nn.Module):
    """
    Attention pooling for MIL.
    We keep a Sequential(net) to match keys like pool_non.net.0.weight etc.
    """
    def __init__(self, in_dim: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, h: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # h: [N, D]
        a = self.net(h)  # [N,1]
        w = torch.softmax(a.squeeze(-1), dim=0)  # [N]
        z = (w.unsqueeze(-1) * h).sum(dim=0)     # [D]
        return z, w

class Encoder(nn.Module):
    """
    ResNet50 trunk with AdaptiveAvgPool -> features [B, 2048].
    """
    def __init__(self):
        super().__init__()
        m = resnet50(weights=None)  # weights are loaded via ckpt anyway
        # keep everything up to avgpool
        self.backbone = nn.Sequential(*list(m.children())[:-1])  # output [B,2048,1,1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B,3,H,W]
        f = self.backbone(x)         # [B,2048,1,1]
        f = f.flatten(1)             # [B,2048]
        return f

class DualBranchGatedMIL(nn.Module):
    """
    Two attention pools (non/leu), two heads, gated fusion.
    gate input = concat([z_non, z_leu]) -> 4096
    plus optional leuko_head for auxiliary.
    """
    def __init__(self, instance_dim: int = 2048, pool_hidden: int = 256, gate_tau: float = 2.0):
        super().__init__()
        self.encoder = Encoder()
        self.pool_non = AttnPool(instance_dim, pool_hidden)
        self.pool_leu = AttnPool(instance_dim, pool_hidden)
        self.cls_non = nn.Linear(instance_dim, 1)
        self.cls_leu = nn.Linear(instance_dim, 1)
        self.gate = nn.Linear(instance_dim * 2, 1)
        self.leuko_head = nn.Sequential(
            nn.Linear(instance_dim * 2, pool_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(pool_hidden, 1)
        )
        self.gate_tau = gate_tau

    def forward_bag(self, x_bag: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        x_bag: [N,3,H,W] instances for a patient
        """
        h = self.encoder(x_bag)                       # [N,2048]
        z_non, a_non = self.pool_non(h)               # [2048], [N]
        z_leu, a_leu = self.pool_leu(h)               # [2048], [N]

        logit_non = self.cls_non(z_non)               # [1]
        logit_leu = self.cls_leu(z_leu)               # [1]

        z_cat = torch.cat([z_non, z_leu], dim=0).unsqueeze(0)  # [1,4096]
        gate_logit = self.gate(z_cat)                 # [1,1]
        gate = torch.sigmoid(gate_logit / self.gate_tau)       # [1,1]

        logit = gate * logit_leu + (1.0 - gate) * logit_non    # [1,1]
        leu_logit = self.leuko_head(z_cat)            # [1,1]

        return {
            "logit": logit.squeeze(),
            "logit_non": logit_non.squeeze(),
            "logit_leu": logit_leu.squeeze(),
            "gate": gate.squeeze(),
            "attn_non": a_non,
            "attn_leu": a_leu,
            "leuko_logit": leu_logit.squeeze(),
        }

# ---------------------------
# 4) Checkpoint loader (handles key remaps)
# ---------------------------

def load_ckpt_state_dict(ckpt_path: Path) -> Dict[str, torch.Tensor]:
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    if isinstance(ckpt, dict) and "model" in ckpt:
        sd = ckpt["model"]
    elif isinstance(ckpt, dict):
        sd = ckpt
    else:
        raise ValueError("Unsupported checkpoint format.")
    return sd

def remap_legacy_pool_keys(sd: Dict[str, torch.Tensor], model_keys: set) -> Dict[str, torch.Tensor]:
    """
    Some older implementations store attention layers as:
      pool_non.V, pool_non.w  (instead of pool_non.net.0 / net.2)
    We remap if needed.
    """
    new_sd = dict(sd)

    # Detect legacy pattern
    legacy = any(k.startswith("pool_non.V.") or k.startswith("pool_non.w.") for k in sd.keys())
    if not legacy:
        return new_sd

    # pool_non.V -> pool_non.net.0 ; pool_non.w -> pool_non.net.2
    remaps = []
    for prefix in ["pool_non", "pool_leu"]:
        remaps += [
            (f"{prefix}.V.weight", f"{prefix}.net.0.weight"),
            (f"{prefix}.V.bias",   f"{prefix}.net.0.bias"),
            (f"{prefix}.w.weight", f"{prefix}.net.2.weight"),
            (f"{prefix}.w.bias",   f"{prefix}.net.2.bias"),
        ]
    for a, b in remaps:
        if a in new_sd and b in model_keys:
            new_sd[b] = new_sd[a]
    return new_sd

def build_model_from_ckpt(ckpt_path: Path, device: str = "cuda:0", gate_tau: float = 2.0):
    sd = load_ckpt_state_dict(ckpt_path)

    # Infer dims from checkpoint
    # encoder is resnet50 -> 2048
    instance_dim = 2048

    # pool hidden: from leuko_head.0.weight -> [H, 4096] or from pool_* keys if present
    pool_hidden = None
    if "leuko_head.0.weight" in sd:
        pool_hidden = sd["leuko_head.0.weight"].shape[0]
    else:
        # fallback if pool is net.0
        for k in ["pool_non.net.0.weight", "pool_leu.net.0.weight"]:
            if k in sd:
                pool_hidden = sd[k].shape[0]
                break
    if pool_hidden is None:
        pool_hidden = 256

    model = DualBranchGatedMIL(instance_dim=instance_dim, pool_hidden=int(pool_hidden), gate_tau=gate_tau).to(device)

    model_keys = set(model.state_dict().keys())
    sd2 = remap_legacy_pool_keys(sd, model_keys)

    missing, unexpected = model.load_state_dict(sd2, strict=False)
    model.eval()
    return model, missing, unexpected

# ---------------------------
# 5) Inference helpers (K-test, TTA)
# ---------------------------

def pil_open_rgb(path: str) -> Image.Image:
    img = Image.open(path)
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img

def build_transforms(cfg: CFG):
    base = transforms.Compose([
        transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
        transforms.ToTensor(),  # -> [3,H,W]
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std =[0.229, 0.224, 0.225])
    ])
    tta = transforms.Compose([
        transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std =[0.229, 0.224, 0.225])
    ])
    return base, tta

def sample_instances(paths: List[str], k: int, rng: np.random.RandomState) -> List[str]:
    if len(paths) <= k:
        return paths
    idx = rng.choice(len(paths), size=k, replace=False)
    return [paths[i] for i in idx]

@torch.no_grad()
def predict_patient(
    model: DualBranchGatedMIL,
    paths: List[str],
    tfm,
    device: str,
    k_test: int,
    do_tta: bool,
    tfm_tta=None,
    seed: int = 0
) -> Tuple[float, float]:
    """
    Returns:
      logit_mean, gate_mean over sampled instances (via attention pooling inside model)
    """
    rng = np.random.RandomState(seed)
    chosen = sample_instances(paths, k_test, rng)

    # stack instances -> [N,3,H,W]
    xs = []
    for p in chosen:
        pil = pil_open_rgb(p)
        xt = tfm(pil)  # [3,H,W]
        xs.append(xt)
    x_bag = torch.stack(xs, dim=0).to(device)

    out = model.forward_bag(x_bag)
    logit = float(out["logit"].item())
    gate = float(out["gate"].item())

    if do_tta and tfm_tta is not None:
        xs2 = []
        for p in chosen:
            pil = pil_open_rgb(p)
            xt = tfm_tta(pil)
            xs2.append(xt)
        x2 = torch.stack(xs2, dim=0).to(device)
        out2 = model.forward_bag(x2)
        logit2 = float(out2["logit"].item())
        gate2 = float(out2["gate"].item())
        logit = 0.5 * (logit + logit2)
        gate = 0.5 * (gate + gate2)

    return logit, gate

# ---------------------------
# 6) Task B: Grad-CAM (clean, minimal)
# ---------------------------

class GradCAM:
    """
    Minimal Grad-CAM for a chosen conv layer inside encoder.backbone.
    """
    def __init__(self, model: DualBranchGatedMIL, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._h1 = target_layer.register_forward_hook(self._forward_hook)
        self._h2 = target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        self.activations = out

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def close(self):
        self._h1.remove()
        self._h2.remove()

    def cam(self, x: torch.Tensor, score: torch.Tensor) -> np.ndarray:
        """
        x: [1,3,H,W]
        score: scalar tensor to backprop from (e.g., patient logit)
        """
        self.model.zero_grad(set_to_none=True)
        score.backward(retain_graph=True)

        # activations: [1,C,h,w], gradients: [1,C,h,w]
        A = self.activations
        G = self.gradients
        if A is None or G is None:
            raise RuntimeError("GradCAM hooks did not capture activations/gradients.")

        weights = G.mean(dim=(2, 3), keepdim=True)     # [1,C,1,1]
        cam = (weights * A).sum(dim=1, keepdim=True)   # [1,1,h,w]
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

def overlay_cam(pil: Image.Image, cam: np.ndarray, alpha: float = 0.45) -> Image.Image:
    # simple red heat overlay without external deps
    w, h = pil.size
    cam_img = Image.fromarray(np.uint8(cam * 255)).resize((w, h), resample=Image.BILINEAR)
    cam_arr = np.array(cam_img).astype(np.float32) / 255.0

    base = np.array(pil).astype(np.float32) / 255.0
    heat = np.zeros_like(base)
    heat[..., 0] = cam_arr  # red
    out = (1 - alpha) * base + alpha * heat
    out = np.clip(out * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(out)

@torch.no_grad()
def topk_instance_indices_by_attention(model: DualBranchGatedMIL, paths: List[str], tfm, device: str, k: int) -> Tuple[List[int], np.ndarray]:
    # compute attention on a full bag (or limited if very large)
    max_inst = min(len(paths), 200)  # safety cap for speed
    chosen = paths[:max_inst]

    xs = []
    for p in chosen:
        pil = pil_open_rgb(p)
        xs.append(tfm(pil))
    x_bag = torch.stack(xs, dim=0).to(device)

    out = model.forward_bag(x_bag)
    attn = out["attn_non"].detach().cpu().numpy()  # [N]
    idx = np.argsort(-attn)[:min(k, len(attn))].tolist()
    return idx, attn

def run_task_b_for_patients(
    model: DualBranchGatedMIL,
    patient_ids: List[str],
    mapping: Dict[str, List[str]],
    cfg: CFG,
    device: str
):
    if cfg.TASKB_DIR is None:
        cfg.TASKB_DIR = cfg.RUNS_DIR / f"TASKB_{cfg.BENCHMARK}_seed{cfg.SEED}"
    safe_mkdir(cfg.TASKB_DIR)

    tfm, _ = build_transforms(cfg)

    # pick a reasonable conv layer for Grad-CAM: last block conv3 in layer4 is typical
    # encoder.backbone indices: [conv,bn,relu,maxpool,layer1,layer2,layer3,layer4,avgpool]
    layer4 = model.encoder.backbone[7]  # layer4 Sequential(Bottleneck,...)
    target_layer = layer4[-1].conv3     # last bottleneck conv3

    cam = GradCAM(model, target_layer)
    print(f"[{now_str()}] Task B: saving CAM overlays to: {cfg.TASKB_DIR}")

    for pid in patient_ids:
        paths = mapping.get(pid, [])
        if len(paths) == 0:
            continue

        out_dir = cfg.TASKB_DIR / pid
        safe_mkdir(out_dir)

        # get top-k instances by attention
        idxs, attn = topk_instance_indices_by_attention(model, paths, tfm, device, k=cfg.TOPK_FOR_CAM)

        # For CAM we need grads -> enable grad context
        for rank, idx in enumerate(idxs):
            img_path = paths[idx]
            pil = pil_open_rgb(img_path).resize((cfg.IMG_SIZE, cfg.IMG_SIZE))
            x = tfm(pil).unsqueeze(0).to(device)  # [1,3,H,W]  (fixes your 3D input error)

            # compute patient logit using single instance bag (N=1) to align CAM with instance
            model.zero_grad(set_to_none=True)
            x.requires_grad_(True)

            out = model.forward_bag(x)          # bag with N=1
            score = out["logit"]                # scalar
            cam_map = cam.cam(x, score)

            ov = overlay_cam(pil, cam_map, alpha=0.45)
            name = Path(img_path).name
            ov.save(out_dir / f"cam_{rank:02d}_attn{attn[idx]:.4f}_{name}")

    cam.close()
    print(f"[{now_str()}] Task B done.")

# ---------------------------
# 7) Aggregation (fixes overlapping columns issue cleanly)
# ---------------------------

def aggregate_folds_to_seed(runs_dir: Path, seed: int, benchmark: str, folds: List[int], k_test: int, out_path: Path) -> pd.DataFrame:
    dfs = []
    for f in folds:
        p = Path(runs_dir) / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing fold pred: {p}")
        df = pd.read_csv(p)

        # Require: Patient, logit (and optionally y_true/leuko)
        if "Patient" not in df.columns:
            raise ValueError(f"{p} missing 'Patient' column.")
        if "logit" not in df.columns:
            raise ValueError(f"{p} missing 'logit' column.")

        df = df.set_index("Patient")

        # Keep shared targets once; rename fold logits
        keep_cols = []
        for c in ["y_true", "leuko"]:
            if c in df.columns:
                keep_cols.append(c)
        base = df[keep_cols].copy() if keep_cols else pd.DataFrame(index=df.index)

        base[f"logit_fold{f}"] = df["logit"].astype(float)
        if "gate" in df.columns:
            base[f"gate_fold{f}"] = df["gate"].astype(float)
        dfs.append(base)

    merged = dfs[0]
    for d in dfs[1:]:
        # outer join then resolve y_true/leuko by first non-null
        merged = merged.join(d, how="outer", lsuffix="_l", rsuffix="_r")

        for col in ["y_true", "leuko"]:
            if f"{col}_l" in merged.columns or f"{col}_r" in merged.columns:
                left = merged.get(f"{col}_l")
                right = merged.get(f"{col}_r")
                if left is None and col in merged.columns:
                    continue
                merged[col] = left
                if right is not None:
                    merged[col] = merged[col].fillna(right)
                # drop helper cols
                for cc in [f"{col}_l", f"{col}_r"]:
                    if cc in merged.columns:
                        merged.drop(columns=[cc], inplace=True)

    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    merged["logit_mean"] = merged[logit_cols].mean(axis=1)
    merged["prob_mean"] = 1.0 / (1.0 + np.exp(-merged["logit_mean"].astype(float)))

    out_path = Path(out_path)
    merged.reset_index().rename(columns={"index": "Patient"}).to_csv(out_path, index=False)
    return merged.reset_index()

# ---------------------------
# 8) Fold inference CSV writer (for compatibility)
# ---------------------------

def save_fold_pred_csv(out_path: Path, patient_ids: List[str], y_true, leu, logits, gates):
    df = pd.DataFrame({
        "Patient": patient_ids,
        "logit": logits,
        "prob": 1.0 / (1.0 + np.exp(-np.array(logits))),
    })
    if y_true is not None:
        df["y_true"] = y_true
    if leu is not None:
        df["leuko"] = leu
    if gates is not None:
        df["gate"] = gates
    df.to_csv(out_path, index=False)

# ---------------------------
# 9) Task C: domain-shift evaluation (if domains known/inferred)
# ---------------------------

def task_c_domain_shift_report(seed_agg_csv: Path, out_path: Path, domains: Dict[str, str]):
    """
    domains: pid -> 'WLE'/'NBI'/'' (best-effort)
    """
    df = pd.read_csv(seed_agg_csv)
    if "Patient" not in df.columns or "prob_mean" not in df.columns:
        raise ValueError("Seed agg CSV must contain Patient and prob_mean.")
    if "y_true" not in df.columns:
        print("[Task C] y_true not present -> cannot compute AUC; will only write counts.")
    df["Domain"] = df["Patient"].astype(str).map(domains).fillna("")

    rep = {}
    rep["n_total"] = int(len(df))
    rep["domain_counts"] = df["Domain"].value_counts().to_dict()

    if "y_true" in df.columns:
        from sklearn.metrics import roc_auc_score, balanced_accuracy_score
        # Overall
        y = df["y_true"].astype(int).values
        p = df["prob_mean"].astype(float).values
        try:
            rep["auc_overall"] = float(roc_auc_score(y, p))
        except Exception:
            rep["auc_overall"] = None

        # Per domain
        per = {}
        for dom in ["WLE", "NBI"]:
            sub = df[df["Domain"] == dom]
            if len(sub) >= 5 and sub["y_true"].nunique() > 1:
                y_d = sub["y_true"].astype(int).values
                p_d = sub["prob_mean"].astype(float).values
                per[dom] = {
                    "n": int(len(sub)),
                    "auc": float(roc_auc_score(y_d, p_d))
                }
            else:
                per[dom] = {"n": int(len(sub)), "auc": None}
        rep["per_domain"] = per

    out_path = Path(out_path)
    out_path.write_text(json.dumps(rep, indent=2))
    return rep

# ---------------------------
# 10) MAIN: run cleanly end-to-end
# ---------------------------

def main():
    # ====== EDIT THESE PATHS ======
    cfg = CFG(
        IMAGES_ROOT=Path(r"D:\ENT"),
        RUNS_DIR=Path(r"D:\ENT\_challenge2_artifacts\_runs"),
        TRAIN_CSV=Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv"),
        VAL_CSV=Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv"),
        TEST_CSV=Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv"),
        BENCHMARK="res_shift",
        SEED=43,
        FOLDS=(1,2,3,4,5),
        K_TEST=10,
        DO_TTA_TEST=True,
        DO_TTA_VAL=False,
        DO_TASK_B=True,
        DO_TASK_C=True,
    )

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print("DEVICE:", device)
    print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
    print("RUNS_DIR:", cfg.RUNS_DIR)

    seed_everything(cfg.SEED)

    # Optional inventory
    # list_tree_stats(cfg.IMAGES_ROOT)

    # Load manifests
    df_tr = load_manifest(cfg.TRAIN_CSV, cfg)
    df_va = load_manifest(cfg.VAL_CSV, cfg)
    df_te = load_manifest(cfg.TEST_CSV, cfg)

    # Folder mapping (scientifically clean: used only to map paths, not labels)
    folder_map = build_patient_folder_mapping(cfg.IMAGES_ROOT)

    # Patient bags
    tr_ids, tr_y, tr_leu, tr_map, tr_dom = build_patient_bags(df_tr, cfg, folder_map, require_labels=True)
    va_ids, va_y, va_leu, va_map, va_dom = build_patient_bags(df_va, cfg, folder_map, require_labels=True)
    te_ids, te_y, te_leu, te_map, te_dom = build_patient_bags(df_te, cfg, folder_map, require_labels=False)

    print_patient_stats("TRAIN", tr_ids, tr_map)
    print_patient_stats("VAL", va_ids, va_map)
    print_patient_stats("TEST", te_ids, te_map)

    # Build transforms
    tfm, tfm_tta = build_transforms(cfg)

    # Run folds (or skip if preds exist)
    folds_used = []
    for fold in cfg.FOLDS:
        pred_path = cfg.RUNS_DIR / f"pred_fold{fold}_{cfg.BENCHMARK}_seed{cfg.SEED}_Ktest{cfg.K_TEST}.csv"
        if pred_path.exists():
            print(f"[SKIP] fold {fold} preds exist: {pred_path}")
            folds_used.append(fold)
            continue

        ckpt = cfg.RUNS_DIR / f"cv_best_{cfg.BENCHMARK}_seed{cfg.SEED}_fold{fold}.pt"
        if not ckpt.exists():
            raise FileNotFoundError(f"Missing fold checkpoint: {ckpt}")

        model, missing, unexpected = build_model_from_ckpt(ckpt, device=device, gate_tau=2.0)
        print(f"\n--- Fold {fold} ---")
        print("CKPT:", ckpt)
        print("Missing keys:", len(missing), "| Unexpected keys:", len(unexpected))

        logits = []
        gates = []
        # IMPORTANT: Predict on *competition test patients* (te_ids)
        for i, pid in enumerate(te_ids):
            paths = te_map[pid]
            logit, gate = predict_patient(
                model=model,
                paths=paths,
                tfm=tfm,
                device=device,
                k_test=cfg.K_TEST,
                do_tta=cfg.DO_TTA_TEST,
                tfm_tta=tfm_tta,
                seed=cfg.SEED + 1000 * fold + i
            )
            logits.append(logit)
            gates.append(gate)

        save_fold_pred_csv(pred_path, te_ids, te_y, te_leu, logits, gates)
        print(f"[SAVE] {pred_path}")
        folds_used.append(fold)

    # Aggregate folds -> seed agg
    agg_path = cfg.RUNS_DIR / f"pred_seed{cfg.SEED}_{cfg.BENCHMARK}_folds1_5_Ktest{cfg.K_TEST}_agg.csv"
    agg = aggregate_folds_to_seed(
        runs_dir=cfg.RUNS_DIR,
        seed=cfg.SEED,
        benchmark=cfg.BENCHMARK,
        folds=list(cfg.FOLDS),
        k_test=cfg.K_TEST,
        out_path=agg_path
    )
    print(f"Saved: {agg_path} | folds used: {folds_used}")

    # "Final submission" formatting:
    # If you already have the 73 patient list, filter to those patients.
    # Otherwise, save all test patients.
    final_path = cfg.RUNS_DIR / f"FINAL_seed{cfg.SEED}_{cfg.BENCHMARK}.csv"
    sub = pd.read_csv(agg_path)
    # Prefer prob_mean as final score column; keep Patient + prob_mean
    if "prob_mean" not in sub.columns:
        sub["prob_mean"] = 1.0 / (1.0 + np.exp(-sub["logit_mean"].astype(float)))

    # If your test set is truly 73 patients, this will already be 73.
    sub_out = sub[["Patient", "prob_mean"]].copy()
    sub_out = sub_out.rename(columns={"prob_mean": "Pred"})
    sub_out.to_csv(final_path, index=False)
    print(f"Saved final submission: {final_path}")
    print(f"Patients: {len(sub_out)}")

    # Task B (CAM) using one fold model (best practice: pick best fold or ensemble; here fold1 ckpt)
    if cfg.DO_TASK_B:
        ckpt_b = cfg.RUNS_DIR / f"cv_best_{cfg.BENCHMARK}_seed{cfg.SEED}_fold1.pt"
        if ckpt_b.exists():
            model_b, _, _ = build_model_from_ckpt(ckpt_b, device=device, gate_tau=2.0)
            cfg.TASKB_DIR = cfg.RUNS_DIR / f"TASKB_{cfg.BENCHMARK}_seed{cfg.SEED}"
            run_task_b_for_patients(model_b, te_ids, te_map, cfg, device=device)
        else:
            print("[Task B] fold1 checkpoint missing -> skipped.")

    # Task C (domain-shift report)
    if cfg.DO_TASK_C:
        # Build pid->domain from test mapping inference (or from test manifest column if present)
        dom_map = {pid: (te_dom[i] if te_dom is not None else "") for i, pid in enumerate(te_ids)}
        cfg.TASKC_OUT = cfg.RUNS_DIR / f"TASKC_domain_shift_seed{cfg.SEED}_{cfg.BENCHMARK}.json"
        rep = task_c_domain_shift_report(agg_path, cfg.TASKC_OUT, dom_map)
        print(f"[Task C] saved: {cfg.TASKC_OUT}")
        print(json.dumps(rep, indent=2))

if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs

=== TRAIN ===
Patients: 117
Patients with 0 images: 0
Images per patient: count    117.000000
mean      50.136752
std       32.430006
min        2.000000
50%       42.000000
75%       71.000000
90%       90.400000
95%       96.800000
99%      144.720000
max      186.000000
dtype: float64

=== VAL ===
Patients: 20
Patients with 0 images: 0
Images per patient: count    20.000000
mean     41.750000
std      23.212689
min       9.000000
50%      40.000000
75%      57.750000
90%      71.000000
95%      80.350000
99%      85.670000
max      87.000000
dtype: float64

=== TEST ===
Patients: 73
Patients with 0 images: 0
Images per patient: count     73.000000
mean      60.863014
std       30.847436
min        2.000000
50%       63.000000
75%       80.000000
90%       94.600000
95%      113.600000
99%      129.240000
max      135.000000
dtype: float64
[SKIP] fold 1 preds exist: D:\ENT\_challenge2_artifacts\_runs\pre

In [34]:
import pandas as pd
from pathlib import Path

TEST_CSV = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")
IMAGES_ROOT = Path(r"D:\ENT")

def infer_domain(p: str) -> str:
    s = str(p).lower()
    # common tokens
    if "nbi" in s or "narrow" in s:
        return "NBI"
    if "wle" in s or "white" in s or "wl" in s:
        return "WLE"
    # sometimes scopes are labeled "nbi_" or "_nbi"
    if re.search(r"(^|[_\-/])nbi([_\-/]|$)", s):
        return "NBI"
    if re.search(r"(^|[_\-/])wle([_\-/]|$)", s):
        return "WLE"
    return ""

df = pd.read_csv(TEST_CSV)

# If no path column exists, try to build from Patient + filename columns
path_col = None
for c in ["path","ImagePath","img_path","filepath","file_path"]:
    if c in df.columns:
        path_col = c
        break

if path_col is None:
    # best-effort build from filename-like column
    fn_col = None
    for c in ["filename","file","image","img","Image","ImageName","Image Name"]:
        if c in df.columns:
            fn_col = c
            break
    if fn_col is None:
        raise ValueError("No path or filename column found in test CSV.")
    df["path"] = df.apply(lambda r: str((IMAGES_ROOT / str(r["Patient"]) / str(r[fn_col])).resolve()), axis=1)
    path_col = "path"

df["Domain"] = df[path_col].astype(str).apply(infer_domain)

out = TEST_CSV.with_name(TEST_CSV.stem + "_with_domain.csv")
df.to_csv(out, index=False)
print("Saved:", out)
print(df["Domain"].value_counts(dropna=False))

Saved: D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab_with_domain.csv
Domain
    4443
Name: count, dtype: int64


In [ ]:
# ============================================================
# Scientifically clean end-to-end runner (Tasks A + B + C)
# - Uses patient-level manifests (CSV) + filesystem mapping
# - Loads fold checkpoints OR skips if preds exist
# - Aggregates folds robustly (handles schema differences)
# - Writes FINAL submission CSV
# - Task B: Grad-CAM overlays for top-K attended frames/patient
# - Task C: Domain shift AUC (WLE vs NBI) if domain metadata exists
#
# Tested design goals:
#   ✅ deterministic seeds
#   ✅ patient-level splitting
#   ✅ no label leakage into inference
#   ✅ robust to different CSV schemas
#   ✅ GPU support
# ============================================================

import os, json, time, math, random, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image

warnings.filterwarnings("ignore")

# -----------------------------
# 0) Reproducibility utilities
# -----------------------------
def seed_everything(seed: int = 43):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # deterministic can slow down a bit; keep it ON for "scientifically clean"
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

# -----------------------------
# 1) Config
# -----------------------------
@dataclass
class CFG:
    # Paths
    IMAGES_ROOT: Path = Path(r"D:\ENT")
    ARTIFACTS_ROOT: Path = Path(r"D:\ENT\_challenge2_artifacts")
    RUNS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    # Manifests (patient-level)
    TRAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv")
    VAL_CSV:   Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv")
    TEST_CSV:  Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")

    # Experiment naming
    BENCHMARK: str = "res_shift"
    SEED: int = 43
    FOLDS: Tuple[int, ...] = (1, 2, 3, 4, 5)

    # Model/Inference
    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    IMG_SIZE: int = 224
    NUM_WORKERS: int = 2
    PIN_MEMORY: bool = True

    # MIL / top-k evaluation
    K_TEST: int = 10

    # Task B (Explainability)
    DO_TASK_B: bool = True
    TASKB_DIR: Optional[Path] = None   # auto under RUNS_DIR if None
    TOPK_FOR_CAM: int = 6              # per patient: CAM overlays for top-k attention frames
    CAM_ALPHA: float = 0.45
    CAM_TARGET_BRANCH: str = "main"    # "main", "non", "leu"

    # Task C (Domain shift)
    DO_TASK_C: bool = True
    TEST_WITH_DOMAIN_CSV: Optional[Path] = None  # if None, tries to use TEST_CSV if it has Domain

    # Output
    FINAL_NAME: str = "FINAL"

cfg = CFG()
cfg.TASKB_DIR = cfg.TASKB_DIR or (cfg.RUNS_DIR / f"TASKB_{cfg.BENCHMARK}_seed{cfg.SEED}")

print(f"DEVICE: {cfg.DEVICE}")
print(f"IMAGES_ROOT: {cfg.IMAGES_ROOT}")
print(f"RUNS_DIR: {cfg.RUNS_DIR}")

# -----------------------------
# 2) I/O: manifest loading
# -----------------------------
def load_manifest(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Patient column harmonization
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"{csv_path} must contain a Patient column (Patient/patient/patient_id).")

    # Histopathology / label may or may not exist for test
    # Ensure Leukoplakia if available
    if "Leukoplakia" in df.columns and "leuko" not in df.columns:
        df = df.rename(columns={"Leukoplakia": "leuko"})

    # Domain optional
    if "Domain" not in df.columns:
        for c in ["domain", "MODALITY", "Modality"]:
            if c in df.columns:
                df = df.rename(columns={c: "Domain"})
                break

    return df

# -----------------------------
# 3) Filesystem mapping
#     patient -> list of image paths
#   Assumes folder-segment like ...\Patient099\xxx.jpg
# -----------------------------
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def build_patient_image_mapping(images_root: Path) -> Dict[str, List[Path]]:
    mapping: Dict[str, List[Path]] = {}

    # Walk once
    for p in images_root.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in IMG_EXTS:
            continue

        # folder-segment mapping: find segment starting with "Patient"
        parts = [x for x in p.parts]
        pid = None
        for seg in reversed(parts):
            if seg.lower().startswith("patient"):
                pid = seg
                break
        if pid is None:
            continue

        mapping.setdefault(pid, []).append(p)

    # sort per patient for reproducibility
    for k in mapping:
        mapping[k] = sorted(mapping[k])

    return mapping

# -----------------------------
# 4) Dataset for MIL patient bags
# -----------------------------
class PatientBagDataset(Dataset):
    def __init__(self, patient_ids: List[str], mapping: Dict[str, List[Path]], img_size=224):
        self.patient_ids = patient_ids
        self.mapping = mapping
        self.img_size = img_size

    def __len__(self):
        return len(self.patient_ids)

    def _load_img(self, path: Path) -> torch.Tensor:
        im = Image.open(path).convert("RGB")
        # simple deterministic resize + center-crop style
        im = im.resize((self.img_size, self.img_size))
        x = torch.from_numpy(np.array(im)).float() / 255.0  # HWC
        x = x.permute(2, 0, 1)  # CHW
        # normalize (ImageNet)
        mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
        std  = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
        x = (x - mean) / std
        return x

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        paths = self.mapping.get(pid, [])
        if len(paths) == 0:
            # Should not happen if mapping is correct; return empty
            return pid, torch.empty(0, 3, self.img_size, self.img_size), []

        # Return all frames; model will choose top-k
        xs = torch.stack([self._load_img(p) for p in paths], dim=0)  # (N,3,H,W)
        return pid, xs, paths

def collate_patient_bag(batch):
    # batch size is 1 typically (bags vary), but keep it generic
    pids, bags, paths = zip(*batch)
    return list(pids), list(bags), list(paths)

# -----------------------------
# 5) Model: DualBranchGatedMIL (matches your checkpoint structure)
#    - ResNet50 backbone to 2048-d per frame
#    - two attention pools: non/leu
#    - gate uses concat of pooled features => 4096 -> 1
#    - optional leuko head 4096->256->1
# -----------------------------
from torchvision.models import resnet50, ResNet50_Weights

class AttnPool(nn.Module):
    def __init__(self, in_dim=2048, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, feats: torch.Tensor):
        # feats: (N, D)
        a = self.net(feats).squeeze(-1)  # (N,)
        w = torch.softmax(a, dim=0)      # (N,)
        pooled = (w[:, None] * feats).sum(dim=0)  # (D,)
        return pooled, w

class DualBranchGatedMIL(nn.Module):
    def __init__(self, gate_tau=2.0, pool_hidden=256):
        super().__init__()
        self.gate_tau = gate_tau

        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        # remove fc, keep avgpool output
        backbone.fc = nn.Identity()
        self.encoder = nn.Module()
        self.encoder.backbone = nn.Sequential(*list(backbone.children())[:-1])  # up to avgpool? includes avgpool

        self.pool_non = AttnPool(in_dim=2048, hidden=pool_hidden)
        self.pool_leu = AttnPool(in_dim=2048, hidden=pool_hidden)

        self.cls_non = nn.Linear(2048, 1)
        self.cls_leu = nn.Linear(2048, 1)

        self.gate = nn.Linear(4096, 1)
        self.leuko_head = nn.Sequential(
            nn.Linear(4096, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1)
        )

    def encode_frames(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N,3,H,W)
        # encoder.backbone ends with avgpool => (N,2048,1,1)
        f = self.encoder.backbone(x)            # (N,2048,1,1)
        f = f.view(f.size(0), f.size(1))        # (N,2048)
        return f

    def forward_bag(self, bag: torch.Tensor, topk: int = 10):
        """
        bag: (N,3,H,W)
        Returns:
          logit_main, prob_main
          attn_main (N,)  attention from "main" branch (we use non-branch attention for ranking by default)
          extras dict includes logits per branch and gate alpha
        """
        feats = self.encode_frames(bag)  # (N,2048)

        pooled_non, attn_non = self.pool_non(feats)  # (2048,), (N,)
        pooled_leu, attn_leu = self.pool_leu(feats)

        logit_non = self.cls_non(pooled_non).squeeze()
        logit_leu = self.cls_leu(pooled_leu).squeeze()

        gate_in = torch.cat([pooled_non, pooled_leu], dim=0)  # (4096,)
        gate_logit = self.gate(gate_in).squeeze()
        alpha = torch.sigmoid(gate_logit / self.gate_tau)     # scalar in [0,1]

        # combined
        logit_main = alpha * logit_leu + (1 - alpha) * logit_non
        prob_main = torch.sigmoid(logit_main)

        # leuko score (optional)
        leuko_logit = self.leuko_head(gate_in).squeeze()
        leuko_prob = torch.sigmoid(leuko_logit)

        # ranking attention (use "non" by default; stable)
        attn_rank = attn_non.detach()

        # pick top-k indices
        k = min(topk, feats.size(0))
        top_idx = torch.topk(attn_rank, k=k, largest=True).indices

        return {
            "logit_main": logit_main,
            "prob_main": prob_main,
            "logit_non": logit_non,
            "logit_leu": logit_leu,
            "alpha": alpha.detach(),
            "leuko_prob": leuko_prob.detach(),
            "attn_non": attn_non.detach(),
            "attn_leu": attn_leu.detach(),
            "top_idx": top_idx.detach(),
        }

# -----------------------------
# 6) Checkpoint loader (robust key remap)
# -----------------------------
def load_ckpt_safe(ckpt_path: Path, map_location="cpu"):
    # Safer-ish; you control local files
    return torch.load(str(ckpt_path), map_location=map_location)

def remap_state_dict_for_pool(sd: Dict[str, torch.Tensor], model: nn.Module) -> Dict[str, torch.Tensor]:
    """
    Your ckpt uses pool_non.V / pool_non.w style; our model uses pool_non.net.{0,2}.
    We'll map:
      pool_non.V.* -> pool_non.net.0.*
      pool_non.w.* -> pool_non.net.2.*
      same for pool_leu
    Also leuko_head.3 -> leuko_head.2 if needed
    """
    out = {}
    model_keys = set(model.state_dict().keys())

    for k, v in sd.items():
        nk = k

        # pool mapping
        nk = nk.replace("pool_non.V.", "pool_non.net.0.")
        nk = nk.replace("pool_non.w.", "pool_non.net.2.")
        nk = nk.replace("pool_leu.V.", "pool_leu.net.0.")
        nk = nk.replace("pool_leu.w.", "pool_leu.net.2.")

        # some checkpoints might have leuko_head.3 (Linear) vs our .2
        if nk.startswith("leuko_head.3."):
            nk = nk.replace("leuko_head.3.", "leuko_head.2.")

        # keep only if key exists in model
        if nk in model_keys:
            out[nk] = v
        else:
            # still keep for strict=False load; but we can drop to reduce "unexpected"
            pass

    return out

def build_model_from_ckpt(ckpt_path: Path, device: str, gate_tau: float = 2.0) -> Tuple[nn.Module, dict]:
    ckpt = load_ckpt_safe(ckpt_path, map_location="cpu")
    if "model" not in ckpt:
        raise ValueError(f"Checkpoint missing 'model' key: {ckpt_path}")
    sd = ckpt["model"]

    model = DualBranchGatedMIL(gate_tau=gate_tau).to(device)

    sd2 = remap_state_dict_for_pool(sd, model)
    missing, unexpected = model.load_state_dict(sd2, strict=False)

    print(f"[CKPT LOAD] {ckpt_path.name}")
    print(f"  missing: {len(missing)}  unexpected: {len(unexpected)}")
    if len(missing) < 15 and len(unexpected) < 15:
        if missing:    print("  missing sample:", missing[:10])
        if unexpected: print("  unexpected sample:", unexpected[:10])

    model.eval()
    return model, ckpt

# -----------------------------
# 7) Predictions per fold
# -----------------------------
@torch.no_grad()
def infer_fold(
    model: DualBranchGatedMIL,
    patient_ids: List[str],
    mapping: Dict[str, List[Path]],
    out_csv: Path,
    device: str,
    k_test: int
):
    ds = PatientBagDataset(patient_ids, mapping, img_size=cfg.IMG_SIZE)
    dl = DataLoader(ds, batch_size=1, shuffle=False,
                    num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY,
                    collate_fn=collate_patient_bag)

    rows = []
    for pids, bags, paths in dl:
        pid = pids[0]
        bag = bags[0]  # (N,3,H,W)
        if bag.numel() == 0:
            continue
        bag = bag.to(device)

        out = model.forward_bag(bag, topk=k_test)

        rows.append({
            "Patient": pid,
            "logit": float(out["logit_main"].cpu().item()),
            "prob":  float(out["prob_main"].cpu().item()),
            "gate_alpha": float(out["alpha"].cpu().item()),
            "leuko": float(out["leuko_prob"].cpu().item()),
        })

    df = pd.DataFrame(rows).set_index("Patient")
    df.to_csv(out_csv)
    print(f"Saved fold test preds: {out_csv}")
    return df

# -----------------------------
# 8) Robust fold aggregation (fixes your gate_alpha KeyError)
# -----------------------------
def _pick_first_existing(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def aggregate_folds_to_seed(
    runs_dir: Path,
    seed: int,
    benchmark: str,
    folds,
    k_test: int,
    out_path: Path
) -> pd.DataFrame:
    dfs = []

    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            raise FileNotFoundError(str(p))

        d = pd.read_csv(p)

        if "Patient" not in d.columns:
            alt = _pick_first_existing(d, ["patient", "patient_id", "PatientID"])
            if alt is None:
                raise ValueError(f"{p} has no Patient column. Columns: {list(d.columns)}")
            d = d.rename(columns={alt: "Patient"})
        d = d.set_index("Patient")

        # labels optional in folds; keep best effort
        if "y_true" not in d.columns:
            d["y_true"] = np.nan
        if "leuko" not in d.columns:
            d["leuko"] = 0.0

        logit_col = _pick_first_existing(d, ["logit", "logit_mean", "score", "pred_logit"])
        prob_col  = _pick_first_existing(d, ["prob", "prob_mean", "proba", "pred_prob"])
        gate_col  = _pick_first_existing(d, ["gate_alpha", "gate_mean", "alpha", "gate"])

        if logit_col is None:
            raise ValueError(f"{p} missing logit column. Columns: {list(d.columns)}")

        keep = ["y_true", "leuko", logit_col]
        if prob_col is not None: keep.append(prob_col)
        if gate_col is not None: keep.append(gate_col)

        d = d[keep].copy()
        d = d.rename(columns={
            logit_col: f"logit_fold{f}",
            **({prob_col: f"prob_fold{f}"} if prob_col else {}),
            **({gate_col: f"gate_fold{f}"} if gate_col else {}),
        })
        dfs.append(d)

    base = dfs[0][["y_true", "leuko"]].copy()
    for d in dfs:
        fold_cols = [c for c in d.columns if c.startswith(("logit_fold", "prob_fold", "gate_fold"))]
        base = base.join(d[fold_cols], how="inner")

    logit_cols = [c for c in base.columns if c.startswith("logit_fold")]
    base["logit_mean"] = base[logit_cols].mean(axis=1).astype(float)

    prob_cols = [c for c in base.columns if c.startswith("prob_fold")]
    if prob_cols:
        base["prob_mean"] = base[prob_cols].mean(axis=1).astype(float)
    else:
        base["prob_mean"] = 1.0 / (1.0 + np.exp(-base["logit_mean"].values.astype(float)))

    gate_cols = [c for c in base.columns if c.startswith("gate_fold")]
    base["gate_mean"] = base[gate_cols].mean(axis=1).astype(float) if gate_cols else np.nan

    base.to_csv(out_path)
    print(f"Saved: {out_path} | folds used: {list(folds)}")
    return base

# -----------------------------
# 9) FINAL submission writer
# -----------------------------
def write_final_submission(agg_df: pd.DataFrame, test_patients: List[str], out_path: Path):
    # Ensure only test patients, keep patient order as provided
    sub = agg_df.loc[[p for p in test_patients if p in agg_df.index]].copy()

    # competition usually expects Patient + probability; keep both prob and logit for traceability
    sub = sub.reset_index()[["Patient", "prob_mean", "logit_mean"]]
    sub = sub.rename(columns={"prob_mean": "prob"})
    sub.to_csv(out_path, index=False)
    print(f"Saved final submission: {out_path}")
    print(f"Patients: {len(sub)}")
    return sub

# -----------------------------
# 10) Task B: Grad-CAM (minimal, no external deps)
# -----------------------------
class GradCAM:
    def __init__(self, model: DualBranchGatedMIL, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.h1 = target_layer.register_forward_hook(self._forward_hook)
        self.h2 = target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        self.activations = out.detach()

    def _backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def remove(self):
        self.h1.remove()
        self.h2.remove()

    def cam(self, x: torch.Tensor, score: torch.Tensor) -> np.ndarray:
        """
        x: (1,3,H,W)
        score: scalar tensor to backprop
        """
        self.model.zero_grad(set_to_none=True)
        score.backward(retain_graph=True)

        # activations: (1,C,h,w) ; gradients: (1,C,h,w)
        A = self.activations
        G = self.gradients
        if A is None or G is None:
            raise RuntimeError("No activations/gradients captured. Wrong layer?")

        w = G.mean(dim=(2, 3), keepdim=True)      # (1,C,1,1)
        cam = (w * A).sum(dim=1, keepdim=True)    # (1,1,h,w)
        cam = F.relu(cam)
        cam = cam[0, 0].cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

def overlay_cam_on_pil(pil: Image.Image, cam: np.ndarray, alpha=0.45) -> Image.Image:
    # cam is (h,w) normalized; resize to image size
    cam_img = Image.fromarray(np.uint8(cam * 255)).resize(pil.size)
    cam_arr = np.array(cam_img).astype(np.float32) / 255.0

    base = np.array(pil).astype(np.float32) / 255.0

    # simple "red heat" overlay
    heat = np.zeros_like(base)
    heat[..., 0] = cam_arr  # red channel

    out = (1 - alpha) * base + alpha * heat
    out = np.clip(out * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(out)

@torch.no_grad()
def select_topk_frames(model: DualBranchGatedMIL, bag: torch.Tensor, k: int) -> torch.Tensor:
    out = model.forward_bag(bag, topk=k)
    return out["top_idx"], out

def run_task_b_explainability(
    model: DualBranchGatedMIL,
    patient_ids: List[str],
    mapping: Dict[str, List[Path]],
    out_dir: Path,
    device: str,
    topk_for_cam: int = 6,
):
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"[{now()}] Task B: saving CAM overlays to: {out_dir}")

    # Choose a reasonable conv layer for CAM: last conv block in ResNet layer4
    # Our encoder.backbone is children[:-1], so layer4 exists inside it.
    # Find last Bottleneck conv3 in layer4:
    # In torchvision resnet50: layer4 is backbone[-2] if avgpool is last; but our seq structure differs.
    # We'll grab the last module with name containing "layer4" if possible.
    target_layer = None
    for name, m in model.named_modules():
        if "encoder.backbone.7" in name and isinstance(m, nn.Conv2d):
            target_layer = m
    # fallback: last Conv2d
    if target_layer is None:
        for name, m in model.named_modules():
            if isinstance(m, nn.Conv2d):
                target_layer = m

    cam = GradCAM(model, target_layer)

    ds = PatientBagDataset(patient_ids, mapping, img_size=cfg.IMG_SIZE)
    dl = DataLoader(ds, batch_size=1, shuffle=False,
                    num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY,
                    collate_fn=collate_patient_bag)

    # IMPORTANT: GradCAM needs gradients => no @torch.no_grad
    model.eval()

    for pids, bags, paths in dl:
        pid = pids[0]
        bag = bags[0]
        img_paths = paths[0]
        if bag.numel() == 0:
            continue

        bag = bag.to(device)

        # get top-k indices using attention
        top_idx, out = select_topk_frames(model, bag, k=topk_for_cam)

        # For each selected frame, compute CAM by doing a forward on single image and backprop the desired score
        # We'll use main score for the bag by recomputing on single frame through encoder + pooled score isn't defined.
        # So: use frame-level proxy = classifier on frame feature (non branch) for CAM stability.
        # (scientifically acceptable proxy; report as "frame-level CAM proxy aligned with MIL attention selection")
        attn = out["attn_non"].detach().cpu().numpy()

        pid_dir = out_dir / pid
        pid_dir.mkdir(parents=True, exist_ok=True)

        for rank, idx in enumerate(top_idx.cpu().numpy()):
            img_path = img_paths[int(idx)]
            pil = Image.open(img_path).convert("RGB").resize((cfg.IMG_SIZE, cfg.IMG_SIZE))

            x = torch.from_numpy(np.array(pil)).float() / 255.0
            x = x.permute(2, 0, 1).unsqueeze(0).to(device)
            mean = torch.tensor([0.485, 0.456, 0.406], device=device)[:, None, None]
            std  = torch.tensor([0.229, 0.224, 0.225], device=device)[:, None, None]
            x = (x - mean) / std

            # forward through encoder with grads
            model.zero_grad(set_to_none=True)
            feats = model.encode_frames(x[0])  # WRONG shape if x[0]; keep x as (1,3,H,W)

            # fix: encode_frames expects (N,3,H,W)
            feats = model.encode_frames(x)  # (1,2048)
            pooled = feats[0]
            score = model.cls_non(pooled).squeeze()  # scalar

            # Need grads: temporarily enable grad
            score = score.requires_grad_()
            # But score came from ops; ensure graph exists: re-run without torch.no_grad (we are in normal)
            model.zero_grad(set_to_none=True)
            feats = model.encode_frames(x)            # (1,2048) with graph
            score = model.cls_non(feats[0]).squeeze() # scalar with graph

            cam_map = cam.cam(x, score)
            overlay = overlay_cam_on_pil(pil, cam_map, alpha=cfg.CAM_ALPHA)

            out_name = f"cam_rank{rank:02d}_attn{attn[int(idx)]:.4f}_{img_path.name}"
            overlay.save(pid_dir / out_name)

    cam.remove()
    print(f"[{now()}] Task B done.")

# -----------------------------
# 11) Task C: Domain shift (AUC per modality)
# -----------------------------
from sklearn.metrics import roc_auc_score

def run_task_c_domain_shift(agg_df: pd.DataFrame, test_meta: pd.DataFrame, out_json: Path):
    """
    Expects:
      agg_df indexed by Patient, includes y_true (if available) and logit_mean/prob_mean
      test_meta includes Patient + Domain + y_true (if available)
    """
    # Merge
    m = test_meta.copy()
    if "Patient" not in m.columns:
        raise ValueError("test_meta must contain Patient column.")
    if "Domain" not in m.columns:
        print("[Task C] No Domain column found. Skipping per-domain AUC.")
        return

    m = m.set_index("Patient")
    merged = m.join(agg_df[["prob_mean"]], how="inner")

    # We need labels for AUC; if not available, produce counts only
    ycol = None
    for c in ["y_true", "Label", "label", "target"]:
        if c in merged.columns:
            ycol = c
            break

    res = {
        "n_total": int(len(merged)),
        "domain_counts": merged["Domain"].fillna("").value_counts().to_dict(),
        "auc_overall": None,
        "per_domain": {}
    }

    if ycol is not None and merged[ycol].notna().any():
        y = merged[ycol].astype(float).values
        p = merged["prob_mean"].astype(float).values
        try:
            res["auc_overall"] = float(roc_auc_score(y, p))
        except Exception:
            res["auc_overall"] = None

        for dom in ["WLE", "NBI"]:
            dd = merged[merged["Domain"] == dom]
            if len(dd) < 5 or dd[ycol].nunique() < 2:
                res["per_domain"][dom] = {"n": int(len(dd)), "auc": None}
            else:
                res["per_domain"][dom] = {
                    "n": int(len(dd)),
                    "auc": float(roc_auc_score(dd[ycol].astype(float).values, dd["prob_mean"].astype(float).values))
                }
    else:
        # labels not present; report counts only
        res["per_domain"] = {
            "WLE": {"n": int((merged["Domain"] == "WLE").sum()), "auc": None},
            "NBI": {"n": int((merged["Domain"] == "NBI").sum()), "auc": None},
        }

    out_json.parent.mkdir(parents=True, exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2)

    print(f"[Task C] saved: {out_json}")
    print(json.dumps(res, indent=2))

# -----------------------------
# 12) Stats + sanity
# -----------------------------
def print_split_stats(name: str, patients: List[str], mapping: Dict[str, List[Path]]):
    counts = [len(mapping.get(p, [])) for p in patients]
    s = pd.Series(counts, dtype=float)
    print(f"\n=== {name} ===")
    print(f"Patients: {len(patients)}")
    print(f"Patients with 0 images: {(s==0).sum()}")
    print("Images per patient:", s.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

# -----------------------------
# 13) Main runner
# -----------------------------
def main():
    seed_everything(cfg.SEED)

    cfg.RUNS_DIR.mkdir(parents=True, exist_ok=True)

    # Load manifests
    df_tr = load_manifest(cfg.TRAIN_CSV)
    df_va = load_manifest(cfg.VAL_CSV)
    df_te = load_manifest(cfg.TEST_CSV)

    # Patient lists
    tr_p = sorted(df_tr["Patient"].astype(str).unique().tolist())
    va_p = sorted(df_va["Patient"].astype(str).unique().tolist())
    te_p = sorted(df_te["Patient"].astype(str).unique().tolist())

    # Build filesystem mapping once
    mapping = build_patient_image_mapping(cfg.IMAGES_ROOT)

    # Print stats (patient-level)
    print_split_stats("TRAIN", tr_p, mapping)
    print_split_stats("VAL", va_p, mapping)
    print_split_stats("TEST", te_p, mapping)

    # --- Per-fold inference (or skip if preds exist)
    for fold in cfg.FOLDS:
        pred_csv = cfg.RUNS_DIR / f"pred_fold{fold}_{cfg.BENCHMARK}_seed{cfg.SEED}_Ktest{cfg.K_TEST}.csv"
        if pred_csv.exists():
            print(f"[SKIP] fold {fold} preds exist: {pred_csv}")
            continue

        ckpt = cfg.RUNS_DIR / f"cv_best_{cfg.BENCHMARK}_seed{cfg.SEED}_fold{fold}.pt"
        if not ckpt.exists():
            # fallback to other naming you have used
            ckpt2 = cfg.RUNS_DIR / f"cv_best_{cfg.BENCHMARK}_seed{cfg.SEED}_fold{fold}.pt"
            if not ckpt2.exists():
                raise FileNotFoundError(f"Missing checkpoint for fold {fold}: {ckpt}")

        model, _ = build_model_from_ckpt(ckpt, device=cfg.DEVICE, gate_tau=2.0)
        infer_fold(model, te_p, mapping, pred_csv, device=cfg.DEVICE, k_test=cfg.K_TEST)

    # --- Aggregate folds to seed
    agg_path = cfg.RUNS_DIR / f"pred_seed{cfg.SEED}_{cfg.BENCHMARK}_folds1_5_Ktest{cfg.K_TEST}_agg.csv"
    agg_df = aggregate_folds_to_seed(
        runs_dir=cfg.RUNS_DIR,
        seed=cfg.SEED,
        benchmark=cfg.BENCHMARK,
        folds=cfg.FOLDS,
        k_test=cfg.K_TEST,
        out_path=agg_path
    )

    # --- Final submission
    final_path = cfg.RUNS_DIR / f"{cfg.FINAL_NAME}_seed{cfg.SEED}_{cfg.BENCHMARK}.csv"
    sub = write_final_submission(agg_df, te_p, final_path)

    # --- Task B (Explainability)
    if cfg.DO_TASK_B:
        # use best available ckpt for CAM (fold1 preferred)
        fold1_ckpt = cfg.RUNS_DIR / f"cv_best_{cfg.BENCHMARK}_seed{cfg.SEED}_fold1.pt"
        if not fold1_ckpt.exists():
            # fallback: any cv_best for this seed
            cands = sorted(cfg.RUNS_DIR.glob(f"cv_best_{cfg.BENCHMARK}_seed{cfg.SEED}_fold*.pt"))
            if len(cands) == 0:
                print("[Task B] No fold checkpoints found. Skipping.")
            else:
                fold1_ckpt = cands[0]

        if fold1_ckpt.exists():
            model, _ = build_model_from_ckpt(fold1_ckpt, device=cfg.DEVICE, gate_tau=2.0)
            run_task_b_explainability(
                model=model,
                patient_ids=te_p,
                mapping=mapping,
                out_dir=cfg.TASKB_DIR,
                device=cfg.DEVICE,
                topk_for_cam=cfg.TOPK_FOR_CAM
            )

    # --- Task C (Domain shift)
    if cfg.DO_TASK_C:
        meta_path = cfg.TEST_WITH_DOMAIN_CSV or cfg.TEST_CSV
        meta = pd.read_csv(meta_path)

        # Ensure Patient/Domain exist
        if "Patient" not in meta.columns:
            for c in ["patient", "patient_id", "PatientID"]:
                if c in meta.columns:
                    meta = meta.rename(columns={c: "Patient"})
                    break

        out_json = cfg.RUNS_DIR / f"TASKC_domain_shift_seed{cfg.SEED}_{cfg.BENCHMARK}.json"
        run_task_c_domain_shift(agg_df, meta, out_json)

    print("\nDONE.")
    print("Agg:", agg_path)
    print("Final:", final_path)
    if cfg.DO_TASK_B:
        print("TaskB dir:", cfg.TASKB_DIR)

if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs

=== TRAIN ===
Patients: 117
Patients with 0 images: 0
Images per patient: count    117.000000
mean      50.247863
std       32.454187
min        2.000000
50%       42.000000
75%       72.000000
90%       90.400000
95%       96.800000
99%      144.720000
max      186.000000
dtype: float64

=== VAL ===
Patients: 20
Patients with 0 images: 0
Images per patient: count    20.000000
mean     41.750000
std      23.212689
min       9.000000
50%      40.000000
75%      57.750000
90%      71.000000
95%      80.350000
99%      85.670000
max      87.000000
dtype: float64

=== TEST ===
Patients: 73
Patients with 0 images: 0
Images per patient: count     73.000000
mean      66.739726
std       31.082787
min        4.000000
50%       69.000000
75%       86.000000
90%      100.600000
95%      119.600000
99%      135.240000
max      141.000000
dtype: float64
[SKIP] fold 1 preds exist: D:\ENT\_challenge2_artifacts\_runs\pre

In [2]:
# ============================================================
# Scientifically-clean FULL PIPELINE (Tasks A+B+C) for ENT endoscopy
# - Patient-level stats + filesystem inventory
# - Robust manifest loader (works even if train/val/test csv has no 'path')
# - Patient->image mapping from filesystem (folder-segment)
# - Loads your fold1 checkpoint model architecture from ckpt keys
# - Aggregates fold predictions safely (no overlapping-column errors)
# - Produces FINAL submission CSV
# - TASK B: Grad-CAM overlays on top-attended instances (robust logging, num_workers=0)
# - TASK C: Domain shift analysis by inferring WLE/NBI from image paths
#
# Works with:
#   DEVICE: cuda:0
#   IMAGES_ROOT: D:\ENT
#   RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs
#
# NOTE:
#   - Assumes your fold prediction files already exist:
#       pred_fold{1..5}_res_shift_seed43_Ktest10.csv
#     If they don't, you need to run training/inference first.
# ============================================================

import os, re, json, time, math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# CONFIG
# -----------------------------
@dataclass
class CFG:
    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"

    # Paths
    IMAGES_ROOT: Path = Path(r"D:\ENT")
    ARTIFACTS_ROOT: Path = Path(r"D:\ENT\_challenge2_artifacts")
    RUNS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    # Your benchmark naming
    BENCHMARK: str = "res_shift"
    SEED: int = 43
    FOLDS: Tuple[int, ...] = (1, 2, 3, 4, 5)
    K_TEST: int = 10

    # Fold1 checkpoint for Task B model loading
    FOLD1_CKPT: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold1.pt")

    # Task toggles
    DO_TASK_B: bool = True
    DO_TASK_C: bool = True

    # Task B settings
    IMG_SIZE: int = 224
    TOPK_FOR_CAM: int = 6
    CAM_ALPHA: float = 0.45
    TASKB_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\TASKB_res_shift_seed43")

    # Outputs
    AGG_PATH: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv")
    FINAL_PATH: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\FINAL_seed43_res_shift.csv")
    TASKC_JSON: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\TASKC_domain_shift_seed43_res_shift.json")


cfg = CFG()

# -----------------------------
# UTIL
# -----------------------------
def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_everything(seed: int = 43):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def print_header():
    print(f"DEVICE: {cfg.DEVICE}")
    print(f"IMAGES_ROOT: {cfg.IMAGES_ROOT}")
    print(f"RUNS_DIR: {cfg.RUNS_DIR}")

# -----------------------------
# FILESYSTEM INVENTORY
# -----------------------------
def inventory_tree(root: Path) -> pd.DataFrame:
    rows = []
    for p in root.rglob("*"):
        if p.is_file():
            rows.append((p.suffix.lower(), 1, p.stat().st_size))
    if not rows:
        return pd.DataFrame(columns=["ext", "count", "bytes"])
    df = pd.DataFrame(rows, columns=["ext", "count", "bytes"])
    out = df.groupby("ext", as_index=False).agg(count=("count","sum"), bytes=("bytes","sum"))
    out = out.sort_values(["count","bytes"], ascending=False)
    return out

# -----------------------------
# PATIENT ID EXTRACTION (folder-segment)
# -----------------------------
_PAT_RE = re.compile(r"(patient\s*[_-]?\s*\d+|patient\d+)", re.IGNORECASE)

def extract_patient_id_from_path(p: Path) -> Optional[str]:
    # Uses folder names and filename to find "Patient###"
    parts = list(p.parts)
    for s in parts[::-1]:
        m = _PAT_RE.search(s)
        if m:
            token = m.group(0)
            token = token.replace(" ", "").replace("_", "").replace("-", "")
            # normalize "patient099" -> "Patient099"
            token = token.lower().replace("patient", "")
            if token.isdigit():
                return f"Patient{int(token):03d}"
            # if already like 099
            try:
                n = int(token)
                return f"Patient{n:03d}"
            except:
                pass
    # last attempt: search in full string
    m = _PAT_RE.search(str(p))
    if m:
        token = m.group(0)
        token = token.replace(" ", "").replace("_", "").replace("-", "")
        token = token.lower().replace("patient", "")
        if token.isdigit():
            return f"Patient{int(token):03d}"
    return None

def build_patient_image_mapping(images_root: Path, exts=(".jpg",".jpeg",".png",".bmp",".tif",".tiff")) -> Dict[str, List[Path]]:
    mapping: Dict[str, List[Path]] = {}
    all_files = []
    for ext in exts:
        all_files.extend(list(images_root.rglob(f"*{ext}")))
    for f in all_files:
        pid = extract_patient_id_from_path(f)
        if pid is None:
            continue
        mapping.setdefault(pid, []).append(f)
    # sort each patient list for determinism
    for pid in mapping:
        mapping[pid] = sorted(mapping[pid])
    return mapping

def describe_patient_split(patients: List[str], mapping: Dict[str, List[Path]], name: str):
    n_imgs = [len(mapping.get(pid, [])) for pid in patients]
    s = pd.Series(n_imgs)
    print(f"\n=== {name} ===")
    print(f"Patients: {len(patients)}")
    print(f"Patients with 0 images: {(s==0).sum()}")
    print("Images per patient:", s.describe(percentiles=[0.5,0.75,0.9,0.95,0.99]))

# -----------------------------
# MANIFEST LOADER (robust)
# -----------------------------
def load_manifest_any(csv_path: Path) -> pd.DataFrame:
    """
    Loads a split CSV even if it lacks 'path'.
    It must contain at least Patient column; label columns optional.
    Accepts common variants:
      - Patient / patient / patient_id / PatientID
      - Histopathology / label / y_true / target
      - Leukoplakia / leuko / leuko_flag
    """
    df = pd.read_csv(csv_path)

    # Patient
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID", "patientid"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"{csv_path} must contain a Patient column (or patient/patient_id/PatientID).")

    # Label (optional)
    if "y_true" not in df.columns:
        for c in ["label", "target", "Histopathology", "histopathology", "Class", "class"]:
            if c in df.columns:
                df = df.rename(columns={c: "y_true"})
                break

    # Leukoplakia (optional)
    if "leuko" not in df.columns:
        for c in ["Leukoplakia", "leukoplakia", "leuko_flag", "Leuko"]:
            if c in df.columns:
                df = df.rename(columns={c: "leuko"})
                break

    return df

def coerce_leuko(series: pd.Series) -> pd.Series:
    """
    Converts Leukoplakia field to 0/1 if it's Yes/No, True/False, 0/1, etc.
    """
    if series is None:
        return series
    s = series.copy()
    if s.dtype == object:
        s2 = s.astype(str).str.strip().str.lower()
        s2 = s2.replace({
            "yes": 1, "y": 1, "true": 1, "1": 1,
            "no": 0, "n": 0, "false": 0, "0": 0,
            "nan": np.nan, "none": np.nan, "": np.nan
        })
        return pd.to_numeric(s2, errors="coerce")
    return pd.to_numeric(s, errors="coerce")

# -----------------------------
# MODEL ARCH (from your ckpt keys)
# -----------------------------
class AttnPool(nn.Module):
    """
    Standard attention pooling: a_i = softmax(w^T tanh(Vh_i))
    """
    def __init__(self, in_dim: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, H: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # H: (N, D)
        a = self.net(H)                   # (N,1)
        a = torch.softmax(a.squeeze(1), dim=0)  # (N,)
        z = (a.unsqueeze(1) * H).sum(dim=0)     # (D,)
        return z, a

class ResNet50Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        # build ResNet50 backbone "like torchvision resnet50", without fc
        from torchvision.models import resnet50
        m = resnet50(weights=None)
        # keep children up to avgpool
        self.backbone = nn.Sequential(
            m.conv1, m.bn1, m.relu, m.maxpool,
            m.layer1, m.layer2, m.layer3, m.layer4,
            m.avgpool
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,3,H,W)
        f = self.backbone(x)              # (B,2048,1,1)
        f = f.flatten(1)                  # (B,2048)
        return f

class DualBranchGatedMIL(nn.Module):
    """
    Matches your checkpoint prefixes:
      encoder.*
      pool_non.* (AttnPool)
      pool_leu.*
      cls_non.*
      cls_leu.*
      gate.*
      leuko_head.*
    """
    def __init__(self, instance_dim=2048, pool_hidden=256, gate_in_dim=4096, leuko_in_dim=4096, gate_tau=2.0):
        super().__init__()
        self.encoder = ResNet50Encoder()
        self.pool_non = AttnPool(instance_dim, pool_hidden)
        self.pool_leu = AttnPool(instance_dim, pool_hidden)
        self.cls_non = nn.Linear(instance_dim, 1, bias=True)
        self.cls_leu = nn.Linear(instance_dim, 1, bias=True)
        self.gate = nn.Linear(gate_in_dim, 1, bias=True)
        self.leuko_head = nn.Sequential(
            nn.Linear(leuko_in_dim, pool_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(pool_hidden, 1)
        )
        self.gate_tau = gate_tau

    @torch.no_grad()
    def encode_frames(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

    def forward_bag(self, bag: torch.Tensor, topk: int = 10) -> Dict[str, torch.Tensor]:
        """
        bag: (N,3,H,W)
        returns dict with logits + attention + top-k indices
        """
        H = self.encoder(bag)  # (N,2048)

        z_non, a_non = self.pool_non(H)
        z_leu, a_leu = self.pool_leu(H)

        logit_non = self.cls_non(z_non)  # (1,)
        logit_leu = self.cls_leu(z_leu)  # (1,)

        g_in = torch.cat([z_non, z_leu], dim=0)        # (4096,)
        gate_logit = self.gate(g_in)                   # (1,)
        gate_alpha = torch.sigmoid(gate_logit / self.gate_tau)  # (1,)

        logit = gate_alpha * logit_non + (1.0 - gate_alpha) * logit_leu

        # top-k based on non branch attention (typical)
        topk = min(topk, H.shape[0])
        top_idx = torch.topk(a_non, k=topk, largest=True).indices

        # leuko auxiliary prediction from concatenated branch reps
        leuko_logit = self.leuko_head(g_in)

        return {
            "logit": logit.squeeze(),
            "prob": torch.sigmoid(logit.squeeze()),
            "logit_non": logit_non.squeeze(),
            "logit_leu": logit_leu.squeeze(),
            "gate_alpha": gate_alpha.squeeze(),
            "attn_non": a_non,
            "attn_leu": a_leu,
            "top_idx": top_idx,
            "leuko_logit": leuko_logit.squeeze(),
        }

# -----------------------------
# CHECKPOINT LOADER (infers dims + remaps pool keys if needed)
# -----------------------------
def build_model_from_ckpt(ckpt_path: Path, device: str, gate_tau: float = 2.0) -> Tuple[DualBranchGatedMIL, dict]:
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    if not isinstance(ckpt, dict) or "model" not in ckpt:
        raise ValueError("Checkpoint must be a dict with key 'model'.")

    sd = ckpt["model"]

    # Infer instance_dim from cls_non.weight shape: (1, D)
    if "cls_non.weight" in sd:
        instance_dim = sd["cls_non.weight"].shape[1]
    else:
        raise KeyError("Checkpoint missing cls_non.weight")

    # Infer pool_hidden from leuko_head.0.weight: (H, 4096) where H=pool_hidden
    if "leuko_head.0.weight" in sd:
        pool_hidden = sd["leuko_head.0.weight"].shape[0]
        leuko_in_dim = sd["leuko_head.0.weight"].shape[1]
    else:
        raise KeyError("Checkpoint missing leuko_head.0.weight")

    # Infer gate input dim
    if "gate.weight" in sd:
        gate_in_dim = sd["gate.weight"].shape[1]
    else:
        raise KeyError("Checkpoint missing gate.weight")

    model = DualBranchGatedMIL(
        instance_dim=instance_dim,
        pool_hidden=pool_hidden,
        gate_in_dim=gate_in_dim,
        leuko_in_dim=leuko_in_dim,
        gate_tau=gate_tau
    ).to(device)

    # Some checkpoints store pool params as pool_non.V / pool_non.w instead of pool_non.net.*
    # Remap if needed.
    model_keys = set(model.state_dict().keys())
    sd2 = {}
    for k, v in sd.items():
        nk = k

        # Remap old naming: pool_non.V.* -> pool_non.net.0.* ; pool_non.w.* -> pool_non.net.2.*
        nk = nk.replace("pool_non.V.", "pool_non.net.0.")
        nk = nk.replace("pool_non.w.", "pool_non.net.2.")
        nk = nk.replace("pool_leu.V.", "pool_leu.net.0.")
        nk = nk.replace("pool_leu.w.", "pool_leu.net.2.")

        # Sometimes leuko_head has extra layer index shift (e.g., .3 instead of .2)
        # We'll keep as-is; strict=False will ignore mismatches if any.
        sd2[nk] = v

    missing, unexpected = model.load_state_dict(sd2, strict=False)
    print(f"[CKPT LOAD] {ckpt_path.name}")
    print(f"  missing: {len(missing)}  unexpected: {len(unexpected)}")
    if len(missing) > 0:
        print("  missing sample:", missing[:10])
    if len(unexpected) > 0:
        print("  unexpected sample:", unexpected[:10])

    model.eval()
    return model, ckpt

# -----------------------------
# AGGREGATE FOLDS SAFELY
# -----------------------------
def read_fold_pred(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"{path} must have Patient column.")

    # common columns:
    # y_true, leuko, logit, prob, (optionally gate_alpha)
    if "prob" not in df.columns:
        # try prob_mean or p
        for c in ["prob_mean", "p", "pred", "proba"]:
            if c in df.columns:
                df = df.rename(columns={c:"prob"})
                break
    if "logit" not in df.columns:
        for c in ["logit_mean", "logits", "score"]:
            if c in df.columns:
                df = df.rename(columns={c:"logit"})
                break

    keep = ["Patient"]
    for c in ["y_true", "leuko", "logit", "prob", "gate_alpha"]:
        if c in df.columns:
            keep.append(c)

    df = df[keep].copy()
    df = df.set_index("Patient")
    return df

def aggregate_folds_to_seed(
    runs_dir: Path,
    seed: int,
    benchmark: str,
    folds: Tuple[int, ...],
    k_test: int,
    out_path: Path
) -> pd.DataFrame:
    dfs = []
    used_folds = []

    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            print(f"[WARN] missing fold file: {p}")
            continue
        d = read_fold_pred(p)

        # Rename fold-specific columns
        ren = {}
        if "logit" in d.columns: ren["logit"] = f"logit_fold{f}"
        if "prob" in d.columns:  ren["prob"]  = f"prob_fold{f}"
        if "gate_alpha" in d.columns: ren["gate_alpha"] = f"gate_fold{f}"
        d = d.rename(columns=ren)

        dfs.append(d)
        used_folds.append(f)

    if len(dfs) == 0:
        raise RuntimeError("No fold prediction files found.")

    # Start from first fold, keep y_true/leuko once, then join only fold-specific cols thereafter
    base = dfs[0].copy()

    # Ensure y_true and leuko exist (optional)
    if "y_true" in base.columns:
        base["y_true"] = pd.to_numeric(base["y_true"], errors="coerce")
    if "leuko" in base.columns:
        base["leuko"] = coerce_leuko(base["leuko"])

    for d in dfs[1:]:
        cols_to_add = [c for c in d.columns if c.startswith("logit_fold") or c.startswith("prob_fold") or c.startswith("gate_fold")]
        base = base.join(d[cols_to_add], how="inner")

    # Mean over folds
    logit_cols = [c for c in base.columns if c.startswith("logit_fold")]
    prob_cols  = [c for c in base.columns if c.startswith("prob_fold")]
    gate_cols  = [c for c in base.columns if c.startswith("gate_fold")]

    base["logit_mean"] = base[logit_cols].mean(axis=1).astype(float)
    if len(prob_cols) > 0:
        base["prob_mean"]  = base[prob_cols].mean(axis=1).astype(float)
    else:
        base["prob_mean"] = 1.0 / (1.0 + np.exp(-base["logit_mean"].astype(float)))

    if len(gate_cols) > 0:
        base["gate_mean"]  = base[gate_cols].mean(axis=1).astype(float)

    safe_mkdir(out_path.parent)
    base.reset_index().to_csv(out_path, index=False)
    print(f"Saved: {out_path} | folds used: {used_folds}")
    return base

# -----------------------------
# FINAL SUBMISSION
# -----------------------------
def make_final_submission(agg_df: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    # Minimal: Patient + prob_mean
    sub = agg_df.reset_index().rename(columns={"prob_mean": "Probability"})[["Patient", "Probability"]]
    sub.to_csv(out_path, index=False)
    print(f"Saved final submission: {out_path}")
    print(f"Patients: {len(sub)}")
    return sub

# -----------------------------
# TASK B: Grad-CAM on top-attended instances
# -----------------------------
def imagenet_norm(x: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor([0.485, 0.456, 0.406], device=x.device)[None, :, None, None]
    std  = torch.tensor([0.229, 0.224, 0.225], device=x.device)[None, :, None, None]
    return (x - mean) / std

def pil_to_tensor(pil: Image.Image, img_size: int, device: str) -> torch.Tensor:
    pil = pil.convert("RGB").resize((img_size, img_size))
    arr = np.array(pil).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(device)
    return imagenet_norm(x)

def overlay_cam_on_pil(pil: Image.Image, cam: np.ndarray, alpha=0.45) -> Image.Image:
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam_img = Image.fromarray(np.uint8(cam * 255)).resize(pil.size)
    cam_arr = np.array(cam_img).astype(np.float32) / 255.0

    base = np.array(pil).astype(np.float32) / 255.0
    heat = np.zeros_like(base)
    heat[..., 0] = cam_arr  # red overlay

    out = (1 - alpha) * base + alpha * heat
    out = np.clip(out * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(out)

class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.h1 = target_layer.register_forward_hook(self._fh)
        self.h2 = target_layer.register_full_backward_hook(self._bh)

    def _fh(self, m, inp, out):
        self.activations = out

    def _bh(self, m, gin, gout):
        self.gradients = gout[0]

    def remove(self):
        self.h1.remove()
        self.h2.remove()

    def compute(self, score: torch.Tensor) -> np.ndarray:
        self.model.zero_grad(set_to_none=True)
        score.backward(retain_graph=False)

        A = self.activations
        G = self.gradients
        if A is None or G is None:
            raise RuntimeError("GradCAM: missing activations/gradients. Wrong layer or no backward executed.")

        w = G.mean(dim=(2,3), keepdim=True)
        cam = (w * A).sum(dim=1, keepdim=True)
        cam = F.relu(cam)[0,0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

def pick_last_conv(model: nn.Module) -> nn.Module:
    last_conv = None
    for _, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            last_conv = m
    if last_conv is None:
        raise RuntimeError("Could not find Conv2d in model for CAM.")
    return last_conv

def run_task_b_explainability(
    model: DualBranchGatedMIL,
    patient_ids: List[str],
    mapping: Dict[str, List[Path]],
    out_dir: Path,
    device: str,
    img_size: int = 224,
    k_test: int = 10,
    topk_for_cam: int = 6,
    alpha: float = 0.45,
    log_every: int = 5
) -> pd.DataFrame:
    out_dir = Path(out_dir)
    safe_mkdir(out_dir)

    model = model.to(device)
    model.eval()

    target_layer = pick_last_conv(model)
    cam = GradCAM(model, target_layer)

    print(f"[{now()}] Task B: saving CAM overlays to: {out_dir}")

    rows = []
    torch.set_grad_enabled(True)

    for i, pid in enumerate(patient_ids, 1):
        paths = mapping.get(pid, [])
        if len(paths) == 0:
            rows.append({"Patient": pid, "n_images": 0, "n_saved": 0})
            continue

        # Build bag (N,3,H,W) normalized
        with torch.no_grad():
            xs = []
            for p in paths:
                pil = Image.open(p).convert("RGB").resize((img_size, img_size))
                arr = np.array(pil).astype(np.float32) / 255.0
                x = torch.from_numpy(arr).permute(2,0,1)
                xs.append(x)
            bag = torch.stack(xs, dim=0).to(device)
            bag = imagenet_norm(bag)

            out = model.forward_bag(bag, topk=k_test)
            top_idx = out["top_idx"].detach().cpu().numpy()
            attn = out["attn_non"].detach().cpu().numpy()

        pid_dir = out_dir / pid
        safe_mkdir(pid_dir)

        saved = 0
        for rank, idx in enumerate(top_idx[:topk_for_cam]):
            img_path = paths[int(idx)]
            pil = Image.open(img_path).convert("RGB").resize((img_size, img_size))
            x = pil_to_tensor(pil, img_size, device)  # (1,3,H,W)

            # frame-level proxy score using non branch classifier
            feats = model.encoder(x)                  # (1,2048)
            score = model.cls_non(feats[0]).squeeze() # scalar tensor

            cam_map = cam.compute(score)
            overlay = overlay_cam_on_pil(pil, cam_map, alpha=alpha)

            out_name = f"cam_rank{rank:02d}_attn{attn[int(idx)]:.4f}_{Path(img_path).name}"
            overlay.save(pid_dir / out_name)
            saved += 1

        rows.append({"Patient": pid, "n_images": len(paths), "n_saved": saved})

        if (i % log_every) == 0 or i == 1 or i == len(patient_ids):
            total_saved = sum(r["n_saved"] for r in rows)
            print(f"[{now()}] Task B progress: {i}/{len(patient_ids)} patients | overlays saved={total_saved}")

    cam.remove()
    torch.set_grad_enabled(False)

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "TASKB_summary.csv", index=False)
    print(f"[{now()}] Task B done.")
    return df

# -----------------------------
# TASK C: Domain shift analysis (infer WLE/NBI from paths)
# -----------------------------
def infer_patient_domain_from_paths(paths: List[Path]) -> str:
    tags = set()
    for p in paths:
        s = str(p).lower()
        if "nbi" in s:
            tags.add("NBI")
        if "wle" in s:
            tags.add("WLE")
    if len(tags) == 0:
        return "UNKNOWN"
    if len(tags) == 1:
        return list(tags)[0]
    return "MIXED"

def auc_safe(y: np.ndarray, p: np.ndarray) -> Optional[float]:
    from sklearn.metrics import roc_auc_score
    if len(y) < 5:
        return None
    if len(np.unique(y[~np.isnan(y)])) < 2:
        return None
    mask = ~np.isnan(y) & ~np.isnan(p)
    if mask.sum() < 5 or len(np.unique(y[mask])) < 2:
        return None
    return float(roc_auc_score(y[mask], p[mask]))

def run_task_c_domain_shift(agg_df: pd.DataFrame, patient_ids: List[str], mapping: Dict[str, List[Path]], out_json: Path):
    meta = []
    for pid in patient_ids:
        dom = infer_patient_domain_from_paths(mapping.get(pid, []))
        meta.append((pid, dom))
    meta = pd.DataFrame(meta, columns=["Patient", "Domain"]).set_index("Patient")

    df = meta.join(agg_df[["prob_mean"]], how="inner")
    if "y_true" in agg_df.columns:
        df = df.join(agg_df[["y_true"]], how="left")

    res = {
        "n_total": int(len(df)),
        "domain_counts": df["Domain"].value_counts().to_dict(),
        "auc_overall": None,
        "per_domain": {}
    }

    if "y_true" in df.columns:
        y = pd.to_numeric(df["y_true"], errors="coerce").values.astype(float)
        p = pd.to_numeric(df["prob_mean"], errors="coerce").values.astype(float)
        res["auc_overall"] = auc_safe(y, p)

        for dom in ["WLE", "NBI", "MIXED", "UNKNOWN"]:
            dd = df[df["Domain"] == dom]
            y_d = pd.to_numeric(dd.get("y_true", pd.Series([np.nan]*len(dd))), errors="coerce").values.astype(float)
            p_d = pd.to_numeric(dd["prob_mean"], errors="coerce").values.astype(float)
            res["per_domain"][dom] = {"n": int(len(dd)), "auc": auc_safe(y_d, p_d)}
    else:
        for dom in ["WLE", "NBI", "MIXED", "UNKNOWN"]:
            res["per_domain"][dom] = {"n": int((df["Domain"] == dom).sum()), "auc": None}

    safe_mkdir(out_json.parent)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2)

    print(f"[Task C] saved: {out_json}")
    print(json.dumps(res, indent=2))
    return res, df

# -----------------------------
# MAIN
# -----------------------------
def main():
    seed_everything(cfg.SEED)
    print_header()

    # 1) Build patient->image mapping from filesystem
    mapping = build_patient_image_mapping(cfg.IMAGES_ROOT)
    if len(mapping) == 0:
        raise RuntimeError(f"No patient images found under {cfg.IMAGES_ROOT} (check path and naming).")

    # 2) If you have split CSVs, load them (optional) - otherwise we just run on agg patients.
    #    You uploaded split_res_shift_train_major_colab.csv to /mnt/data too; on your PC use D:\... paths.
    #    Here: we only need test patients for Task B/C and final output.
    #    We'll infer test patients from aggregated predictions.

    # 3) Aggregate folds to seed
    agg_df = aggregate_folds_to_seed(
        runs_dir=cfg.RUNS_DIR,
        seed=cfg.SEED,
        benchmark=cfg.BENCHMARK,
        folds=cfg.FOLDS,
        k_test=cfg.K_TEST,
        out_path=cfg.AGG_PATH
    )

    # 4) Final submission
    sub = make_final_submission(agg_df, cfg.FINAL_PATH)

    # test patients
    te_p = sub["Patient"].astype(str).tolist()

    # 5) Patient split stats (for the 73 test patients)
    describe_patient_split(te_p, mapping, "TEST")

    # 6) Task B (Explainability)
    if cfg.DO_TASK_B:
        if not cfg.FOLD1_CKPT.exists():
            raise FileNotFoundError(f"Fold1 ckpt not found: {cfg.FOLD1_CKPT}")
        model, _ = build_model_from_ckpt(cfg.FOLD1_CKPT, device=cfg.DEVICE, gate_tau=2.0)

        run_task_b_explainability(
            model=model,
            patient_ids=te_p,
            mapping=mapping,
            out_dir=cfg.TASKB_DIR,
            device=cfg.DEVICE,
            img_size=cfg.IMG_SIZE,
            k_test=cfg.K_TEST,
            topk_for_cam=cfg.TOPK_FOR_CAM,
            alpha=cfg.CAM_ALPHA,
            log_every=5
        )

    # 7) Task C (Domain shift)
    if cfg.DO_TASK_C:
        run_task_c_domain_shift(
            agg_df=agg_df,
            patient_ids=te_p,
            mapping=mapping,
            out_json=cfg.TASKC_JSON
        )

    print("\nDONE.")
    print("Agg:", cfg.AGG_PATH)
    print("Final:", cfg.FINAL_PATH)
    if cfg.DO_TASK_B:
        print("TaskB dir:", cfg.TASKB_DIR)
    if cfg.DO_TASK_C:
        print("TaskC json:", cfg.TASKC_JSON)

if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs
Saved: D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv | folds used: [1, 2, 3, 4, 5]
Saved final submission: D:\ENT\_challenge2_artifacts\_runs\FINAL_seed43_res_shift.csv
Patients: 73

=== TEST ===
Patients: 73
Patients with 0 images: 0
Images per patient: count     73.000000
mean      66.739726
std       31.082787
min        4.000000
50%       69.000000
75%       86.000000
90%      100.600000
95%      119.600000
99%      135.240000
max      141.000000
dtype: float64


C:\Users\admin\AppData\Local\Temp\ipykernel_14828\2104212108.py:336: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_location="cpu")


[CKPT LOAD] cv_best_res_shift_seed43_fold1.pt
  missing: 2  unexpected: 2
  missing sample: ['leuko_head.2.weight', 'leuko_head.2.bias']
  unexpected sample: ['leuko_head.3.weight', 'leuko_head.3.bias']
[2026-02-27 22:07:20] Task B: saving CAM overlays to: D:\ENT\_challenge2_artifacts\_runs\TASKB_res_shift_seed43


C:\Users\admin\miniconda3\envs\tensorflow\lib\site-packages\torch\autograd\graph.py:825: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[2026-02-27 22:07:25] Task B progress: 1/73 patients | overlays saved=6
[2026-02-27 22:07:33] Task B progress: 5/73 patients | overlays saved=30
[2026-02-27 22:07:43] Task B progress: 10/73 patients | overlays saved=60
[2026-02-27 22:07:52] Task B progress: 15/73 patients | overlays saved=90
[2026-02-27 22:08:01] Task B progress: 20/73 patients | overlays saved=120
[2026-02-27 22:08:12] Task B progress: 25/73 patients | overlays saved=150
[2026-02-27 22:08:20] Task B progress: 30/73 patients | overlays saved=180
[2026-02-27 22:08:28] Task B progress: 35/73 patients | overlays saved=210
[2026-02-27 22:08:37] Task B progress: 40/73 patients | overlays saved=240
[2026-02-27 22:08:49] Task B progress: 45/73 patients | overlays saved=270
[2026-02-27 22:08:59] Task B progress: 50/73 patients | overlays saved=300
[2026-02-27 22:09:15] Task B progress: 55/73 patients | overlays saved=330
[2026-02-27 22:09:25] Task B progress: 60/73 patients | overlays saved=358
[2026-02-27 22:09:33] Task B pro

In [5]:
import os
import json
import time
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models.resnet import Bottleneck

# ---------------------------
# Reproducibility
# ---------------------------
def seed_everything(seed: int = 43):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

def now_str():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def read_image_rgb(path: Path) -> Image.Image:
    with Image.open(path) as im:
        return im.convert("RGB")

def list_files_recursive(root: Path, exts=(".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")) -> List[Path]:
    out = []
    for dp, _, fnames in os.walk(root):
        for f in fnames:
            if f.lower().endswith(exts):
                out.append(Path(dp) / f)
    return out

# ---------------------------
# Data: manifests + patient mapping
# ---------------------------
def load_manifest(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"{csv_path} missing Patient column")

    # Optional domain column
    if "Domain" not in df.columns:
        for c in ["domain", "Modality", "modality"]:
            if c in df.columns:
                df = df.rename(columns={c: "Domain"})
                break

    df["Patient"] = df["Patient"].astype(str)
    if "Domain" in df.columns:
        df["Domain"] = df["Domain"].astype(str).str.strip().replace({"": "UNKNOWN"})
    return df

def build_patient_mapping(images_root: Path) -> Dict[str, List[Path]]:
    all_imgs = list_files_recursive(images_root)
    mapping: Dict[str, List[Path]] = {}
    for p in all_imgs:
        pid = None
        for seg in p.parts:
            s = seg.lower().replace(" ", "")
            if s.startswith("patient") and any(ch.isdigit() for ch in s):
                pid = seg.replace(" ", "")
                break
        if pid is None:
            continue
        mapping.setdefault(pid, []).append(p)
    for k in list(mapping.keys()):
        mapping[k] = sorted(mapping[k])
    return mapping

def patient_stats_from_mapping(patient_ids: List[str], mapping: Dict[str, List[Path]], title: str):
    counts = []
    missing = 0
    for pid in patient_ids:
        n = len(mapping.get(pid, []))
        if n == 0:
            missing += 1
        counts.append(n)
    s = pd.Series(counts, dtype=float)

    print(f"\n=== {title} ===")
    print(f"Patients: {len(patient_ids)}")
    print(f"Patients with 0 images: {missing}")
    print("Images per patient:", s.describe())

# ---------------------------
# Model blocks: Attention Pool
# ---------------------------
class AttnPool(nn.Module):
    """
    Your ckpt uses pool_non.V / pool_non.w naming.
    We'll implement those exact parameter names so strict load is possible.
    """
    def __init__(self, in_dim: int, hidden: int):
        super().__init__()
        self.V = nn.Linear(in_dim, hidden)
        self.w = nn.Linear(hidden, 1)

    def forward(self, H: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # H: [N, D]
        a = self.w(torch.tanh(self.V(H)))  # [N,1]
        a = torch.softmax(a.squeeze(-1), dim=0)  # [N]
        z = (a.unsqueeze(-1) * H).sum(dim=0)     # [D]
        return z, a

# ---------------------------
# Encoder: Sequential-style ResNet50
# Matches checkpoint keys like:
# encoder.backbone.0.weight, 1.weight, 4.0.conv1.weight, ..., 8.*
# ---------------------------
class SequentialResNet50Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        # ResNet50 stem
        conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        bn1   = nn.BatchNorm2d(64)
        relu  = nn.ReLU(inplace=True)
        maxp  = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # layers with Bottleneck blocks: [3,4,6,3]
        layer1 = self._make_layer(64,  64, blocks=3, stride=1)   # out 256
        layer2 = self._make_layer(256, 128, blocks=4, stride=2)  # out 512
        layer3 = self._make_layer(512, 256, blocks=6, stride=2)  # out 1024
        layer4 = self._make_layer(1024,512, blocks=3, stride=2)  # out 2048

        avgp = nn.AdaptiveAvgPool2d((1,1))

        # IMPORTANT: Put into Sequential indices that match your ckpt
        # 0 conv1, 1 bn1, 2 relu, 3 maxpool, 4 layer1, 5 layer2, 6 layer3, 7 layer4, 8 avgpool
        self.backbone = nn.Sequential(
            conv1, bn1, relu, maxp,
            layer1, layer2, layer3, layer4,
            avgp
        )

    def _make_layer(self, inplanes: int, planes: int, blocks: int, stride: int):
        downsample = None
        outplanes = planes * Bottleneck.expansion  # 4x
        if stride != 1 or inplanes != outplanes:
            downsample = nn.Sequential(
                nn.Conv2d(inplanes, outplanes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(outplanes),
            )

        layers = []
        layers.append(Bottleneck(inplanes, planes, stride=stride, downsample=downsample))
        inplanes = outplanes
        for _ in range(1, blocks):
            layers.append(Bottleneck(inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [N,3,H,W] -> [N,2048,1,1] -> [N,2048]
        y = self.backbone(x)
        y = torch.flatten(y, 1)
        return y

# ---------------------------
# Full model: matches your ckpt prefixes:
# encoder, pool_non, pool_leu, leuko_head, cls_non, cls_leu, gate
# ---------------------------
class DualBranchGatedMIL(nn.Module):
    def __init__(self, instance_dim=2048, pool_hidden=256, gate_tau=2.0, leuko_in_dim=4096, gate_in_dim=4096):
        super().__init__()
        self.gate_tau = float(gate_tau)

        self.encoder = nn.Module()
        self.encoder.backbone = SequentialResNet50Encoder().backbone  # expose as encoder.backbone.*

        self.pool_non = AttnPool(instance_dim, pool_hidden)
        self.pool_leu = AttnPool(instance_dim, pool_hidden)

        self.cls_non = nn.Linear(instance_dim, 1)
        self.cls_leu = nn.Linear(instance_dim, 1)

        self.gate = nn.Linear(gate_in_dim, 1)

        # Your ckpt sometimes stores last layer as leuko_head.3 not .2; we’ll remap during load.
        self.leuko_head = nn.Sequential(
            nn.Linear(leuko_in_dim, pool_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(pool_hidden, 1),
        )

        # For CAM: use layer4 which is backbone[7]
        self._cam_target_layer = self.encoder.backbone[7]

    def encode_instances(self, x: torch.Tensor) -> torch.Tensor:
        # run through sequential encoder backbone and flatten
        y = self.encoder.backbone(x)  # [N,2048,1,1]
        y = torch.flatten(y, 1)       # [N,2048]
        return y

    def forward(self, X: torch.Tensor) -> Dict[str, torch.Tensor]:
        feats = self.encode_instances(X)  # [N,2048]

        z_non, a_non = self.pool_non(feats)
        z_leu, a_leu = self.pool_leu(feats)

        logit_non = self.cls_non(z_non)  # [1]
        logit_leu = self.cls_leu(z_leu)  # [1]

        z_cat = torch.cat([z_non, z_leu], dim=0).unsqueeze(0)  # [1,4096]

        gate_logit = self.gate(z_cat)  # [1,1]
        gate_alpha = torch.sigmoid(gate_logit / max(self.gate_tau, 1e-6))  # [1,1]

        logit = (1.0 - gate_alpha) * logit_non + gate_alpha * logit_leu
        prob = torch.sigmoid(logit)

        leuko_logit = self.leuko_head(z_cat)

        return {
            "logit": logit.view(()),
            "prob": prob.view(()),
            "gate_alpha": gate_alpha.view(()),
            "logit_non": logit_non.view(()),
            "logit_leu": logit_leu.view(()),
            "leuko_logit": leuko_logit.view(()),
            "a_non": a_non,
            "a_leu": a_leu,
        }

# ---------------------------
# CKPT loader (strict match + remaps)
# ---------------------------
def build_model_from_ckpt(ckpt_path: Path, device: str, gate_tau: float = 2.0):
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    sd = ckpt["model"]

    # Infer dims from ckpt
    instance_dim = sd["cls_non.weight"].shape[1]      # 2048
    pool_hidden  = sd["leuko_head.0.weight"].shape[0] # 256
    leuko_in_dim = sd["leuko_head.0.weight"].shape[1] # 4096
    gate_in_dim  = sd["gate.weight"].shape[1]         # 4096

    model = DualBranchGatedMIL(
        instance_dim=instance_dim,
        pool_hidden=pool_hidden,
        gate_tau=gate_tau,
        leuko_in_dim=leuko_in_dim,
        gate_in_dim=gate_in_dim
    ).to(device)

    # Remap leuko_head last layer index if needed: leuko_head.3 -> leuko_head.2
    sd2 = {}
    for k, v in sd.items():
        nk = k.replace("leuko_head.3.", "leuko_head.2.")
        sd2[nk] = v

    missing, unexpected = model.load_state_dict(sd2, strict=False)

    print(f"[CKPT LOAD] {ckpt_path.name}")
    print(f"  missing: {len(missing)}  unexpected: {len(unexpected)}")
    if missing:
        print("  missing sample:", missing[:10])
    if unexpected:
        print("  unexpected sample:", unexpected[:10])

    model.eval()
    return model, ckpt

# ---------------------------
# Task A: aggregate folds (robust to missing columns)
# ---------------------------
def aggregate_folds_to_seed(
    runs_dir: Path,
    seed: int,
    benchmark: str,
    folds: List[int],
    k_test: int,
    out_path: Path
) -> pd.DataFrame:
    dfs = []
    used = []
    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            continue

        d = pd.read_csv(p)
        if "Patient" not in d.columns:
            raise ValueError(f"{p} missing Patient column")
        d = d.set_index("Patient")

        # Normalize columns
        d = d.rename(columns={
            "logit_mean": "logit",
            "prob_mean": "prob",
        })

        base_cols = [c for c in ["y_true", "leuko"] if c in d.columns]
        fold_cols = [c for c in ["logit", "prob", "gate_alpha"] if c in d.columns]
        d = d[base_cols + fold_cols].copy()

        ren = {}
        if "logit" in d.columns: ren["logit"] = f"logit_fold{f}"
        if "prob" in d.columns:  ren["prob"]  = f"prob_fold{f}"
        if "gate_alpha" in d.columns: ren["gate_alpha"] = f"gate_fold{f}"
        d = d.rename(columns=ren)

        dfs.append(d)
        used.append(f)

    if not dfs:
        raise FileNotFoundError("No fold prediction files found.")

    merged = dfs[0].copy()
    for d in dfs[1:]:
        # drop duplicate base columns
        drop = [c for c in ["y_true","leuko"] if c in d.columns and c in merged.columns]
        merged = merged.join(d.drop(columns=drop, errors="ignore"), how="inner")

    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    prob_cols  = [c for c in merged.columns if c.startswith("prob_fold")]
    gate_cols  = [c for c in merged.columns if c.startswith("gate_fold")]

    merged["logit_mean"] = merged[logit_cols].mean(axis=1).astype(float)
    if prob_cols:
        merged["prob_mean"] = merged[prob_cols].mean(axis=1).astype(float)
    else:
        merged["prob_mean"] = 1.0 / (1.0 + np.exp(-merged["logit_mean"].values))

    if gate_cols:
        merged["gate_mean"] = merged[gate_cols].mean(axis=1).astype(float)

    merged = merged.reset_index()
    safe_mkdir(out_path.parent)
    merged.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | folds used: {used}")
    return merged

def save_final_submission(agg_df: pd.DataFrame, out_csv: Path):
    sub = agg_df[["Patient", "prob_mean"]].rename(columns={"prob_mean":"Probability"})
    sub.to_csv(out_csv, index=False)
    print(f"Saved final submission: {out_csv}")
    print(f"Patients: {len(sub)}")
    return sub

# ---------------------------
# Task C: domain shift analysis (from *_with_domain.csv)
# ---------------------------
def auc_safe(y: np.ndarray, p: np.ndarray) -> Optional[float]:
    from sklearn.metrics import roc_auc_score
    mask = ~np.isnan(y) & ~np.isnan(p)
    if mask.sum() < 5:
        return None
    if len(np.unique(y[mask])) < 2:
        return None
    return float(roc_auc_score(y[mask], p[mask]))

def run_task_c_domain_shift_from_csv(agg_df: pd.DataFrame, domain_csv: Path, out_json: Path):
    meta = load_manifest(domain_csv)
    if "Domain" not in meta.columns:
        raise ValueError(f"{domain_csv} must contain Domain column")

    meta = meta.drop_duplicates("Patient").set_index("Patient")
    agg = agg_df.set_index("Patient")

    cols = ["prob_mean"]
    if "y_true" in agg.columns:
        cols.append("y_true")

    df = meta[["Domain"]].join(agg[cols], how="inner")

    res = {
        "n_total": int(len(df)),
        "domain_counts": df["Domain"].value_counts().to_dict(),
        "auc_overall": None,
        "per_domain": {}
    }

    if "y_true" in df.columns:
        y = pd.to_numeric(df["y_true"], errors="coerce").values.astype(float)
        p = pd.to_numeric(df["prob_mean"], errors="coerce").values.astype(float)
        res["auc_overall"] = auc_safe(y, p)

        for dom, dd in df.groupby("Domain"):
            y_d = pd.to_numeric(dd["y_true"], errors="coerce").values.astype(float)
            p_d = pd.to_numeric(dd["prob_mean"], errors="coerce").values.astype(float)
            res["per_domain"][dom] = {"n": int(len(dd)), "auc": auc_safe(y_d, p_d)}
    else:
        for dom, dd in df.groupby("Domain"):
            res["per_domain"][dom] = {"n": int(len(dd)), "auc": None}

    safe_mkdir(out_json.parent)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2)

    print(f"[Task C] saved: {out_json}")
    print(json.dumps(res, indent=2))
    return res

# ---------------------------
# Task B: Grad-CAM (NO no_grad here!)
# ---------------------------
class GradCAM:
    def __init__(self, model: DualBranchGatedMIL, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.hook_a = target_layer.register_forward_hook(self._forward_hook)
        self.hook_g = target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        self.activations = out

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def close(self):
        self.hook_a.remove()
        self.hook_g.remove()

    def cam_for_tensor(self, x: torch.Tensor) -> np.ndarray:
        """
        x: [1,3,H,W]
        returns cam [H,W] in [0,1]
        """
        self.model.zero_grad(set_to_none=True)

        out = self.model(x)          # forward with grads
        logit = out["logit"]         # scalar
        logit.backward()             # create grads

        A = self.activations
        G = self.gradients
        if A is None or G is None:
            raise RuntimeError("Hooks did not capture activations/gradients.")

        # A, G shapes: [1,C,h,w]
        weights = G.mean(dim=(2,3), keepdim=True)     # [1,C,1,1]
        cam = (weights * A).sum(dim=1, keepdim=False) # [1,h,w]
        cam = F.relu(cam)[0]                          # [h,w]

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        cam = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=(x.shape[-2], x.shape[-1]),
                            mode="bilinear", align_corners=False)[0,0]
        return cam.detach().cpu().numpy()

def overlay_cam_on_pil(img: Image.Image, cam: np.ndarray, alpha: float = 0.45) -> Image.Image:
    cam_u8 = np.uint8(np.clip(cam * 255, 0, 255))
    cam_img = Image.fromarray(cam_u8).resize(img.size, resample=Image.BILINEAR)

    base = np.array(img).astype(np.float32)
    heat = np.array(cam_img).astype(np.float32)
    heat3 = np.stack([heat, np.zeros_like(heat), np.zeros_like(heat)], axis=-1)  # red overlay

    out = (1 - alpha) * base + alpha * heat3
    out = np.uint8(np.clip(out, 0, 255))
    return Image.fromarray(out)

def pick_topk_images_for_patient(mapping: Dict[str, List[Path]], pid: str, k: int) -> List[Path]:
    imgs = mapping.get(pid, [])
    return imgs[:k] if imgs else []

def run_task_b_explainability(
    model: DualBranchGatedMIL,
    patient_ids: List[str],
    mapping: Dict[str, List[Path]],
    out_dir: Path,
    device: str,
    img_size: int = 224,
    topk_for_cam: int = 6
):
    safe_mkdir(out_dir)
    print(f"[{now_str()}] Task B: saving CAM overlays to: {out_dir}")

    tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

    model.train(False)  # eval mode, but gradients still allowed
    cam = GradCAM(model, model._cam_target_layer)

    saved = 0
    rows = []

    for i, pid in enumerate(patient_ids, 1):
        paths = pick_topk_images_for_patient(mapping, pid, topk_for_cam)
        if not paths:
            continue

        pdir = out_dir / pid
        safe_mkdir(pdir)

        for j, img_path in enumerate(paths):
            pil = read_image_rgb(img_path)
            x = tfm(pil).unsqueeze(0).to(device)

            # IMPORTANT: ensure gradients are enabled
            with torch.enable_grad():
                cam_map = cam.cam_for_tensor(x)

            pil_resized = pil.resize((img_size, img_size))
            overlay = overlay_cam_on_pil(pil_resized, cam_map, alpha=0.45)

            out_name = f"cam_{j:02d}_{img_path.name}"
            overlay.save(pdir / out_name)
            saved += 1

            rows.append({
                "Patient": pid,
                "image_path": str(img_path),
                "overlay_path": str(pdir / out_name),
            })

        if i % 5 == 0 or i == len(patient_ids):
            print(f"[{now_str()}] Task B progress: {i}/{len(patient_ids)} patients | overlays saved={saved}")

    cam.close()

    summ = pd.DataFrame(rows)
    summ_path = out_dir / "TASKB_summary.csv"
    summ.to_csv(summ_path, index=False)
    print(f"[{now_str()}] Task B done.")
    return summ_path

# ---------------------------
# Config + main
# ---------------------------
@dataclass
class CFG:
    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"

    IMAGES_ROOT: Path = Path(r"D:\ENT")
    RUNS_DIR: Path   = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    BENCHMARK: str = "res_shift"
    SEED: int = 43
    FOLDS: Tuple[int, ...] = (1,2,3,4,5)
    K_TEST: int = 10

    TRAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv")
    VAL_CSV:   Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv")
    TEST_CSV:  Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")

    TEST_WITH_DOMAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab_with_domain.csv")

    DO_TASK_B: bool = True
    DO_TASK_C: bool = True

    CKPT_FOR_TASKB: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold1.pt")
    GATE_TAU: float = 2.0

    IMG_SIZE: int = 224
    TOPK_FOR_CAM: int = 6

def main():
    cfg = CFG()
    seed_everything(cfg.SEED)

    print("DEVICE:", cfg.DEVICE)
    print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
    print("RUNS_DIR:", cfg.RUNS_DIR)

    # manifests
    df_tr = load_manifest(cfg.TRAIN_CSV)
    df_va = load_manifest(cfg.VAL_CSV)
    df_te = load_manifest(cfg.TEST_CSV)

    tr_p = sorted(df_tr["Patient"].unique().tolist())
    va_p = sorted(df_va["Patient"].unique().tolist())
    te_p = sorted(df_te["Patient"].unique().tolist())

    # mapping
    mapping = build_patient_mapping(cfg.IMAGES_ROOT)

    patient_stats_from_mapping(tr_p, mapping, "TRAIN")
    patient_stats_from_mapping(va_p, mapping, "VAL")
    patient_stats_from_mapping(te_p, mapping, "TEST")

    # Task A: aggregate + final
    agg_path = cfg.RUNS_DIR / f"pred_seed{cfg.SEED}_{cfg.BENCHMARK}_folds1_5_Ktest{cfg.K_TEST}_agg.csv"
    agg_df = aggregate_folds_to_seed(
        runs_dir=cfg.RUNS_DIR,
        seed=cfg.SEED,
        benchmark=cfg.BENCHMARK,
        folds=list(cfg.FOLDS),
        k_test=cfg.K_TEST,
        out_path=agg_path
    )

    final_path = cfg.RUNS_DIR / f"FINAL_seed{cfg.SEED}_{cfg.BENCHMARK}.csv"
    save_final_submission(agg_df, final_path)

    # Task B
    if cfg.DO_TASK_B:
        taskb_dir = cfg.RUNS_DIR / f"TASKB_{cfg.BENCHMARK}_seed{cfg.SEED}"
        model, _ = build_model_from_ckpt(cfg.CKPT_FOR_TASKB, device=cfg.DEVICE, gate_tau=cfg.GATE_TAU)
        run_task_b_explainability(
            model=model,
            patient_ids=te_p,
            mapping=mapping,
            out_dir=taskb_dir,
            device=cfg.DEVICE,
            img_size=cfg.IMG_SIZE,
            topk_for_cam=cfg.TOPK_FOR_CAM
        )

    # Task C
    if cfg.DO_TASK_C:
        out_json = cfg.RUNS_DIR / f"TASKC_domain_shift_seed{cfg.SEED}_{cfg.BENCHMARK}.json"
        run_task_c_domain_shift_from_csv(agg_df, cfg.TEST_WITH_DOMAIN_CSV, out_json)

    print("\nDONE.")
    print("Agg:", agg_path)
    print("Final:", final_path)
    if cfg.DO_TASK_B:
        print("TaskB dir:", cfg.RUNS_DIR / f"TASKB_{cfg.BENCHMARK}_seed{cfg.SEED}")
    if cfg.DO_TASK_C:
        print("TaskC json:", cfg.RUNS_DIR / f"TASKC_domain_shift_seed{cfg.SEED}_{cfg.BENCHMARK}.json")

if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs

=== TRAIN ===
Patients: 117
Patients with 0 images: 0
Images per patient: count    117.000000
mean      50.247863
std       32.454187
min        2.000000
25%       26.000000
50%       42.000000
75%       72.000000
max      186.000000
dtype: float64

=== VAL ===
Patients: 20
Patients with 0 images: 0
Images per patient: count    20.000000
mean     41.750000
std      23.212689
min       9.000000
25%      21.500000
50%      40.000000
75%      57.750000
max      87.000000
dtype: float64

=== TEST ===
Patients: 73
Patients with 0 images: 0
Images per patient: count     73.000000
mean      72.712329
std       31.139685
min        8.000000
25%       52.000000
50%       75.000000
75%       92.000000
max      147.000000
dtype: float64
Saved: D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv | folds used: [1, 2, 3, 4, 5]
Saved final submission: D:\ENT\_challenge2_artifacts\_runs\FINAL

C:\Users\admin\AppData\Local\Temp\ipykernel_14828\3115030341.py:249: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_location="cpu")


[CKPT LOAD] cv_best_res_shift_seed43_fold1.pt
  missing: 50  unexpected: 0
  missing sample: ['_cam_target_layer.0.conv1.weight', '_cam_target_layer.0.bn1.weight', '_cam_target_layer.0.bn1.bias', '_cam_target_layer.0.bn1.running_mean', '_cam_target_layer.0.bn1.running_var', '_cam_target_layer.0.conv2.weight', '_cam_target_layer.0.bn2.weight', '_cam_target_layer.0.bn2.bias', '_cam_target_layer.0.bn2.running_mean', '_cam_target_layer.0.bn2.running_var']
[2026-02-27 22:53:27] Task B: saving CAM overlays to: D:\ENT\_challenge2_artifacts\_runs\TASKB_res_shift_seed43
[2026-02-27 22:53:28] Task B progress: 5/73 patients | overlays saved=30
[2026-02-27 22:53:29] Task B progress: 10/73 patients | overlays saved=60
[2026-02-27 22:53:30] Task B progress: 15/73 patients | overlays saved=90
[2026-02-27 22:53:31] Task B progress: 20/73 patients | overlays saved=120
[2026-02-27 22:53:32] Task B progress: 25/73 patients | overlays saved=150
[2026-02-27 22:53:34] Task B progress: 30/73 patients | overl

In [7]:
import os
import pandas as pd
from pathlib import Path

# -----------------------
# CONFIG
# -----------------------
IMAGES_ROOT = Path(r"D:\ENT")
TEST_CSV = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")
OUT_CSV = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab_with_domain.csv")

# -----------------------
# 1) Build patient -> image paths mapping
# -----------------------
def list_files_recursive(root, exts=(".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
    out = []
    for dp, _, fnames in os.walk(root):
        for f in fnames:
            if f.lower().endswith(exts):
                out.append(Path(dp) / f)
    return out

def build_patient_mapping(images_root):
    all_imgs = list_files_recursive(images_root)
    mapping = {}
    for p in all_imgs:
        pid = None
        for seg in p.parts:
            s = seg.lower().replace(" ", "")
            if s.startswith("patient") and any(ch.isdigit() for ch in s):
                pid = seg.replace(" ", "")
                break
        if pid is None:
            continue
        mapping.setdefault(pid, []).append(p)

    for k in mapping:
        mapping[k] = sorted(mapping[k])

    return mapping

print("Building mapping from:", IMAGES_ROOT)
mapping = build_patient_mapping(IMAGES_ROOT)
print("Patients found in image root:", len(mapping))

# -----------------------
# 2) Domain inference logic
# -----------------------
def infer_domain_from_paths(paths):
    if not paths:
        return "UNKNOWN"

    votes = []
    for p in paths:
        s = str(p).lower()
        if "nbi" in s:
            votes.append("NBI")
        elif "wle" in s or "white" in s or "wl" in s:
            votes.append("WLE")
        else:
            votes.append("UNKNOWN")

    vc = pd.Series(votes).value_counts()

    if len(vc) == 0:
        return "UNKNOWN"

    # Prefer real domain over UNKNOWN
    for dom in ["NBI", "WLE"]:
        if dom in vc.index:
            return dom

    return "UNKNOWN"

# -----------------------
# 3) Load test CSV
# -----------------------
df = pd.read_csv(TEST_CSV)

if "Patient" not in df.columns:
    for c in ["patient", "patient_id", "PatientID"]:
        if c in df.columns:
            df = df.rename(columns={c: "Patient"})
            break

if "Patient" not in df.columns:
    raise ValueError("Test CSV missing Patient column")

df["Patient"] = df["Patient"].astype(str)

# -----------------------
# 4) Assign domain per patient
# -----------------------
domains = []
for pid in df["Patient"]:
    paths = mapping.get(pid, [])
    domains.append(infer_domain_from_paths(paths))

df["Domain"] = domains

# -----------------------
# 5) Save
# -----------------------
df.to_csv(OUT_CSV, index=False)

print("\nSaved:", OUT_CSV)
print("\nDomain distribution:")
print(df["Domain"].value_counts(dropna=False))

Building mapping from: D:\ENT
Patients found in image root: 210

Saved: D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab_with_domain.csv

Domain distribution:
Domain
UNKNOWN    4443
Name: count, dtype: int64


In [8]:
"""
Scientifically-clean end-to-end runner (Tasks A/B/C finalization)
- Builds robust patient->image mapping from IMAGES_ROOT
- Loads train/val/test CSVs (supports many column name variants)
- Patient-level stats + file inventory (optional)
- Aggregates existing fold prediction CSVs -> seed-level agg CSV
- Produces FINAL submission CSV (73 patients in your case)
- Task B: Grad-CAM overlays for test patients using a CV fold checkpoint
- Task C: Domain-shift AUC by WLE/NBI (requires test_with_domain CSV; robust domain inference)
Designed to be DROP-IN and tolerant to your prior issues:
- No assumption that CSV has "path"
- No assumption of patient ID formatting
- No assumption of fold CSV has gate_alpha
- No assumption Domain is already filled
"""

import os, re, json, math, time, random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from torchvision import transforms

# ----------------------------
# Config
# ----------------------------
@dataclass
class CFG:
    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"

    BENCHMARK: str = "res_shift"
    SEED: int = 43
    FOLDS: Tuple[int, ...] = (1,2,3,4,5)
    K_TEST: int = 10

    # Paths (set these)
    IMAGES_ROOT: Path = Path(r"D:\ENT")
    RUNS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    TRAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv")
    VAL_CSV:   Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv")
    TEST_CSV:  Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")
    # For Task C, prefer the *_with_domain.csv you created:
    TEST_WITH_DOMAIN_CSV: Optional[Path] = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab_with_domain.csv")

    # Task B
    DO_TASK_B: bool = True
    TOPK_FOR_CAM: int = 6
    IMG_SIZE: int = 224
    TASKB_DIRNAME: str = "TASKB_res_shift_seed43"  # will be placed inside RUNS_DIR
    # pick a checkpoint that exists:
    CKPT_FOR_TASKB: Optional[Path] = Path(r"D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold1.pt")

    # Task C
    DO_TASK_C: bool = True
    TASKC_JSON_NAME: str = "TASKC_domain_shift_seed43_res_shift.json"

cfg = CFG()


# ----------------------------
# Utils
# ----------------------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def list_files_recursive(root: Path, exts=(".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp")) -> List[Path]:
    out = []
    root = Path(root)
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            out.append(p)
    return out

def normalize_patient_id(x: str) -> str:
    """
    Robust patient-id normalization:
    - extracts digits if present and returns standardized "Patient###" style
    - if no digits, returns upper-stripped string
    """
    s = str(x).strip()
    m = re.search(r"(\d+)", s)
    if m:
        num = int(m.group(1))
        return f"Patient{num:03d}"
    return s.upper()

def infer_domain_from_path(p: Path) -> str:
    """
    Domain inference from filepath segments.
    Searches for WLE/NBI keywords in any parent folder or filename.
    """
    text = " ".join([*p.parts, p.name]).lower()
    # common variants
    if "nbi" in text:
        return "NBI"
    if "wle" in text or "white" in text or "wl" in text:
        return "WLE"
    return "UNKNOWN"

def build_patient_mapping(images_root: Path) -> Dict[str, List[Path]]:
    """
    Robust mapping:
    - scans image files under images_root
    - finds patient token anywhere in path segment: "patient099" / "Patient_99" / "099" etc
    - normalizes to Patient### to match your Excel-based convention
    """
    imgs = list_files_recursive(images_root)
    mapping: Dict[str, List[Path]] = {}
    for p in imgs:
        pid = None
        # look for segment containing 'patient' first
        for seg in p.parts:
            low = seg.lower().replace(" ", "")
            if "patient" in low:
                pid = normalize_patient_id(seg)
                break
        # fallback: any numeric segment
        if pid is None:
            for seg in p.parts:
                m = re.search(r"\d+", seg)
                if m:
                    pid = normalize_patient_id(m.group(0))
                    break
        if pid is None:
            continue
        mapping.setdefault(pid, []).append(p)

    # sort paths for deterministic order
    for k in list(mapping.keys()):
        mapping[k] = sorted(mapping[k])
    return mapping

def find_first_existing(paths: List[Path]) -> Optional[Path]:
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


# ----------------------------
# CSV loaders (tolerant)
# ----------------------------
def load_manifest(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Patient column
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID", "pid", "PATIENT"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"{csv_path} missing Patient column (tried common aliases). Columns={list(df.columns)}")

    # Normalize patient IDs to Patient###
    df["Patient"] = df["Patient"].astype(str).map(normalize_patient_id)

    # y_true label column (optional for test)
    # Accept common label names: y, label, target, leuko, Leukoplakia, Histopathology etc.
    if "y_true" not in df.columns:
        for c in ["y", "label", "target", "Target", "Label", "Leukoplakia", "leuko", "Histopathology"]:
            if c in df.columns:
                df = df.rename(columns={c: "y_true"})
                break

    # leuko column (optional)
    if "leuko" not in df.columns:
        for c in ["Leukoplakia", "LEUKO", "leukoplakia"]:
            if c in df.columns:
                df = df.rename(columns={c: "leuko"})
                break

    # Domain column (optional)
    if "Domain" not in df.columns:
        for c in ["domain", "MODALITY", "Modality", "modality"]:
            if c in df.columns:
                df = df.rename(columns={c: "Domain"})
                break

    return df


# ----------------------------
# Stats
# ----------------------------
def print_split_stats(name: str, df: pd.DataFrame, mapping: Dict[str, List[Path]]):
    pids = df["Patient"].unique().tolist()
    counts = []
    zero = 0
    for pid in pids:
        n = len(mapping.get(pid, []))
        counts.append(n)
        if n == 0:
            zero += 1
    s = pd.Series(counts)
    print(f"\n=== {name.upper()} ===")
    print("Patients:", len(pids))
    print("Patients with 0 images:", zero)
    print("Images per patient:", s.describe())

def ensure_test_domain_csv(test_csv: Path, mapping: Dict[str, List[Path]], out_csv: Path) -> Path:
    df = pd.read_csv(test_csv)

    # Patient column normalize
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID", "pid"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError("Cannot create domain csv: no Patient column in test csv")

    df["Patient"] = df["Patient"].astype(str).map(normalize_patient_id)

    # infer patient-level domain
    domains = []
    for pid in df["Patient"].tolist():
        paths = mapping.get(pid, [])
        if not paths:
            domains.append("UNKNOWN")
            continue
        # majority vote across that patient's images
        votes = [infer_domain_from_path(p) for p in paths]
        # pick most frequent non-UNKNOWN if possible
        vc = pd.Series(votes).value_counts()
        if len(vc) == 0:
            domains.append("UNKNOWN")
        else:
            # prefer non-UNKNOWN
            if "UNKNOWN" in vc.index and len(vc) > 1:
                vc2 = vc.drop(index=["UNKNOWN"])
                domains.append(vc2.index[0] if len(vc2) else "UNKNOWN")
            else:
                domains.append(vc.index[0])

    df["Domain"] = domains
    df.to_csv(out_csv, index=False)

    print(f"\nSaved: {out_csv}")
    print("\nDomain distribution:")
    print(df["Domain"].value_counts(dropna=False))
    return out_csv


# ----------------------------
# Aggregation (fold -> seed)
# ----------------------------
def read_fold_pred_csv(p: Path) -> pd.DataFrame:
    d = pd.read_csv(p)
    # Patient column
    if "Patient" not in d.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in d.columns:
                d = d.rename(columns={c: "Patient"})
                break
    if "Patient" not in d.columns:
        raise ValueError(f"{p} missing Patient col")

    d["Patient"] = d["Patient"].astype(str).map(normalize_patient_id)
    d = d.set_index("Patient")

    # standardize columns
    col_map = {}
    if "logit_mean" in d.columns and "logit" not in d.columns:
        col_map["logit_mean"] = "logit"
    if "prob_mean" in d.columns and "prob" not in d.columns:
        col_map["prob_mean"] = "prob"
    d = d.rename(columns=col_map)

    # must have logits
    if "logit" not in d.columns:
        # try "logits"
        if "logits" in d.columns:
            d = d.rename(columns={"logits":"logit"})
        else:
            raise ValueError(f"{p} has no logit/logit_mean column. cols={list(d.columns)}")

    # y_true/leuko optional
    keep = []
    for c in ["y_true", "leuko", "logit", "prob", "gate_alpha", "gate_mean", "gate"]:
        if c in d.columns:
            keep.append(c)

    # guarantee prob exists
    if "prob" not in d.columns:
        d["prob"] = 1/(1+np.exp(-d["logit"].astype(float)))

    return d[keep]

def aggregate_folds_to_seed(
    runs_dir: Path,
    seed: int,
    benchmark: str,
    folds: Tuple[int,...],
    k_test: int,
    out_path: Path
) -> pd.DataFrame:
    dfs = []
    used = []
    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            continue
        d = read_fold_pred_csv(p)
        # rename fold-specific
        rename = {"logit": f"logit_fold{f}", "prob": f"prob_fold{f}"}
        if "gate_alpha" in d.columns:
            rename["gate_alpha"] = f"gate_fold{f}"
        d = d.rename(columns=rename)
        dfs.append(d)
        used.append(f)

    if not dfs:
        raise FileNotFoundError("No fold prediction CSVs found for aggregation.")

    # merge on index
    merged = dfs[0]
    for d in dfs[1:]:
        # keep y_true/leuko only from left; join only fold-specific cols from right
        right_cols = [c for c in d.columns if c.startswith("logit_fold") or c.startswith("prob_fold") or c.startswith("gate_fold")]
        merged = merged.join(d[right_cols], how="inner")

    # compute means
    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    prob_cols  = [c for c in merged.columns if c.startswith("prob_fold")]
    gate_cols  = [c for c in merged.columns if c.startswith("gate_fold")]

    merged["logit_mean"] = merged[logit_cols].mean(axis=1).astype(float)
    merged["prob_mean"]  = merged[prob_cols].mean(axis=1).astype(float)
    if gate_cols:
        merged["gate_mean"] = merged[gate_cols].mean(axis=1).astype(float)

    merged = merged.reset_index()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | folds used: {used}")
    return merged

def make_final_submission(agg_df: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    """
    FINAL CSV: Patient, prob
    """
    sub = agg_df[["Patient", "prob_mean"]].rename(columns={"prob_mean":"prob"})
    sub.to_csv(out_path, index=False)
    print(f"Saved final submission: {out_path}")
    print("Patients:", len(sub))
    return sub


# ----------------------------
# Model definitions to load your checkpoint (DualBranchGatedMIL)
# ----------------------------
class AttnPool(nn.Module):
    """
    Generic attention pooling: scores per instance -> softmax -> weighted sum.
    Matches checkpoint style where pool layers are stored under pool_non.V/w or under pool_non.net.* depending on version.
    We'll implement with net Sequential (Linear->Tanh->Linear) and also support V/w naming in load.
    """
    def __init__(self, in_dim: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):  # x: [N, D]
        a = self.net(x).squeeze(-1)          # [N]
        w = torch.softmax(a, dim=0)          # [N]
        z = (w.unsqueeze(-1) * x).sum(dim=0) # [D]
        return z, w

class ResNet50Encoder(nn.Module):
    """
    Minimal ResNet50 backbone (torchvision-style layout differs from your checkpoint which is Sequential)
    Instead, we will *not* rebuild torchvision resnet; we will load the checkpoint into a generic "encoder.backbone" Sequential.
    For Task B we mainly need feature maps from a target layer and logits from heads.
    We'll build a Sequential backbone matching checkpoint *keys*, which are "encoder.backbone.0,1,2,...".
    """
    def __init__(self):
        super().__init__()
        # placeholder; we'll fill by reconstructing from checkpoint? Not feasible without original code.
        # Instead: For Task B, we DO NOT need to rebuild exact modules if checkpoint loads cleanly.
        # Your working run shows it CAN load when we match state_dict keys (encoder.backbone.0 etc).
        # We'll implement a lightweight reconstruction that matches those keys using torchvision resnet and then wrap.
        import torchvision.models as models
        m = models.resnet50(weights=None)
        # build Sequential to match checkpoint key style used in your ckpt print (0..8)
        self.backbone = nn.Sequential(
            m.conv1, m.bn1, m.relu, m.maxpool,
            m.layer1, m.layer2, m.layer3, m.layer4,
            nn.AdaptiveAvgPool2d((1,1))
        )

    def forward(self, x):
        feats = self.backbone(x)   # [B,2048,1,1]
        return feats

class DualBranchGatedMIL(nn.Module):
    """
    Matches your printed architecture:
    - encoder.backbone (Sequential 0..8)
    - pool_non, pool_leu: AttnPool(2048->256->1)
    - cls_non, cls_leu: Linear(2048->1)
    - gate: Linear(4096->1)
    - leuko_head: Linear(4096->256->1)
    """
    def __init__(self, gate_tau=2.0):
        super().__init__()
        self.encoder = nn.Module()
        self.encoder.backbone = ResNet50Encoder().backbone  # make keys encoder.backbone.0... ok

        self.pool_non = AttnPool(2048, 256)
        self.pool_leu = AttnPool(2048, 256)
        self.cls_non = nn.Linear(2048, 1)
        self.cls_leu = nn.Linear(2048, 1)
        self.gate = nn.Linear(4096, 1)
        self.leuko_head = nn.Sequential(
            nn.Linear(4096, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1),
        )
        self.gate_tau = gate_tau

        # will be set for CAM hook
        self._cam_target_layer = None

    def _encode_instances(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B,3,H,W] -> instance embedding [B,2048]
        """
        feats = self.encoder.backbone(x)          # [B,2048,1,1]
        feats = feats.flatten(1)                  # [B,2048]
        return feats

    def forward(self, x_non: torch.Tensor, x_leu: torch.Tensor):
        """
        Inputs are instance tensors; in practice for CAM we will pass a single image as both branches.
        """
        h_non = self._encode_instances(x_non)     # [B,2048]
        h_leu = self._encode_instances(x_leu)     # [B,2048]

        # MIL pooling expects [N,D] per patient; for single-image CAM, N=B=1
        z_non, a_non = self.pool_non(h_non)
        z_leu, a_leu = self.pool_leu(h_leu)

        logit_non = self.cls_non(z_non)  # [1]
        logit_leu = self.cls_leu(z_leu)  # [1]

        fused = torch.cat([z_non, z_leu], dim=0).unsqueeze(0)  # [1,4096]
        gate_logit = self.gate(fused)                          # [1,1]
        alpha = torch.sigmoid(gate_logit / self.gate_tau)      # [1,1]

        # final logit: interpolate between branches
        logit = (1 - alpha) * logit_non + alpha * logit_leu    # [1,1]
        leuko_logit = self.leuko_head(fused)                   # [1,1]

        return {
            "logit": logit.squeeze(),
            "prob": torch.sigmoid(logit).squeeze(),
            "alpha": alpha.squeeze(),
            "logit_non": logit_non.squeeze(),
            "logit_leu": logit_leu.squeeze(),
            "leuko_logit": leuko_logit.squeeze(),
            "attn_non": a_non.detach().cpu(),
            "attn_leu": a_leu.detach().cpu(),
        }

def remap_state_dict(sd: Dict[str, torch.Tensor], model_keys: set) -> Dict[str, torch.Tensor]:
    """
    Fixes common naming mismatches:
    - Some checkpoints store pool params as pool_non.V / pool_non.w instead of pool_non.net.*
    We'll remap those into net.{0,2} equivalents.
    """
    out = {}
    for k,v in sd.items():
        nk = k
        # pool V/w -> net linear weights
        # pool_non.V.weight maps to pool_non.net.0.weight
        nk = nk.replace("pool_non.V.", "pool_non.net.0.")
        nk = nk.replace("pool_non.w.", "pool_non.net.2.")
        nk = nk.replace("pool_leu.V.", "pool_leu.net.0.")
        nk = nk.replace("pool_leu.w.", "pool_leu.net.2.")

        # leuko_head.3 -> leuko_head.2 (some versions have an extra identity layer)
        nk = nk.replace("leuko_head.3.", "leuko_head.2.")

        out[nk] = v

    # filter only keys that exist in model
    out2 = {k:v for k,v in out.items() if k in model_keys}
    return out2

def build_model_from_ckpt(ckpt_path: Path, device: str, gate_tau: float=2.0) -> Tuple[nn.Module, dict]:
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    sd = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    model = DualBranchGatedMIL(gate_tau=gate_tau).to(device)
    model_keys = set(model.state_dict().keys())

    sd2 = remap_state_dict(sd, model_keys)
    missing, unexpected = model.load_state_dict(sd2, strict=False)

    print(f"[CKPT LOAD] {Path(ckpt_path).name}")
    print("  missing:", len(missing), " unexpected:", len(unexpected))
    if len(missing):
        print("  missing sample:", missing[:10])
    if len(unexpected):
        print("  unexpected sample:", unexpected[:10])

    model.eval()
    return model, ckpt


# ----------------------------
# Grad-CAM
# ----------------------------
class GradCAM:
    def __init__(self, model: DualBranchGatedMIL, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.h1 = target_layer.register_forward_hook(self._fwd_hook)
        self.h2 = target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, m, inp, out):
        self.activations = out

    def _bwd_hook(self, m, grad_in, grad_out):
        self.gradients = grad_out[0]

    def close(self):
        self.h1.remove()
        self.h2.remove()

    @torch.enable_grad()
    def cam_for_tensor(self, x: torch.Tensor) -> np.ndarray:
        """
        x: [1,3,H,W]
        Returns CAM in [H',W'] normalized to [0,1].
        """
        self.model.zero_grad(set_to_none=True)

        # Ensure grad enabled and model in eval
        self.model.eval()
        x = x.requires_grad_(True)

        out = self.model(x, x)        # use same image for both branches
        logit = out["logit"]
        if not torch.is_tensor(logit):
            logit = torch.tensor(logit, device=x.device, dtype=torch.float32)

        logit.backward(retain_graph=False)

        A = self.activations          # [1,C,h,w]
        G = self.gradients            # [1,C,h,w]
        if A is None or G is None:
            raise RuntimeError("GradCAM hooks did not capture activations/gradients. Check target layer.")

        weights = G.mean(dim=(2,3), keepdim=True)  # [1,C,1,1]
        cam = (weights * A).sum(dim=1, keepdim=False)  # [1,h,w]
        cam = F.relu(cam)
        cam = cam[0].detach().cpu().numpy()

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam

def overlay_cam_on_pil(pil: Image.Image, cam: np.ndarray, alpha=0.45) -> Image.Image:
    """
    Simple heat overlay without matplotlib to keep dependencies low.
    """
    # resize cam to pil size
    cam_img = Image.fromarray(np.uint8(cam*255)).resize(pil.size, resample=Image.BILINEAR).convert("L")
    cam_np = np.array(cam_img).astype(np.float32) / 255.0

    # create red heatmap-like overlay
    base = np.array(pil).astype(np.float32)
    heat = np.zeros_like(base)
    heat[...,0] = cam_np * 255.0  # red channel

    out = (1-alpha)*base + alpha*heat
    out = np.clip(out, 0, 255).astype(np.uint8)
    return Image.fromarray(out)


# ----------------------------
# Task B runner
# ----------------------------
def run_task_b_explainability(
    model: DualBranchGatedMIL,
    patient_ids: List[str],
    mapping: Dict[str, List[Path]],
    out_dir: Path,
    device: str,
    img_size: int,
    topk_for_cam: int
):
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"[{now()}] Task B: saving CAM overlays to: {out_dir}")

    # Transform
    tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

    # choose CAM target layer: last block of layer4 in sequential backbone is encoder.backbone[7]
    # Use its last Bottleneck block (index 2) and its conv3 for strong spatial maps:
    # encoder.backbone[7] is layer4 (Sequential of bottlenecks)
    target_layer = model.encoder.backbone[7][-1].conv3
    cam = GradCAM(model, target_layer)

    saved = 0
    for i, pid in enumerate(patient_ids, 1):
        paths = mapping.get(pid, [])
        if not paths:
            continue

        # take first K images (or random subset) for overlays
        take = paths[:topk_for_cam]

        for j, img_path in enumerate(take, 1):
            try:
                pil = Image.open(img_path).convert("RGB")
            except Exception:
                continue

            x = tfm(pil).unsqueeze(0).to(device)

            # Need gradients for CAM
            with torch.enable_grad():
                cam_map = cam.cam_for_tensor(x)

            overlay = overlay_cam_on_pil(pil.resize((img_size, img_size)), cam_map, alpha=0.45)
            out_name = f"{pid}_img{j:02d}_{img_path.name}"
            overlay.save(out_dir / out_name)
            saved += 1

        if i % 5 == 0 or i == len(patient_ids):
            print(f"[{now()}] Task B progress: {i}/{len(patient_ids)} patients | overlays saved={saved}")

    cam.close()
    print(f"[{now()}] Task B done.")


# ----------------------------
# Task C (Domain shift)
# ----------------------------
def auc_roc(y_true: np.ndarray, y_score: np.ndarray) -> Optional[float]:
    """
    Computes AUC without sklearn (scientifically clean & portable).
    Returns None if only one class present.
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(np.unique(y_true)) < 2:
        return None

    # rank-based AUC (Mann–Whitney U)
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(y_score)+1)

    pos = y_true == 1
    n_pos = pos.sum()
    n_neg = len(y_true) - n_pos
    sum_ranks_pos = ranks[pos].sum()

    auc = (sum_ranks_pos - n_pos*(n_pos+1)/2) / (n_pos*n_neg + 1e-12)
    return float(auc)

def run_task_c_domain_shift(agg_df: pd.DataFrame, meta_df: pd.DataFrame, out_json: Path):
    """
    agg_df must have Patient, y_true (optional), prob_mean, logit_mean
    meta_df must have Patient, Domain, and ideally y_true as well.
    We compute per-domain AUC using y_true from agg_df if available else from meta_df.
    """
    # normalize Patient
    meta = meta_df.copy()
    if "Patient" not in meta.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in meta.columns:
                meta = meta.rename(columns={c:"Patient"})
                break
    meta["Patient"] = meta["Patient"].astype(str).map(normalize_patient_id)

    if "Domain" not in meta.columns:
        raise ValueError("Task C meta csv must include Domain column.")

    # merge
    agg = agg_df.copy()
    agg["Patient"] = agg["Patient"].astype(str).map(normalize_patient_id)

    merged = pd.merge(agg, meta[["Patient","Domain"] + (["y_true"] if "y_true" in meta.columns else [])], on="Patient", how="left")

    # choose y_true source
    if "y_true" in merged.columns and merged["y_true"].notna().any():
        y = merged["y_true"]
    elif "y_true_y" in merged.columns:
        y = merged["y_true_y"]
    else:
        raise ValueError("Task C needs y_true either in agg_df or meta_df.")

    # normalize y_true to 0/1 if it contains strings
    y_ser = y.copy()
    if y_ser.dtype == object:
        # map common variants
        y_ser = y_ser.astype(str).str.strip().str.lower().replace({
            "yes":1, "no":0, "true":1, "false":0, "1":1, "0":0
        })
    y_ser = pd.to_numeric(y_ser, errors="coerce").fillna(0).astype(int)

    merged["y_true_bin"] = y_ser
    merged["Domain"] = merged["Domain"].fillna("UNKNOWN").astype(str)

    overall_auc = auc_roc(merged["y_true_bin"].values, merged["prob_mean"].values)

    per = {}
    for dom, g in merged.groupby("Domain"):
        per[dom] = {
            "n": int(len(g)),
            "auc": auc_roc(g["y_true_bin"].values, g["prob_mean"].values)
        }

    out = {
        "n_total": int(len(merged)),
        "domain_counts": merged["Domain"].value_counts(dropna=False).to_dict(),
        "auc_overall": overall_auc,
        "per_domain": per
    }

    out_json.parent.mkdir(parents=True, exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)

    print(f"[Task C] saved: {out_json}")
    print(json.dumps(out, indent=2))


# ----------------------------
# MAIN
# ----------------------------
def main():
    print("DEVICE:", cfg.DEVICE)
    print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
    print("RUNS_DIR:", cfg.RUNS_DIR)

    seed_everything(cfg.SEED)

    # 1) mapping
    mapping = build_patient_mapping(cfg.IMAGES_ROOT)
    print("\nPatients found in image root:", len(mapping))

    # 2) load manifests
    df_tr = load_manifest(cfg.TRAIN_CSV)
    df_va = load_manifest(cfg.VAL_CSV)
    df_te = load_manifest(cfg.TEST_CSV)

    # 3) stats
    print_split_stats("train", df_tr, mapping)
    print_split_stats("val", df_va, mapping)
    print_split_stats("test", df_te, mapping)

    # 4) ensure test_with_domain
    test_with_domain = None
    if cfg.DO_TASK_C:
        # prefer provided file, else create
        if cfg.TEST_WITH_DOMAIN_CSV is not None and cfg.TEST_WITH_DOMAIN_CSV.exists():
            test_with_domain = cfg.TEST_WITH_DOMAIN_CSV
        else:
            out_csv = cfg.TEST_CSV.with_name(cfg.TEST_CSV.stem + "_with_domain.csv")
            test_with_domain = ensure_test_domain_csv(cfg.TEST_CSV, mapping, out_csv)

    # 5) aggregate folds -> seed
    agg_path = cfg.RUNS_DIR / f"pred_seed{cfg.SEED}_{cfg.BENCHMARK}_folds1_5_Ktest{cfg.K_TEST}_agg.csv"
    agg_df = aggregate_folds_to_seed(
        runs_dir=cfg.RUNS_DIR,
        seed=cfg.SEED,
        benchmark=cfg.BENCHMARK,
        folds=cfg.FOLDS,
        k_test=cfg.K_TEST,
        out_path=agg_path
    )

    # 6) final submission
    final_path = cfg.RUNS_DIR / f"FINAL_seed{cfg.SEED}_{cfg.BENCHMARK}.csv"
    _ = make_final_submission(agg_df, final_path)

    # 7) Task B
    if cfg.DO_TASK_B:
        if cfg.CKPT_FOR_TASKB is None or not cfg.CKPT_FOR_TASKB.exists():
            raise FileNotFoundError(f"Task B ckpt not found: {cfg.CKPT_FOR_TASKB}")

        model, _ = build_model_from_ckpt(cfg.CKPT_FOR_TASKB, device=cfg.DEVICE, gate_tau=2.0)
        taskb_dir = cfg.RUNS_DIR / cfg.TASKB_DIRNAME

        te_p = df_te["Patient"].unique().tolist()
        run_task_b_explainability(
            model=model,
            patient_ids=te_p,
            mapping=mapping,
            out_dir=taskb_dir,
            device=cfg.DEVICE,
            img_size=cfg.IMG_SIZE,
            topk_for_cam=cfg.TOPK_FOR_CAM
        )

    # 8) Task C
    if cfg.DO_TASK_C:
        meta = pd.read_csv(test_with_domain)
        out_json = cfg.RUNS_DIR / cfg.TASKC_JSON_NAME
        run_task_c_domain_shift(agg_df, meta, out_json)

    print("\nDONE.")
    print("Agg:", agg_path)
    print("Final:", final_path)
    if cfg.DO_TASK_B:
        print("TaskB dir:", cfg.RUNS_DIR / cfg.TASKB_DIRNAME)
    if cfg.DO_TASK_C:
        print("TaskC json:", cfg.RUNS_DIR / cfg.TASKC_JSON_NAME)

if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs

Patients found in image root: 210

=== TRAIN ===
Patients: 117
Patients with 0 images: 0
Images per patient: count    117.000000
mean      50.247863
std       32.454187
min        2.000000
25%       26.000000
50%       42.000000
75%       72.000000
max      186.000000
dtype: float64

=== VAL ===
Patients: 20
Patients with 0 images: 0
Images per patient: count    20.000000
mean     41.750000
std      23.212689
min       9.000000
25%      21.500000
50%      40.000000
75%      57.750000
max      87.000000
dtype: float64

=== TEST ===
Patients: 73
Patients with 0 images: 0
Images per patient: count     73.000000
mean      78.712329
std       31.139685
min       14.000000
25%       58.000000
50%       81.000000
75%       98.000000
max      153.000000
dtype: float64
Saved: D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv | folds used: [1, 2, 3, 4, 5]
Saved final submission: D:\EN

C:\Users\admin\AppData\Local\Temp\ipykernel_14828\4223735821.py:510: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_location="cpu")


[CKPT LOAD] cv_best_res_shift_seed43_fold1.pt
  missing: 0  unexpected: 0
[2026-02-28 10:08:15] Task B: saving CAM overlays to: D:\ENT\_challenge2_artifacts\_runs\TASKB_res_shift_seed43
[2026-02-28 10:08:17] Task B progress: 5/73 patients | overlays saved=30
[2026-02-28 10:08:19] Task B progress: 10/73 patients | overlays saved=60
[2026-02-28 10:08:21] Task B progress: 15/73 patients | overlays saved=90
[2026-02-28 10:08:23] Task B progress: 20/73 patients | overlays saved=120
[2026-02-28 10:08:26] Task B progress: 25/73 patients | overlays saved=150
[2026-02-28 10:08:28] Task B progress: 30/73 patients | overlays saved=180
[2026-02-28 10:08:30] Task B progress: 35/73 patients | overlays saved=210
[2026-02-28 10:08:32] Task B progress: 40/73 patients | overlays saved=240
[2026-02-28 10:08:34] Task B progress: 45/73 patients | overlays saved=270
[2026-02-28 10:08:36] Task B progress: 50/73 patients | overlays saved=300
[2026-02-28 10:08:39] Task B progress: 55/73 patients | overlays sav

In [9]:
# ============================================================
# ENT Challenge 2 — Scientifically Clean End-to-End Script
# Tasks:
#   A) Patient-level inference + final submission (patient=73)
#   B) Explainability (Grad-CAM overlays saved per patient)
#   C) Domain shift evaluation (WLE/NBI) at PATIENT level
#
# Key fixes included:
#   - Robust manifest loader (no required 'path' column)
#   - Patient-level mapping from filesystem (folder-segment)
#   - Robust domain inference + PATIENT-level domain CSV builder
#   - Robust fold-aggregation (works even if some columns missing)
#   - Checkpoint -> model builder matched to your ckpt key prefixes
#   - Grad-CAM that actually backprops (ensures requires_grad)
#
# Configure paths in Config section and run main().
# ============================================================

import os
import re
import json
import math
import time
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms as T

warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------
# Config
# -----------------------------
@dataclass
class Config:
    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"

    # Root containing patient folders and images
    IMAGES_ROOT: Path = Path(r"D:\ENT")

    # Challenge artifacts root and run directory (preds, ckpts, outputs)
    ARTIFACTS_ROOT: Path = Path(r"D:\ENT\_challenge2_artifacts")
    RUNS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    BENCHMARK: str = "res_shift"
    SEED: int = 43
    FOLDS: Tuple[int, ...] = (1, 2, 3, 4, 5)

    # CSV splits (can be image-level lists; we only need Patient lists robustly)
    TRAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv")
    VAL_CSV: Path   = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv")
    TEST_CSV: Path  = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")

    # Task C uses PATIENT-level domain meta; we build it from TEST_CSV + filesystem mapping
    TEST_PATIENT_DOMAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_PATIENT_with_domain.csv")

    # Inference config
    IMG_SIZE: int = 224
    K_TEST: int = 10
    TTA_TEST: bool = True

    # Task B config
    DO_TASK_B: bool = True
    TOPK_FOR_CAM: int = 6  # overlays per patient
    TASKB_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\TASKB_res_shift_seed43")

    # Task C config
    DO_TASK_C: bool = True
    TASKC_JSON: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\TASKC_domain_shift_seed43_res_shift.json")

    # Checkpoint for Task B CAM model
    CKPT_FOR_TASKB: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold1.pt")

    # Outputs
    AGG_OUT: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv")
    FINAL_OUT: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\FINAL_seed43_res_shift.csv")


cfg = Config()
cfg.RUNS_DIR.mkdir(parents=True, exist_ok=True)
cfg.TASKB_DIR.mkdir(parents=True, exist_ok=True)

print("DEVICE:", cfg.DEVICE)
print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
print("RUNS_DIR:", cfg.RUNS_DIR)


# -----------------------------
# Utils
# -----------------------------
def now_str():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def normalize_patient_id(x: str) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    # normalize common patterns
    s = re.sub(r"\s+", "", s)
    # "Patient099" stays; "patient099" -> "Patient099"
    if re.match(r"(?i)^patient\d+$", s):
        num = re.findall(r"\d+", s)[0]
        return f"Patient{int(num):03d}"
    return s

def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

def robust_read_image(path: Path) -> Image.Image:
    img = Image.open(path).convert("RGB")
    return img

def list_all_files(root: Path, exts=(".jpg", ".jpeg", ".png", ".bmp")) -> List[Path]:
    out = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            out.append(p)
    return out


# -----------------------------
# Manifest loader (robust)
# -----------------------------
def load_manifest(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Patient column
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID", "pid"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        # fallback: if first col looks like patient id
        df = df.rename(columns={df.columns[0]: "Patient"})

    df["Patient"] = df["Patient"].astype(str).map(normalize_patient_id)

    # Label column
    if "y_true" not in df.columns:
        for c in ["Histopathology", "label", "Label", "target", "Target"]:
            if c in df.columns:
                df = df.rename(columns={c: "y_true"})
                break

    # Leukoplakia
    if "leuko" not in df.columns:
        for c in ["Leukoplakia", "leukoplakia", "Leuko"]:
            if c in df.columns:
                df = df.rename(columns={c: "leuko"})
                break

    # ensure leuko numeric if possible (handles 'No'/'Yes')
    if "leuko" in df.columns:
        def _to_leuko(v):
            if pd.isna(v): return np.nan
            s = str(v).strip().lower()
            if s in ["1", "true", "yes", "y", "pos", "positive"]: return 1
            if s in ["0", "false", "no", "n", "neg", "negative"]: return 0
            return safe_float(v, np.nan)
        df["leuko"] = df["leuko"].map(_to_leuko)

    return df


# -----------------------------
# Filesystem mapping (folder-segment)
# -----------------------------
def build_patient_image_mapping(images_root: Path) -> Dict[str, List[Path]]:
    """
    mapping[PatientXXX] = list of image paths under that patient's folder subtree.
    Assumes patient folders contain 'Patient' in name (e.g., Patient099).
    """
    mapping: Dict[str, List[Path]] = {}
    img_exts = {".jpg", ".jpeg", ".png", ".bmp"}

    # find all candidate patient dirs
    for d in images_root.iterdir():
        if d.is_dir():
            m = re.match(r"(?i)^patient\d+$", d.name.replace(" ", ""))
            if m:
                pid = normalize_patient_id(d.name)
                mapping[pid] = []

    # also handle nested patient dirs
    for d in images_root.rglob("*"):
        if d.is_dir():
            name = d.name.replace(" ", "")
            if re.match(r"(?i)^patient\d+$", name):
                pid = normalize_patient_id(name)
                mapping.setdefault(pid, [])

    # collect images
    for pid in list(mapping.keys()):
        # locate the directory path(s) that match pid
        # choose any dir named exactly pid
        dirs = [p for p in images_root.rglob(pid) if p.is_dir()]
        if not dirs:
            # fallback: case-insensitive
            dirs = [p for p in images_root.rglob("*") if p.is_dir() and p.name.lower() == pid.lower()]
        for d in dirs:
            for f in d.rglob("*"):
                if f.is_file() and f.suffix.lower() in img_exts:
                    mapping[pid].append(f)

        # unique and sort
        mapping[pid] = sorted(list(set(mapping[pid])))

    # remove empty keys (rare)
    mapping = {k: v for k, v in mapping.items() if len(v) > 0}

    print("Patients found in image root:", len(mapping))
    return mapping


def print_patient_split_stats(name: str, patient_ids: List[str], mapping: Dict[str, List[Path]]):
    counts = [len(mapping.get(pid, [])) for pid in patient_ids]
    s = pd.Series(counts, dtype=float)
    print(f"\n=== {name} ===")
    print("Patients:", len(patient_ids))
    print("Patients with 0 images:", int((s == 0).sum()))
    print("Images per patient:", s.describe())


# -----------------------------
# Domain inference (paths -> WLE/NBI/MIXED/UNKNOWN)
# -----------------------------
def infer_domain_from_path(p: Path) -> str:
    s = str(p).lower()
    # common tokens
    if "nbi" in s or "narrowband" in s or "narrow_band" in s:
        return "NBI"
    if "wle" in s or "white" in s or "wl" in s or "whitelight" in s or "white_light" in s:
        return "WLE"
    # if folder names include modality like "mode1/mode2"
    if "mode_nbi" in s:
        return "NBI"
    if "mode_wle" in s:
        return "WLE"
    return "UNKNOWN"


def infer_patient_domain(paths: List[Path]) -> str:
    if not paths:
        return "UNKNOWN"
    votes = [infer_domain_from_path(p) for p in paths]
    vc = pd.Series(votes).value_counts()
    if "UNKNOWN" in vc.index and len(vc) > 1:
        vc = vc.drop(index=["UNKNOWN"])
    if len(vc) == 0:
        return "UNKNOWN"
    if len(vc) == 1:
        return vc.index[0]
    # multiple domains
    top = vc.index[0]
    # if strong evidence for both WLE and NBI, label mixed
    if ("WLE" in vc.index) and ("NBI" in vc.index):
        return "MIXED"
    return top


def build_test_patient_domain_csv(test_csv: Path, mapping: Dict[str, List[Path]], out_csv: Path) -> Path:
    df = load_manifest(test_csv)
    pats = df[["Patient"]].drop_duplicates().copy()

    domains = []
    for pid in pats["Patient"].tolist():
        dom = infer_patient_domain(mapping.get(pid, []))
        domains.append(dom)
    pats["Domain"] = domains

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pats.to_csv(out_csv, index=False)

    print("\nSaved patient-level domain csv:", out_csv)
    print("Rows:", len(pats))
    print("\nDomain distribution:\n", pats["Domain"].value_counts(dropna=False))
    return out_csv


# -----------------------------
# Model: Attention pooling + Dual-branch gated MIL (matches your ckpt prefixes)
# -----------------------------
class AttnPool(nn.Module):
    """
    Uses Sequential(net): Linear->Tanh->Linear, producing attention weights for instances.
    """
    def __init__(self, in_dim=2048, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, H: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # H: [N, D]
        a = self.net(H)            # [N,1]
        w = torch.softmax(a.squeeze(-1), dim=0)  # [N]
        M = torch.sum(H * w.unsqueeze(-1), dim=0)  # [D]
        return M, w


class ResNet50Encoder(nn.Module):
    """
    Minimal ResNet50 backbone compatible with torchvision's resnet50 style.
    For your ckpt, keys show encoder.backbone is a Sequential resembling resnet layers.
    We'll recreate by loading weights into a small wrapper that contains backbone as Sequential.
    """
    def __init__(self):
        super().__init__()
        # We'll build resnet50 from torchvision without classifier.
        from torchvision.models import resnet50
        m = resnet50(weights=None)
        # Feature extractor to 2048 vector:
        self.backbone = nn.Sequential(
            m.conv1, m.bn1, m.relu, m.maxpool,
            m.layer1, m.layer2, m.layer3, m.layer4,
            m.avgpool
        )

    def forward(self, x):
        f = self.backbone(x)  # [B,2048,1,1]
        f = torch.flatten(f, 1)  # [B,2048]
        return f


class DualBranchGatedMIL(nn.Module):
    """
    Produces two branch logits (non/leuko) and a gate that mixes them.
    Output dict includes:
      logit_non, logit_leu, gate_alpha (sigmoid), logit (fused), prob (sigmoid)
    """
    def __init__(self, gate_tau=2.0, pool_hidden=256):
        super().__init__()
        self.encoder = ResNet50Encoder()
        self.pool_non = AttnPool(in_dim=2048, hidden=pool_hidden)
        self.pool_leu = AttnPool(in_dim=2048, hidden=pool_hidden)

        self.cls_non = nn.Linear(2048, 1)
        self.cls_leu = nn.Linear(2048, 1)

        # gate uses concatenated pooled vectors => 4096
        self.gate = nn.Linear(4096, 1)
        self.gate_tau = gate_tau

        # optional head (exists in your ckpt; can be missing harmlessly)
        self.leuko_head = nn.Sequential(
            nn.Linear(4096, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1)
        )

    def forward(self, x: torch.Tensor, leuko_flag: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        # x: [B,3,H,W] OR [N,3,H,W] instances; we treat batch as instance list (MIL bag)
        # We'll compute features for all instances and pool.
        H = self.encoder(x)  # [N,2048] if N instances

        M_non, attn_non = self.pool_non(H)
        M_leu, attn_leu = self.pool_leu(H)

        logit_non = self.cls_non(M_non.unsqueeze(0)).squeeze(0)  # [1]
        logit_leu = self.cls_leu(M_leu.unsqueeze(0)).squeeze(0)  # [1]

        fused = torch.cat([M_non, M_leu], dim=0)  # [4096]
        gate_logit = self.gate(fused.unsqueeze(0)).squeeze(0)  # [1]
        gate_alpha = torch.sigmoid(gate_logit / self.gate_tau)  # [1]

        logit = gate_alpha * logit_leu + (1 - gate_alpha) * logit_non
        prob = torch.sigmoid(logit)

        out = {
            "logit_non": logit_non.view(()),
            "logit_leu": logit_leu.view(()),
            "gate_alpha": gate_alpha.view(()),
            "logit": logit.view(()),
            "prob": prob.view(()),
            "attn_non": attn_non,
            "attn_leu": attn_leu,
        }
        return out


def _strip_prefix(sd: Dict[str, torch.Tensor], prefix: str) -> Dict[str, torch.Tensor]:
    out = {}
    for k, v in sd.items():
        if k.startswith(prefix):
            out[k[len(prefix):]] = v
        else:
            out[k] = v
    return out


def build_model_from_ckpt(ckpt_path: Path, device: str, gate_tau: float = 2.0) -> Tuple[nn.Module, dict]:
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    if isinstance(ckpt, dict) and "model" in ckpt:
        sd = ckpt["model"]
    elif isinstance(ckpt, dict):
        sd = ckpt
    else:
        raise ValueError("Unsupported checkpoint format.")

    # Create model
    model = DualBranchGatedMIL(gate_tau=gate_tau).to(device)

    # Remap keys if needed:
    # Your ckpt example: encoder.backbone.0.weight etc.
    # Our model encoder backbone is also Sequential => encoder.backbone.0.weight etc.
    # So likely no remap needed.

    # Handle leuko_head last layer mismatch cases (sometimes 2 vs 3 index)
    # We'll attempt strict=False load, but also allow shifting leuko_head.3 -> leuko_head.2 if needed.
    model_keys = set(model.state_dict().keys())

    sd2 = dict(sd)

    # If checkpoint has leuko_head.3.* but model has leuko_head.2.*, remap
    if any(k.startswith("leuko_head.3.") for k in sd2.keys()) and any(k.startswith("leuko_head.2.") for k in model_keys):
        for wname in ["weight", "bias"]:
            k_old = f"leuko_head.3.{wname}"
            k_new = f"leuko_head.2.{wname}"
            if k_old in sd2 and k_new in model_keys and k_new not in sd2:
                sd2[k_new] = sd2[k_old]

    missing, unexpected = model.load_state_dict(sd2, strict=False)
    print(f"[CKPT LOAD] {ckpt_path.name}")
    print(f"  missing: {len(missing)}  unexpected: {len(unexpected)}")
    if len(missing) > 0:
        print("  missing sample:", missing[:10])
    if len(unexpected) > 0:
        print("  unexpected sample:", unexpected[:10])

    model.eval()
    return model, ckpt


# -----------------------------
# Inference: patient-level top-K aggregation
# -----------------------------
def get_test_transform(img_size: int) -> T.Compose:
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
    ])

def tta_variants(pil: Image.Image, img_size: int) -> List[Image.Image]:
    # lightweight TTA: original + hflip
    pil = pil.resize((img_size, img_size))
    return [pil, pil.transpose(Image.FLIP_LEFT_RIGHT)]

@torch.no_grad()
def infer_patient(model: nn.Module, img_paths: List[Path], device: str, img_size: int, k_test: int, tta: bool) -> Dict[str, float]:
    tfm = get_test_transform(img_size)

    if len(img_paths) == 0:
        return {"logit": np.nan, "prob": np.nan, "gate_alpha": np.nan}

    # sample up to k_test images
    if len(img_paths) > k_test:
        # deterministic selection for reproducibility (sorted already)
        img_paths = img_paths[:k_test]

    # collect instance tensors
    xs = []
    for p in img_paths:
        try:
            pil = robust_read_image(p)
        except Exception:
            continue
        if tta:
            for pil2 in tta_variants(pil, img_size):
                xs.append(tfm(pil2))
        else:
            xs.append(tfm(pil))
    if len(xs) == 0:
        return {"logit": np.nan, "prob": np.nan, "gate_alpha": np.nan}

    x = torch.stack(xs, dim=0).to(device)  # [N,3,H,W]
    out = model(x)

    return {
        "logit": float(out["logit"].detach().cpu().item()),
        "prob": float(out["prob"].detach().cpu().item()),
        "gate_alpha": float(out["gate_alpha"].detach().cpu().item()),
    }


# -----------------------------
# Fold prediction file aggregation (robust to missing columns)
# -----------------------------
def aggregate_folds_to_seed(runs_dir: Path, seed: int, benchmark: str, folds: Tuple[int, ...], k_test: int, out_path: Path) -> pd.DataFrame:
    dfs = []
    used = []
    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            continue
        d = pd.read_csv(p)

        if "Patient" not in d.columns:
            for c in ["patient", "patient_id", "PatientID"]:
                if c in d.columns:
                    d = d.rename(columns={c: "Patient"})
                    break
        d["Patient"] = d["Patient"].astype(str).map(normalize_patient_id)
        d = d.set_index("Patient")

        # choose available columns
        base_cols = []
        for c in ["y_true", "leuko"]:
            if c in d.columns:
                base_cols.append(c)

        fold_cols = {}
        if "logit" in d.columns:
            fold_cols["logit"] = f"logit_fold{f}"
        if "prob" in d.columns:
            fold_cols["prob"] = f"prob_fold{f}"
        if "gate_alpha" in d.columns:
            fold_cols["gate_alpha"] = f"gate_fold{f}"

        keep_cols = base_cols + list(fold_cols.keys())
        d2 = d[keep_cols].rename(columns=fold_cols)
        dfs.append(d2)
        used.append(f)

    if len(dfs) == 0:
        raise FileNotFoundError("No fold prediction files found to aggregate.")

    merged = dfs[0]
    for d in dfs[1:]:
        # keep y_true/leuko from first; join only fold-specific columns from later dfs
        join_cols = [c for c in d.columns if c not in ["y_true", "leuko"]]
        merged = merged.join(d[join_cols], how="inner")

    # compute mean logit/prob across folds that have them
    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    prob_cols  = [c for c in merged.columns if c.startswith("prob_fold")]
    gate_cols  = [c for c in merged.columns if c.startswith("gate_fold")]

    if len(logit_cols) > 0:
        merged["logit_mean"] = merged[logit_cols].mean(axis=1)
        merged["prob_mean"] = 1 / (1 + np.exp(-merged["logit_mean"].astype(float)))
    elif len(prob_cols) > 0:
        merged["prob_mean"] = merged[prob_cols].mean(axis=1)
        merged["logit_mean"] = np.log(merged["prob_mean"] / (1 - merged["prob_mean"] + 1e-8) + 1e-8)
    else:
        raise ValueError("No logit/prob columns found in fold preds.")

    if len(gate_cols) > 0:
        merged["gate_mean"] = merged[gate_cols].mean(axis=1)

    merged = merged.reset_index()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | folds used: {used}")
    return merged


# -----------------------------
# Final submission writer
# -----------------------------
def write_final_submission(agg_df: pd.DataFrame, out_path: Path):
    # expected patient-level: Patient, prob_mean (or logit_mean)
    if "prob_mean" not in agg_df.columns:
        raise ValueError("agg_df missing prob_mean.")
    sub = agg_df[["Patient", "prob_mean"]].rename(columns={"prob_mean": "prob"})
    sub.to_csv(out_path, index=False)
    print("Saved final submission:", out_path)
    print("Patients:", len(sub))


# -----------------------------
# Metrics
# -----------------------------
def fast_auc(y_true: np.ndarray, y_score: np.ndarray) -> Optional[float]:
    # robust AUC without sklearn
    y_true = np.asarray(y_true).astype(float)
    y_score = np.asarray(y_score).astype(float)
    mask = np.isfinite(y_true) & np.isfinite(y_score)
    y_true = y_true[mask]
    y_score = y_score[mask]
    # binary only
    uniq = np.unique(y_true)
    if len(uniq) < 2:
        return None
    # rank-based AUC
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(y_score)) + 1
    pos = y_true > 0.5
    n_pos = pos.sum()
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    auc = (ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return float(auc)


# -----------------------------
# Grad-CAM (Task B)
# -----------------------------
class GradCAM:
    def __init__(self, model: DualBranchGatedMIL, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.h1 = target_layer.register_forward_hook(self._forward_hook)
        self.h2 = target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        self.activations = out

    def _backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def remove(self):
        self.h1.remove()
        self.h2.remove()

    def cam_for_tensor(self, x: torch.Tensor) -> np.ndarray:
        """
        x: [1,3,H,W] (single image)
        returns CAM [H,W] normalized 0..1
        """
        self.model.zero_grad(set_to_none=True)

        # Ensure grad enabled
        with torch.enable_grad():
            out = self.model(x)  # uses MIL but with 1 instance OK
            logit = out["logit"]
            # ensure requires grad
            if not logit.requires_grad:
                logit = logit.clone().requires_grad_(True)

            logit.backward(retain_graph=False)

            A = self.activations          # [1,C,h,w]
            G = self.gradients            # [1,C,h,w]
            if A is None or G is None:
                raise RuntimeError("GradCAM hooks did not capture activations/gradients.")

            w = G.mean(dim=(2, 3), keepdim=True)  # [1,C,1,1]
            cam = (w * A).sum(dim=1, keepdim=False)  # [1,h,w]
            cam = F.relu(cam)
            cam = cam.squeeze(0)
            cam = cam.detach().cpu().numpy()

            cam = cam - cam.min()
            cam = cam / (cam.max() + 1e-8)
            return cam


def overlay_cam_on_pil(pil: Image.Image, cam: np.ndarray, alpha=0.45) -> Image.Image:
    # convert cam -> heatmap RGB using simple colormap (no external deps)
    cam_img = Image.fromarray(np.uint8(cam * 255)).resize(pil.size)
    cam_arr = np.array(cam_img).astype(np.float32) / 255.0

    base = np.array(pil).astype(np.float32) / 255.0

    # pseudo-jet: red increases with cam, blue decreases
    heat = np.zeros_like(base)
    heat[..., 0] = cam_arr  # R
    heat[..., 1] = np.clip(1.5 * cam_arr, 0, 1) * 0.3  # small G
    heat[..., 2] = 1 - cam_arr  # B

    out = (1 - alpha) * base + alpha * heat
    out = np.uint8(np.clip(out * 255, 0, 255))
    return Image.fromarray(out)


def pick_cam_layer(model: DualBranchGatedMIL) -> nn.Module:
    """
    Use last Bottleneck block conv3 output as CAM layer.
    For our encoder.backbone Sequential, layer4 is index 7 and is a Sequential of Bottlenecks.
    We target the last bottleneck: encoder.backbone[7][-1].conv3
    """
    layer4 = model.encoder.backbone[7]          # Sequential(layer4)
    last_block = layer4[-1]                     # Bottleneck
    return last_block.conv3


def run_task_b_explainability(model: DualBranchGatedMIL,
                              patient_ids: List[str],
                              mapping: Dict[str, List[Path]],
                              out_dir: Path,
                              device: str,
                              img_size: int,
                              topk_for_cam: int):
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"[{now_str()}] Task B: saving CAM overlays to: {out_dir}")

    tfm = get_test_transform(img_size)

    cam_layer = pick_cam_layer(model)
    cam = GradCAM(model, cam_layer)

    saved = 0
    for i, pid in enumerate(patient_ids):
        paths = mapping.get(pid, [])
        if not paths:
            continue

        # choose topk_for_cam images deterministically
        paths = paths[:topk_for_cam]
        pdir = out_dir / pid
        pdir.mkdir(parents=True, exist_ok=True)

        for rank, img_path in enumerate(paths, start=1):
            try:
                pil = robust_read_image(img_path).resize((img_size, img_size))
            except Exception:
                continue

            x = tfm(pil).unsqueeze(0).to(device)
            # ensure model parameters require grad for CAM
            for p in model.parameters():
                p.requires_grad_(True)

            cam_map = cam.cam_for_tensor(x)
            overlay = overlay_cam_on_pil(pil, cam_map, alpha=0.45)
            out_name = pdir / f"cam_rank{rank:02d}_{img_path.name}"
            overlay.save(out_name)
            saved += 1

        if (i + 1) % 5 == 0 or (i + 1) == len(patient_ids):
            print(f"[{now_str()}] Task B progress: {i+1}/{len(patient_ids)} patients | overlays saved={saved}")

    cam.remove()
    print(f"[{now_str()}] Task B done.")


# -----------------------------
# Task C: Domain-shift evaluation at PATIENT level
# -----------------------------
def run_task_c_domain_shift(agg_df: pd.DataFrame, patient_meta: pd.DataFrame, out_json: Path):
    """
    patient_meta must have columns: Patient, Domain
    agg_df must have Patient, y_true, logit_mean (or prob_mean)
    """
    m = patient_meta.copy()
    if "Patient" not in m.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in m.columns:
                m = m.rename(columns={c: "Patient"})
                break
    if "Domain" not in m.columns:
        raise ValueError("patient_meta must include Domain column.")

    m["Patient"] = m["Patient"].astype(str).map(normalize_patient_id)
    m["Domain"] = m["Domain"].astype(str).fillna("UNKNOWN")

    a = agg_df.copy()
    a["Patient"] = a["Patient"].astype(str).map(normalize_patient_id)

    # merge patient-level
    df = a.merge(m[["Patient", "Domain"]], on="Patient", how="inner")

    # choose score
    if "logit_mean" in df.columns:
        score = df["logit_mean"].astype(float).values
    elif "prob_mean" in df.columns:
        score = df["prob_mean"].astype(float).values
    else:
        raise ValueError("agg_df must include logit_mean or prob_mean.")

    y_true = df["y_true"].astype(float).values if "y_true" in df.columns else None

    out = {
        "n_total": int(len(df)),
        "domain_counts": df["Domain"].value_counts(dropna=False).to_dict(),
        "auc_overall": None,
        "per_domain": {}
    }

    if y_true is not None and np.isfinite(y_true).sum() > 0:
        out["auc_overall"] = fast_auc(y_true, score)

        for dom, g in df.groupby("Domain"):
            yt = g["y_true"].astype(float).values
            sc = (g["logit_mean"].astype(float).values if "logit_mean" in g.columns else g["prob_mean"].astype(float).values)
            out["per_domain"][str(dom)] = {
                "n": int(len(g)),
                "auc": fast_auc(yt, sc)
            }
    else:
        # if no y_true, still report counts
        for dom, g in df.groupby("Domain"):
            out["per_domain"][str(dom)] = {"n": int(len(g)), "auc": None}

    out_json.parent.mkdir(parents=True, exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)

    print(f"[Task C] saved: {out_json}")
    print(json.dumps(out, indent=2))


# -----------------------------
# Main
# -----------------------------
def main():
    seed_everything(cfg.SEED)

    # 1) Build filesystem mapping
    mapping = build_patient_image_mapping(cfg.IMAGES_ROOT)

    # 2) Load splits and get patient lists (unique)
    df_tr = load_manifest(cfg.TRAIN_CSV)
    df_va = load_manifest(cfg.VAL_CSV)
    df_te = load_manifest(cfg.TEST_CSV)

    tr_p = sorted(df_tr["Patient"].dropna().astype(str).map(normalize_patient_id).unique().tolist())
    va_p = sorted(df_va["Patient"].dropna().astype(str).map(normalize_patient_id).unique().tolist())
    te_p = sorted(df_te["Patient"].dropna().astype(str).map(normalize_patient_id).unique().tolist())

    # filter to patients that actually exist in filesystem mapping (scientifically clean)
    tr_p = [p for p in tr_p if p in mapping]
    va_p = [p for p in va_p if p in mapping]
    te_p = [p for p in te_p if p in mapping]

    # stats
    print_patient_split_stats("TRAIN", tr_p, mapping)
    print_patient_split_stats("VAL", va_p, mapping)
    print_patient_split_stats("TEST", te_p, mapping)

    # 3) Aggregate fold predictions (Task A result already exists; this is safe)
    agg_df = aggregate_folds_to_seed(
        runs_dir=cfg.RUNS_DIR,
        seed=cfg.SEED,
        benchmark=cfg.BENCHMARK,
        folds=cfg.FOLDS,
        k_test=cfg.K_TEST,
        out_path=cfg.AGG_OUT
    )

    # 4) Final submission
    write_final_submission(agg_df, cfg.FINAL_OUT)

    # 5) Task B (Explainability): run CAM overlays using fold1 ckpt (or chosen ckpt)
    if cfg.DO_TASK_B:
        if not cfg.CKPT_FOR_TASKB.exists():
            raise FileNotFoundError(f"Task B ckpt not found: {cfg.CKPT_FOR_TASKB}")

        model, _ = build_model_from_ckpt(cfg.CKPT_FOR_TASKB, device=cfg.DEVICE, gate_tau=2.0)
        run_task_b_explainability(
            model=model,
            patient_ids=te_p,
            mapping=mapping,
            out_dir=cfg.TASKB_DIR,
            device=cfg.DEVICE,
            img_size=cfg.IMG_SIZE,
            topk_for_cam=cfg.TOPK_FOR_CAM
        )

    # 6) Task C (Domain shift) — PATIENT LEVEL (same 73 patients)
    if cfg.DO_TASK_C:
        # build patient-domain csv from your TEST list (73), not image-level csv (4443)
        build_test_patient_domain_csv(cfg.TEST_CSV, mapping, cfg.TEST_PATIENT_DOMAIN_CSV)
        meta = pd.read_csv(cfg.TEST_PATIENT_DOMAIN_CSV)
        run_task_c_domain_shift(agg_df, meta, cfg.TASKC_JSON)

    print("\nDONE.")
    print("Agg:", cfg.AGG_OUT)
    print("Final:", cfg.FINAL_OUT)
    if cfg.DO_TASK_B:
        print("TaskB dir:", cfg.TASKB_DIR)
    if cfg.DO_TASK_C:
        print("TaskC json:", cfg.TASKC_JSON)


if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs
Patients found in image root: 210

=== TRAIN ===
Patients: 117
Patients with 0 images: 0
Images per patient: count    117.000000
mean      50.247863
std       32.454187
min        2.000000
25%       26.000000
50%       42.000000
75%       72.000000
max      186.000000
dtype: float64

=== VAL ===
Patients: 20
Patients with 0 images: 0
Images per patient: count    20.000000
mean     41.750000
std      23.212689
min       9.000000
25%      21.500000
50%      40.000000
75%      57.750000
max      87.000000
dtype: float64

=== TEST ===
Patients: 73
Patients with 0 images: 0
Images per patient: count     73.000000
mean      78.712329
std       31.139685
min       14.000000
25%       58.000000
50%       81.000000
75%       98.000000
max      153.000000
dtype: float64
Saved: D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv | folds used: [1, 2, 3, 4, 5]
Saved final submission: D:\ENT

C:\Users\admin\AppData\Local\Temp\ipykernel_14828\1578070635.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_location="cpu")


[CKPT LOAD] cv_best_res_shift_seed43_fold1.pt
  missing: 8  unexpected: 10
  missing sample: ['pool_non.net.0.weight', 'pool_non.net.0.bias', 'pool_non.net.2.weight', 'pool_non.net.2.bias', 'pool_leu.net.0.weight', 'pool_leu.net.0.bias', 'pool_leu.net.2.weight', 'pool_leu.net.2.bias']
  unexpected sample: ['pool_non.V.weight', 'pool_non.V.bias', 'pool_non.w.weight', 'pool_non.w.bias', 'pool_leu.V.weight', 'pool_leu.V.bias', 'pool_leu.w.weight', 'pool_leu.w.bias', 'leuko_head.3.weight', 'leuko_head.3.bias']
[2026-02-28 10:48:01] Task B: saving CAM overlays to: D:\ENT\_challenge2_artifacts\_runs\TASKB_res_shift_seed43
[2026-02-28 10:48:02] Task B progress: 5/73 patients | overlays saved=30
[2026-02-28 10:48:04] Task B progress: 10/73 patients | overlays saved=60
[2026-02-28 10:48:05] Task B progress: 15/73 patients | overlays saved=90
[2026-02-28 10:48:06] Task B progress: 20/73 patients | overlays saved=120
[2026-02-28 10:48:07] Task B progress: 25/73 patients | overlays saved=150
[2026

In [11]:
import re, json, time, random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# =========================
# CONFIG
# =========================
@dataclass
class Config:
    SEED: int = 43
    BENCHMARK: str = "res_shift"
    K_TEST: int = 10
    FOLDS: Tuple[int, ...] = (1,2,3,4,5)

    IMAGES_ROOT: Path = Path(r"D:\ENT")
    ARTIFACTS_ROOT: Path = Path(r"D:\ENT\_challenge2_artifacts")
    RUNS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    TRAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv")
    VAL_CSV: Path   = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv")
    TEST_CSV: Path  = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")

    # outputs
    PATIENT_DOMAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_PATIENT_with_domain.csv")
    DOMAIN_DEBUG_XLSX: Path  = Path(r"D:\ENT\_challenge2_artifacts\DOMAIN_NEEDS_LABELS.xlsx")
    TASKC_JSON: Path         = Path(r"D:\ENT\_challenge2_artifacts\_runs\TASKC_domain_shift_seed43_res_shift.json")

    AGG_OUT: Path   = Path(r"D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv")
    FINAL_OUT: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\FINAL_seed43_res_shift.csv")


cfg = Config()
cfg.RUNS_DIR.mkdir(parents=True, exist_ok=True)
cfg.ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def normalize_patient_id(x: str) -> str:
    s = str(x).strip()
    s = re.sub(r"\s+", "", s)
    if re.match(r"(?i)^patient\d+$", s):
        num = re.findall(r"\d+", s)[0]
        return f"Patient{int(num):03d}"
    return s


# =========================
# LOAD MANIFEST (robust)
# =========================
def load_manifest(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Patient column
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID", "pid", "ID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        df = df.rename(columns={df.columns[0]: "Patient"})
    df["Patient"] = df["Patient"].astype(str).map(normalize_patient_id)

    # y_true (optional)
    if "y_true" not in df.columns:
        for c in ["Histopathology", "label", "Label", "target", "Target"]:
            if c in df.columns:
                df = df.rename(columns={c: "y_true"})
                break

    return df


# =========================
# BUILD PATIENT->FILES MAP
# =========================
def build_patient_image_mapping(images_root: Path) -> Dict[str, List[Path]]:
    mapping: Dict[str, List[Path]] = {}
    img_exts = {".jpg", ".jpeg", ".png", ".bmp"}

    # find patient dirs
    for d in images_root.rglob("*"):
        if d.is_dir():
            name = d.name.replace(" ", "")
            if re.match(r"(?i)^patient\d+$", name):
                pid = normalize_patient_id(name)
                mapping.setdefault(pid, [])

    # collect images under each patient directory
    for pid in list(mapping.keys()):
        dirs = [p for p in images_root.rglob(pid) if p.is_dir()]
        if not dirs:
            dirs = [p for p in images_root.rglob("*") if p.is_dir() and p.name.lower() == pid.lower()]
        for d in dirs:
            for f in d.rglob("*"):
                if f.is_file() and f.suffix.lower() in img_exts:
                    mapping[pid].append(f)
        mapping[pid] = sorted(list(set(mapping[pid])))

    mapping = {k: v for k, v in mapping.items() if len(v) > 0}
    print("Patients found in image root:", len(mapping))
    return mapping


# =========================
# DOMAIN EXTRACTION (correct)
# 1) Use existing columns if present
# 2) Else use filename tokens if present
# 3) Else export for manual labeling (scientifically honest)
# =========================
CANDIDATE_DOMAIN_COLS = [
    "Domain","domain","Modality","modality","MODALITY",
    "Scope","scope","Type","type","Mode","mode",
    "Imaging","imaging","Technique","technique",
    "NBI","WLE"
]

def detect_domain_column(df: pd.DataFrame) -> Optional[str]:
    for c in CANDIDATE_DOMAIN_COLS:
        if c in df.columns:
            return c
    return None

def normalize_domain(v) -> str:
    if pd.isna(v):
        return "UNKNOWN"
    s = str(v).strip().lower()

    # common encodings
    if s in ["nbi", "nb", "narrowband", "narrow band", "narrow-band", "1", "true", "yes"]:
        return "NBI"
    if s in ["wle", "wl", "white light", "whitelight", "0", "false", "no"]:
        return "WLE"

    # allow already normalized or mixed
    if "nbi" in s:
        return "NBI"
    if "wle" in s or "white" in s:
        return "WLE"
    if "mixed" in s:
        return "MIXED"
    return "UNKNOWN"

def infer_domain_from_filenames(paths: List[Path]) -> str:
    if not paths:
        return "UNKNOWN"

    # look only at names (not directories) to avoid false positives
    votes = []
    for p in paths[:200]:  # cap scan for speed
        name = p.name.lower()

        # strict tokens
        if re.search(r"(^|[_\-.])nbi([_\-.]|$)", name):
            votes.append("NBI")
        elif re.search(r"(^|[_\-.])wle([_\-.]|$)", name):
            votes.append("WLE")
        elif re.search(r"(^|[_\-.])wl([_\-.]|$)", name):
            votes.append("WLE")
        elif "narrow" in name and "band" in name:
            votes.append("NBI")
        elif "white" in name and "light" in name:
            votes.append("WLE")

    if len(votes) == 0:
        return "UNKNOWN"
    vc = pd.Series(votes).value_counts()
    if "NBI" in vc.index and "WLE" in vc.index:
        return "MIXED"
    return vc.index[0]

def build_patient_domain_table(
    test_df: pd.DataFrame,
    mapping: Dict[str, List[Path]],
    out_csv: Path,
    debug_xlsx: Path
) -> Path:
    pats = test_df[["Patient"]].drop_duplicates().copy()

    # try column-based domain from CSV (if exists)
    dom_col = detect_domain_column(test_df)
    if dom_col is not None:
        # patient-level mode from that column
        tmp = test_df[["Patient", dom_col]].copy()
        tmp[dom_col] = tmp[dom_col].map(normalize_domain)
        dom = tmp.groupby("Patient")[dom_col].agg(lambda x: x.value_counts().index[0]).reset_index()
        dom = dom.rename(columns={dom_col: "Domain"})
        pats = pats.merge(dom, on="Patient", how="left")
        pats["Domain"] = pats["Domain"].fillna("UNKNOWN")
        print(f"[Domain] Using column '{dom_col}' from TEST_CSV.")
    else:
        # fallback: filename tokens
        print("[Domain] No modality/domain column found in TEST_CSV. Trying filename-based inference...")
        pats["Domain"] = [infer_domain_from_filenames(mapping.get(pid, [])) for pid in pats["Patient"].tolist()]

    # save patient domain csv
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pats.to_csv(out_csv, index=False)
    print("\nSaved patient-level domain csv:", out_csv)
    print("Rows:", len(pats))
    print("\nDomain distribution:\n", pats["Domain"].value_counts(dropna=False))

    # if still all UNKNOWN => export labeling helper
    if (pats["Domain"] == "UNKNOWN").all():
        rows = []
        for pid in pats["Patient"].tolist():
            files = mapping.get(pid, [])
            examples = [f.name for f in files[:8]]
            rows.append({
                "Patient": pid,
                "ExampleFiles": " | ".join(examples),
                "SuggestedDomain(WLE/NBI/MIXED)": ""
            })
        dbg = pd.DataFrame(rows)
        debug_xlsx.parent.mkdir(parents=True, exist_ok=True)
        dbg.to_excel(debug_xlsx, index=False)
        print("\n[IMPORTANT] Domain is UNKNOWN for all patients.")
        print("This means your dataset paths/filenames do not encode modality.")
        print("I exported a labeling helper file here:\n ", debug_xlsx)

    return out_csv


# =========================
# AGGREGATION + TASK C
# =========================
def aggregate_folds_to_seed(runs_dir: Path, seed: int, benchmark: str, folds: Tuple[int, ...], k_test: int, out_path: Path) -> pd.DataFrame:
    dfs = []
    used = []
    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            continue
        d = pd.read_csv(p)

        if "Patient" not in d.columns:
            for c in ["patient", "patient_id", "PatientID"]:
                if c in d.columns:
                    d = d.rename(columns={c: "Patient"})
                    break
        d["Patient"] = d["Patient"].astype(str).map(normalize_patient_id)
        d = d.set_index("Patient")

        base_cols = [c for c in ["y_true","leuko"] if c in d.columns]
        fold_cols = {}
        if "logit" in d.columns: fold_cols["logit"] = f"logit_fold{f}"
        if "prob" in d.columns:  fold_cols["prob"]  = f"prob_fold{f}"
        if "gate_alpha" in d.columns: fold_cols["gate_alpha"] = f"gate_fold{f}"

        keep_cols = base_cols + list(fold_cols.keys())
        d2 = d[keep_cols].rename(columns=fold_cols)
        dfs.append(d2)
        used.append(f)

    if len(dfs) == 0:
        raise FileNotFoundError("No fold prediction files found to aggregate.")

    merged = dfs[0]
    for d in dfs[1:]:
        join_cols = [c for c in d.columns if c not in ["y_true","leuko"]]
        merged = merged.join(d[join_cols], how="inner")

    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    if logit_cols:
        merged["logit_mean"] = merged[logit_cols].mean(axis=1)
        merged["prob_mean"] = 1 / (1 + np.exp(-merged["logit_mean"].astype(float)))
    else:
        prob_cols = [c for c in merged.columns if c.startswith("prob_fold")]
        merged["prob_mean"] = merged[prob_cols].mean(axis=1)
        merged["logit_mean"] = np.log(merged["prob_mean"] / (1 - merged["prob_mean"] + 1e-8) + 1e-8)

    merged = merged.reset_index()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | folds used: {used}")
    return merged

def write_final_submission(agg_df: pd.DataFrame, out_path: Path):
    sub = agg_df[["Patient","prob_mean"]].rename(columns={"prob_mean":"prob"})
    sub.to_csv(out_path, index=False)
    print("Saved final submission:", out_path)
    print("Patients:", len(sub))

def fast_auc(y_true: np.ndarray, y_score: np.ndarray) -> Optional[float]:
    y_true = np.asarray(y_true).astype(float)
    y_score = np.asarray(y_score).astype(float)
    m = np.isfinite(y_true) & np.isfinite(y_score)
    y_true, y_score = y_true[m], y_score[m]
    if len(np.unique(y_true)) < 2:
        return None
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(y_score)) + 1
    pos = y_true > 0.5
    n_pos = pos.sum()
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    auc = (ranks[pos].sum() - n_pos*(n_pos+1)/2) / (n_pos*n_neg)
    return float(auc)

def run_task_c_domain_shift(agg_df: pd.DataFrame, patient_meta: pd.DataFrame, out_json: Path):
    meta = patient_meta.copy()
    meta["Patient"] = meta["Patient"].astype(str).map(normalize_patient_id)
    meta["Domain"] = meta["Domain"].astype(str).fillna("UNKNOWN")

    df = agg_df.copy()
    df["Patient"] = df["Patient"].astype(str).map(normalize_patient_id)
    df = df.merge(meta[["Patient","Domain"]], on="Patient", how="inner")

    out = {
        "n_total": int(len(df)),
        "domain_counts": df["Domain"].value_counts(dropna=False).to_dict(),
        "auc_overall": None,
        "per_domain": {}
    }

    if "y_true" in df.columns:
        y_true = df["y_true"].astype(float).values
        score = df["logit_mean"].astype(float).values
        out["auc_overall"] = fast_auc(y_true, score)
        for dom, g in df.groupby("Domain"):
            yt = g["y_true"].astype(float).values
            sc = g["logit_mean"].astype(float).values
            out["per_domain"][str(dom)] = {"n": int(len(g)), "auc": fast_auc(yt, sc)}
    else:
        for dom, g in df.groupby("Domain"):
            out["per_domain"][str(dom)] = {"n": int(len(g)), "auc": None}

    out_json.parent.mkdir(parents=True, exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)

    print(f"[Task C] saved: {out_json}")
    print(json.dumps(out, indent=2))


# =========================
# MAIN
# =========================
def main():
    seed_everything(cfg.SEED)
    print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
    print("RUNS_DIR:", cfg.RUNS_DIR)

    mapping = build_patient_image_mapping(cfg.IMAGES_ROOT)

    df_te = load_manifest(cfg.TEST_CSV)

    # Build patient-domain table (scientifically correct)
    patient_domain_csv = build_patient_domain_table(
        test_df=df_te,
        mapping=mapping,
        out_csv=cfg.PATIENT_DOMAIN_CSV,
        debug_xlsx=cfg.DOMAIN_DEBUG_XLSX
    )

    # Aggregate + final submission (already working)
    agg_df = aggregate_folds_to_seed(cfg.RUNS_DIR, cfg.SEED, cfg.BENCHMARK, cfg.FOLDS, cfg.K_TEST, cfg.AGG_OUT)
    write_final_submission(agg_df, cfg.FINAL_OUT)

    # Task C only if domain has signal
    meta = pd.read_csv(patient_domain_csv)
    if (meta["Domain"] == "UNKNOWN").all():
        print("\n[STOP] Task C is not valid because domain is UNKNOWN for all patients.")
        print("Please label domains in:", cfg.DOMAIN_DEBUG_XLSX)
        print("Then rerun, or provide a modality column in the TEST_CSV.")
        return

    run_task_c_domain_shift(agg_df, meta, cfg.TASKC_JSON)

    print("\nDONE.")
    print("Agg:", cfg.AGG_OUT)
    print("Final:", cfg.FINAL_OUT)
    print("Domain meta:", cfg.PATIENT_DOMAIN_CSV)
    print("TaskC json:", cfg.TASKC_JSON)


if __name__ == "__main__":
    main()

IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs
Patients found in image root: 210
[Domain] Using column 'modality' from TEST_CSV.

Saved patient-level domain csv: D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_PATIENT_with_domain.csv
Rows: 73

Domain distribution:
 Domain
UNKNOWN    73
Name: count, dtype: int64

[IMPORTANT] Domain is UNKNOWN for all patients.
This means your dataset paths/filenames do not encode modality.
I exported a labeling helper file here:
  D:\ENT\_challenge2_artifacts\DOMAIN_NEEDS_LABELS.xlsx
Saved: D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv | folds used: [1, 2, 3, 4, 5]
Saved final submission: D:\ENT\_challenge2_artifacts\_runs\FINAL_seed43_res_shift.csv
Patients: 73

[STOP] Task C is not valid because domain is UNKNOWN for all patients.
Please label domains in: D:\ENT\_challenge2_artifacts\DOMAIN_NEEDS_LABELS.xlsx
Then rerun, or provide a modality column in the TEST_CSV.


In [12]:
# ============================================================
# Scientifically-clean FULL PIPELINE (Tasks A + B + C + finalize)
# - Patient-level splits (from your provided CSVs)
# - Aggregates existing fold predictions to seed-level (robust to column names)
# - Creates FINAL submission CSV (patients=73)
# - Task B: Grad-CAM overlays (uses fold1 ckpt)
# - Task C: Domain shift AUC per modality (WLE/NBI/MIXED), with strict stop if all UNKNOWN
#   -> builds a DOMAIN_NEEDS_LABELS.xlsx helper if modality cannot be inferred
#
# Paths are Windows-friendly; run in Jupyter or as a .py file.
# ============================================================

import os
import re
import json
import time
import math
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

from PIL import Image, ImageOps
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms


# ----------------------------
# Config
# ----------------------------
@dataclass
class CFG:
    # Core I/O
    IMAGES_ROOT: Path = Path(r"D:\ENT")
    ARTIFACTS_ROOT: Path = Path(r"D:\ENT\_challenge2_artifacts")
    RUNS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    # Manifests (patient-level)
    TRAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv")
    VAL_CSV: Path   = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv")
    TEST_CSV: Path  = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")

    # If you already generated a patient-level domain CSV, put it here.
    # Otherwise this script will attempt to infer from TEST_CSV 'modality' or create helper XLSX.
    TEST_PATIENT_DOMAIN_CSV: Optional[Path] = None  # e.g. Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_PATIENT_with_domain.csv")

    # Run params
    BENCHMARK: str = "res_shift"
    SEED: int = 43
    FOLDS: Tuple[int, ...] = (1, 2, 3, 4, 5)
    K_TEST: int = 10

    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"

    # Task B (explainability)
    DO_TASK_B: bool = True
    TOPK_FOR_CAM: int = 6
    IMG_SIZE: int = 224
    TASKB_DIR_NAME: str = "TASKB_res_shift_seed43"
    CKPT_FOR_TASKB: Optional[Path] = None  # if None -> auto: cv_best_{BENCHMARK}_seed{SEED}_fold1.pt

    # Task C (domain shift)
    DO_TASK_C: bool = True
    DOMAIN_HELPER_XLSX: Path = Path(r"D:\ENT\_challenge2_artifacts\DOMAIN_NEEDS_LABELS.xlsx")

    # Final outputs
    FINAL_SUBMISSION_NAME: str = "FINAL_seed43_res_shift.csv"


cfg = CFG()


# ----------------------------
# Utilities
# ----------------------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def now_str():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def norm_patient(x: str) -> str:
    s = str(x).strip().replace(" ", "")
    if s.lower().startswith("patient"):
        num = "".join([c for c in s if c.isdigit()])
        if num:
            return f"Patient{int(num):03d}"
    return s


def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)


# ----------------------------
# Loading manifests (robust)
# ----------------------------
def load_manifest(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # normalize Patient column
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID", "patientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"{csv_path} must contain a Patient/patient_id column.")
    df["Patient"] = df["Patient"].astype(str).map(norm_patient)

    # normalize label if present
    if "y_true" not in df.columns:
        for c in ["y", "label", "Label", "target", "Target"]:
            if c in df.columns:
                df = df.rename(columns={c: "y_true"})
                break

    # Some splits are patient-only CSVs (no path). That is OK.
    return df


# ----------------------------
# Build patient -> image paths mapping from filesystem
# ----------------------------
def build_patient_image_mapping(images_root: Path) -> Dict[str, List[Path]]:
    """
    Assumes each patient has its own folder somewhere under IMAGES_ROOT,
    with a folder name containing "PatientXYZ" or exactly "PatientXYZ".
    We'll map by folder segments.
    """
    images_root = Path(images_root)
    mapping: Dict[str, List[Path]] = {}

    # collect candidate patient folders
    # Strategy: any path segment matching Patient\d+
    pat_re = re.compile(r"^Patient(\d+)$", re.IGNORECASE)

    for p in images_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]:
            # find a Patient### segment in parents
            pid = None
            for seg in p.parts:
                m = pat_re.match(seg)
                if m:
                    pid = f"Patient{int(m.group(1)):03d}"
                    break
            if pid is None:
                continue
            mapping.setdefault(pid, []).append(p)

    # sort
    for k in mapping:
        mapping[k] = sorted(mapping[k])
    return mapping


# ----------------------------
# Basic stats
# ----------------------------
def report_split_stats(name: str, df: pd.DataFrame, mapping: Dict[str, List[Path]]):
    pats = sorted(df["Patient"].unique().tolist())
    counts = []
    zero = 0
    for pid in pats:
        n = len(mapping.get(pid, []))
        counts.append(n)
        if n == 0:
            zero += 1
    s = pd.Series(counts, dtype=float)
    print(f"\n=== {name.upper()} ===")
    print("Patients:", len(pats))
    print("Patients with 0 images:", zero)
    print("Images per patient:", s.describe())
    return pats


# ----------------------------
# AUC (no sklearn dependency)
# ----------------------------
def fast_auc(y_true, y_score) -> Optional[float]:
    y_true = np.asarray(y_true).astype(float)
    y_score = np.asarray(y_score).astype(float)
    m = np.isfinite(y_true) & np.isfinite(y_score)
    y_true, y_score = y_true[m], y_score[m]
    if len(np.unique(y_true)) < 2:
        return None
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(y_score)) + 1
    pos = y_true > 0.5
    n_pos = pos.sum()
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    return float((ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


# ----------------------------
# Fold predictions aggregation (ROBUST)
# ----------------------------
def _detect_pred_columns(df: pd.DataFrame) -> Dict[str, str]:
    """
    Your fold pred CSVs can have different column names.
    We detect:
      - patient id: Patient
      - y_true: y_true (optional)
      - leuko: leuko (optional)
      - logit: logit OR logit_mean OR logits OR score
      - prob:  prob OR prob_mean OR p
      - gate:  gate_alpha (optional)
    """
    cols = df.columns.tolist()
    out = {}

    # patient
    if "Patient" in cols:
        out["Patient"] = "Patient"
    else:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in cols:
                out["Patient"] = c
                break
    if "Patient" not in out:
        raise ValueError("Pred file missing Patient column.")

    # y_true
    for c in ["y_true", "y", "label", "Label", "target", "Target"]:
        if c in cols:
            out["y_true"] = c
            break

    # leuko
    for c in ["leuko", "Leukoplakia", "leukoplakia"]:
        if c in cols:
            out["leuko"] = c
            break

    # logit-like
    for c in ["logit", "logit_mean", "logits", "score", "pred", "raw_logit"]:
        if c in cols:
            out["logit"] = c
            break
    if "logit" not in out:
        # sometimes only prob exists
        for c in ["prob", "prob_mean", "p"]:
            if c in cols:
                out["prob"] = c
                break
        if "prob" not in out:
            raise ValueError(f"Pred file columns do not include a logit/prob column. cols={cols}")

    # prob
    for c in ["prob", "prob_mean", "p"]:
        if c in cols:
            out["prob"] = c
            break

    # gate
    for c in ["gate_alpha", "gate", "alpha", "gate_mean"]:
        if c in cols:
            out["gate"] = c
            break

    return out


def aggregate_folds_to_seed(runs_dir: Path, seed: int, benchmark: str, folds: Tuple[int, ...], k_test: int, out_path: Path) -> pd.DataFrame:
    dfs = []
    used = []
    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            continue
        d = pd.read_csv(p)
        det = _detect_pred_columns(d)
        d = d.rename(columns={det["Patient"]: "Patient"})
        d["Patient"] = d["Patient"].astype(str).map(norm_patient)
        d = d.set_index("Patient")

        keep = {}
        if "y_true" in det: keep["y_true"] = det["y_true"]
        if "leuko"  in det: keep["leuko"]  = det["leuko"]
        if "prob"   in det: keep["prob"]   = det["prob"]
        if "logit"  in det: keep["logit"]  = det["logit"]
        if "gate"   in det: keep["gate"]   = det["gate"]

        d2 = d[list(keep.values())].rename(columns={v: k for k, v in keep.items()})

        # rename fold-specific
        rename = {}
        if "logit" in d2.columns: rename["logit"] = f"logit_fold{f}"
        if "prob"  in d2.columns: rename["prob"]  = f"prob_fold{f}"
        if "gate"  in d2.columns: rename["gate"]  = f"gate_fold{f}"
        d2 = d2.rename(columns=rename)

        dfs.append(d2)
        used.append(f)

    if len(dfs) == 0:
        raise FileNotFoundError("No fold prediction files found for aggregation.")

    merged = dfs[0].copy()
    for d in dfs[1:]:
        # keep y_true/leuko only once (if present)
        for col in ["y_true", "leuko"]:
            if col in d.columns and col in merged.columns:
                d = d.drop(columns=[col])
        merged = merged.join(d, how="outer")

    # compute means
    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    prob_cols  = [c for c in merged.columns if c.startswith("prob_fold")]
    gate_cols  = [c for c in merged.columns if c.startswith("gate_fold")]

    if logit_cols:
        merged["logit_mean"] = merged[logit_cols].mean(axis=1)
        merged["prob_mean"]  = 1.0 / (1.0 + np.exp(-merged["logit_mean"].astype(float)))
    elif prob_cols:
        merged["prob_mean"] = merged[prob_cols].mean(axis=1)
        # avoid inf
        eps = 1e-6
        p = np.clip(merged["prob_mean"].astype(float), eps, 1 - eps)
        merged["logit_mean"] = np.log(p / (1 - p))
    else:
        raise ValueError("Neither logit_fold* nor prob_fold* exist after merging.")

    if gate_cols:
        merged["gate_mean"] = merged[gate_cols].mean(axis=1)

    merged = merged.reset_index()
    ensure_dir(out_path.parent)
    merged.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | folds used: {used}")
    return merged


# ----------------------------
# Final submission creation
# ----------------------------
def make_final_submission(agg_df: pd.DataFrame, out_csv: Path) -> pd.DataFrame:
    """
    Creates FINAL csv with at least:
      Patient, prob (or prob_mean)
    If competition requires specific columns, adjust here.
    """
    df = agg_df.copy()
    if "prob_mean" not in df.columns:
        # derive from logit_mean
        df["prob_mean"] = 1.0 / (1.0 + np.exp(-df["logit_mean"].astype(float)))
    sub = df[["Patient", "prob_mean"]].rename(columns={"prob_mean": "prob"})
    sub.to_csv(out_csv, index=False)
    print(f"Saved final submission: {out_csv}")
    print(f"Patients: {len(sub)}")
    return sub


# ----------------------------
# Model components (Gated MIL style)
# ----------------------------
class AttnPool(nn.Module):
    # Matches your model structure with .net = Sequential(Linear->Tanh->Linear)
    def __init__(self, in_dim: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, feats: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # feats: [N, C]
        a = self.net(feats)  # [N,1]
        a = a.squeeze(1)     # [N]
        w = torch.softmax(a, dim=0)  # [N]
        pooled = (w.unsqueeze(1) * feats).sum(dim=0)  # [C]
        return pooled, w


class DualBranchGatedMIL(nn.Module):
    """
    Inferred from your checkpoint prefixes:
      encoder.backbone.*
      pool_non, pool_leu  (AttnPool)
      cls_non, cls_leu    (Linear 2048->1)
      gate               (Linear 4096->1)
      leuko_head         (MLP 4096->256->1)
    """
    def __init__(self, backbone: nn.Module, feat_dim: int = 2048, pool_hidden: int = 256, gate_tau: float = 2.0):
        super().__init__()
        self.encoder = nn.Module()
        self.encoder.backbone = backbone  # should output [B,2048,1,1]
        self.pool_non = AttnPool(feat_dim, pool_hidden)
        self.pool_leu = AttnPool(feat_dim, pool_hidden)
        self.cls_non = nn.Linear(feat_dim, 1)
        self.cls_leu = nn.Linear(feat_dim, 1)
        self.gate = nn.Linear(2 * feat_dim, 1)
        self.leuko_head = nn.Sequential(
            nn.Linear(2 * feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1),
        )
        self.gate_tau = gate_tau

        # convenience handle for CAM layer
        self._cam_target_layer = None

    def set_cam_target_layer(self, layer: nn.Module):
        self._cam_target_layer = layer

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # backbone ends with AdaptiveAvgPool2d(1) in your printout
        out = self.encoder.backbone(x)  # [B,2048,1,1]
        if out.dim() == 4:
            out = out.flatten(1)        # [B,2048]
        return out

    def forward(self, x_bag: torch.Tensor, leuko_mask: Optional[torch.Tensor] = None):
        """
        x_bag: [N,3,H,W] images of a patient
        leuko_mask: optional [N] boolean (True if leukoplakia image)
        """
        feats = self.encode(x_bag)  # [N,2048]

        if leuko_mask is None:
            # if unknown, treat all as "non"
            non_feats = feats
            leu_feats = feats
        else:
            leuko_mask = leuko_mask.bool()
            non_feats = feats[~leuko_mask] if (~leuko_mask).any() else feats
            leu_feats = feats[leuko_mask] if leuko_mask.any() else feats

        z_non, attn_non = self.pool_non(non_feats)  # [2048], [N_non]
        z_leu, attn_leu = self.pool_leu(leu_feats)  # [2048], [N_leu]

        logit_non = self.cls_non(z_non)  # [1]
        logit_leu = self.cls_leu(z_leu)  # [1]

        z = torch.cat([z_non, z_leu], dim=0).unsqueeze(0)  # [1,4096]
        gate_logit = self.gate(z) / max(self.gate_tau, 1e-6)
        alpha = torch.sigmoid(gate_logit).squeeze(0)  # [1] -> scalar-ish

        # fused logit
        logit = alpha * logit_leu + (1 - alpha) * logit_non  # [1]

        # auxiliary leuko logit (optional in your training)
        leuko_logit = self.leuko_head(z)  # [1,1]

        return {
            "logit": logit.squeeze(0),        # scalar tensor
            "alpha": alpha.squeeze(),
            "logit_non": logit_non.squeeze(0),
            "logit_leu": logit_leu.squeeze(0),
            "attn_non": attn_non.detach(),
            "attn_leu": attn_leu.detach(),
            "leuko_logit": leuko_logit.squeeze(0),
        }


# ----------------------------
# Backbone builder compatible with your checkpoint keys (Sequential style)
# ----------------------------
def build_resnet50_sequential_backbone() -> nn.Module:
    """
    Your checkpoint showed encoder.backbone as nn.Sequential:
      0: conv1, 1: bn1, 2: relu, 3: maxpool,
      4: layer1 (Sequential of Bottleneck),
      5: layer2,
      6: layer3,
      7: layer4,
      8: AdaptiveAvgPool2d(1)
    We will reconstruct that using torchvision's resnet50 pieces.
    """
    from torchvision.models import resnet50, ResNet50_Weights
    m = resnet50(weights=None)  # weights not needed; we load ckpt
    backbone = nn.Sequential(
        m.conv1,
        m.bn1,
        m.relu,
        m.maxpool,
        m.layer1,
        m.layer2,
        m.layer3,
        m.layer4,
        nn.AdaptiveAvgPool2d((1, 1)),
    )
    return backbone


# ----------------------------
# Load model from checkpoint (robust key remap)
# ----------------------------
def build_model_from_ckpt(ckpt_path: Path, device: str, gate_tau: float = 2.0) -> Tuple[DualBranchGatedMIL, Dict]:
    ckpt_path = Path(ckpt_path)
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    sd = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    backbone = build_resnet50_sequential_backbone()
    model = DualBranchGatedMIL(backbone=backbone, feat_dim=2048, pool_hidden=256, gate_tau=gate_tau)

    # Handle two variants of attention pool keying:
    # Variant A (your earlier successful one): pool_non.V / pool_non.w etc
    # Variant B: pool_non.net.0, pool_non.net.2
    # We'll remap if needed.
    model_keys = set(model.state_dict().keys())
    new_sd = {}

    for k, v in sd.items():
        nk = k

        # If someone saved backbone as encoder.backbone.conv1.weight etc (torchvision style), we need to map to sequential indices.
        # Your ckpt already uses encoder.backbone.0.weight etc, so mostly no-op.
        # But we support a small mapping anyway.
        if nk.startswith("encoder.backbone.conv1."):
            nk = nk.replace("encoder.backbone.conv1.", "encoder.backbone.0.")
        if nk.startswith("encoder.backbone.bn1."):
            nk = nk.replace("encoder.backbone.bn1.", "encoder.backbone.1.")
        if nk.startswith("encoder.backbone.relu."):
            nk = nk.replace("encoder.backbone.relu.", "encoder.backbone.2.")
        if nk.startswith("encoder.backbone.maxpool."):
            nk = nk.replace("encoder.backbone.maxpool.", "encoder.backbone.3.")
        # layer1..layer4 mapping would require more; but your ckpt is already sequential.

        # Attention pool remap:
        # If ckpt has pool_non.V/w and model expects pool_non.net.0/net.2
        # We can convert V->net.0 and w->net.2
        if nk.startswith("pool_non.V."):
            nk = nk.replace("pool_non.V.", "pool_non.net.0.")
        if nk.startswith("pool_non.w."):
            nk = nk.replace("pool_non.w.", "pool_non.net.2.")
        if nk.startswith("pool_leu.V."):
            nk = nk.replace("pool_leu.V.", "pool_leu.net.0.")
        if nk.startswith("pool_leu.w."):
            nk = nk.replace("pool_leu.w.", "pool_leu.net.2.")

        # leuko_head layer index mismatch (sometimes 0,1,3 vs 0,1,2)
        # If ckpt uses leuko_head.3.* but model uses leuko_head.2.*
        if nk.startswith("leuko_head.3."):
            nk = nk.replace("leuko_head.3.", "leuko_head.2.")

        new_sd[nk] = v

    missing, unexpected = model.load_state_dict(new_sd, strict=False)
    model = model.to(device)
    model.eval()

    print(f"[CKPT LOAD] {ckpt_path.name}")
    print(f"  missing: {len(missing)}  unexpected: {len(unexpected)}")
    if len(missing) > 0:
        print("  missing sample:", list(missing)[:10])
    if len(unexpected) > 0:
        print("  unexpected sample:", list(unexpected)[:10])

    # choose a good CAM target layer: last bottleneck block in layer4
    # In sequential backbone, layer4 is at index 7, which is m.layer4 (Sequential of Bottleneck).
    # We'll use the last bottleneck conv3 inside layer4[-1].
    try:
        target = model.encoder.backbone[7][-1].conv3
        model.set_cam_target_layer(target)
    except Exception:
        model.set_cam_target_layer(None)

    return model, ckpt


# ----------------------------
# Grad-CAM (single layer)
# ----------------------------
class GradCAM:
    def __init__(self, model: DualBranchGatedMIL, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._h1 = None
        self._h2 = None

    def _fwd_hook(self, module, inp, out):
        self.activations = out.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __enter__(self):
        self._h1 = self.target_layer.register_forward_hook(self._fwd_hook)
        self._h2 = self.target_layer.register_full_backward_hook(self._bwd_hook)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self._h1: self._h1.remove()
        if self._h2: self._h2.remove()

    @torch.no_grad()
    def _normalize(self, cam: np.ndarray) -> np.ndarray:
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam

    def cam_for_tensor(self, x: torch.Tensor) -> np.ndarray:
        """
        x: [1,3,H,W]
        Returns cam: [H,W] normalized 0..1
        """
        self.model.zero_grad(set_to_none=True)

        # Need gradients
        was_training = self.model.training
        self.model.train(False)

        x = x.requires_grad_(True)

        out = self.model(x)  # bag with single image
        logit = out["logit"]
        if not logit.requires_grad:
            # force graph
            logit = logit * 1.0

        logit.backward(retain_graph=False)

        A = self.activations  # [1,C,h,w]
        G = self.gradients    # [1,C,h,w]
        if A is None or G is None:
            raise RuntimeError("GradCAM hooks did not capture activations/gradients.")

        w = G.mean(dim=(2, 3), keepdim=True)  # [1,C,1,1]
        cam = (w * A).sum(dim=1, keepdim=False)  # [1,h,w]
        cam = F.relu(cam)
        cam = cam[0].cpu().numpy()
        cam = self._normalize(cam)

        # restore
        if was_training:
            self.model.train(True)

        return cam


def overlay_cam_on_pil(pil: Image.Image, cam: np.ndarray, alpha: float = 0.45) -> Image.Image:
    # cam: [h,w] 0..1, resize to image
    cam_res = cv2.resize(cam, pil.size, interpolation=cv2.INTER_LINEAR)
    heat = (cam_res * 255).astype(np.uint8)
    heat = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    heat_pil = Image.fromarray(heat)

    blended = Image.blend(pil.convert("RGB"), heat_pil, alpha=alpha)
    return blended


# ----------------------------
# Task B: Explainability (CAM overlays)
# ----------------------------
def run_task_b_explainability(
    model: DualBranchGatedMIL,
    patient_ids: List[str],
    mapping: Dict[str, List[Path]],
    out_dir: Path,
    device: str,
    img_size: int,
    topk_for_cam: int
):
    ensure_dir(out_dir)
    print(f"[{now_str()}] Task B: saving CAM overlays to: {out_dir}")

    tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

    # decide CAM target
    if model._cam_target_layer is None:
        raise RuntimeError("CAM target layer not set. Cannot run Task B.")
    cam_layer = model._cam_target_layer

    saved = 0
    with GradCAM(model, cam_layer) as cam:
        for i, pid in enumerate(patient_ids, 1):
            imgs = mapping.get(pid, [])
            if len(imgs) == 0:
                continue

            # choose topK by simple heuristic: evenly spaced or first K
            chosen = imgs[:topk_for_cam] if len(imgs) >= topk_for_cam else imgs

            # folder per patient
            pdir = out_dir / pid
            ensure_dir(pdir)

            for j, img_path in enumerate(chosen):
                try:
                    pil = Image.open(img_path).convert("RGB")
                except Exception:
                    continue

                x = tfm(pil).unsqueeze(0).to(device)

                # must allow grads
                torch.set_grad_enabled(True)
                cam_map = cam.cam_for_tensor(x)
                torch.set_grad_enabled(False)

                pil_resized = pil.resize((img_size, img_size))
                overlay = overlay_cam_on_pil(pil_resized, cam_map, alpha=0.45)

                out_name = f"cam_{j:02d}_{img_path.name}"
                overlay.save(pdir / out_name)
                saved += 1

            if i % 5 == 0 or i == len(patient_ids):
                print(f"[{now_str()}] Task B progress: {i}/{len(patient_ids)} patients | overlays saved={saved}")

    print(f"[{now_str()}] Task B done.")


# ----------------------------
# Task C: Domain shift (WLE vs NBI)
# ----------------------------
def normalize_domain(v) -> str:
    if pd.isna(v):
        return "UNKNOWN"
    s = str(v).strip()
    if s == "":
        return "UNKNOWN"
    sl = s.lower()

    # direct mentions
    if "nbi" in sl or "narrow" in sl:
        return "NBI"
    if "wle" in sl or "white" in sl or sl in ["wl", "w"]:
        return "WLE"
    if "mix" in sl:
        return "MIXED"
    # numeric encodings (common)
    if sl in ["0", "false", "no"]:
        return "WLE"
    if sl in ["1", "true", "yes"]:
        return "NBI"
    return "UNKNOWN"


def build_patient_domain_from_test(
    test_csv: Path,
    out_patient_csv: Path,
    helper_xlsx: Path
) -> Tuple[pd.DataFrame, bool]:
    """
    Produces patient-level domain CSV with columns: Patient, Domain
    Returns (patient_df, ok_for_task_c)
    ok_for_task_c is False if all UNKNOWN (needs manual labels)
    """
    df = pd.read_csv(test_csv)

    # normalize Patient column
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError("TEST_CSV missing Patient column.")
    df["Patient"] = df["Patient"].astype(str).map(norm_patient)

    # locate modality/domain column
    dom_col = None
    for c in ["Domain", "domain", "modality", "Modality", "MODALITY", "source", "Source"]:
        if c in df.columns:
            dom_col = c
            break

    if dom_col is None:
        # cannot infer; make helper
        pats = sorted(df["Patient"].unique().tolist())
        helper = pd.DataFrame({"Patient": pats, "Domain": ""})
        helper.to_excel(helper_xlsx, index=False)
        print(f"\n[IMPORTANT] No modality/domain column found in TEST_CSV.")
        print(f"I exported a labeling helper file here:\n  {helper_xlsx}")
        return pd.DataFrame({"Patient": pats, "Domain": ["UNKNOWN"] * len(pats)}), False

    print(f"[Domain] Using column '{dom_col}' from TEST_CSV.")
    df["Domain_img"] = df[dom_col].map(normalize_domain)

    # patient aggregation
    def patient_domain(domains: pd.Series) -> str:
        s = domains.dropna().astype(str)
        s = s[s != ""]
        if len(s) == 0:
            return "UNKNOWN"
        vals = set(s.tolist())
        if "WLE" in vals and "NBI" in vals:
            return "MIXED"
        vc = s.value_counts()
        return vc.index[0] if len(vc) else "UNKNOWN"

    pat = df.groupby("Patient")["Domain_img"].apply(patient_domain).reset_index()
    pat = pat.rename(columns={"Domain_img": "Domain"})
    pat.to_csv(out_patient_csv, index=False)

    print(f"\nSaved patient-level domain csv: {out_patient_csv}")
    print("Rows:", len(pat))
    print("\nDomain distribution:\n", pat["Domain"].value_counts(dropna=False))

    all_unknown = (pat["Domain"].astype(str) == "UNKNOWN").all()
    if all_unknown:
        # export helper with image-level hints
        pats = sorted(pat["Patient"].tolist())
        helper = pd.DataFrame({"Patient": pats, "Domain": ""})
        helper.to_excel(helper_xlsx, index=False)
        print("\n[IMPORTANT] Domain is UNKNOWN for all patients.")
        print("This means your dataset paths/filenames do not encode modality OR the modality values are not mappable.")
        print("I exported a labeling helper file here:")
        print(f"  {helper_xlsx}")
        return pat, False

    return pat, True


def run_task_c_domain_shift(agg_df: pd.DataFrame, patient_domain_df: pd.DataFrame, out_json: Path):
    """
    agg_df must contain Patient, y_true, logit_mean
    patient_domain_df contains Patient, Domain
    """
    df = agg_df.merge(patient_domain_df, on="Patient", how="inner")
    if "y_true" not in df.columns:
        raise ValueError("agg_df missing y_true (needed for AUC).")

    out = {
        "n_total": int(len(df)),
        "domain_counts": df["Domain"].value_counts(dropna=False).to_dict(),
        "auc_overall": fast_auc(df["y_true"], df["logit_mean"]),
        "per_domain": {}
    }

    for dom, g in df.groupby("Domain"):
        out["per_domain"][str(dom)] = {
            "n": int(len(g)),
            "auc": fast_auc(g["y_true"], g["logit_mean"])
        }

    ensure_dir(out_json.parent)
    with open(out_json, "w") as f:
        json.dump(out, f, indent=2)

    print(f"[Task C] saved: {out_json}")
    print(json.dumps(out, indent=2))


# ----------------------------
# Main
# ----------------------------
def main():
    seed_everything(cfg.SEED)

    print("DEVICE:", cfg.DEVICE)
    print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
    print("RUNS_DIR:", cfg.RUNS_DIR)

    # Build mapping from filesystem
    mapping = build_patient_image_mapping(cfg.IMAGES_ROOT)
    print("Patients found in image root:", len(mapping))

    # Load manifests (patient-level)
    df_tr = load_manifest(cfg.TRAIN_CSV)
    df_va = load_manifest(cfg.VAL_CSV)
    df_te = load_manifest(cfg.TEST_CSV)

    tr_p = report_split_stats("train", df_tr, mapping)
    va_p = report_split_stats("val", df_va, mapping)
    te_p = report_split_stats("test", df_te, mapping)

    # 1) Aggregate folds -> seed
    agg_path = cfg.RUNS_DIR / f"pred_seed{cfg.SEED}_{cfg.BENCHMARK}_folds1_5_Ktest{cfg.K_TEST}_agg.csv"
    agg_df = aggregate_folds_to_seed(
        runs_dir=cfg.RUNS_DIR,
        seed=cfg.SEED,
        benchmark=cfg.BENCHMARK,
        folds=cfg.FOLDS,
        k_test=cfg.K_TEST,
        out_path=agg_path
    )

    # 2) Final submission
    final_path = cfg.RUNS_DIR / cfg.FINAL_SUBMISSION_NAME
    sub = make_final_submission(agg_df, final_path)

    # 3) Task B (Grad-CAM overlays)
    taskb_dir = cfg.RUNS_DIR / cfg.TASKB_DIR_NAME
    if cfg.DO_TASK_B:
        if cfg.CKPT_FOR_TASKB is None:
            ckpt = cfg.RUNS_DIR / f"cv_best_{cfg.BENCHMARK}_seed{cfg.SEED}_fold1.pt"
        else:
            ckpt = cfg.CKPT_FOR_TASKB

        if not ckpt.exists():
            raise FileNotFoundError(f"Task B ckpt not found: {ckpt}")

        model, _ = build_model_from_ckpt(ckpt, device=cfg.DEVICE, gate_tau=2.0)
        run_task_b_explainability(
            model=model,
            patient_ids=te_p,
            mapping=mapping,
            out_dir=taskb_dir,
            device=cfg.DEVICE,
            img_size=cfg.IMG_SIZE,
            topk_for_cam=cfg.TOPK_FOR_CAM
        )

    # 4) Task C (Domain shift)
    if cfg.DO_TASK_C:
        # patient-level domain csv path (auto)
        patient_domain_csv = cfg.ARTIFACTS_ROOT / "split_res_shift_test_minor_PATIENT_with_domain.csv"

        if cfg.TEST_PATIENT_DOMAIN_CSV is not None and Path(cfg.TEST_PATIENT_DOMAIN_CSV).exists():
            patient_domain_df = pd.read_csv(cfg.TEST_PATIENT_DOMAIN_CSV)
            ok = not (patient_domain_df["Domain"].astype(str) == "UNKNOWN").all()
            if not ok:
                print("\n[STOP] Provided patient-domain csv has all UNKNOWN; Task C not valid.")
        else:
            patient_domain_df, ok = build_patient_domain_from_test(
                test_csv=cfg.TEST_CSV,
                out_patient_csv=patient_domain_csv,
                helper_xlsx=cfg.DOMAIN_HELPER_XLSX
            )

        if not ok:
            print(f"\n[STOP] Task C is not valid because domain is UNKNOWN for all patients.")
            print(f"Please label domains in: {cfg.DOMAIN_HELPER_XLSX}")
            print(f"Then rerun Task C using TEST_PATIENT_DOMAIN_CSV pointing to the labeled CSV/XLSX export.")
        else:
            out_json = cfg.RUNS_DIR / f"TASKC_domain_shift_seed{cfg.SEED}_{cfg.BENCHMARK}.json"
            run_task_c_domain_shift(agg_df, patient_domain_df, out_json)

    print("\nDONE.")
    print("Agg:", agg_path)
    print("Final:", final_path)
    if cfg.DO_TASK_B:
        print("TaskB dir:", taskb_dir)
    if cfg.DO_TASK_C:
        print("Patient-domain meta:", cfg.ARTIFACTS_ROOT / "split_res_shift_test_minor_PATIENT_with_domain.csv")
        print("TaskC json:", cfg.RUNS_DIR / f"TASKC_domain_shift_seed{cfg.SEED}_{cfg.BENCHMARK}.json")


if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs
Patients found in image root: 210

=== TRAIN ===
Patients: 117
Patients with 0 images: 0
Images per patient: count    117.000000
mean      50.247863
std       32.454187
min        2.000000
25%       26.000000
50%       42.000000
75%       72.000000
max      186.000000
dtype: float64

=== VAL ===
Patients: 20
Patients with 0 images: 0
Images per patient: count    20.000000
mean     41.750000
std      23.212689
min       9.000000
25%      21.500000
50%      40.000000
75%      57.750000
max      87.000000
dtype: float64

=== TEST ===
Patients: 73
Patients with 0 images: 0
Images per patient: count     73.000000
mean      84.712329
std       31.139685
min       20.000000
25%       64.000000
50%       87.000000
75%      104.000000
max      159.000000
dtype: float64
Saved: D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv | folds used: [1, 2, 3, 4, 5]
Saved final submission: D:\ENT

C:\Users\admin\AppData\Local\Temp\ipykernel_14828\730240387.py:508: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_location="cpu")


[CKPT LOAD] cv_best_res_shift_seed43_fold1.pt
  missing: 0  unexpected: 0
[2026-02-28 13:18:22] Task B: saving CAM overlays to: D:\ENT\_challenge2_artifacts\_runs\TASKB_res_shift_seed43
[2026-02-28 13:18:23] Task B progress: 5/73 patients | overlays saved=30
[2026-02-28 13:18:25] Task B progress: 10/73 patients | overlays saved=60
[2026-02-28 13:18:26] Task B progress: 15/73 patients | overlays saved=90
[2026-02-28 13:18:27] Task B progress: 20/73 patients | overlays saved=120
[2026-02-28 13:18:29] Task B progress: 25/73 patients | overlays saved=150
[2026-02-28 13:18:30] Task B progress: 30/73 patients | overlays saved=180
[2026-02-28 13:18:31] Task B progress: 35/73 patients | overlays saved=210
[2026-02-28 13:18:33] Task B progress: 40/73 patients | overlays saved=240
[2026-02-28 13:18:34] Task B progress: 45/73 patients | overlays saved=270
[2026-02-28 13:18:36] Task B progress: 50/73 patients | overlays saved=300
[2026-02-28 13:18:37] Task B progress: 55/73 patients | overlays sav

In [14]:
# ============================================================
# TASK C (Domain Shift) — Updated code (NO missing file issues)
# Works with either:
#   A) DOMAIN_NEEDS_LABELS.xlsx  (preferred; fill it)
#   B) DOMAIN_LABELS_FILLED.csv  (if you export to csv)
#
# Assumptions:
# - You already have agg predictions:
#   D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv
# - Your Excel/CSV has at least: Patient, Domain
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score

RUNS_DIR = Path(r"D:\ENT\_challenge2_artifacts\_runs")
AGG_PATH = RUNS_DIR / "pred_seed43_res_shift_folds1_5_Ktest10_agg.csv"
OUT_JSON = RUNS_DIR / "TASKC_domain_shift_seed43_res_shift.json"

# --- pick ONE of these (Excel recommended)
DOMAIN_XLSX = Path(r"D:\ENT\_challenge2_artifacts\DOMAIN_NEEDS_LABELS.xlsx")     # fill this file
DOMAIN_CSV  = Path(r"D:\ENT\_challenge2_artifacts\DOMAIN_LABELS_FILLED.csv")    # optional export

def _load_domain_labels() -> pd.DataFrame:
    """
    Loads patient-level domain labels from XLSX or CSV.
    Enforces: columns Patient, Domain.
    """
    if DOMAIN_XLSX.exists():
        df = pd.read_excel(DOMAIN_XLSX)
        src = str(DOMAIN_XLSX)
    elif DOMAIN_CSV.exists():
        df = pd.read_csv(DOMAIN_CSV)
        src = str(DOMAIN_CSV)
    else:
        raise FileNotFoundError(
            "No domain label file found.\n"
            f"Expected one of:\n  {DOMAIN_XLSX}\n  {DOMAIN_CSV}\n\n"
            "Fill DOMAIN_NEEDS_LABELS.xlsx (Domain column: WLE/NBI/MIXED) or export to DOMAIN_LABELS_FILLED.csv."
        )

    # normalize column names
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break

    if "Domain" not in df.columns:
        for c in ["domain", "modality", "Modality"]:
            if c in df.columns:
                df = df.rename(columns={c: "Domain"})
                break

    if "Patient" not in df.columns or "Domain" not in df.columns:
        raise ValueError(f"Domain labels file missing required columns. Found: {list(df.columns)} | Source: {src}")

    df = df[["Patient", "Domain"]].copy()
    df["Patient"] = df["Patient"].astype(str).str.strip()

    # clean and standardize domains
    df["Domain"] = df["Domain"].astype(str).str.strip().str.upper()
    # map common variants
    mapping = {
        "WHITE LIGHT": "WLE",
        "WHITE-LIGHT": "WLE",
        "WL": "WLE",
        "NARROW BAND": "NBI",
        "NARROW-BAND": "NBI",
        "NB": "NBI",
        "NBI/WLE": "MIXED",
        "WLE/NBI": "MIXED",
        "BOTH": "MIXED",
        "UNKNOWN": "UNKNOWN",
        "NAN": "UNKNOWN",
        "NONE": "UNKNOWN",
        "": "UNKNOWN",
    }
    df["Domain"] = df["Domain"].replace(mapping)

    # drop duplicates (keep first)
    df = df.dropna(subset=["Patient"]).drop_duplicates(subset=["Patient"], keep="first")

    return df

def _load_agg_preds() -> pd.DataFrame:
    """
    Loads aggregated patient-level predictions.
    Expects at least: Patient, y_true, logit_mean OR prob_mean OR prob.
    """
    if not AGG_PATH.exists():
        raise FileNotFoundError(f"Agg predictions not found: {AGG_PATH}")

    agg = pd.read_csv(AGG_PATH)

    if "Patient" not in agg.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in agg.columns:
                agg = agg.rename(columns={c: "Patient"})
                break
    if "Patient" not in agg.columns:
        raise ValueError(f"Agg file must contain Patient column. Found: {list(agg.columns)}")

    # pick a score column
    score_col = None
    for c in ["prob_mean", "prob", "p", "y_pred_prob", "Pred", "pred_prob"]:
        if c in agg.columns:
            score_col = c
            break
    if score_col is None:
        # try logits
        for c in ["logit_mean", "logit", "Logit"]:
            if c in agg.columns:
                score_col = c
                break

    if score_col is None:
        raise ValueError(f"Could not find prob/logit column in agg. Found: {list(agg.columns)}")

    # y_true column
    y_col = None
    for c in ["y_true", "Y", "label", "Label", "target"]:
        if c in agg.columns:
            y_col = c
            break
    if y_col is None:
        raise ValueError(f"Could not find y_true column in agg. Found: {list(agg.columns)}")

    agg = agg[["Patient", y_col, score_col]].copy()
    agg = agg.rename(columns={y_col: "y_true", score_col: "score"})
    agg["Patient"] = agg["Patient"].astype(str).str.strip()

    # coerce y_true
    agg["y_true"] = pd.to_numeric(agg["y_true"], errors="coerce")
    if agg["y_true"].isna().any():
        raise ValueError("y_true contains non-numeric values after coercion. Fix labels in agg file.")
    agg["y_true"] = agg["y_true"].astype(int)

    # coerce score
    agg["score"] = pd.to_numeric(agg["score"], errors="coerce")
    if agg["score"].isna().any():
        # if score has NaN, drop those rows
        agg = agg.dropna(subset=["score"]).copy()

    return agg

def _safe_auc(y_true, score):
    y_true = np.asarray(y_true)
    score = np.asarray(score)
    # AUC undefined if only one class present
    if len(np.unique(y_true)) < 2:
        return None
    return float(roc_auc_score(y_true, score))

def run_task_c_domain_shift():
    dom = _load_domain_labels()
    agg = _load_agg_preds()

    merged = agg.merge(dom, on="Patient", how="left")
    merged["Domain"] = merged["Domain"].fillna("UNKNOWN").astype(str).str.strip().str.upper()
    merged["Domain"] = merged["Domain"].replace({"NAN": "UNKNOWN", "": "UNKNOWN"})

    domain_counts = merged["Domain"].value_counts(dropna=False).to_dict()

    # Overall AUC
    auc_overall = _safe_auc(merged["y_true"], merged["score"])

    per_domain = {}
    for d in sorted(merged["Domain"].unique()):
        sub = merged[merged["Domain"] == d]
        per_domain[d] = {
            "n": int(len(sub)),
            "auc": _safe_auc(sub["y_true"], sub["score"])
        }

    report = {
        "n_total": int(len(merged)),
        "domain_counts": {str(k): int(v) for k, v in domain_counts.items()},
        "auc_overall": auc_overall,
        "per_domain": per_domain,
        "agg_path": str(AGG_PATH),
        "domain_labels_used": str(DOMAIN_XLSX if DOMAIN_XLSX.exists() else DOMAIN_CSV),
        "note": (
            "If WLE/NBI are still UNKNOWN, fill Domain column in the label file with WLE or NBI (or MIXED)."
        )
    }

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print("[Task C] saved:", OUT_JSON)
    print(json.dumps(report, indent=2))
    return report

if __name__ == "__main__":
    run_task_c_domain_shift()

[Task C] saved: D:\ENT\_challenge2_artifacts\_runs\TASKC_domain_shift_seed43_res_shift.json
{
  "n_total": 73,
  "domain_counts": {
    "UNKNOWN": 73
  },
  "auc_overall": 0.8408695652173913,
  "per_domain": {
    "UNKNOWN": {
      "n": 73,
      "auc": 0.8408695652173913
    }
  },
  "agg_path": "D:\\ENT\\_challenge2_artifacts\\_runs\\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv",
  "domain_labels_used": "D:\\ENT\\_challenge2_artifacts\\DOMAIN_NEEDS_LABELS.xlsx",
  "note": "If WLE/NBI are still UNKNOWN, fill Domain column in the label file with WLE or NBI (or MIXED)."
}


In [15]:
# ============================================================
# Scientifically-clean end-to-end script (Tasks A/B/C finalization)
# - Patient-level split & stats
# - Fold preds aggregation -> seed agg
# - Final submission (patient-level)
# - Task B: Grad-CAM overlays (top-K images per patient)
# - Task C: Domain shift analysis (requires patient-level domain labels)
#
# IMPORTANT:
#   1) Your fold prediction files must exist:
#        RUNS_DIR/pred_fold{f}_{BENCHMARK}_seed{SEED}_Ktest{K_TEST}.csv
#   2) Your test manifest (image-level) must exist:
#        TEST_CSV = ...split_res_shift_test_minor_colab.csv
#      It can be "patient-level" too; we only need Patient ids to drive Task B.
#   3) Task C requires patient-level domain labels (WLE/NBI/MIXED).
#      If not available, the code exports DOMAIN_NEEDS_LABELS.xlsx
#      You fill it, then rerun with TEST_PATIENT_DOMAIN_PATH pointing to it.
# ============================================================

import os
import re
import json
import math
import time
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image

# -----------------------
# Config
# -----------------------
@dataclass
class CFG:
    # Paths
    IMAGES_ROOT: Path = Path(r"D:\ENT")
    ARTIFACTS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts")
    RUNS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    # Manifests (image-level is ok)
    TRAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv")
    VAL_CSV: Path   = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv")
    TEST_CSV: Path  = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")

    # Optional patient-level domain labels for Task C (xlsx/csv)
    # If None or missing -> will export DOMAIN_NEEDS_LABELS.xlsx and STOP Task C.
    TEST_PATIENT_DOMAIN_PATH: Optional[Path] = None
    # Example when filled:
    # TEST_PATIENT_DOMAIN_PATH = Path(r"D:\ENT\_challenge2_artifacts\DOMAIN_NEEDS_LABELS.xlsx")

    # Run
    BENCHMARK: str = "res_shift"
    SEED: int = 43
    FOLDS: Tuple[int, ...] = (1, 2, 3, 4, 5)
    K_TEST: int = 10

    # Task B
    DO_TASK_B: bool = True
    CKPT_FOR_TASKB: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold1.pt")
    TASKB_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\TASKB_res_shift_seed43")
    TOPK_FOR_CAM: int = 6          # overlays per patient
    IMG_SIZE: int = 224            # inference resize for CAM

    # Task C
    DO_TASK_C: bool = True
    DOMAIN_LABEL_EXPORT_XLSX: Path = Path(r"D:\ENT\_challenge2_artifacts\DOMAIN_NEEDS_LABELS.xlsx")

    # Device
    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"

cfg = CFG()


# ============================================================
# Utilities
# ============================================================
def now_ts():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_everything(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def safe_int(x):
    try:
        return int(x)
    except Exception:
        return None

def print_header(title: str):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

# ============================================================
# Manifest loading (robust)
# ============================================================
def load_manifest(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # normalize Patient column
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID", "patientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"{csv_path} must contain a Patient column (or patient/patient_id/PatientID).")

    # try to normalize image path column (not strictly required for Task A/C aggregation)
    # but needed for mapping if you want path-based domain inference (we mostly do folder scan instead)
    if "path" not in df.columns:
        for c in ["ImagePath", "img_path", "filepath", "file_path", "image_path"]:
            if c in df.columns:
                df = df.rename(columns={c: "path"})
                break

    # normalize label if present
    if "y_true" not in df.columns:
        for c in ["label", "Label", "Histopathology", "target", "Target", "malignant", "Malignant"]:
            if c in df.columns:
                df = df.rename(columns={c: "y_true"})
                break

    # normalize Leukoplakia if present
    if "leuko" not in df.columns:
        for c in ["Leukoplakia", "leukoplakia", "Leuko", "leu"]:
            if c in df.columns:
                df = df.rename(columns={c: "leuko"})
                break

    # clean
    df["Patient"] = df["Patient"].astype(str).str.strip()
    return df

# ============================================================
# Patient stats
# ============================================================
def patient_stats(df: pd.DataFrame, name: str, mapping: Dict[str, List[Path]]):
    pats = sorted(df["Patient"].unique().tolist())
    counts = [len(mapping.get(p, [])) for p in pats]
    s = pd.Series(counts, index=pats)

    print(f"\n=== {name} ===")
    print(f"Patients: {len(pats)}")
    print(f"Patients with 0 images: {(s==0).sum()}")
    print("Images per patient:", s.describe(percentiles=[0.25,0.5,0.75,0.9,0.95,0.99]))

# ============================================================
# File-system mapping: Patient -> list[image paths]
# "folder-segment" heuristic:
#   any path segment containing 'Patient' + digits OR starting with 'Patient'
#   e.g. ...\Patient099\xxx.jpg
# ============================================================
PAT_RE = re.compile(r"(patient\s*0*\d+)", re.IGNORECASE)

def build_patient_image_mapping(images_root: Path) -> Dict[str, List[Path]]:
    mapping: Dict[str, List[Path]] = {}
    all_files = list(images_root.rglob("*.jpg"))
    for fp in all_files:
        parts = [p for p in fp.parts]
        pid = None
        for seg in parts:
            m = PAT_RE.search(seg)
            if m:
                pid = m.group(1)
                pid = re.sub(r"\s+", "", pid)          # remove spaces
                pid = pid[0].upper() + pid[1:]         # Patient...
                # normalize Patient000 -> Patient000
                # keep as found
                break
        if pid is None:
            continue
        mapping.setdefault(pid, []).append(fp)

    # sort each patient's images for stable selection
    for k in mapping:
        mapping[k] = sorted(mapping[k])
    print(f"Patients found in image root: {len(mapping)}")
    return mapping

# ============================================================
# Model: DualBranchGatedMIL (matches your checkpoint)
# - ResNet50 backbone -> instance feature dim 2048
# - Two attention pools (non/leu)
# - Two classifiers (non/leu): 2048 -> 1
# - Gate: 4096 -> 1
# - leuko_head: 4096 -> 256 -> 1
# ============================================================
from torchvision.models import resnet50

class ResNet50Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        m = resnet50(weights=None)
        # keep conv1..layer4, avgpool
        self.backbone = nn.Sequential(
            m.conv1, m.bn1, m.relu, m.maxpool,
            m.layer1, m.layer2, m.layer3, m.layer4,
            m.avgpool
        )

    def forward(self, x):
        # x: [B,3,H,W]
        feat = self.backbone(x)        # [B,2048,1,1]
        feat = feat.flatten(1)         # [B,2048]
        return feat

class AttnPool(nn.Module):
    """MIL attention pooling a la Ilse et al. (tanh -> linear)"""
    def __init__(self, in_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, H):
        # H: [N, D]
        a = self.net(H).squeeze(1)           # [N]
        w = torch.softmax(a, dim=0)          # [N]
        z = torch.sum(w.unsqueeze(1) * H, dim=0)  # [D]
        return z, w

class DualBranchGatedMIL(nn.Module):
    def __init__(self, gate_tau: float = 2.0, instance_dim: int = 2048, pool_hidden: int = 256):
        super().__init__()
        self.gate_tau = gate_tau
        self.encoder = nn.Module()
        self.encoder.backbone = ResNet50Encoder().backbone  # keep same key prefix: encoder.backbone...

        self.pool_non = AttnPool(instance_dim, pool_hidden)
        self.pool_leu = AttnPool(instance_dim, pool_hidden)

        self.cls_non = nn.Linear(instance_dim, 1)
        self.cls_leu = nn.Linear(instance_dim, 1)

        self.gate = nn.Linear(instance_dim * 2, 1)
        self.leuko_head = nn.Sequential(
            nn.Linear(instance_dim * 2, pool_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(pool_hidden, 1)
        )

        # for CAM target selection (we’ll set later)
        self._cam_target_layer = None

    def encode_instances(self, x):
        # x: [N,3,H,W]
        feat = self.encoder.backbone(x)      # [N,2048,1,1]
        feat = feat.flatten(1)               # [N,2048]
        return feat

    def forward_bag(self, x_non, x_leu):
        # x_non: [N1,3,H,W], x_leu: [N2,3,H,W]
        Hn = self.encode_instances(x_non) if x_non is not None else None
        Hl = self.encode_instances(x_leu) if x_leu is not None else None

        zn, an = self.pool_non(Hn) if Hn is not None else (None, None)
        zl, al = self.pool_leu(Hl) if Hl is not None else (None, None)

        logit_non = self.cls_non(zn).squeeze(0) if zn is not None else None
        logit_leu = self.cls_leu(zl).squeeze(0) if zl is not None else None

        # fuse
        zcat = torch.cat([zn, zl], dim=0)  # [4096]
        gate_logit = self.gate(zcat).squeeze(0)  # scalar
        alpha = torch.sigmoid(gate_logit / self.gate_tau)

        # main logit (fused)
        # if one branch is missing, fall back
        if logit_non is None and logit_leu is None:
            main = gate_logit * 0.0
        elif logit_non is None:
            main = logit_leu
        elif logit_leu is None:
            main = logit_non
        else:
            main = alpha * logit_leu + (1.0 - alpha) * logit_non

        leuko_logit = self.leuko_head(zcat).squeeze(0)

        return {
            "logit": main,
            "prob": torch.sigmoid(main),
            "logit_non": logit_non,
            "logit_leu": logit_leu,
            "alpha": alpha.detach(),
            "attn_non": an.detach() if an is not None else None,
            "attn_leu": al.detach() if al is not None else None,
            "leuko_logit": leuko_logit,
            "leuko_prob": torch.sigmoid(leuko_logit),
        }

# ============================================================
# Build model from checkpoint (strict scientific compatibility)
# Handles key naming differences seen in your logs:
# - Some checkpoints use pool_non.V / pool_non.w (older style)
# - Some use pool_non.net.0 / net.2 (ours)
# - We remap if needed and load strict=False, but we verify critical modules load.
# ============================================================
def remap_pool_vw_to_net(sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    # If keys exist: pool_non.V.weight -> pool_non.net.0.weight, pool_non.w.weight -> pool_non.net.2.weight
    out = dict(sd)
    for branch in ["pool_non", "pool_leu"]:
        Vw = f"{branch}.V.weight"
        Vb = f"{branch}.V.bias"
        ww = f"{branch}.w.weight"
        wb = f"{branch}.w.bias"
        if Vw in out:
            out[f"{branch}.net.0.weight"] = out.pop(Vw)
        if Vb in out:
            out[f"{branch}.net.0.bias"] = out.pop(Vb)
        if ww in out:
            out[f"{branch}.net.2.weight"] = out.pop(ww)
        if wb in out:
            out[f"{branch}.net.2.bias"] = out.pop(wb)
    return out

def remap_leuko_head_3_to_2(sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    # Some checkpoints have leuko_head.3.* instead of leuko_head.2.*
    out = dict(sd)
    for w in ["weight", "bias"]:
        k3 = f"leuko_head.3.{w}"
        k2 = f"leuko_head.2.{w}"
        if k3 in out and k2 not in out:
            out[k2] = out.pop(k3)
    return out

def build_model_from_ckpt(ckpt_path: Path, device: str, gate_tau: float = 2.0) -> Tuple[DualBranchGatedMIL, dict]:
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    sd = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    sd = remap_pool_vw_to_net(sd)
    sd = remap_leuko_head_3_to_2(sd)

    model = DualBranchGatedMIL(gate_tau=gate_tau).to(device)

    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"[CKPT LOAD] {ckpt_path.name}")
    print(f"  missing: {len(missing)}  unexpected: {len(unexpected)}")
    if len(missing):
        print("  missing sample:", missing[:10])
    if len(unexpected):
        print("  unexpected sample:", unexpected[:10])

    model.eval()
    return model, ckpt

# ============================================================
# Grad-CAM (scientifically correct: enable grads for target layer)
# ============================================================
class GradCAM:
    def __init__(self, model: DualBranchGatedMIL, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.h1 = target_layer.register_forward_hook(self._fwd_hook)
        self.h2 = target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, module, inp, out):
        self.activations = out

    def _bwd_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def remove(self):
        self.h1.remove()
        self.h2.remove()

    @torch.enable_grad()
    def cam_for_tensor(self, x_non: torch.Tensor, x_leu: torch.Tensor) -> np.ndarray:
        """
        x_non: [N,3,H,W], x_leu: [N,3,H,W]
        returns CAM for the *top* instance used to compute grads:
          We'll compute CAM for each instance by doing per-instance forward,
          but for speed we do one instance at a time in Task B loop.
        """
        self.model.zero_grad(set_to_none=True)

        out = self.model.forward_bag(x_non, x_leu)
        logit = out["logit"]

        # ensure grad path exists
        if not logit.requires_grad:
            logit = logit.clone().requires_grad_(True)

        logit.backward(retain_graph=False)

        A = self.activations     # [1,C,h,w] if single image through backbone layer
        G = self.gradients       # [1,C,h,w]
        if A is None or G is None:
            raise RuntimeError("GradCAM hooks did not capture activations/gradients.")

        w = G.mean(dim=(2, 3), keepdim=True)           # [1,C,1,1]
        cam = (w * A).sum(dim=1, keepdim=True)         # [1,1,h,w]
        cam = F.relu(cam)
        cam = cam.squeeze().detach().cpu().numpy()     # [h,w]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam

def overlay_cam_on_pil(pil: Image.Image, cam: np.ndarray, alpha: float = 0.45) -> Image.Image:
    # cam is [h,w] in 0..1
    cam_img = Image.fromarray(np.uint8(cam * 255)).resize(pil.size, resample=Image.BILINEAR)
    cam_rgb = Image.merge("RGB", (cam_img, cam_img, cam_img))
    out = Image.blend(pil.convert("RGB"), cam_rgb, alpha=alpha)
    return out

# ============================================================
# Fold prediction aggregation -> seed agg
# Robust to column name differences.
# ============================================================
def read_fold_pred(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    # normalize Patient
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"{path} must contain Patient column.")

    # normalize logit/prob columns
    if "logit" not in df.columns:
        for c in ["logit_mean", "logits", "pred_logit"]:
            if c in df.columns:
                df = df.rename(columns={c: "logit"})
                break
    if "prob" not in df.columns:
        for c in ["prob_mean", "pred_prob", "p"]:
            if c in df.columns:
                df = df.rename(columns={c: "prob"})
                break

    # y_true & leuko (optional but expected for test)
    if "y_true" not in df.columns:
        for c in ["label", "Label", "target", "Target"]:
            if c in df.columns:
                df = df.rename(columns={c: "y_true"})
                break
    if "leuko" not in df.columns:
        for c in ["Leukoplakia", "leukoplakia", "leu"]:
            if c in df.columns:
                df = df.rename(columns={c: "leuko"})
                break

    # gate alpha is optional (many of your pred files don't have it)
    if "gate_alpha" not in df.columns:
        for c in ["alpha", "gate", "gate_mean", "gate_prob"]:
            if c in df.columns:
                df = df.rename(columns={c: "gate_alpha"})
                break

    df["Patient"] = df["Patient"].astype(str).str.strip()
    return df

def auc_roc(y_true, y_score) -> float:
    # minimal sklearn-free AUC for binary labels
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    # handle NaN
    m = ~np.isnan(y_score)
    y_true = y_true[m]
    y_score = y_score[m]
    if len(np.unique(y_true)) < 2:
        return float("nan")

    # rank-based AUC
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(y_score)) + 1

    pos = (y_true == 1)
    n_pos = pos.sum()
    n_neg = (~pos).sum()
    sum_ranks_pos = ranks[pos].sum()
    auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg + 1e-12)
    return float(auc)

def aggregate_folds_to_seed(runs_dir: Path, seed: int, benchmark: str, folds: Tuple[int, ...], k_test: int, out_path: Path) -> pd.DataFrame:
    dfs = []
    used = []
    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            print(f"[WARN] missing fold file: {p}")
            continue
        d = read_fold_pred(p).set_index("Patient")
        keep = [c for c in ["y_true", "leuko", "logit", "prob", "gate_alpha"] if c in d.columns]
        d = d[keep].copy()
        # rename fold-specific columns
        if "logit" in d.columns:
            d = d.rename(columns={"logit": f"logit_fold{f}"})
        if "prob" in d.columns:
            d = d.rename(columns={"prob": f"prob_fold{f}"})
        if "gate_alpha" in d.columns:
            d = d.rename(columns={"gate_alpha": f"gate_fold{f}"})
        dfs.append(d)
        used.append(f)

    if not dfs:
        raise FileNotFoundError("No fold prediction files found to aggregate.")

    merged = dfs[0]
    for d in dfs[1:]:
        # keep y_true/leuko from first if present
        cols_to_join = [c for c in d.columns if c.startswith("logit_fold") or c.startswith("prob_fold") or c.startswith("gate_fold")]
        merged = merged.join(d[cols_to_join], how="inner")

    # average logits across folds
    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    if not logit_cols:
        raise ValueError("No logit_fold* columns found after aggregation.")
    merged["logit_mean"] = merged[logit_cols].mean(axis=1)
    merged["prob_mean"] = 1.0 / (1.0 + np.exp(-merged["logit_mean"].astype(float)))

    # compute AUC if y_true exists
    if "y_true" in merged.columns:
        merged["y_true"] = pd.to_numeric(merged["y_true"], errors="coerce")
        auc = auc_roc(merged["y_true"].values, merged["logit_mean"].values)
        print(f"AGG TEST AUC: {auc}")

    merged = merged.reset_index()
    ensure_dir(out_path.parent)
    merged.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | folds used: {used}")
    return merged

# ============================================================
# Final submission creation (patient-level)
# ============================================================
def make_final_submission(agg_df: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    # standard final: Patient, prob_mean (or logit_mean), optional leuko
    out = agg_df.copy()
    if "prob_mean" not in out.columns and "logit_mean" in out.columns:
        out["prob_mean"] = 1.0 / (1.0 + np.exp(-out["logit_mean"].astype(float)))

    keep = ["Patient"]
    if "prob_mean" in out.columns:
        keep.append("prob_mean")
    elif "logit_mean" in out.columns:
        keep.append("logit_mean")
    if "leuko" in out.columns:
        keep.append("leuko")
    if "y_true" in out.columns:
        keep.append("y_true")

    out = out[keep]
    out.to_csv(out_path, index=False)
    print(f"Saved final submission: {out_path}")
    print(f"Patients: {len(out)}")
    return out

# ============================================================
# Domain labeling / Task C
# ============================================================
def export_domain_needs_labels(patients: List[str], out_xlsx: Path):
    df = pd.DataFrame({"Patient": patients, "Domain": ""})
    df.to_excel(out_xlsx, index=False)
    print("\n[IMPORTANT] Domain is UNKNOWN for all patients or missing.")
    print("I exported a labeling helper file here:")
    print(f"  {out_xlsx}")

def load_patient_domains(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Domain" not in df.columns:
        for c in ["domain", "modality", "Modality"]:
            if c in df.columns:
                df = df.rename(columns={c: "Domain"})
                break
    if "Patient" not in df.columns or "Domain" not in df.columns:
        raise ValueError(f"{path} must contain Patient and Domain columns.")

    df["Patient"] = df["Patient"].astype(str).str.strip()
    df["Domain_raw"] = df["Domain"].astype(str)

    # strict mapping to WLE/NBI/MIXED/UNKNOWN
    df["Domain"] = (
        df["Domain"].astype(str)
        .str.strip().str.upper()
        .replace({
            "WHITE LIGHT": "WLE", "WHITE-LIGHT": "WLE", "WL": "WLE", "WLE": "WLE",
            "NARROW BAND": "NBI", "NARROW-BAND": "NBI", "NB": "NBI", "NBI": "NBI",
            "BOTH": "MIXED", "MIX": "MIXED", "MIXED": "MIXED", "WLE/NBI": "MIXED", "NBI/WLE": "MIXED",
            "": "UNKNOWN", "NONE": "UNKNOWN", "NAN": "UNKNOWN", "NO": "UNKNOWN"
        })
    )
    return df[["Patient", "Domain", "Domain_raw"]]

def run_task_c_domain_shift(agg_df: pd.DataFrame, patient_domains: pd.DataFrame, out_json: Path, note_extra: str = ""):
    merged = agg_df.merge(patient_domains[["Patient", "Domain"]], on="Patient", how="left")
    merged["Domain"] = merged["Domain"].fillna("UNKNOWN")

    domain_counts = merged["Domain"].value_counts(dropna=False).to_dict()

    # must have at least one WLE and one NBI for a valid domain shift analysis
    has_wle = (merged["Domain"] == "WLE").sum() > 0
    has_nbi = (merged["Domain"] == "NBI").sum() > 0

    # compute overall auc
    auc_overall = None
    if "y_true" in merged.columns and "logit_mean" in merged.columns:
        auc_overall = auc_roc(merged["y_true"].values, merged["logit_mean"].values)

    per_domain = {}
    for dom, sub in merged.groupby("Domain"):
        auc_dom = None
        if "y_true" in sub.columns and "logit_mean" in sub.columns and len(sub) > 1:
            auc_dom = auc_roc(sub["y_true"].values, sub["logit_mean"].values)
        per_domain[dom] = {"n": int(len(sub)), "auc": auc_dom}

    payload = {
        "n_total": int(len(merged)),
        "domain_counts": {k: int(v) for k, v in domain_counts.items()},
        "auc_overall": auc_overall,
        "per_domain": per_domain,
    }

    ensure_dir(out_json.parent)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print(f"[Task C] saved: {out_json}")
    print(json.dumps(payload, indent=2))

    if not (has_wle and has_nbi):
        print("\n[STOP] Task C is not valid because you do not have both WLE and NBI labeled.")
        print("Fill DOMAIN_NEEDS_LABELS.xlsx Domain column with WLE/NBI/MIXED.")
        if note_extra:
            print(note_extra)

# ============================================================
# Task B: Grad-CAM overlays per patient
#   - pick top-K images by attention or by uniform sampling
#   - run CAM per image (single instance bag for gradient correctness)
# ============================================================
def image_transform(img_size: int):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

def pick_k_images(paths: List[Path], k: int) -> List[Path]:
    if len(paths) <= k:
        return paths
    # uniform sampling for scientific coverage
    idx = np.linspace(0, len(paths) - 1, num=k, dtype=int)
    return [paths[i] for i in idx]

def find_cam_target_layer(model: DualBranchGatedMIL) -> nn.Module:
    # Use last block of layer4: encoder.backbone[7] is layer4 sequential
    # We'll target the last Bottleneck inside it: layer4[-1]
    # In our encoder: backbone = conv1,bn1,relu,maxpool,layer1,layer2,layer3,layer4,avgpool
    layer4 = model.encoder.backbone[7]
    return layer4[-1]

def run_task_b_explainability(
    model: DualBranchGatedMIL,
    patient_ids: List[str],
    mapping: Dict[str, List[Path]],
    out_dir: Path,
    device: str,
    img_size: int,
    topk_for_cam: int
):
    ensure_dir(out_dir)
    tfm = image_transform(img_size)

    # IMPORTANT: CAM needs grads. Ensure model parameters require_grad True.
    model.train()  # not for BN stats; but enables grad flow in general
    for p in model.parameters():
        p.requires_grad_(True)

    target_layer = find_cam_target_layer(model)
    cam = GradCAM(model, target_layer)

    print(f"[{now_ts()}] Task B: saving CAM overlays to: {out_dir}")

    saved = 0
    for i, pid in enumerate(patient_ids, start=1):
        paths = mapping.get(pid, [])
        if not paths:
            continue

        chosen = pick_k_images(paths, topk_for_cam)
        pid_dir = ensure_dir(out_dir / pid)

        for j, img_path in enumerate(chosen, start=1):
            try:
                pil = Image.open(img_path).convert("RGB")
            except Exception:
                continue

            x = tfm(pil).unsqueeze(0).to(device)  # [1,3,H,W]

            # We use single-instance bag for both branches (scientific: CAM explains single image effect)
            cam_map = cam.cam_for_tensor(x_non=x, x_leu=x)

            overlay = overlay_cam_on_pil(pil.resize((img_size, img_size)), cam_map, alpha=0.45)
            out_name = pid_dir / f"cam_{j:02d}_{img_path.name}"
            overlay.save(out_name)
            saved += 1

        if i % 5 == 0 or i == len(patient_ids):
            print(f"[{now_ts()}] Task B progress: {i}/{len(patient_ids)} patients | overlays saved={saved}")

    cam.remove()
    print(f"[{now_ts()}] Task B done.")

# ============================================================
# MAIN
# ============================================================
def main():
    print("DEVICE:", cfg.DEVICE)
    print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
    print("RUNS_DIR:", cfg.RUNS_DIR)

    seed_everything(cfg.SEED)

    # 1) Build patient->images mapping from filesystem
    mapping = build_patient_image_mapping(cfg.IMAGES_ROOT)

    # 2) Load manifests (patient ids) and print stats
    df_tr = load_manifest(cfg.TRAIN_CSV)
    df_va = load_manifest(cfg.VAL_CSV)
    df_te = load_manifest(cfg.TEST_CSV)

    patient_stats(df_tr, "TRAIN", mapping)
    patient_stats(df_va, "VAL", mapping)
    patient_stats(df_te, "TEST", mapping)

    tr_p = sorted(df_tr["Patient"].unique().tolist())
    va_p = sorted(df_va["Patient"].unique().tolist())
    te_p = sorted(df_te["Patient"].unique().tolist())

    # 3) Aggregate folds -> seed agg
    agg_path = cfg.RUNS_DIR / f"pred_seed{cfg.SEED}_{cfg.BENCHMARK}_folds1_5_Ktest{cfg.K_TEST}_agg.csv"
    agg_df = aggregate_folds_to_seed(
        runs_dir=cfg.RUNS_DIR,
        seed=cfg.SEED,
        benchmark=cfg.BENCHMARK,
        folds=cfg.FOLDS,
        k_test=cfg.K_TEST,
        out_path=agg_path
    )

    # 4) Final submission
    final_path = cfg.RUNS_DIR / f"FINAL_seed{cfg.SEED}_{cfg.BENCHMARK}.csv"
    make_final_submission(agg_df, final_path)

    # 5) Task B
    if cfg.DO_TASK_B:
        if not cfg.CKPT_FOR_TASKB.exists():
            raise FileNotFoundError(f"Task B ckpt not found: {cfg.CKPT_FOR_TASKB}")

        model, _ = build_model_from_ckpt(cfg.CKPT_FOR_TASKB, device=cfg.DEVICE, gate_tau=2.0)
        run_task_b_explainability(
            model=model,
            patient_ids=te_p,
            mapping=mapping,
            out_dir=cfg.TASKB_DIR,
            device=cfg.DEVICE,
            img_size=cfg.IMG_SIZE,
            topk_for_cam=cfg.TOPK_FOR_CAM
        )

    # 6) Task C
    if cfg.DO_TASK_C:
        # need patient-level domains
        dom_path = cfg.TEST_PATIENT_DOMAIN_PATH

        # If not provided, export helper and STOP Task C.
        if dom_path is None or (not Path(dom_path).exists()):
            export_domain_needs_labels(te_p, cfg.DOMAIN_LABEL_EXPORT_XLSX)
            print("\n[STOP] Task C not run because TEST_PATIENT_DOMAIN_PATH is missing.")
            print("Fill the exported file Domain column using: WLE / NBI / MIXED")
            print("Then set cfg.TEST_PATIENT_DOMAIN_PATH to that XLSX/CSV and rerun.")
        else:
            patient_domains = load_patient_domains(Path(dom_path))

            # show distribution
            print("\nPatient-level Domain distribution:")
            print(patient_domains["Domain"].value_counts(dropna=False))

            out_json = cfg.RUNS_DIR / f"TASKC_domain_shift_seed{cfg.SEED}_{cfg.BENCHMARK}.json"
            run_task_c_domain_shift(
                agg_df=agg_df,
                patient_domains=patient_domains,
                out_json=out_json,
                note_extra=f"Domain labels used: {dom_path}"
            )

    print("\nDONE.")
    print("Agg:", agg_path)
    print("Final:", final_path)
    if cfg.DO_TASK_B:
        print("TaskB dir:", cfg.TASKB_DIR)
    if cfg.DO_TASK_C:
        print("TaskC json:", cfg.RUNS_DIR / f"TASKC_domain_shift_seed{cfg.SEED}_{cfg.BENCHMARK}.json")

if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs
Patients found in image root: 210

=== TRAIN ===
Patients: 117
Patients with 0 images: 0
Images per patient: count    117.000000
mean      50.247863
std       32.454187
min        2.000000
25%       26.000000
50%       42.000000
75%       72.000000
90%       90.400000
95%       96.800000
99%      144.720000
max      186.000000
dtype: float64

=== VAL ===
Patients: 20
Patients with 0 images: 0
Images per patient: count    20.000000
mean     41.750000
std      23.212689
min       9.000000
25%      21.500000
50%      40.000000
75%      57.750000
90%      71.000000
95%      80.350000
99%      85.670000
max      87.000000
dtype: float64

=== TEST ===
Patients: 73
Patients with 0 images: 0
Images per patient: count     73.000000
mean      95.712329
std       31.139685
min       31.000000
25%       75.000000
50%       98.000000
75%      115.000000
90%      129.600000
95%      148.600000
99%      164.240000
max    

C:\Users\admin\AppData\Local\Temp\ipykernel_14828\2104101371.py:354: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_location="cpu")


[CKPT LOAD] cv_best_res_shift_seed43_fold1.pt
  missing: 0  unexpected: 0
[2026-02-28 14:37:09] Task B: saving CAM overlays to: D:\ENT\_challenge2_artifacts\_runs\TASKB_res_shift_seed43
[2026-02-28 14:37:12] Task B progress: 5/73 patients | overlays saved=30
[2026-02-28 14:37:15] Task B progress: 10/73 patients | overlays saved=60
[2026-02-28 14:37:17] Task B progress: 15/73 patients | overlays saved=90
[2026-02-28 14:37:20] Task B progress: 20/73 patients | overlays saved=120
[2026-02-28 14:37:22] Task B progress: 25/73 patients | overlays saved=150
[2026-02-28 14:37:25] Task B progress: 30/73 patients | overlays saved=180
[2026-02-28 14:37:27] Task B progress: 35/73 patients | overlays saved=210
[2026-02-28 14:37:30] Task B progress: 40/73 patients | overlays saved=240
[2026-02-28 14:37:33] Task B progress: 45/73 patients | overlays saved=270
[2026-02-28 14:37:35] Task B progress: 50/73 patients | overlays saved=300
[2026-02-28 14:37:38] Task B progress: 55/73 patients | overlays sav

In [18]:
import pandas as pd
from pathlib import Path
import os
import random

IMAGES_ROOT = Path(r"D:\ENT")
TEST_PATIENT_CSV = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")
OUT_XLSX = Path(r"D:\ENT\_challenge2_artifacts\DOMAIN_NEEDS_LABELS.xlsx")

random.seed(0)

def list_images_for_patient(images_root: Path, patient_id: str):
    # assumes patient images live under folders containing patient_id, as you used earlier
    # robust search: any file under IMAGES_ROOT with patient_id in its path
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    hits = []
    pid = str(patient_id).strip()
    for p in images_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts and pid in str(p):
            hits.append(p)
    return sorted(hits)

def load_test_patients(csv_path: Path):
    df = pd.read_csv(csv_path)
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"No Patient column in {csv_path}. Columns={list(df.columns)}")
    return sorted(df["Patient"].astype(str).str.strip().unique().tolist())

test_patients = load_test_patients(TEST_PATIENT_CSV)

rows = []
for pid in test_patients:
    imgs = list_images_for_patient(IMAGES_ROOT, pid)
    if len(imgs) == 0:
        ex1 = ex2 = ex3 = ""
    else:
        # pick 3 spread-out examples (deterministic)
        picks = [imgs[0], imgs[len(imgs)//2], imgs[-1]] if len(imgs) >= 3 else imgs
        picks = (picks + ["", "", ""])[:3]
        ex1, ex2, ex3 = [str(x) if x != "" else "" for x in picks]

    rows.append({
        "Patient": pid,
        "Domain": "",   # <-- you will fill: WLE / NBI / MIXED
        "ExamplePath1": ex1,
        "ExamplePath2": ex2,
        "ExamplePath3": ex3,
        "Notes": ""
    })

out_df = pd.DataFrame(rows)
out_df.to_excel(OUT_XLSX, index=False)
print("Wrote:", OUT_XLSX)
print("Rows:", len(out_df))
print(out_df.head(5))

Wrote: D:\ENT\_challenge2_artifacts\DOMAIN_NEEDS_LABELS.xlsx
Rows: 73
      Patient Domain                                       ExamplePath1  \
0  Patient070         D:\ENT\_challenge2_artifacts\_runs\TASKB_res_s...   
1  Patient071         D:\ENT\_challenge2_artifacts\_runs\TASKB_res_s...   
2  Patient100         D:\ENT\_challenge2_artifacts\_runs\TASKB_res_s...   
3  Patient106         D:\ENT\_challenge2_artifacts\_runs\TASKB_res_s...   
4  Patient107         D:\ENT\_challenge2_artifacts\_runs\TASKB_res_s...   

                                        ExamplePath2  \
0  D:\ENT\Benign\Reinkes_edema\Patient070\P070 (2...   
1        D:\ENT\Benign\Cyst\Patient071\P071 (15).jpg   
2       D:\ENT\Malignant\SCC\Patient100\P100 (2).jpg   
3  D:\ENT\Benign\Hyperkeratosis\Patient106\P106 (...   
4  D:\ENT\_challenge2_artifacts\_runs\TASKB_res_s...   

                                        ExamplePath3 Notes  
0  D:\ENT\Benign\Reinkes_edema\Patient070\P070 (9...        
1         D:\ENT\Ben

In [42]:
# ==============================
# Q1-ready Evaluation Bundle
# Bootstrap CI + Calibration + Temp Scaling + Uncertainty + Thresholds + DCA
# Works with your agg file columns:
# Patient,y_true,prob_fold1..5,logit_fold1..5,prob_mean,logit_mean
# ==============================

import os, json
import numpy as np
import pandas as pd

from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Tuple

from sklearn.metrics import (
    roc_auc_score, roc_curve,
    brier_score_loss, confusion_matrix
)
from sklearn.model_selection import StratifiedShuffleSplit
from scipy.special import expit  # sigmoid
from scipy.optimize import minimize

# ------------------------------
# Config
# ------------------------------
@dataclass
class Cfg:
    RUNS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs")
    AGG_CSV: Path  = Path(r"D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv")
    SEED: int = 43

    # Bootstrap
    N_BOOT: int = 5000

    # ECE
    ECE_BINS: int = 10

    # Temp scaling split (because you only have 73 patients in agg)
    # For paper-quality: later replace with CV-val predictions and fit T only on val folds.
    CALIB_FRAC: float = 0.30  # 30% held out to tune T, evaluate on remaining 70%

    # Threshold targets
    TARGET_SENS: float = 0.90
    TARGET_SPEC: float = 0.85

    # Decision curve thresholds
    DCA_POINTS: int = 101  # 0..1

cfg = Cfg()

# ------------------------------
# Helpers
# ------------------------------
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def bootstrap_auc_ci(y_true: np.ndarray, y_score: np.ndarray, n_boot=5000, seed=0) -> Dict:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        ys = y_score[idx]
        if np.unique(yt).size < 2:
            continue
        aucs.append(roc_auc_score(yt, ys))
    aucs = np.array(aucs, dtype=float)
    return {
        "auc_mean_boot": float(np.mean(aucs)),
        "auc_ci95": [float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))],
        "n_boot_used": int(len(aucs))
    }

def ece_score(y_true: np.ndarray, prob: np.ndarray, n_bins=10) -> Dict:
    # equal-width bins in [0,1]
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ids = np.digitize(prob, bins) - 1
    ids = np.clip(ids, 0, n_bins - 1)

    ece = 0.0
    rows = []
    n = len(y_true)

    for b in range(n_bins):
        mask = ids == b
        nb = int(mask.sum())
        if nb == 0:
            rows.append({"bin": b, "n": 0, "conf_mean": np.nan, "acc_mean": np.nan, "gap": np.nan})
            continue
        conf = float(np.mean(prob[mask]))
        acc  = float(np.mean(y_true[mask]))
        gap  = abs(acc - conf)
        ece += (nb / n) * gap
        rows.append({"bin": b, "n": nb, "conf_mean": conf, "acc_mean": acc, "gap": gap})

    return {"ece": float(ece), "bins": rows}

def fit_temperature_from_logits(logits: np.ndarray, y_true: np.ndarray) -> float:
    # minimize NLL of sigmoid(logit / T)
    # constrain T>0
    def nll(logT):
        T = np.exp(logT)  # positive
        p = expit(logits / T)
        eps = 1e-9
        p = np.clip(p, eps, 1 - eps)
        return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

    res = minimize(nll, x0=np.array([0.0], dtype=float), method="L-BFGS-B")
    T = float(np.exp(res.x[0]))
    return T

def threshold_at_target_sens(y_true: np.ndarray, prob: np.ndarray, target_sens=0.90) -> float:
    fpr, tpr, thr = roc_curve(y_true, prob)
    # roc_curve returns thresholds from inf -> -inf; tpr increases.
    # choose smallest threshold achieving tpr>=target_sens (more conservative)
    idx = np.where(tpr >= target_sens)[0]
    if len(idx) == 0:
        return float(thr[-1])
    return float(thr[idx[-1]])  # lowest threshold still meeting sens

def threshold_at_target_spec(y_true: np.ndarray, prob: np.ndarray, target_spec=0.85) -> float:
    fpr, tpr, thr = roc_curve(y_true, prob)
    spec = 1 - fpr
    idx = np.where(spec >= target_spec)[0]
    if len(idx) == 0:
        return float(thr[0])
    return float(thr[idx[-1]])

def metrics_at_threshold(y_true: np.ndarray, prob: np.ndarray, thr: float) -> Dict:
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    ppv  = tp / (tp + fp) if (tp + fp) else np.nan
    npv  = tn / (tn + fn) if (tn + fn) else np.nan
    acc  = (tp + tn) / (tp + tn + fp + fn)
    return {
        "threshold": float(thr),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "sensitivity": float(sens),
        "specificity": float(spec),
        "ppv": float(ppv),
        "npv": float(npv),
        "accuracy": float(acc),
    }

def decision_curve_net_benefit(y_true: np.ndarray, prob: np.ndarray, thresholds: np.ndarray) -> pd.DataFrame:
    # NB = TP/N - FP/N * (pt/(1-pt))
    y = y_true.astype(int)
    N = len(y)
    rows = []
    prev = np.mean(y)  # biopsy-all reference
    for pt in thresholds:
        if pt <= 0 or pt >= 1:
            continue
        pred = (prob >= pt).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
        nb_model = (tp / N) - (fp / N) * (pt / (1 - pt))
        nb_all   = (prev) - (1 - prev) * (pt / (1 - pt))
        nb_none  = 0.0
        rows.append({
            "threshold": float(pt),
            "nb_model": float(nb_model),
            "nb_all": float(nb_all),
            "nb_none": float(nb_none),
            "tp": int(tp), "fp": int(fp)
        })
    return pd.DataFrame(rows)

def compute_uncertainty(df: pd.DataFrame, k=5) -> pd.DataFrame:
    # Uses prob_fold1..k and logit_fold1..k if present
    prob_cols = [f"prob_fold{i}" for i in range(1, k+1) if f"prob_fold{i}" in df.columns]
    logit_cols = [f"logit_fold{i}" for i in range(1, k+1) if f"logit_fold{i}" in df.columns]

    out = df[["Patient","y_true","prob_mean","logit_mean"]].copy()

    # variance across folds (prob)
    P = df[prob_cols].to_numpy(dtype=float)
    out["prob_var_folds"] = np.var(P, axis=1)

    # predictive entropy of mean prob
    p = np.clip(out["prob_mean"].to_numpy(dtype=float), 1e-9, 1-1e-9)
    out["entropy_meanprob"] = -(p*np.log(p) + (1-p)*np.log(1-p))

    # mutual information proxy using fold probs:
    # MI ≈ H(E[p]) - E[H(p_i)]
    # H(p_i) for Bernoulli
    Hp_i = -(np.clip(P,1e-9,1-1e-9)*np.log(np.clip(P,1e-9,1-1e-9)) +
             (1-np.clip(P,1e-9,1-1e-9))*np.log(1-np.clip(P,1e-9,1-1e-9)))
    out["mi_proxy"] = out["entropy_meanprob"].to_numpy() - np.mean(Hp_i, axis=1)

    # error indicator
    out["pred_label_0p5"] = (out["prob_mean"] >= 0.5).astype(int)
    out["error_0p5"] = (out["pred_label_0p5"] != out["y_true"]).astype(int)

    return out

def summarize_uncertainty(unc_df: pd.DataFrame) -> Dict:
    # compare uncertainty for correct vs incorrect
    res = {}
    for col in ["prob_var_folds","entropy_meanprob","mi_proxy"]:
        a = unc_df.loc[unc_df["error_0p5"]==0, col].dropna().to_numpy()
        b = unc_df.loc[unc_df["error_0p5"]==1, col].dropna().to_numpy()
        res[col] = {
            "mean_correct": float(np.mean(a)) if len(a) else None,
            "mean_incorrect": float(np.mean(b)) if len(b) else None,
            "n_correct": int(len(a)),
            "n_incorrect": int(len(b))
        }
    return res

# ------------------------------
# Main
# ------------------------------
def main():
    ensure_dir(cfg.RUNS_DIR)

    df = pd.read_csv(cfg.AGG_CSV)
    # sanity
    needed = {"Patient","y_true","prob_mean","logit_mean"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["y_true"] = df["y_true"].astype(int)
    y = df["y_true"].to_numpy()
    prob = df["prob_mean"].to_numpy(dtype=float)
    logit = df["logit_mean"].to_numpy(dtype=float)

    # ---- AUC + CI
    auc = float(roc_auc_score(y, prob))
    auc_ci = bootstrap_auc_ci(y, prob, n_boot=cfg.N_BOOT, seed=cfg.SEED)

    # ---- Calibration (raw)
    brier = float(brier_score_loss(y, prob))
    ece_raw = ece_score(y, prob, n_bins=cfg.ECE_BINS)

    # ---- Temperature scaling (calibration split)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=cfg.CALIB_FRAC, random_state=cfg.SEED)
    idx_train, idx_cal = next(sss.split(np.zeros_like(y), y))

    # Fit T on calibration subset (idx_cal) to avoid tuning on the same set we report on
    # Here: "cal" is the tuning set; we report on "train" portion. (You can flip if you prefer.)
    T = fit_temperature_from_logits(logit[idx_cal], y[idx_cal])

    prob_T = expit(logit / T)
    prob_T_report = prob_T[idx_train]
    y_report = y[idx_train]

    # Report-calibration metrics (post-temp scaling)
    auc_T = float(roc_auc_score(y_report, prob_T_report))
    brier_T = float(brier_score_loss(y_report, prob_T_report))
    ece_T = ece_score(y_report, prob_T_report, n_bins=cfg.ECE_BINS)

    # ---- Clinical thresholds (use full set, and also the report split)
    thr_sens = threshold_at_target_sens(y, prob, cfg.TARGET_SENS)
    thr_spec = threshold_at_target_spec(y, prob, cfg.TARGET_SPEC)

    met_sens = metrics_at_threshold(y, prob, thr_sens)
    met_spec = metrics_at_threshold(y, prob, thr_spec)

    # ---- Decision curve
    thresholds = np.linspace(0.01, 0.99, cfg.DCA_POINTS)
    dca = decision_curve_net_benefit(y, prob, thresholds)

    # ---- Uncertainty
    unc_df = compute_uncertainty(df, k=5)
    unc_summary = summarize_uncertainty(unc_df)

    # ---- Save artifacts
    out = {
        "agg_csv": str(cfg.AGG_CSV),
        "n_patients": int(len(df)),
        "auc_prob_mean": auc,
        "auc_bootstrap": auc_ci,
        "calibration_raw": {
            "brier": brier,
            "ece": ece_raw["ece"],
            "ece_bins": ece_raw["bins"]
        },
        "temperature_scaling": {
            "split_note": f"T fit on {cfg.CALIB_FRAC*100:.0f}% stratified subset; metrics reported on remaining {(1-cfg.CALIB_FRAC)*100:.0f}%",
            "T": float(T),
            "auc_after_T_report_split": auc_T,
            "brier_after_T_report_split": brier_T,
            "ece_after_T_report_split": ece_T["ece"],
            "ece_bins_after_T_report_split": ece_T["bins"]
        },
        "thresholds": {
            f"sens>={cfg.TARGET_SENS}": met_sens,
            f"spec>={cfg.TARGET_SPEC}": met_spec
        },
        "uncertainty_summary": unc_summary
    }

    out_json = cfg.RUNS_DIR / "Q1_EVAL_summary_seed43.json"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)

    out_unc_csv = cfg.RUNS_DIR / "Q1_uncertainty_table_seed43.csv"
    unc_df.to_csv(out_unc_csv, index=False)

    out_dca_csv = cfg.RUNS_DIR / "Q1_decision_curve_seed43.csv"
    dca.to_csv(out_dca_csv, index=False)

    # Also save temp-scaled probabilities for transparency
    df_out = df[["Patient","y_true","prob_mean","logit_mean"]].copy()
    df_out["prob_T"] = prob_T
    df_out["T_used"] = float(T)
    out_prob_csv = cfg.RUNS_DIR / "Q1_probs_with_temperature_seed43.csv"
    df_out.to_csv(out_prob_csv, index=False)

    # print key results
    print("Saved:", out_json)
    print("Saved:", out_unc_csv)
    print("Saved:", out_dca_csv)
    print("Saved:", out_prob_csv)

    print("\n=== KEY RESULTS (RAW) ===")
    print("AUC:", auc)
    print("AUC 95% CI:", auc_ci["auc_ci95"])
    print("Brier:", brier)
    print("ECE:", ece_raw["ece"])

    print("\n=== TEMP SCALING ===")
    print("T:", T)
    print("AUC (report split):", auc_T)
    print("Brier (report split):", brier_T)
    print("ECE (report split):", ece_T["ece"])

    print("\n=== THRESHOLDS (RAW probs) ===")
    print(f"Threshold for Sens≥{cfg.TARGET_SENS}:", met_sens["threshold"], met_sens)
    print(f"Threshold for Spec≥{cfg.TARGET_SPEC}:", met_spec["threshold"], met_spec)

    print("\n=== UNCERTAINTY (0.5 cutoff) ===")
    for k,v in unc_summary.items():
        print(k, v)

if __name__ == "__main__":
    main()

Saved: D:\ENT\_challenge2_artifacts\_runs\Q1_EVAL_summary_seed43.json
Saved: D:\ENT\_challenge2_artifacts\_runs\Q1_uncertainty_table_seed43.csv
Saved: D:\ENT\_challenge2_artifacts\_runs\Q1_decision_curve_seed43.csv
Saved: D:\ENT\_challenge2_artifacts\_runs\Q1_probs_with_temperature_seed43.csv

=== KEY RESULTS (RAW) ===
AUC: 0.8408695652173913
AUC 95% CI: [0.7454545454545455, 0.9198036006546645]
Brier: 0.20377330820036005
ECE: 0.20394909209025122

=== TEMP SCALING ===
T: 2.241094080648076
AUC (report split): 0.8839285714285714
Brier (report split): 0.16541025673900392
ECE (report split): 0.19428120937663443

=== THRESHOLDS (RAW probs) ===
Threshold for Sens≥0.9: 0.0039010962897429 {'threshold': 0.0039010962897429, 'tp': 23, 'fp': 50, 'tn': 0, 'fn': 0, 'sensitivity': 1.0, 'specificity': 0.0, 'ppv': 0.3150684931506849, 'npv': nan, 'accuracy': 0.3150684931506849}
Threshold for Spec≥0.85: 0.1991468647761961 {'threshold': 0.1991468647761961, 'tp': 12, 'fp': 7, 'tn': 43, 'fn': 11, 'sensitivit

In [44]:
# q1_clean_ent_pipeline.py
# ============================================================
# Scientifically clean pipeline for:
#   Task A: Patient-level MIL classification + fold aggregation + final submission
#   Task B: Explainability via Grad-CAM overlays (top-k images per patient)
#   Task C: Domain-shift evaluation (WLE vs NBI) using labeled patient-domain file
#   Q1 extras: CI bootstraps, calibration (ECE), Brier, decision curve, uncertainty proxies
# ============================================================

from __future__ import annotations
import os
import re
import json
import math
import time
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

# torchvision is used for transforms & resnet blocks
import torchvision
from torchvision import transforms

warnings.filterwarnings("ignore")


# ----------------------------
# Config
# ----------------------------
@dataclass
class CFG:
    # core paths
    IMAGES_ROOT: Path = Path(r"D:\ENT")
    ARTIFACTS_ROOT: Path = Path(r"D:\ENT\_challenge2_artifacts")
    RUNS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    # manifests (your split CSVs)
    TRAIN_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_train_major_colab.csv")
    VAL_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_val_major_colab.csv")
    TEST_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")

    # task switches
    DO_TASK_A: bool = True
    DO_TASK_B: bool = True
    DO_TASK_C: bool = True
    DO_Q1_EVAL: bool = True

    # run params
    SEED: int = 43
    BENCHMARK: str = "res_shift"
    FOLDS: Tuple[int, ...] = (1, 2, 3, 4, 5)
    K_TEST: int = 10  # not used here for inference if preds already exist; used for file naming

    # model / ckpt
    # use one fold ckpt for Task B CAM (Fold1 by default)
    CKPT_FOR_TASKB: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold1.pt")

    # image
    IMG_SIZE: int = 224

    # Task B
    TOPK_FOR_CAM: int = 6  # overlays per patient

    # Device
    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"

    # Domain labeling
    # If you have a filled file, point to it (xlsx or csv). Otherwise the code exports a template.
    TEST_PATIENT_DOMAIN_PATH: Optional[Path] = None  # e.g. Path(r"D:\ENT\_challenge2_artifacts\DOMAIN_LABELS_FILLED.xlsx")

    # Output dirs
    TASKB_DIR: Optional[Path] = None  # auto-filled

cfg = CFG()


# ----------------------------
# Reproducibility
# ----------------------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ----------------------------
# Robust CSV loader
# ----------------------------
def load_manifest(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # normalize common columns
    # patient id
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID", "id", "case", "CaseID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break

    # image path
    if "path" not in df.columns:
        for c in ["ImagePath", "img_path", "image_path", "filepath", "file", "Path"]:
            if c in df.columns:
                df = df.rename(columns={c: "path"})
                break

    # label
    if "y_true" not in df.columns:
        for c in ["label", "Label", "target", "Target", "Histopathology", "GT", "y"]:
            if c in df.columns:
                df = df.rename(columns={c: "y_true"})
                break

    # leuko (optional auxiliary)
    if "leuko" not in df.columns:
        for c in ["Leukoplakia", "leukoplakia", "leuko_label"]:
            if c in df.columns:
                df = df.rename(columns={c: "leuko"})
                break

    # allow manifests that contain only patient list (no paths)
    if "Patient" not in df.columns:
        raise ValueError(f"{csv_path} must contain a patient column (Patient / patient / patient_id).")

    return df


def as_binary_label(x) -> int:
    # robust label parsing
    if pd.isna(x):
        return 0
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        return int(x)
    s = str(x).strip().lower()
    if s in ["1", "true", "yes", "y", "malignant", "scc", "cancer", "carcinoma"]:
        return 1
    if s in ["0", "false", "no", "n", "benign"]:
        return 0
    # fallback: try int
    try:
        return int(float(s))
    except:
        return 0


def summarize_split(df: pd.DataFrame, name: str, mapping: Dict[str, List[Path]]):
    pats = sorted(df["Patient"].astype(str).unique().tolist())
    counts = [len(mapping.get(p, [])) for p in pats]
    print(f"\n=== {name.upper()} ===")
    print("Patients:", len(pats))
    print("Patients with 0 images:", sum([c == 0 for c in counts]))
    ser = pd.Series(counts)
    print("Images per patient:", ser.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))


# ----------------------------
# Mapping: patient -> list of image paths
# ----------------------------
def normalize_path(p: str, images_root: Path) -> Path:
    # if already absolute, keep; else join to images_root
    p = str(p).strip().strip('"').strip("'")
    pp = Path(p)
    if pp.is_absolute():
        return pp
    return images_root / pp


def build_mapping_from_manifests(
    df_list: List[pd.DataFrame],
    images_root: Path,
) -> Dict[str, List[Path]]:
    mapping: Dict[str, List[Path]] = {}
    for df in df_list:
        if "path" not in df.columns:
            # manifests without per-image rows; nothing to add here
            continue
        for _, r in df.iterrows():
            pid = str(r["Patient"])
            ip = normalize_path(r["path"], images_root)
            mapping.setdefault(pid, []).append(ip)

    # de-dup and keep existing files only
    for pid in list(mapping.keys()):
        uniq = []
        seen = set()
        for p in mapping[pid]:
            s = str(p)
            if s in seen:
                continue
            seen.add(s)
            if p.exists():
                uniq.append(p)
        mapping[pid] = uniq
    return mapping


def build_mapping_from_root_folder(images_root: Path) -> Dict[str, List[Path]]:
    """
    Fallback: scan images_root and match folder segments containing PatientXXX.
    This is slower but useful when CSV paths aren't present.
    """
    pat_re = re.compile(r"(Patient\d+)", re.IGNORECASE)
    mapping: Dict[str, List[Path]] = {}
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    for p in images_root.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in exts:
            continue
        m = pat_re.search(str(p))
        if not m:
            continue
        pid = m.group(1)
        mapping.setdefault(pid, []).append(p)

    # sort stable
    for pid in mapping:
        mapping[pid] = sorted(mapping[pid], key=lambda x: str(x))
    print("Patients found in image root:", len(mapping))
    return mapping


# ----------------------------
# Model: DualBranchGatedMIL + AttnPool
# ----------------------------
class AttnPool(nn.Module):
    """
    Classic MIL attention (Ilse et al.). Here implemented as:
      a_i = softmax( w(tanh(V h_i)) )
      bag = sum_i a_i h_i
    """
    def __init__(self, in_dim: int, hidden: int):
        super().__init__()
        self.V = nn.Linear(in_dim, hidden)
        self.w = nn.Linear(hidden, 1)

    def forward(self, H: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # H: [N, D]
        A = self.w(torch.tanh(self.V(H)))  # [N,1]
        A = torch.softmax(A.squeeze(-1), dim=0)  # [N]
        bag = torch.sum(A.unsqueeze(-1) * H, dim=0)  # [D]
        return bag, A


class DualBranchGatedMIL(nn.Module):
    """
    Encoder: ResNet50 backbone -> 2048D per-image feature.
    Two MIL branches (non/leuko) + gating on concatenated bag representations.
    Output: fused logit + branch logits + gate alpha + attention weights.
    """
    def __init__(self, gate_tau: float = 2.0):
        super().__init__()
        self.gate_tau = gate_tau

        # encoder backbone: torchvision resnet50
        resnet = torchvision.models.resnet50(weights=None)
        # feature extractor up to avgpool
        self.encoder = nn.Module()
        self.encoder.backbone = nn.Sequential(*list(resnet.children())[:-1])  # outputs [B,2048,1,1]
        self.instance_dim = 2048

        # attention pools
        self.pool_non = AttnPool(self.instance_dim, hidden=256)
        self.pool_leu = AttnPool(self.instance_dim, hidden=256)

        # branch classifiers
        self.cls_non = nn.Linear(self.instance_dim, 1)
        self.cls_leu = nn.Linear(self.instance_dim, 1)

        # gate uses concatenated bags [4096] -> 1
        self.gate = nn.Linear(self.instance_dim * 2, 1)

        # auxiliary head for leuko from concatenated bags (optional)
        self.leuko_head = nn.Sequential(
            nn.Linear(self.instance_dim * 2, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1)
        )

        # CAM target layer (last conv block): hook this
        # We'll set this dynamically as the final Bottleneck block in layer4
        self._cam_target_layer = None
        self._set_default_cam_target()

    def _set_default_cam_target(self):
        # encoder.backbone is Sequential: [conv,bn,relu,maxpool,layer1,layer2,layer3,layer4,avgpool]
        layer4 = self.encoder.backbone[7]  # resnet layer4
        # target last bottleneck
        self._cam_target_layer = layer4[-1]

    def encode_images(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B,3,H,W] -> [B,2048]
        feat = self.encoder.backbone(x)     # [B,2048,1,1]
        feat = feat.flatten(1)              # [B,2048]
        return feat

    def forward_patient(self, X: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        X: [N,3,H,W] where N=instances for a patient
        """
        H = self.encode_images(X)  # [N,2048]

        bag_non, attn_non = self.pool_non(H)
        bag_leu, attn_leu = self.pool_leu(H)

        logit_non = self.cls_non(bag_non)  # [1]
        logit_leu = self.cls_leu(bag_leu)  # [1]

        g_in = torch.cat([bag_non, bag_leu], dim=0).unsqueeze(0)  # [1,4096]
        gate_logit = self.gate(g_in) / max(self.gate_tau, 1e-6)   # [1,1]
        gate_alpha = torch.sigmoid(gate_logit).squeeze(0).squeeze(0)  # scalar

        fused_logit = gate_alpha * logit_leu.squeeze(0) + (1.0 - gate_alpha) * logit_non.squeeze(0)

        leuko_logit = self.leuko_head(g_in).squeeze(0).squeeze(0)

        return {
            "logit": fused_logit.squeeze(),
            "logit_non": logit_non.squeeze(),
            "logit_leu": logit_leu.squeeze(),
            "leuko_logit": leuko_logit.squeeze(),
            "gate_alpha": gate_alpha,
            "attn_non": attn_non.detach(),
            "attn_leu": attn_leu.detach(),
        }


# ----------------------------
# Checkpoint loading (fixes your UnpicklingError)
# ----------------------------
def safe_torch_load(path: Path):
    """
    Try weights_only=True first; if it fails due to numpy globals, fall back to weights_only=False.
    This is safe ONLY for trusted checkpoints (your own training).
    """
    try:
        return torch.load(str(path), map_location="cpu", weights_only=True)
    except Exception as e:
        msg = str(e)
        if ("Weights only load failed" in msg) or ("WeightsUnpickler error" in msg) or ("Unsupported global" in msg):
            print("[WARN] weights_only=True failed; falling back to weights_only=False for trusted ckpt.")
            return torch.load(str(path), map_location="cpu", weights_only=False)
        # older torch versions
        if "weights_only" in msg and ("unexpected keyword" in msg or "got an unexpected keyword" in msg):
            return torch.load(str(path), map_location="cpu")
        raise


def build_model_from_ckpt(ckpt_path: Path, device: str, gate_tau: float = 2.0) -> Tuple[DualBranchGatedMIL, dict]:
    ckpt = safe_torch_load(ckpt_path)
    sd = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    model = DualBranchGatedMIL(gate_tau=gate_tau).to(device)

    # Some ckpts store AttnPool as V/w, while our AttnPool matches that exactly.
    # Some ckpts store leuko_head with extra layer indices (2 vs 3). We'll remap if needed.
    model_sd_keys = set(model.state_dict().keys())
    sd2 = {}

    # Key remap for leuko_head if needed
    for k, v in sd.items():
        kk = k

        # handle leuko_head.3 -> leuko_head.2
        if kk.startswith("leuko_head.3.") and ("leuko_head.2." in " ".join(model_sd_keys)):
            kk = kk.replace("leuko_head.3.", "leuko_head.2.")

        # handle pool_non.net.* vs pool_non.V/w (we use V/w)
        # If checkpoint has pool_non.net.0 etc, translate:
        if kk.startswith("pool_non.net.0."):
            kk = kk.replace("pool_non.net.0.", "pool_non.V.")
        if kk.startswith("pool_non.net.2."):
            kk = kk.replace("pool_non.net.2.", "pool_non.w.")
        if kk.startswith("pool_leu.net.0."):
            kk = kk.replace("pool_leu.net.0.", "pool_leu.V.")
        if kk.startswith("pool_leu.net.2."):
            kk = kk.replace("pool_leu.net.2.", "pool_leu.w.")

        sd2[kk] = v

    missing, unexpected = model.load_state_dict(sd2, strict=False)
    print(f"[CKPT LOAD] {ckpt_path.name}")
    print("  missing:", len(missing), " unexpected:", len(unexpected))
    if len(missing) > 0:
        print("  missing sample:", missing[:10])
    if len(unexpected) > 0:
        print("  unexpected sample:", unexpected[:10])

    model.eval()
    return model, ckpt


# ----------------------------
# Image transforms / reading
# ----------------------------
def get_transform(img_size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


def read_pil(path: Path) -> Image.Image:
    img = Image.open(path).convert("RGB")
    return img


# ----------------------------
# Task A: aggregate folds + final submission
# ----------------------------
def aggregate_folds_to_seed(
    runs_dir: Path,
    seed: int,
    benchmark: str,
    folds: Tuple[int, ...],
    k_test: int,
    out_path: Path,
) -> pd.DataFrame:
    dfs = []
    used_folds = []
    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            # allow other naming (some of yours used pred_fold4_res_shift_seed42...)
            # skip if missing
            continue
        d = pd.read_csv(p)
        # normalize expected columns
        if "Patient" not in d.columns:
            raise ValueError(f"{p} missing Patient column.")
        d = d.set_index("Patient")

        # allow various logit/prob naming
        logit_col = "logit" if "logit" in d.columns else ("logit_mean" if "logit_mean" in d.columns else None)
        prob_col  = "prob"  if "prob"  in d.columns else ("prob_mean"  if "prob_mean"  in d.columns else None)
        if logit_col is None or prob_col is None:
            raise ValueError(f"{p} must have logit/prob columns. Found: {d.columns.tolist()}")

        base_cols = []
        for c in ["y_true", "leuko"]:
            if c in d.columns:
                base_cols.append(c)

        keep = base_cols + [logit_col, prob_col]
        dd = d[keep].copy()
        dd = dd.rename(columns={logit_col: f"logit_fold{f}", prob_col: f"prob_fold{f}"})

        dfs.append(dd)
        used_folds.append(f)

    if len(dfs) == 0:
        raise FileNotFoundError("No fold prediction CSVs were found to aggregate.")

    merged = dfs[0]
    for dd in dfs[1:]:
        # merge only fold-specific cols
        fold_cols = [c for c in dd.columns if c.startswith("logit_fold") or c.startswith("prob_fold")]
        merged = merged.join(dd[fold_cols], how="inner")

    # compute mean
    logit_cols = [c for c in merged.columns if c.startswith("logit_fold")]
    prob_cols  = [c for c in merged.columns if c.startswith("prob_fold")]

    merged["logit_mean"] = merged[logit_cols].mean(axis=1)
    merged["prob_mean"]  = merged[prob_cols].mean(axis=1)

    merged = merged.reset_index()

    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | folds used: {used_folds}")
    return merged


def make_final_submission(agg_df: pd.DataFrame, out_csv: Path) -> pd.DataFrame:
    if "Patient" not in agg_df.columns:
        raise ValueError("agg_df must have Patient column.")
    if "prob_mean" not in agg_df.columns:
        raise ValueError("agg_df must have prob_mean column.")

    sub = agg_df[["Patient", "prob_mean"]].copy()
    sub = sub.rename(columns={"prob_mean": "prob"})
    sub.to_csv(out_csv, index=False)
    print(f"Saved final submission: {out_csv}")
    print("Patients:", len(sub))
    return sub


# ----------------------------
# Task B: Grad-CAM (per-image) + overlay
# ----------------------------
class GradCAM:
    """
    Minimal Grad-CAM for a chosen conv layer.
    We compute CAM for fused patient logit but per-image forward is done as:
      - run model.encode_images for images
      - run attention + gating to get fused logit
    We need gradients wrt activations of target layer inside encoder backbone.
    """
    def __init__(self, model: DualBranchGatedMIL, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        def fwd_hook(_, __, out):
            self.activations = out

        def bwd_hook(_, grad_in, grad_out):
            # grad_out[0] matches activation gradient
            self.gradients = grad_out[0]

        self.h1 = target_layer.register_forward_hook(fwd_hook)
        self.h2 = target_layer.register_full_backward_hook(bwd_hook)

    def close(self):
        self.h1.remove()
        self.h2.remove()

    @torch.no_grad()
    def _normalize_cam(self, cam: np.ndarray) -> np.ndarray:
        cam = cam - cam.min()
        if cam.max() > 1e-8:
            cam = cam / cam.max()
        return cam

    def cam_for_patient_tensor(self, X: torch.Tensor, instance_index: int) -> np.ndarray:
        """
        X: [N,3,H,W] patient bag
        instance_index: which instance to compute CAM for (we will forward that single image)
        We'll compute CAM for fused logit *conditioned on full patient bag*:
          - forward full bag (needs grad)
          - backprop fused logit
          - take cam at chosen instance activation map (from forward hook)
        """
        self.model.zero_grad(set_to_none=True)

        # ensure grad enabled
        was_grad = torch.is_grad_enabled()
        torch.set_grad_enabled(True)

        # forward full bag through encoder with hooks (activations saved from last forward call)
        out = self.model.forward_patient(X)  # uses encoder.backbone internally

        logit = out["logit"]
        if not logit.requires_grad:
            # safety: ensure something requires grad
            logit = logit.clone().requires_grad_(True)

        logit.backward(retain_graph=False)

        # activations & gradients correspond to full batch forward within encode_images
        A = self.activations         # [N,C,h,w] or [1,C,h,w] depending on encoder path
        G = self.gradients

        # In our setup, encoder.backbone processes input X in one batch, so A shape should be [N,C,h,w]
        if A is None or G is None:
            torch.set_grad_enabled(was_grad)
            raise RuntimeError("GradCAM hooks did not capture activations/gradients. Check target layer.")

        if A.dim() == 4 and A.shape[0] == X.shape[0]:
            a = A[instance_index]  # [C,h,w]
            g = G[instance_index]  # [C,h,w]
        elif A.dim() == 4 and A.shape[0] == 1:
            a = A[0]
            g = G[0]
        else:
            torch.set_grad_enabled(was_grad)
            raise RuntimeError(f"Unexpected activation shape: {A.shape}")

        # channel weights
        w = g.mean(dim=(1, 2))  # [C]
        cam = torch.sum(w[:, None, None] * a, dim=0)  # [h,w]
        cam = F.relu(cam).detach().cpu().numpy()
        cam = self._normalize_cam(cam)

        torch.set_grad_enabled(was_grad)
        return cam


def overlay_cam_on_pil(pil: Image.Image, cam: np.ndarray, alpha: float = 0.45) -> Image.Image:
    """
    cam: [h,w] normalized 0..1
    We'll map to red heatmap (simple) without external libs.
    """
    # resize cam to image
    cam_img = Image.fromarray(np.uint8(cam * 255)).resize(pil.size, resample=Image.BILINEAR)
    cam_arr = np.array(cam_img).astype(np.float32) / 255.0

    img_arr = np.array(pil).astype(np.float32) / 255.0

    # heat in red channel
    heat = np.zeros_like(img_arr)
    heat[..., 0] = cam_arr  # red
    out = (1 - alpha) * img_arr + alpha * heat
    out = np.clip(out * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(out)


def run_task_b_explainability(
    model: DualBranchGatedMIL,
    patient_ids: List[str],
    mapping: Dict[str, List[Path]],
    out_dir: Path,
    device: str,
    img_size: int,
    topk_for_cam: int = 6,
):
    out_dir.mkdir(parents=True, exist_ok=True)
    tfm = get_transform(img_size)

    # cam target
    cam = GradCAM(model, model._cam_target_layer)

    saved = 0
    t0 = time.time()
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Task B: saving CAM overlays to: {out_dir}")

    for i, pid in enumerate(patient_ids, 1):
        paths = mapping.get(pid, [])
        if len(paths) == 0:
            continue

        # pick topK instances by a cheap proxy: first K (or you can compute attention)
        # We'll compute attention by forward pass and then select topk by attention of fused branch.
        imgs = []
        valid_paths = []
        for p in paths:
            try:
                pil = read_pil(p)
                imgs.append(tfm(pil))
                valid_paths.append(p)
            except:
                continue

        if len(imgs) == 0:
            continue

        X = torch.stack(imgs, dim=0).to(device)  # [N,3,H,W]

        # compute attention scores to select topK
        with torch.no_grad():
            out = model.forward_patient(X)
            attn = out["attn_non"].detach().cpu().numpy()  # [N]
            # select topk by attention
            order = np.argsort(-attn)
            pick = order[:min(topk_for_cam, len(order))]

        # compute CAM overlays for selected instances
        for rank, idx in enumerate(pick, 1):
            p = valid_paths[int(idx)]
            pil = read_pil(p).resize((img_size, img_size))

            # Need gradients: do not use torch.no_grad here
            cam_map = cam.cam_for_patient_tensor(X, instance_index=int(idx))
            overlay = overlay_cam_on_pil(pil, cam_map, alpha=0.45)

            out_name = f"{pid}_rank{rank:02d}_attn{attn[int(idx)]:.4f}__{p.name}"
            overlay.save(out_dir / out_name)
            saved += 1

        if i % 5 == 0 or i == len(patient_ids):
            dt = time.time() - t0
            print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Task B progress: {i}/{len(patient_ids)} patients | overlays saved={saved} | elapsed={dt:.1f}s")

    cam.close()
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Task B done. Total overlays: {saved}")


# ----------------------------
# Task C: Domain shift evaluation
# ----------------------------
def clean_domain(x) -> str:
    if pd.isna(x):
        return "UNKNOWN"
    s = str(x).strip().upper()
    if s in ["WLE", "WHITE", "WHITE LIGHT", "WL", "W"]:
        return "WLE"
    if s in ["NBI", "NARROW", "NARROW BAND", "NB", "N"]:
        return "NBI"
    if s in ["MIXED", "BOTH", "WLE+NBI", "NBI+WLE"]:
        return "MIXED"
    if s == "" or s == "UNKNOWN" or s == "NAN":
        return "UNKNOWN"
    return "UNKNOWN"


def export_domain_label_template(
    patient_ids: List[str],
    mapping: Dict[str, List[Path]],
    out_xlsx: Path,
):
    rows = []
    for pid in patient_ids:
        paths = mapping.get(pid, [])
        ex = [str(p) for p in paths[:3]] + ["", "", ""]
        rows.append({
            "Patient": pid,
            "Domain": "",
            "ExamplePath1": ex[0],
            "ExamplePath2": ex[1],
            "ExamplePath3": ex[2],
            "Notes": ""
        })
    df = pd.DataFrame(rows)
    out_xlsx.parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(out_xlsx, index=False)
    print(f"Wrote: {out_xlsx}")
    print("Rows:", len(df))


def load_patient_domains(domain_labels_path: Path) -> pd.DataFrame:
    if domain_labels_path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(domain_labels_path)
    else:
        df = pd.read_csv(domain_labels_path)

    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Domain" not in df.columns:
        for c in ["domain", "modality", "Modality"]:
            if c in df.columns:
                df = df.rename(columns={c: "Domain"})
                break

    if "Patient" not in df.columns or "Domain" not in df.columns:
        raise ValueError("Domain label file must have columns: Patient, Domain")

    df["Patient"] = df["Patient"].astype(str)
    df["Domain"] = df["Domain"].apply(clean_domain)
    return df[["Patient", "Domain"]]


def auc_roc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    # fast AUC without sklearn (rank-based)
    y_true = y_true.astype(int)
    order = np.argsort(y_score)
    y_true_sorted = y_true[order]
    n1 = y_true_sorted.sum()
    n0 = len(y_true_sorted) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    ranks = np.arange(1, len(y_true_sorted) + 1)
    sum_ranks_pos = ranks[y_true_sorted == 1].sum()
    auc = (sum_ranks_pos - n1 * (n1 + 1) / 2) / (n1 * n0)
    return float(auc)


def run_task_c_domain_shift_partial_ok(
    agg_df: pd.DataFrame,
    domain_labels_path: Path,
    out_json_path: Path,
    logit_col: str = "logit_mean",
):
    """
    Domain shift valid if at least one WLE and one NBI patient labeled.
    Per-domain AUC computed only if that domain has both classes.
    """
    if "Patient" not in agg_df.columns:
        raise ValueError("agg_df must have Patient column.")
    if logit_col not in agg_df.columns:
        raise ValueError(f"agg_df must have '{logit_col}' column.")

    labels = load_patient_domains(domain_labels_path)

    merged = agg_df.merge(labels, on="Patient", how="left")
    merged["Domain"] = merged["Domain"].fillna("UNKNOWN").apply(clean_domain)

    # y_true
    if "y_true" in merged.columns:
        y = merged["y_true"].apply(as_binary_label).values.astype(int)
    else:
        raise ValueError("agg_df must contain y_true for Task C evaluation.")

    s = merged[logit_col].values.astype(float)

    # overall
    overall_auc = auc_roc(y, s)

    # counts
    dom_counts = merged["Domain"].value_counts().to_dict()

    # per-domain
    per_dom = {}
    for dom, sub in merged.groupby("Domain"):
        yy = sub["y_true"].apply(as_binary_label).values.astype(int)
        ss = sub[logit_col].values.astype(float)
        a = auc_roc(yy, ss)
        per_dom[dom] = {"n": int(len(sub)), "auc": (None if np.isnan(a) else float(a))}

    # domain shift valid requires at least 1 WLE and 1 NBI present
    n_wle = int((merged["Domain"] == "WLE").sum())
    n_nbi = int((merged["Domain"] == "NBI").sum())
    domain_shift_valid = (n_wle > 0 and n_nbi > 0)

    out = {
        "n_total": int(len(merged)),
        "domain_counts": {k: int(v) for k, v in dom_counts.items()},
        "auc_overall": None if np.isnan(overall_auc) else float(overall_auc),
        "per_domain": per_dom,
        "domain_shift_valid": bool(domain_shift_valid),
        "note": "Per-domain AUC is computed only when that domain subset has both classes. "
                "Domain-shift VALID requires at least 1 WLE and 1 NBI patient labeled.",
        "domain_labels_used": str(domain_labels_path),
        "agg_path": None,
    }

    out_json_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_json_path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)

    print(f"[Task C partial-ok] saved: {out_json_path}")
    print(json.dumps(out, indent=2))


# ----------------------------
# Q1 Evaluation bundle
# ----------------------------
def brier_score(y_true: np.ndarray, p: np.ndarray) -> float:
    return float(np.mean((p - y_true) ** 2))


def expected_calibration_error(y_true: np.ndarray, p: np.ndarray, n_bins: int = 10) -> float:
    # ECE with equal-width bins
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.mean()) * abs(acc - conf)
    return float(ece)


def bootstrap_auc_ci(y_true: np.ndarray, y_score: np.ndarray, n_boot: int = 2000, seed: int = 43) -> Tuple[float, float]:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yy = y_true[idx]
        ss = y_score[idx]
        a = auc_roc(yy, ss)
        if not np.isnan(a):
            aucs.append(a)
    if len(aucs) == 0:
        return (float("nan"), float("nan"))
    lo = float(np.quantile(aucs, 0.025))
    hi = float(np.quantile(aucs, 0.975))
    return lo, hi


def decision_curve_table(y_true: np.ndarray, p: np.ndarray, thresholds: np.ndarray) -> pd.DataFrame:
    # Net benefit = (TP/N) - (FP/N) * (pt/(1-pt))
    rows = []
    N = len(y_true)
    for pt in thresholds:
        pred = (p >= pt).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        nb = (tp / N) - (fp / N) * (pt / max(1e-9, (1 - pt)))
        rows.append({"threshold": float(pt), "tp": tp, "fp": fp, "net_benefit": float(nb)})
    return pd.DataFrame(rows)


def uncertainty_table_from_folds(agg_df: pd.DataFrame) -> pd.DataFrame:
    # compute fold-prob variance and entropy of mean prob
    prob_cols = [c for c in agg_df.columns if c.startswith("prob_fold")]
    out = agg_df.copy()

    if len(prob_cols) >= 2:
        out["prob_var_folds"] = out[prob_cols].var(axis=1)
    else:
        out["prob_var_folds"] = np.nan

    p = out["prob_mean"].values
    eps = 1e-9
    out["entropy_meanprob"] = -(p * np.log(p + eps) + (1 - p) * np.log(1 - p + eps))

    # mutual-information proxy (approx): entropy(mean) - mean(entropy(folds))
    if len(prob_cols) >= 2:
        ent_folds = []
        for c in prob_cols:
            pf = out[c].values
            ent = -(pf * np.log(pf + eps) + (1 - pf) * np.log(1 - pf + eps))
            ent_folds.append(ent)
        ent_folds = np.stack(ent_folds, axis=1)
        out["mi_proxy"] = out["entropy_meanprob"].values - ent_folds.mean(axis=1)
    else:
        out["mi_proxy"] = np.nan

    return out[["Patient", "y_true", "prob_mean", "prob_var_folds", "entropy_meanprob", "mi_proxy"]]


def run_q1_eval_bundle(agg_df: pd.DataFrame, out_dir: Path, seed: int = 43):
    out_dir.mkdir(parents=True, exist_ok=True)

    y = agg_df["y_true"].apply(as_binary_label).values.astype(int)
    p = agg_df["prob_mean"].values.astype(float)
    s = agg_df["logit_mean"].values.astype(float)

    auc = auc_roc(y, s)
    ci_lo, ci_hi = bootstrap_auc_ci(y, s, n_boot=2000, seed=seed)
    brier = brier_score(y, p)
    ece = expected_calibration_error(y, p, n_bins=10)

    # decision curve
    thr = np.linspace(0.01, 0.99, 99)
    dca = decision_curve_table(y, p, thr)

    # uncertainty
    unc = uncertainty_table_from_folds(agg_df)

    summary = {
        "AUC": None if np.isnan(auc) else float(auc),
        "AUC_95CI": [None if np.isnan(ci_lo) else float(ci_lo), None if np.isnan(ci_hi) else float(ci_hi)],
        "Brier": float(brier),
        "ECE": float(ece),
        "n": int(len(y)),
    }

    # save
    summ_path = out_dir / f"Q1_EVAL_summary_seed{seed}.json"
    unc_path  = out_dir / f"Q1_uncertainty_table_seed{seed}.csv"
    dca_path  = out_dir / f"Q1_decision_curve_seed{seed}.csv"

    with open(summ_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    unc.to_csv(unc_path, index=False)
    dca.to_csv(dca_path, index=False)

    print(f"Saved: {summ_path}")
    print(f"Saved: {unc_path}")
    print(f"Saved: {dca_path}")
    print("\n=== KEY RESULTS (RAW) ===")
    print("AUC:", summary["AUC"])
    print("AUC 95% CI:", summary["AUC_95CI"])
    print("Brier:", summary["Brier"])
    print("ECE:", summary["ECE"])


# ----------------------------
# Main
# ----------------------------
def main():
    seed_everything(cfg.SEED)

    cfg.TASKB_DIR = cfg.RUNS_DIR / f"TASKB_{cfg.BENCHMARK}_seed{cfg.SEED}"

    print("DEVICE:", cfg.DEVICE)
    print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
    print("RUNS_DIR:", cfg.RUNS_DIR)

    # Load manifests
    df_tr = load_manifest(cfg.TRAIN_CSV)
    df_va = load_manifest(cfg.VAL_CSV)
    df_te = load_manifest(cfg.TEST_CSV)

    # Ensure labels are binary if present
    if "y_true" in df_tr.columns:
        df_tr["y_true"] = df_tr["y_true"].apply(as_binary_label)
    if "y_true" in df_va.columns:
        df_va["y_true"] = df_va["y_true"].apply(as_binary_label)
    if "y_true" in df_te.columns:
        df_te["y_true"] = df_te["y_true"].apply(as_binary_label)

    # Build mapping
    mapping = build_mapping_from_manifests([df_tr, df_va, df_te], cfg.IMAGES_ROOT)

    # If manifests did not provide paths, fallback to scanning root
    if len(mapping) == 0:
        mapping = build_mapping_from_root_folder(cfg.IMAGES_ROOT)
    else:
        # still useful to report root patient count quickly
        root_map = build_mapping_from_root_folder(cfg.IMAGES_ROOT)
        print("Patients found in image root:", len(root_map))

    # patient lists
    tr_p = sorted(df_tr["Patient"].astype(str).unique().tolist())
    va_p = sorted(df_va["Patient"].astype(str).unique().tolist())
    te_p = sorted(df_te["Patient"].astype(str).unique().tolist())

    # Split stats
    summarize_split(df_tr, "train", mapping)
    summarize_split(df_va, "val", mapping)
    summarize_split(df_te, "test", mapping)

    # --- Task A: aggregate folds -> seed table, final submission
    agg_path = cfg.RUNS_DIR / f"pred_seed{cfg.SEED}_{cfg.BENCHMARK}_folds1_5_Ktest{cfg.K_TEST}_agg.csv"
    final_path = cfg.RUNS_DIR / f"FINAL_seed{cfg.SEED}_{cfg.BENCHMARK}.csv"

    agg_df = None
    if cfg.DO_TASK_A:
        agg_df = aggregate_folds_to_seed(
            runs_dir=cfg.RUNS_DIR,
            seed=cfg.SEED,
            benchmark=cfg.BENCHMARK,
            folds=cfg.FOLDS,
            k_test=cfg.K_TEST,
            out_path=agg_path
        )
        # If AUC desired quickly:
        if "y_true" in agg_df.columns:
            auc = auc_roc(agg_df["y_true"].values.astype(int), agg_df["logit_mean"].values.astype(float))
            print("AGG TEST AUC:", auc)

        make_final_submission(agg_df, final_path)

    # --- Task B: CAM overlays
    if cfg.DO_TASK_B:
        if not cfg.CKPT_FOR_TASKB.exists():
            raise FileNotFoundError(f"Task B ckpt not found: {cfg.CKPT_FOR_TASKB}")
        model, _ = build_model_from_ckpt(cfg.CKPT_FOR_TASKB, device=cfg.DEVICE, gate_tau=2.0)

        run_task_b_explainability(
            model=model,
            patient_ids=te_p,
            mapping=mapping,
            out_dir=cfg.TASKB_DIR,
            device=cfg.DEVICE,
            img_size=cfg.IMG_SIZE,
            topk_for_cam=cfg.TOPK_FOR_CAM
        )

    # --- Task C: Domain shift
    if cfg.DO_TASK_C:
        if agg_df is None:
            # load if missing
            agg_df = pd.read_csv(agg_path)

        # Require a patient-domain file. If missing, export template.
        template_path = cfg.ARTIFACTS_ROOT / "DOMAIN_NEEDS_LABELS.xlsx"
        if cfg.TEST_PATIENT_DOMAIN_PATH is None or (not Path(cfg.TEST_PATIENT_DOMAIN_PATH).exists()):
            # export template and stop Task C
            export_domain_label_template(te_p, mapping, template_path)
            print("\n[STOP] Task C not run because TEST_PATIENT_DOMAIN_PATH is missing.")
            print(f"Fill the exported file Domain column using: WLE / NBI / MIXED")
            print(f"Then set cfg.TEST_PATIENT_DOMAIN_PATH to that XLSX/CSV and rerun.")
        else:
            # run partial-ok evaluator (valid only if both WLE and NBI exist)
            out_json = cfg.RUNS_DIR / f"TASKC_domain_shift_seed{cfg.SEED}_{cfg.BENCHMARK}.json"
            run_task_c_domain_shift_partial_ok(
                agg_df=agg_df,
                domain_labels_path=Path(cfg.TEST_PATIENT_DOMAIN_PATH),
                out_json_path=out_json,
                logit_col="logit_mean"
            )

    # --- Q1 eval bundle
    if cfg.DO_Q1_EVAL:
        if agg_df is None:
            agg_df = pd.read_csv(agg_path)
        run_q1_eval_bundle(agg_df, cfg.RUNS_DIR, seed=cfg.SEED)

    print("\nDONE.")
    print("Agg:", agg_path)
    print("Final:", final_path)
    if cfg.DO_TASK_B:
        print("TaskB dir:", cfg.TASKB_DIR)


if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs
Patients found in image root: 210
Patients found in image root: 210

=== TRAIN ===
Patients: 117
Patients with 0 images: 0
Images per patient: count    117.000000
mean      50.136752
std       32.430006
min        2.000000
50%       42.000000
75%       71.000000
90%       90.400000
95%       96.800000
99%      144.720000
max      186.000000
dtype: float64

=== VAL ===
Patients: 20
Patients with 0 images: 0
Images per patient: count    20.000000
mean     41.750000
std      23.212689
min       9.000000
50%      40.000000
75%      57.750000
90%      71.000000
95%      80.350000
99%      85.670000
max      87.000000
dtype: float64

=== TEST ===
Patients: 73
Patients with 0 images: 0
Images per patient: count     73.000000
mean      60.863014
std       30.847436
min        2.000000
50%       63.000000
75%       80.000000
90%       94.600000
95%      113.600000
99%      129.240000
max      135.000000
dtype: float

In [45]:
import torch
import torch.nn as nn
import torchvision
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from torchvision import transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGES_ROOT = Path(r"D:\ENT")
TEST_CSV = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")
CKPT_PATH = Path(r"D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold1.pt")
OUT_PATH = Path(r"D:\ENT\_challenge2_artifacts\_runs\BASELINE_singleframe_seed43.csv")

# ---------- Load test manifest ----------
df = pd.read_csv(TEST_CSV)

# normalize columns
if "Patient" not in df.columns:
    for c in ["patient", "patient_id"]:
        if c in df.columns:
            df.rename(columns={c: "Patient"}, inplace=True)

if "path" not in df.columns:
    for c in ["ImagePath", "image_path", "filepath"]:
        if c in df.columns:
            df.rename(columns={c: "path"}, inplace=True)

df["Patient"] = df["Patient"].astype(str)

# ---------- Build model (single-frame classifier) ----------
model = torchvision.models.resnet50(weights=None)
model.fc = nn.Linear(2048, 1)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
sd = ckpt["model"] if "model" in ckpt else ckpt

# filter only encoder + cls_non weights if needed
filtered_sd = {k.replace("encoder.backbone.", ""): v 
               for k, v in sd.items() 
               if k.startswith("encoder.backbone.")}

model.load_state_dict(filtered_sd, strict=False)
model = model.to(DEVICE)
model.eval()

# ---------- Transform ----------
tfm = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# ---------- Inference ----------
patient_scores = {}

for pid, group in df.groupby("Patient"):
    probs = []
    for _, row in group.iterrows():
        img_path = Path(row["path"])
        if not img_path.is_absolute():
            img_path = IMAGES_ROOT / img_path

        if not img_path.exists():
            continue

        img = Image.open(img_path).convert("RGB")
        x = tfm(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logit = model(x).squeeze()
            prob = torch.sigmoid(logit).item()
        probs.append(prob)

    if len(probs) == 0:
        continue

    patient_scores[pid] = {
        "prob_mean": float(np.mean(probs)),
        "prob_max": float(np.max(probs))
    }

baseline_df = pd.DataFrame.from_dict(patient_scores, orient="index").reset_index()
baseline_df.rename(columns={"index": "Patient"}, inplace=True)

baseline_df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

Saved: D:\ENT\_challenge2_artifacts\_runs\BASELINE_singleframe_seed43.csv


In [47]:
import numpy as np
import pandas as pd
from pathlib import Path

MIL_PATH = Path(r"D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv")
BASE_PATH = Path(r"D:\ENT\_challenge2_artifacts\_runs\BASELINE_singleframe_seed43.csv")

mil = pd.read_csv(MIL_PATH)
base = pd.read_csv(BASE_PATH)

# Rename baseline columns to avoid collision
base = base.rename(columns={
    "prob_mean": "prob_mean_base",
    "prob_max": "prob_max_base"
})

merged = mil.merge(base, on="Patient", how="inner")

print("Columns after merge:")
print(merged.columns.tolist())

y = merged["y_true"].values.astype(int)

# MIL score (logit is better for AUC ranking)
s_mil = merged["logit_mean"].values.astype(float)

# Baseline score (use mean pooling)
s_base = merged["prob_mean_base"].values.astype(float)

def auc_roc(y, s):
    order = np.argsort(s)
    y_sorted = y[order]
    n1 = y_sorted.sum()
    n0 = len(y_sorted) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    ranks = np.arange(1, len(y_sorted)+1)
    return (ranks[y_sorted==1].sum() - n1*(n1+1)/2) / (n1*n0)

# Compute AUCs
auc_mil = auc_roc(y, s_mil)
auc_base = auc_roc(y, s_base)

# Bootstrap ΔAUC
rng = np.random.default_rng(43)
deltas = []

for _ in range(2000):
    idx = rng.integers(0, len(y), len(y))
    d = auc_roc(y[idx], s_mil[idx]) - auc_roc(y[idx], s_base[idx])
    deltas.append(d)

deltas = np.array(deltas)

ci_lo = np.quantile(deltas, 0.025)
ci_hi = np.quantile(deltas, 0.975)

# two-sided bootstrap p-value
p_value = 2 * min(
    np.mean(deltas <= 0),
    np.mean(deltas >= 0)
)

print("\n=== Statistical Comparison ===")
print("MIL AUC:", auc_mil)
print("Baseline AUC:", auc_base)
print("ΔAUC:", auc_mil - auc_base)
print("95% CI:", ci_lo, ci_hi)
print("Bootstrap p-value:", p_value)

Columns after merge:
['Patient', 'y_true', 'leuko', 'logit_fold1', 'prob_fold1', 'logit_fold2', 'prob_fold2', 'logit_fold3', 'prob_fold3', 'logit_fold4', 'prob_fold4', 'logit_fold5', 'prob_fold5', 'logit_mean', 'prob_mean', 'prob_mean_base', 'prob_max_base']

=== Statistical Comparison ===
MIL AUC: 0.8408695652173913
Baseline AUC: 0.5791304347826087
ΔAUC: 0.2617391304347826
95% CI: 0.10495967023172904 0.431822412687474
Bootstrap p-value: 0.002


In [48]:
import pandas as pd
import numpy as np
from pathlib import Path

MIL_PATH = Path(r"D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv")

df = pd.read_csv(MIL_PATH)

y = df["y_true"].values.astype(int)
p = df["prob_mean"].values

bins = np.linspace(0,1,11)
rows = []

for i in range(len(bins)-1):
    lo, hi = bins[i], bins[i+1]
    mask = (p >= lo) & (p < hi)
    if mask.sum() == 0:
        continue
    rows.append({
        "bin_low": lo,
        "bin_high": hi,
        "n": int(mask.sum()),
        "mean_confidence": float(p[mask].mean()),
        "empirical_accuracy": float(y[mask].mean())
    })

rel_df = pd.DataFrame(rows)
rel_path = Path(r"D:\ENT\_challenge2_artifacts\_runs\Q1_reliability_curve_seed43.csv")
rel_df.to_csv(rel_path, index=False)

print("Saved:", rel_path)
rel_df

Saved: D:\ENT\_challenge2_artifacts\_runs\Q1_reliability_curve_seed43.csv


,bin_low,bin_high,n,mean_confidence,empirical_accuracy
0,0.0,0.1,19,0.085213,0.000000
1,0.1,0.2,25,0.135114,0.240000
2,0.2,0.3,15,0.235084,0.533333
3,0.3,0.4,6,0.347698,0.500000
4,0.4,0.5,5,0.426614,1.000000
5,0.5,0.6,1,0.541589,0.000000
6,0.6,0.7,2,0.606498,0.500000


In [52]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    roc_curve,
)

# -----------------------------
# CONFIG (EDIT THESE 3 LINES)
# -----------------------------
MIL_AGG_CSV = r"D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv"

# Put your baseline predictions CSV here (must have Patient + baseline prob column)
BASELINE_CSV = r"D:\ENT\_challenge2_artifacts\_runs\BASELINE_singleframe_seed43.csv"  # e.g. r"D:\ENT\_challenge2_artifacts\_runs\baseline_preds.csv"

OUT_DIR = r"D:\ENT\_challenge2_artifacts\_runs"

# Column names
PATIENT_COL = "Patient"
Y_COL = "y_true"

# MIL prob column candidates (prefers the first one found; else uses sigmoid(logit_mean))
MIL_PROB_CANDS = ["prob_mean", "prob_mil", "prob"]
MIL_LOGIT_FALLBACK = "logit_mean"

# Baseline prob column candidates inside BASELINE_CSV
BASE_PROB_CANDS = ["prob_mean_base", "prob_base", "prob_mean", "prob", "baseline_prob"]

# Bootstrap settings
N_BOOT = 5000
SEED = 43

# Reliability bins
N_BINS = 10


# -----------------------------
# Helpers
# -----------------------------
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))

def ensure_probs(df, prob_col=None, logit_col=None):
    if prob_col is not None and prob_col in df.columns:
        p = df[prob_col].astype(float).values
        return np.clip(p, 0.0, 1.0)
    if logit_col is not None and logit_col in df.columns:
        p = sigmoid(df[logit_col].astype(float).values)
        return np.clip(p, 0.0, 1.0)
    raise KeyError(f"Could not get probabilities. prob_col={prob_col}, logit_col={logit_col}")

def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    j = tpr - fpr
    k = int(np.argmax(j))
    return float(thr[k])

def bacc_at(y, p, thr):
    pred = (p >= thr).astype(int)
    return float(balanced_accuracy_score(y, pred))

def reliability_table(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            m = (p >= lo) & (p < hi)
        else:
            m = (p >= lo) & (p <= hi)
        n = int(m.sum())
        if n == 0:
            rows.append([lo, hi, 0, np.nan, np.nan])
            continue
        mean_conf = float(np.mean(p[m]))
        emp_pos_rate = float(np.mean(y[m] == 1))
        rows.append([lo, hi, n, mean_conf, emp_pos_rate])
    return pd.DataFrame(rows, columns=["bin_low", "bin_high", "n", "mean_confidence", "empirical_accuracy"])

def plot_reliability(rel_df, out_png, title):
    x = rel_df["mean_confidence"].values
    y = rel_df["empirical_accuracy"].values
    m = ~np.isnan(x) & ~np.isnan(y)
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.scatter(x[m], y[m])
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Empirical positive rate")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def bootstrap_delta_auc(y, p_mil, p_base, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)
    p_mil = np.asarray(p_mil).astype(float)
    p_base = np.asarray(p_base).astype(float)
    n = len(y)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yy = y[idx]
        if yy.min() == yy.max():
            continue
        a_m = roc_auc_score(yy, p_mil[idx])
        a_b = roc_auc_score(yy, p_base[idx])
        deltas.append(a_m - a_b)
    deltas = np.array(deltas, dtype=float)
    if len(deltas) < 50:
        raise RuntimeError("Too few valid bootstrap samples.")
    delta = float(np.mean(deltas))
    ci_lo = float(np.quantile(deltas, 0.025))
    ci_hi = float(np.quantile(deltas, 0.975))
    p_le = float(np.mean(deltas <= 0))
    p_ge = float(np.mean(deltas >= 0))
    p_val = float(min(1.0, 2.0 * min(p_le, p_ge)))
    return delta, (ci_lo, ci_hi), p_val

# -----------------------------
# Main
# -----------------------------
def main():
    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    mil = pd.read_csv(MIL_AGG_CSV)
    if PATIENT_COL not in mil.columns:
        raise ValueError(f"MIL_AGG_CSV must contain '{PATIENT_COL}'.")

    # MIL probabilities
    mil_prob_col = pick_col(mil, MIL_PROB_CANDS)
    p_mil = ensure_probs(mil, prob_col=mil_prob_col, logit_col=MIL_LOGIT_FALLBACK)

    # Labels
    if Y_COL not in mil.columns:
        raise ValueError(f"MIL_AGG_CSV must contain '{Y_COL}'.")
    y = mil[Y_COL].astype(int).values

    # Baseline loading
    if not BASELINE_CSV or not Path(BASELINE_CSV).exists():
        raise FileNotFoundError(
            "BASELINE_CSV is missing or does not exist.\n"
            "Set BASELINE_CSV to a CSV with columns: Patient + baseline prob column (e.g., 'prob')."
        )

    base = pd.read_csv(BASELINE_CSV)
    if PATIENT_COL not in base.columns:
        raise ValueError(f"BASELINE_CSV must contain '{PATIENT_COL}'.")

    base_prob_col = pick_col(base, BASE_PROB_CANDS)
    if base_prob_col is None:
        raise KeyError(
            f"Could not find baseline prob column in BASELINE_CSV. "
            f"Tried: {BASE_PROB_CANDS}. "
            f"Columns are: {list(base.columns)[:50]}"
        )

    # Merge to align patients
    merged = mil[[PATIENT_COL, Y_COL]].copy().merge(
        base[[PATIENT_COL, base_prob_col]].copy(),
        on=PATIENT_COL,
        how="inner"
    )
    if len(merged) < len(mil):
        print(f"[WARN] Baseline CSV has fewer patients than MIL AGG: matched {len(merged)}/{len(mil)}")

    y_m = merged[Y_COL].astype(int).values
    p_base = np.clip(merged[base_prob_col].astype(float).values, 0.0, 1.0)

    # Need both classes
    if y_m.min() == y_m.max():
        raise ValueError("After merge, y_true has only one class; cannot compute AUC/AP.")

    # Metrics
    auc_mil = float(roc_auc_score(y, p_mil))  # MIL on its own file
    # AUC needs aligned vectors; compute MIL aligned as well:
    mil_aligned = mil[[PATIENT_COL]].copy().merge(merged[[PATIENT_COL]], on=PATIENT_COL, how="inner")
    # build aligned p_mil
    mil_map = dict(zip(mil[PATIENT_COL].tolist(), p_mil.tolist()))
    p_mil_aligned = np.array([mil_map[p] for p in merged[PATIENT_COL].tolist()], dtype=float)

    auc_mil_aligned = float(roc_auc_score(y_m, p_mil_aligned))
    auc_base = float(roc_auc_score(y_m, p_base))

    ap_mil = float(average_precision_score(y_m, p_mil_aligned))
    ap_base = float(average_precision_score(y_m, p_base))

    thr_mil = youden_threshold(y_m, p_mil_aligned)
    thr_base = youden_threshold(y_m, p_base)

    bacc_mil = bacc_at(y_m, p_mil_aligned, thr_mil)
    bacc_base = bacc_at(y_m, p_base, thr_base)

    # Bootstrap ΔAUC
    delta_auc, (ci_lo, ci_hi), p_val = bootstrap_delta_auc(y_m, p_mil_aligned, p_base, n_boot=N_BOOT, seed=SEED)

    # Reliability curves
    rel_m = reliability_table(y_m, p_mil_aligned, n_bins=N_BINS)
    rel_b = reliability_table(y_m, p_base, n_bins=N_BINS)

    rel_m_csv = out_dir / "Q1_reliability_curve_MIL.csv"
    rel_b_csv = out_dir / "Q1_reliability_curve_BASELINE.csv"
    rel_m_png = out_dir / "Q1_reliability_curve_MIL.png"
    rel_b_png = out_dir / "Q1_reliability_curve_BASELINE.png"

    rel_m.to_csv(rel_m_csv, index=False)
    rel_b.to_csv(rel_b_csv, index=False)
    plot_reliability(rel_m, rel_m_png, "Reliability Curve (MIL)")
    plot_reliability(rel_b, rel_b_png, "Reliability Curve (Baseline)")

    summary = {
        "mil_agg_csv": MIL_AGG_CSV,
        "baseline_csv": BASELINE_CSV,
        "n_matched": int(len(merged)),
        "mil_prob_col_used": mil_prob_col if mil_prob_col else f"sigmoid({MIL_LOGIT_FALLBACK})",
        "baseline_prob_col_used": base_prob_col,
        "roc_auc": {"mil": auc_mil_aligned, "baseline": auc_base, "delta": auc_mil_aligned - auc_base},
        "pr_auc_average_precision": {"mil": ap_mil, "baseline": ap_base, "delta": ap_mil - ap_base},
        "balanced_accuracy_youden": {"mil": bacc_mil, "baseline": bacc_base},
        "youden_thresholds": {"mil": thr_mil, "baseline": thr_base},
        "bootstrap_delta_auc": {"delta_mean": delta_auc, "ci95": [ci_lo, ci_hi], "p_value": p_val, "n_boot": N_BOOT},
        "reliability_outputs": {
            "mil_csv": str(rel_m_csv),
            "mil_png": str(rel_m_png),
            "baseline_csv": str(rel_b_csv),
            "baseline_png": str(rel_b_png),
        },
    }

    out_json = out_dir / "Q1_compare_MIL_vs_baseline_summary.json"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\n=== Q1: MIL vs BASELINE (Two-CSV) ===")
    print(f"Matched patients: {len(merged)}")
    print(f"ROC-AUC: MIL={auc_mil_aligned:.6f} | BASE={auc_base:.6f} | Δ={auc_mil_aligned-auc_base:.6f}")
    print(f"PR-AUC(AP): MIL={ap_mil:.6f} | BASE={ap_base:.6f} | Δ={ap_mil-ap_base:.6f}")
    print(f"Balanced Acc (Youden): MIL={bacc_mil:.6f} | BASE={bacc_base:.6f}")
    print(f"ΔAUC bootstrap: mean={delta_auc:.6f} | 95%CI=[{ci_lo:.6f},{ci_hi:.6f}] | p={p_val:.6f}")
    print(f"Saved: {out_json}")
    print(f"Reliability MIL: {rel_m_csv} + {rel_m_png}")
    print(f"Reliability BASE: {rel_b_csv} + {rel_b_png}\n")

if __name__ == "__main__":
    main()


=== Q1: MIL vs BASELINE (Two-CSV) ===
Matched patients: 73
ROC-AUC: MIL=0.844348 | BASE=0.579130 | Δ=0.265217
PR-AUC(AP): MIL=0.629041 | BASE=0.355107 | Δ=0.273934
Balanced Acc (Youden): MIL=0.800000 | BASE=0.634783
ΔAUC bootstrap: mean=0.267142 | 95%CI=[0.098438,0.434939] | p=0.001200
Saved: D:\ENT\_challenge2_artifacts\_runs\Q1_compare_MIL_vs_baseline_summary.json
Reliability MIL: D:\ENT\_challenge2_artifacts\_runs\Q1_reliability_curve_MIL.csv + D:\ENT\_challenge2_artifacts\_runs\Q1_reliability_curve_MIL.png
Reliability BASE: D:\ENT\_challenge2_artifacts\_runs\Q1_reliability_curve_BASELINE.csv + D:\ENT\_challenge2_artifacts\_runs\Q1_reliability_curve_BASELINE.png



In [61]:
import os
import time
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
import matplotlib.pyplot as plt


# ============================================================
# 0) REPRO
# ============================================================
def seed_everything(seed: int = 43):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))


# ============================================================
# 1) CONFIG
# ============================================================
@dataclass
class CFG:
    SEED: int = 43
    DEVICE: str = "cuda:0" if torch.cuda.is_available() else "cpu"

    IMAGES_ROOT: Path = Path(r"D:\ENT")
    ARTIFACTS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts")
    RUNS_DIR: Path = Path(r"D:\ENT\_challenge2_artifacts\_runs")

    # Patient lists (these can be unlabeled)
    TEST_CSV: Path = Path(r"D:\ENT\_challenge2_artifacts\split_res_shift_test_minor_colab.csv")

    BENCHMARK: str = "res_shift"
    FOLDS: Tuple[int, ...] = (1, 2, 3, 4, 5)
    K_TEST: int = 10

    IMG_SIZE: int = 224
    BATCH_SIZE_IMG: int = 16

    TRUST_CKPT: bool = True

    CKPT_TEMPLATE: str = r"D:\ENT\_challenge2_artifacts\_runs\cv_best_res_shift_seed43_fold{fold}.pt"

    # If you already have an AGG CSV with y_true, we can evaluate from it directly:
    # (This is the file you printed earlier with y_true/leuko/logit_fold*/prob_fold*/logit_mean/prob_mean)
    EVAL_AGG_CSV: Optional[Path] = Path(r"D:\ENT\_challenge2_artifacts\_runs\pred_seed43_res_shift_folds1_5_Ktest10_agg.csv")

cfg = CFG()


# ============================================================
# 2) LOAD PATIENT LIST (labels optional)
# ============================================================
def load_patient_list(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if "Patient" not in df.columns:
        for c in ["patient", "patient_id", "PatientID"]:
            if c in df.columns:
                df = df.rename(columns={c: "Patient"})
                break
    if "Patient" not in df.columns:
        raise ValueError(f"{csv_path} missing Patient column.")
    df = df[["Patient"]].drop_duplicates().reset_index(drop=True)
    return df


# ============================================================
# 3) BUILD PATIENT->IMAGE PATHS MAPPING
# ============================================================
def build_patient_image_mapping(images_root: Path) -> Dict[str, List[str]]:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    mapping: Dict[str, List[str]] = {}

    for root, dirs, files in os.walk(images_root):
        base = os.path.basename(root)
        if base.lower().startswith("patient"):
            pid = base
            img_paths = []
            for fn in files:
                p = os.path.join(root, fn)
                if Path(p).suffix.lower() in exts:
                    img_paths.append(p)
            if len(img_paths) > 0:
                img_paths.sort()
                mapping[pid] = img_paths

    print(f"Patients found in image root: {len(mapping)}")
    return mapping


# ============================================================
# 4) TRANSFORMS
# ============================================================
def make_tfm(img_size: int):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


def pil_load_rgb(p: str) -> Image.Image:
    return Image.open(p).convert("RGB")


# ============================================================
# 5) MODEL: YOU MUST USE YOUR WORKING build_model_from_ckpt
# ============================================================
def build_model_from_ckpt(ckpt_path: Path, device: str, gate_tau: float = 2.0):
    """
    Replace with YOUR working function (already exists in your notebook).
    """
    raise NotImplementedError(
        "Paste your working build_model_from_ckpt() + model definition here."
    )


@torch.no_grad()
def _extract_instance_features(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    # matches your prior architecture where encoder.backbone exists
    if hasattr(model, "encoder") and hasattr(model.encoder, "backbone"):
        y = model.encoder.backbone(x)
        if y.dim() == 4:
            y = y.flatten(1)
        return y
    raise AttributeError("Update _extract_instance_features() to match your model.")


def _mil_forward_from_feats(model: nn.Module, feats: torch.Tensor) -> Dict[str, torch.Tensor]:
    def attn_pool(pool_module, feats_):
        scores = pool_module.net(feats_).squeeze(-1)  # [N]
        w = torch.softmax(scores, dim=0)
        z = (w.unsqueeze(-1) * feats_).sum(dim=0)
        return z, w

    needed = ["pool_non", "pool_leu", "cls_non", "cls_leu", "gate"]
    for n in needed:
        if not hasattr(model, n):
            raise AttributeError(f"Model missing submodule: {n}")

    z_non, w_non = attn_pool(model.pool_non, feats)
    z_leu, w_leu = attn_pool(model.pool_leu, feats)

    logit_non = model.cls_non(z_non).squeeze()
    logit_leu = model.cls_leu(z_leu).squeeze()

    fused = torch.cat([z_non, z_leu], dim=0).unsqueeze(0)  # [1,2D]
    alpha_logit = model.gate(fused).squeeze()
    alpha = torch.sigmoid(alpha_logit)

    logit_gated = alpha * logit_leu + (1.0 - alpha) * logit_non
    logit_nogate = 0.5 * logit_leu + 0.5 * logit_non

    return {
        "logit_gated": logit_gated,
        "logit_nogate": logit_nogate,
        "alpha": alpha,
    }


def infer_one_patient(
    model: nn.Module,
    img_paths: List[str],
    tfm,
    device: str,
    k_test: int,
    batch_size_img: int,
) -> Dict[str, float]:
    if len(img_paths) == 0:
        return {"logit": np.nan, "prob": np.nan, "logit_nogate": np.nan, "prob_nogate": np.nan, "alpha": np.nan}

    sel = img_paths[: min(k_test, len(img_paths))]

    xs = []
    for p in sel:
        pil = pil_load_rgb(p)
        xs.append(tfm(pil))
    x = torch.stack(xs, dim=0).to(device)

    feats_list = []
    model.eval()
    with torch.no_grad():
        for i in range(0, x.size(0), batch_size_img):
            xb = x[i:i+batch_size_img]
            fb = _extract_instance_features(model, xb)
            feats_list.append(fb)
    feats = torch.cat(feats_list, dim=0)

    out = _mil_forward_from_feats(model, feats)

    lg = float(out["logit_gated"].detach().cpu().item())
    ln = float(out["logit_nogate"].detach().cpu().item())
    a  = float(out["alpha"].detach().cpu().item())

    return {
        "logit": lg,
        "prob": float(sigmoid_np(lg)),
        "logit_nogate": ln,
        "prob_nogate": float(sigmoid_np(ln)),
        "alpha": a,
    }


def run_fold_inference_and_save(
    fold: int,
    ckpt_path: Path,
    patients: List[str],
    mapping: Dict[str, List[str]],
    device: str,
    out_csv: Path,
    tfm,
    k_test: int,
    batch_size_img: int,
):
    print(f"\n[FOLD {fold}] CKPT: {ckpt_path.name}")
    model, _ = build_model_from_ckpt(ckpt_path, device=device, gate_tau=2.0)
    model.to(device).eval()

    rows = []
    t0 = time.time()

    for idx, pid in enumerate(patients, 1):
        img_paths = mapping.get(pid, [])
        r = infer_one_patient(model, img_paths, tfm, device, k_test, batch_size_img)
        rows.append({
            "Patient": pid,
            "logit": r["logit"],
            "prob": r["prob"],
            "logit_nogate": r["logit_nogate"],
            "prob_nogate": r["prob_nogate"],
            "alpha": r["alpha"],
        })

        if idx % 10 == 0 or idx == len(patients):
            print(f"  progress {idx}/{len(patients)}  elapsed={time.time()-t0:.1f}s")

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print("Saved:", out_csv)


# ============================================================
# 6) AGGREGATE FOLDS (ABlation)
# ============================================================
def aggregate_folds_with_ablation(
    runs_dir: Path,
    seed: int,
    benchmark: str,
    folds: Tuple[int, ...],
    k_test: int,
    out_path: Path,
) -> pd.DataFrame:
    dfs = []
    for f in folds:
        p = runs_dir / f"pred_fold{f}_{benchmark}_seed{seed}_Ktest{k_test}.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing fold preds: {p}")
        d = pd.read_csv(p).set_index("Patient")

        # required for ablation aggregation
        for req in ["prob", "logit", "prob_nogate", "logit_nogate", "alpha"]:
            if req not in d.columns:
                raise KeyError(f"{p} missing required column: {req}")

        keep = d[["prob", "logit", "prob_nogate", "logit_nogate", "alpha"]].rename(columns={
            "prob": f"prob_fold{f}",
            "logit": f"logit_fold{f}",
            "prob_nogate": f"prob_nogate_fold{f}",
            "logit_nogate": f"logit_nogate_fold{f}",
            "alpha": f"alpha_fold{f}",
        })
        dfs.append(keep)

    merged = dfs[0]
    for dd in dfs[1:]:
        merged = merged.join(dd, how="inner")

    prob_cols = [f"prob_fold{f}" for f in folds]
    logit_cols = [f"logit_fold{f}" for f in folds]
    ng_prob_cols = [f"prob_nogate_fold{f}" for f in folds]
    ng_logit_cols = [f"logit_nogate_fold{f}" for f in folds]
    a_cols = [f"alpha_fold{f}" for f in folds]

    merged["prob_mean"] = merged[prob_cols].mean(axis=1)
    merged["logit_mean"] = merged[logit_cols].mean(axis=1)
    merged["prob_nogate_mean"] = merged[ng_prob_cols].mean(axis=1)
    merged["logit_nogate_mean"] = merged[ng_logit_cols].mean(axis=1)
    merged["alpha_mean"] = merged[a_cols].mean(axis=1)

    merged = merged.reset_index()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return merged


# ============================================================
# 7) METRICS (need y_true -> use AGG EVAL CSV)
# ============================================================
def roc_auc(y: np.ndarray, s: np.ndarray) -> float:
    y = y.astype(int)
    s = s.astype(float)
    order = np.argsort(s)
    y2 = y[order]
    pos = (y2 == 1)
    neg = (y2 == 0)
    n_pos = pos.sum()
    n_neg = neg.sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = np.arange(1, len(y2) + 1)
    sum_ranks_pos = ranks[pos].sum()
    auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc)


def pr_auc_ap(y: np.ndarray, p: np.ndarray) -> float:
    y = y.astype(int)
    p = p.astype(float)
    order = np.argsort(-p)
    y2 = y[order]
    tp = 0
    fp = 0
    P = (y2 == 1).sum()
    if P == 0:
        return float("nan")
    ap = 0.0
    prev_rec = 0.0
    for i in range(len(y2)):
        if y2[i] == 1:
            tp += 1
        else:
            fp += 1
        prec = tp / (tp + fp)
        rec = tp / P
        ap += prec * (rec - prev_rec)
        prev_rec = rec
    return float(ap)


def youden_threshold(y: np.ndarray, p: np.ndarray):
    y = y.astype(int)
    p = p.astype(float)
    thrs = np.unique(p)
    bestJ = -1e9
    bestT = 0.5
    bestBal = 0.0
    for t in thrs:
        pred = (p >= t).astype(int)
        tp = ((pred == 1) & (y == 1)).sum()
        tn = ((pred == 0) & (y == 0)).sum()
        fp = ((pred == 1) & (y == 0)).sum()
        fn = ((pred == 0) & (y == 1)).sum()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        J = sens + spec - 1.0
        bal = 0.5 * (sens + spec)
        if J > bestJ:
            bestJ = J
            bestT = t
            bestBal = bal
    return float(bestT), float(bestBal)


def reliability_curve(y: np.ndarray, p: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
    y = y.astype(int)
    p = p.astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        n = int(mask.sum())
        if n == 0:
            rows.append({"bin_low": lo, "bin_high": hi, "n": 0,
                         "mean_confidence": np.nan, "empirical_accuracy": np.nan})
        else:
            rows.append({
                "bin_low": lo,
                "bin_high": hi,
                "n": n,
                "mean_confidence": float(p[mask].mean()),
                "empirical_accuracy": float(y[mask].mean()),
            })
    return pd.DataFrame(rows)


def save_reliability_plot(df_rel: pd.DataFrame, out_png: Path, title: str):
    out_png.parent.mkdir(parents=True, exist_ok=True)
    x = df_rel["mean_confidence"].values
    y = df_rel["empirical_accuracy"].values

    plt.figure()
    plt.plot([0, 1], [0, 1])
    plt.plot(x, y, marker="o")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Empirical fraction positive")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close()


def eval_from_labeled_agg(agg_csv: Path, out_prefix: Path):
    df = pd.read_csv(agg_csv)
    if "y_true" not in df.columns:
        raise ValueError(f"EVAL_AGG_CSV must contain y_true. Missing in: {agg_csv}")

    # prefer prob_mean/prob_nogate_mean if available; else fall back
    if "prob_mean" not in df.columns and "prob" in df.columns:
        df["prob_mean"] = df["prob"]
    if "prob_nogate_mean" not in df.columns and "prob_nogate" in df.columns:
        df["prob_nogate_mean"] = df["prob_nogate"]

    for col in ["prob_mean", "prob_nogate_mean"]:
        if col not in df.columns:
            raise KeyError(f"Missing {col} in {agg_csv}. Need it for gated vs no-gate evaluation.")

    y = df["y_true"].values.astype(int)
    p_g = df["prob_mean"].values.astype(float)
    p_ng = df["prob_nogate_mean"].values.astype(float)

    res = {}
    for name, p in [("gated", p_g), ("nogate", p_ng)]:
        auc = roc_auc(y, p)
        ap = pr_auc_ap(y, p)
        thr, bal = youden_threshold(y, p)
        rel = reliability_curve(y, p, n_bins=10)
        rel_csv = out_prefix.parent / f"{out_prefix.name}_reliability_{name}.csv"
        rel_png = out_prefix.parent / f"{out_prefix.name}_reliability_{name}.png"
        rel.to_csv(rel_csv, index=False)
        save_reliability_plot(rel, rel_png, title=f"Reliability ({name})")
        res[name] = {"roc_auc": auc, "pr_auc_ap": ap, "youden_thr": thr, "balanced_acc": bal,
                     "reliability_csv": str(rel_csv), "reliability_png": str(rel_png)}

    out_json = out_prefix.parent / f"{out_prefix.name}_summary.json"
    with open(out_json, "w") as f:
        json.dump({"source": str(agg_csv), **res}, f, indent=2)

    print("\n=== ABLATION (from labeled AGG) ===")
    print(f"ROC-AUC: gated={res['gated']['roc_auc']:.6f} | nogate={res['nogate']['roc_auc']:.6f}")
    print(f"PR-AUC(AP): gated={res['gated']['pr_auc_ap']:.6f} | nogate={res['nogate']['pr_auc_ap']:.6f}")
    print(f"Balanced Acc: gated={res['gated']['balanced_acc']:.6f} | nogate={res['nogate']['balanced_acc']:.6f}")
    print("Saved:", out_json)


# ============================================================
# 8) MAIN
# ============================================================
def main():
    seed_everything(cfg.SEED)
    cfg.RUNS_DIR.mkdir(parents=True, exist_ok=True)

    print("DEVICE:", cfg.DEVICE)
    print("IMAGES_ROOT:", cfg.IMAGES_ROOT)
    print("RUNS_DIR:", cfg.RUNS_DIR)

    # patient list only (no labels needed)
    df_te = load_patient_list(cfg.TEST_CSV)
    patients = df_te["Patient"].tolist()

    mapping = build_patient_image_mapping(cfg.IMAGES_ROOT)
    tfm = make_tfm(cfg.IMG_SIZE)

    # 1) Per-fold inference (writes prob_nogate + alpha)
    for fold in cfg.FOLDS:
        ckpt_path = Path(cfg.CKPT_TEMPLATE.format(fold=fold))
        out_csv = cfg.RUNS_DIR / f"pred_fold{fold}_{cfg.BENCHMARK}_seed{cfg.SEED}_Ktest{cfg.K_TEST}.csv"

        if out_csv.exists():
            print(f"[SKIP] fold {fold} exists: {out_csv}")
            continue
        if not ckpt_path.exists():
            raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")

        run_fold_inference_and_save(
            fold=fold,
            ckpt_path=ckpt_path,
            patients=patients,
            mapping=mapping,
            device=cfg.DEVICE,
            out_csv=out_csv,
            tfm=tfm,
            k_test=cfg.K_TEST,
            batch_size_img=cfg.BATCH_SIZE_IMG,
        )

    # 2) Aggregate folds with ablation
    agg_path = cfg.RUNS_DIR / f"pred_seed{cfg.SEED}_{cfg.BENCHMARK}_folds1_5_Ktest{cfg.K_TEST}_agg_with_nogate.csv"
    aggregate_folds_with_ablation(
        runs_dir=cfg.RUNS_DIR,
        seed=cfg.SEED,
        benchmark=cfg.BENCHMARK,
        folds=cfg.FOLDS,
        k_test=cfg.K_TEST,
        out_path=agg_path
    )

    # 3) Evaluate from labeled AGG if available
    if cfg.EVAL_AGG_CSV is not None and cfg.EVAL_AGG_CSV.exists():
        out_prefix = cfg.RUNS_DIR / f"Q1_ablation_gated_vs_nogate_seed{cfg.SEED}"
        eval_from_labeled_agg(cfg.EVAL_AGG_CSV, out_prefix)
    else:
        print("\n[INFO] No labeled EVAL_AGG_CSV found. Inference + aggregation done.")
        print("Agg with nogate:", agg_path)
        print("To evaluate, set cfg.EVAL_AGG_CSV to a CSV that contains y_true + prob_mean/prob_nogate_mean (or prob/prob_nogate).")

    print("\nDONE.")
    print("Agg with nogate:", agg_path)


if __name__ == "__main__":
    main()

DEVICE: cuda:0
IMAGES_ROOT: D:\ENT
RUNS_DIR: D:\ENT\_challenge2_artifacts\_runs
Patients found in image root: 210
[SKIP] fold 1 exists: D:\ENT\_challenge2_artifacts\_runs\pred_fold1_res_shift_seed43_Ktest10.csv
[SKIP] fold 2 exists: D:\ENT\_challenge2_artifacts\_runs\pred_fold2_res_shift_seed43_Ktest10.csv
[SKIP] fold 3 exists: D:\ENT\_challenge2_artifacts\_runs\pred_fold3_res_shift_seed43_Ktest10.csv
[SKIP] fold 4 exists: D:\ENT\_challenge2_artifacts\_runs\pred_fold4_res_shift_seed43_Ktest10.csv
[SKIP] fold 5 exists: D:\ENT\_challenge2_artifacts\_runs\pred_fold5_res_shift_seed43_Ktest10.csv


KeyError: 'D:\\ENT\\_challenge2_artifacts\\_runs\\pred_fold1_res_shift_seed43_Ktest10.csv missing required column: prob_nogate'